# COMP90042 2026 Project — final system (v8)

Climate-claim fact-checking pipeline.

**Architecture:**
1. Stage-1a — BM25 (k1=1.5, b=0.5) top-500 candidates.
2. Stage-1b — BGE-base-en-v1.5 dense retrieval top-500 (cached evidence embedding, fp16).
3. Stage-2 — fine-tuned cross-encoder reranker (MiniLM-L-6-v2) on BM25 candidates only
   (negatives drawn from BM25 top-500 to avoid dense-side false-negative supervision).
4. Selector — fuses reranker probability with dense cosine: `alpha * sigmoid(logit) +
   (1-alpha) * dense_norm`. Alpha tuned on dev (best = 0.7).
5. Classifier — MiniLM-L-6-v2 multiclass head over `[claim, [SEP], concatenated evidences]`,
   uniform class weights, model selection by dev accuracy with REFUTES/DISPUTED tie-break.
6. Safety — fall back to majority label if classifier dev H underperforms majority H by
   at least the configured tolerance.

**Final dev numbers** (verified with `python eval.py --predictions outputs_notebook_v8_mini2v2/dev-predictions.json --groundtruth data/dev-claims.json`):

```
Evidence Retrieval F-score (F)    = 0.255726
Claim Classification Accuracy (A) = 0.441558
Harmonic Mean of F and A          = 0.323879
```

vs **v3 / v6 baseline** F=0.2106, A=0.5000, H=0.2963 — net **+2.8 H** with retrieval F up
4.5 points and classifier falling back to majority because uniform-weight MiniLM-L-6 ties
the majority baseline. Full ablation including the negative results (bge-reranker-base
regression, DeBERTa-NLI classifier collapse, bf16+SDPA fine-tuning collapse) is in the
top-level `NIGHT_RUN_SUMMARY.md`.

The cells below are the executed pipeline. Outputs were captured to
`v8_mini2v2_run.log` and the complete log is reproduced as the final markdown appendix.


## Dependencies

In [ ]:
# Optional dependency installation. Safe in Colab; usually skipped locally if packages already exist.
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "bm25s": "bm25s",
    "transformers": "transformers",
    "accelerate": "accelerate",
    "psutil": "psutil",
    "sentencepiece": "sentencepiece",  # required by DeBERTa-v3 tokenizer
    "protobuf": "google.protobuf",
}

for pip_name, import_name in REQUIRED_PACKAGES.items():
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_name} ...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])
        except Exception as exc:
            # Some managed envs (e.g. uv-created venvs) ship without pip. In that case the user
            # should pre-install dependencies, so we surface a clear hint instead of dying mid-cell.
            print(f"  [warn] could not auto-install {pip_name}: {exc}. "
                  f"Install manually if downstream imports fail.")


## Imports, config, seeding

In [ ]:
from pathlib import Path
import json
import random
import time
import pickle
from collections import Counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# -------------------------
# Global config (v7)
# -------------------------
SEED = 42
FAST_DEV_MODE = False  # True = quick smoke test; False = full project run

DATA_DIR = Path("data")
OUTPUT_DIR = Path("outputs_notebook_v8_mini2v2")
CACHE_DIR = Path("outputs_notebook_v7") / "cache"
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
CACHE_DIR.mkdir(exist_ok=True, parents=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

# -------------------------
# Retrieval config
# -------------------------
# Stage-1 BM25. k1=1.5, b=0.5 was the best in v5 retrieval-branch sweep
# (mean_recall@500 = 0.650 vs default 0.618 at b=0.75).
BM25_K1 = 1.5
BM25_B = 0.5
BM25_CANDIDATE_K = 500 if not FAST_DEV_MODE else 50

# Stage-1b dense retrieval. BGE-base-en-v1.5 is a 109M-param open dual encoder
# that complements BM25 on paraphrastic claims where lexical overlap fails.
# p10_recall@500 = 0 in v3 (BM25 alone) is the symptom this addresses.
DENSE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
DENSE_MAX_LEN = 256
DENSE_ENCODE_BATCH = 128 if not FAST_DEV_MODE else 16
DENSE_TOP_K = 500 if not FAST_DEV_MODE else 50

# Stage-2 cross-encoder reranker. Upgrade from MiniLM-L-6 (22M) to bge-reranker-base (278M)
# trained on a more diverse retrieval mixture; we still fine-tune on FEVER-style claims.
RERANKER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
RERANKER_MAX_LEN = 256
RERANKER_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
RERANKER_EVAL_BATCH_SIZE = 128 if not FAST_DEV_MODE else 16
RERANKER_EPOCHS = 3 if not FAST_DEV_MODE else 1
RERANKER_LR = 1e-5
NEGATIVES_PER_POSITIVE = 4 if not FAST_DEV_MODE else 2
HARD_NEGATIVE_POOL = min(200, BM25_CANDIDATE_K)

# Selector grids (same as v3).
FIXED_K_GRID = [2, 3, 4, 5]
THRESHOLD_GRID = [round(float(x), 2) for x in np.arange(0.02, 0.52, 0.02)] + [0.60, 0.70, 0.80, 0.90]
RELATIVE_LOGIT_DELTA_GRID = [0.25, 0.50, 0.75, 1.00, 1.50, 2.00, 3.00]
MIN_FINAL_K = 1
MAX_FINAL_K = 5
FINAL_RETRIEVAL_POLICY = "prefer_dynamic"
DYNAMIC_RETRIEVAL_TOLERANCE = 0.02

# -------------------------
# Classification config
# -------------------------
LABELS = ["SUPPORTS", "REFUTES", "NOT_ENOUGH_INFO", "DISPUTED"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = {i: label for label, i in LABEL2ID.items()}
# DeBERTa-v3-base pretrained on MNLI + FEVER + ANLI. Picks up entailment/contradiction
# semantics that the MS-MARCO reranker simply did not encode (v3 REFUTES/DISPUTED acc = 0).
CLASSIFIER_MODEL_NAME = "cross-encoder/ms-marco-MiniLM-L-6-v2"
MAX_SEQ_LEN = 256
TRAIN_BATCH_SIZE = 16 if not FAST_DEV_MODE else 4
EVAL_BATCH_SIZE = 32 if not FAST_DEV_MODE else 8
CLASSIFIER_EPOCHS = 4 if not FAST_DEV_MODE else 1
CLASSIFIER_LR = 1e-5
CLASSIFIER_USE_CLASS_WEIGHTS = False
CLASSIFIER_CLASS_WEIGHT_POWER = 0.5
CLASSIFIER_TRAIN_EVIDENCE_SOURCE = "retrieved"
FALLBACK_TO_MAJORITY_IF_CLASSIFIER_WORSE = True
CLASSIFIER_MIN_H_IMPROVEMENT = 0.005
WEIGHT_DECAY = 0.01

# -------------------------
# Performance switches
# -------------------------
# bf16 autocast: no GradScaler needed, near-identical numerics for fine-tuning.
# Falls back to fp32 on hardware without bf16 (pre-Ampere).
USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
AUTOCAST_DTYPE = torch.bfloat16 if USE_BF16 else torch.float32
# PyTorch native scaled-dot-product attention. Selected per-model where the architecture
# supports it (BertModel, XLM-RoBERTa). DeBERTa-v2's disentangled attention is not
# SDPA-compatible in transformers 5.8.x, so the classifier keeps the default kernel.
ATTN_IMPL = "sdpa"
# Fused AdamW uses CUDA fused kernels and is faster than the default impl with no risk.
FUSED_OPTIM = torch.cuda.is_available()
print(f"perf: bf16={USE_BF16}, attn={ATTN_IMPL}, fused_optim={FUSED_OPTIM}")


def set_seed(seed: int = SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


set_seed(SEED)


## Load data and run lightweight EDA

In [ ]:
def load_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


train_claims = load_json(DATA_DIR / "train-claims.json")
dev_claims = load_json(DATA_DIR / "dev-claims.json")
test_claims = load_json(DATA_DIR / "test-claims-unlabelled.json")
evidence = load_json(DATA_DIR / "evidence.json")

if FAST_DEV_MODE:
    train_claims = dict(list(train_claims.items())[:80])
    dev_claims = dict(list(dev_claims.items())[:30])
    test_claims = dict(list(test_claims.items())[:30])

print(f"train:    {len(train_claims)}")
print(f"dev:      {len(dev_claims)}")
print(f"test:     {len(test_claims)}")
print(f"evidence: {len(evidence)}")

# Lightweight EDA used to justify later choices.
label_dist = Counter(c["claim_label"] for c in train_claims.values())
print("Train label distribution:")
for lbl in LABELS:
    cnt = label_dist[lbl]
    print(f"  {lbl:20s} {cnt:5d} ({cnt / len(train_claims):6.2%})")

gt_counts = [len(c["evidences"]) for c in train_claims.values()]
print("\nGround-truth evidence count per train claim:")
print("  min=", min(gt_counts), "max=", max(gt_counts), "mean=", round(float(np.mean(gt_counts)), 3))
print("  exact counts:", sorted(Counter(gt_counts).items()))

# Official-style metrics. These mirror eval.py's logic and let us tune inside the notebook.


## Submission-format evaluation helpers (mirroring eval.py)

In [ ]:
def evidence_f1_for_claim(pred_eids, gold_eids):
    pred_eids = list(pred_eids)
    gold_eids = list(gold_eids)
    if len(pred_eids) == 0:
        return 0.0
    pred_set = set(pred_eids)
    correct = sum(1 for eid in gold_eids if eid in pred_set)
    if correct == 0:
        return 0.0
    precision = correct / len(pred_eids)
    recall = correct / len(gold_eids)
    return 2 * precision * recall / (precision + recall)


def evaluate_submission(predictions, gold_claims, verbose=True):
    f_scores = []
    correct_labels = 0
    total = 0
    for cid, gold in gold_claims.items():
        pred = predictions[cid]
        f_scores.append(evidence_f1_for_claim(pred["evidences"], gold["evidences"]))
        correct_labels += int(pred["claim_label"] == gold["claim_label"])
        total += 1
    F = float(np.mean(f_scores))
    A = correct_labels / total
    H = 0.0 if (F + A) == 0 else 2 * F * A / (F + A)
    if verbose:
        print(f"Evidence Retrieval F-score (F)    = {F:.6f}")
        print(f"Claim Classification Accuracy (A) = {A:.6f}")
        print(f"Harmonic Mean of F and A          = {H:.6f}")
    return {"F": F, "A": A, "H": H}


def evaluate_retrieval_only(retrieval, gold_claims):
    return float(
        np.mean(
            [
                evidence_f1_for_claim(retrieval[cid], claim["evidences"])
                for cid, claim in gold_claims.items()
            ]
        )
    )


def majority_label(claims):
    return Counter(c["claim_label"] for c in claims.values()).most_common(1)[0][0]


def validate_retrieval_coverage(claims_dict, retrieval, split_name="split"):
    """Fail early with a helpful message if retrieval is incomplete or empty."""
    missing_cids = [cid for cid in claims_dict if cid not in retrieval]
    if missing_cids:
        raise KeyError(f"{split_name}: retrieval missing {len(missing_cids)} claim ids, e.g. {missing_cids[:3]}")

    empty_cids = [cid for cid in claims_dict if len(retrieval[cid]) == 0]
    if empty_cids:
        raise ValueError(f"{split_name}: retrieval has empty evidence lists, e.g. {empty_cids[:3]}")

    bad_eids = []
    for cid in claims_dict:
        for eid in retrieval[cid]:
            if eid not in evidence:
                bad_eids.append((cid, eid))
                if len(bad_eids) >= 3:
                    break
        if len(bad_eids) >= 3:
            break
    if bad_eids:
        raise KeyError(f"{split_name}: retrieval contains unknown evidence ids, e.g. {bad_eids}")


def build_predictions(claims, retrieval, label_predictions=None, default_label=None):
    if label_predictions is None:
        assert default_label is not None
        label_predictions = {cid: default_label for cid in claims.keys()}
    out = {}
    fallback_eid = next(iter(evidence.keys()))
    for cid in claims.keys():
        eids = list(retrieval.get(cid, []))
        if len(eids) == 0:
            # Assignment requires at least one evidence. This fallback should rarely trigger.
            eids = [fallback_eid]
        out[cid] = {"claim_label": label_predictions[cid], "evidences": eids}
    return out


def write_predictions(predictions, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    # ensure_ascii=True keeps the file readable even when Windows uses a non-UTF-8 default encoding.
    with open(path, "w", encoding="utf-8") as f:
        json.dump(predictions, f, indent=2, ensure_ascii=True)
    print("Wrote", path)


## Stage-1a BM25 — index + cached retrieval

In [ ]:
import bm25s

# bm25s may print "resource module not available on Windows". It is harmless.
evidence_ids = list(evidence.keys())
evidence_texts = [evidence[eid] for eid in evidence_ids]

print(f"Tokenizing {len(evidence_texts):,} evidence passages...")
t0 = time.time()
corpus_tokens = bm25s.tokenize(evidence_texts, stopwords="en", stemmer=None)
print(f"Tokenization time: {time.time() - t0:.1f}s")

print(f"Building BM25 index (k1={BM25_K1}, b={BM25_B})...")
t0 = time.time()
bm25_retriever = bm25s.BM25(k1=BM25_K1, b=BM25_B)
bm25_retriever.index(corpus_tokens)
print(f"BM25 index time: {time.time() - t0:.1f}s")


def bm25_retrieve_with_scores(claims_dict, k=BM25_CANDIDATE_K):
    cids = list(claims_dict.keys())
    queries = [claims_dict[cid]["claim_text"] for cid in cids]
    query_tokens = bm25s.tokenize(queries, stopwords="en", stemmer=None)
    results, scores = bm25_retriever.retrieve(query_tokens, k=k)

    out = {}
    for i, cid in enumerate(cids):
        pairs = []
        for j, score in zip(results[i], scores[i]):
            eid = evidence_ids[int(j)]
            pairs.append((eid, float(score)))
        out[cid] = pairs
    return out


def strip_scores(candidate_cache, k):
    return {cid: [eid for eid, _ in pairs[:k]] for cid, pairs in candidate_cache.items()}


def compute_or_load_bm25_candidates(claims_dict, split_name, k=BM25_CANDIDATE_K):
    cache_file = CACHE_DIR / f"{split_name}_bm25_top{k}.pkl"
    if cache_file.exists():
        print("Loading cached BM25 candidates:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    print(f"Computing BM25 candidates for {split_name}...")
    candidates = bm25_retrieve_with_scores(claims_dict, k=k)
    with open(cache_file, "wb") as f:
        pickle.dump(candidates, f)
    return candidates

print(f"Retrieving BM25 top-{BM25_CANDIDATE_K} for dev...")
t0 = time.time()
dev_bm25_candidates = compute_or_load_bm25_candidates(dev_claims, "dev", BM25_CANDIDATE_K)
print(f"BM25 dev retrieval time: {time.time() - t0:.1f}s")

rows = []
for k in [5, 10, 20, 50, 100, 200, 500]:
    if k <= BM25_CANDIDATE_K:
        retr = strip_scores(dev_bm25_candidates, k)
        rows.append({"method": "BM25", "k": k, "retrieval_F": evaluate_retrieval_only(retr, dev_claims)})
pd.DataFrame(rows)

# Build BM25 candidates for the test split too so the hybrid pool covers all three splits in one place.
print(f"Retrieving BM25 top-{BM25_CANDIDATE_K} for test...")
t0 = time.time()
test_bm25_candidates = compute_or_load_bm25_candidates(test_claims, "test", BM25_CANDIDATE_K)
print(f"BM25 test retrieval time: {time.time() - t0:.1f}s")


## Stage-1b BGE dense — encode corpus once and cache

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F

print("Loading dense encoder:", DENSE_MODEL_NAME, "attn=", ATTN_IMPL)
dense_tokenizer = AutoTokenizer.from_pretrained(DENSE_MODEL_NAME)
dense_model = AutoModel.from_pretrained(DENSE_MODEL_NAME, attn_implementation=ATTN_IMPL).to(DEVICE).eval()


@torch.no_grad()
def bge_encode(texts, batch_size=DENSE_ENCODE_BATCH, max_len=DENSE_MAX_LEN, desc=None):
    """Encode a list of texts with BGE: take [CLS] then L2-normalize.

    Returns fp16 numpy array of shape (N, dim). fp16 halves memory; cosine sim is stable.
    Forward pass uses bf16 autocast on supported GPUs; output is cast back to fp16.
    """
    chunks = []
    iterator = range(0, len(texts), batch_size)
    if desc:
        iterator = tqdm(iterator, desc=desc, total=(len(texts) + batch_size - 1) // batch_size)
    for start in iterator:
        batch = texts[start:start + batch_size]
        enc = dense_tokenizer(
            batch, padding=True, truncation=True, max_length=max_len, return_tensors="pt"
        ).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_BF16):
            out = dense_model(**enc).last_hidden_state[:, 0]  # CLS
        out = F.normalize(out.float(), p=2, dim=-1)
        chunks.append(out.detach().to(torch.float16).cpu().numpy())
    return np.concatenate(chunks, axis=0)


evidence_emb_path = CACHE_DIR / "evidence_bge_fp16.npy"
if evidence_emb_path.exists():
    print("Loading cached evidence embeddings:", evidence_emb_path)
    evidence_emb = np.load(evidence_emb_path)
else:
    print(f"Encoding {len(evidence_texts):,} evidence passages with BGE-base ...")
    t0 = time.time()
    evidence_emb = bge_encode(evidence_texts, desc="evidence encode")
    print(f"Evidence encoding time: {time.time() - t0:.1f}s | shape={evidence_emb.shape}")
    np.save(evidence_emb_path, evidence_emb)
    print("Saved:", evidence_emb_path)

print(f"Evidence embedding tensor: {evidence_emb.shape} {evidence_emb.dtype} "
      f"({evidence_emb.nbytes / 1e9:.2f} GB)")


## Stage-1b BGE dense — per-claim top-K retrieval + diagnostics

In [ ]:
# Move evidence embeddings to GPU once. fp16 keeps it within VRAM budget.
# On a 4060 8GB or T4 16GB this fits comfortably (~1.8 GB for 1.2M x 768 fp16).
evidence_emb_gpu = torch.from_numpy(evidence_emb).to(DEVICE)
print("Evidence embeddings on", evidence_emb_gpu.device, evidence_emb_gpu.shape, evidence_emb_gpu.dtype)


@torch.no_grad()
def dense_retrieve_with_scores(claims_dict, k=DENSE_TOP_K, query_batch=64):
    cids = list(claims_dict.keys())
    queries = [claims_dict[cid]["claim_text"] for cid in cids]

    out = {}
    for start in tqdm(range(0, len(queries), query_batch), desc="dense query encode"):
        batch_cids = cids[start:start + query_batch]
        batch_qs = queries[start:start + query_batch]
        enc = dense_tokenizer(
            batch_qs, padding=True, truncation=True, max_length=DENSE_MAX_LEN, return_tensors="pt"
        ).to(DEVICE)
        with torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_BF16):
            q_emb = dense_model(**enc).last_hidden_state[:, 0]
        q_emb = F.normalize(q_emb.float(), p=2, dim=-1).to(torch.float16)
        # cosine sim because both sides are L2-normalized
        sims = q_emb @ evidence_emb_gpu.T  # (B, N)
        top_scores, top_idx = sims.topk(k, dim=-1)
        top_scores = top_scores.float().cpu().numpy()
        top_idx = top_idx.cpu().numpy()
        for row_i, cid in enumerate(batch_cids):
            pairs = []
            for j, score in zip(top_idx[row_i], top_scores[row_i]):
                pairs.append((evidence_ids[int(j)], float(score)))
            out[cid] = pairs
    return out


def compute_or_load_dense_candidates(claims_dict, split_name, k=DENSE_TOP_K):
    cache_file = CACHE_DIR / f"{split_name}_dense_top{k}.pkl"
    if cache_file.exists():
        print("Loading cached dense candidates:", cache_file)
        with open(cache_file, "rb") as f:
            return pickle.load(f)
    print(f"Computing dense candidates for {split_name} ...")
    cands = dense_retrieve_with_scores(claims_dict, k=k)
    with open(cache_file, "wb") as f:
        pickle.dump(cands, f)
    return cands


t0 = time.time()
dev_dense_candidates = compute_or_load_dense_candidates(dev_claims, "dev")
print(f"Dense dev retrieval time: {time.time() - t0:.1f}s")

# Diagnostic: dense recall at several K vs BM25 recall (already shown above).
rows = []
for k in [5, 10, 20, 50, 100, 200, 500]:
    if k <= DENSE_TOP_K:
        retr = strip_scores(dev_dense_candidates, k)
        rows.append({"method": "Dense (BGE)", "k": k, "retrieval_F": evaluate_retrieval_only(retr, dev_claims)})
pd.DataFrame(rows)


## Hybrid pool (BM25 ∪ Dense) — full-pool diagnostics, then keep dense scores for fusion only

In [ ]:
def union_candidates(bm25_pairs, dense_pairs, max_per_source=BM25_CANDIDATE_K):
    """Dedup union preserving best rank from either source.

    Returns list of (eid, score_dict) where score_dict has 'bm25' and 'dense' keys
    (NaN if absent from that source). Order: BM25 rank first, then dense-only candidates
    in dense-score order. This ordering is just for stability; the reranker will re-score.
    """
    seen = {}
    out = []
    for eid, score in bm25_pairs[:max_per_source]:
        if eid in seen:
            continue
        seen[eid] = {"bm25": float(score), "dense": float("nan")}
        out.append(eid)
    for eid, score in dense_pairs[:max_per_source]:
        if eid in seen:
            seen[eid]["dense"] = float(score)
            continue
        seen[eid] = {"bm25": float("nan"), "dense": float(score)}
        out.append(eid)
    return [(eid, seen[eid]) for eid in out]


def build_hybrid_candidates(bm25_cache, dense_cache):
    out = {}
    for cid, bm25_pairs in bm25_cache.items():
        dense_pairs = dense_cache[cid]
        out[cid] = union_candidates(bm25_pairs, dense_pairs)
    return out


def hybrid_to_score_pairs(hybrid_cache):
    """Reduce hybrid (eid, {bm25, dense}) entries to (eid, bm25_score) for downstream
    code that expects v3-style (eid, score) pairs. The bm25 score is used only when the
    item came from BM25; otherwise we fall back to the dense score (re-normalized below)
    so the candidate is not dropped. The reranker re-scores everything anyway, so the
    intermediate scalar only matters for negative sampling sort order.
    """
    out = {}
    for cid, pairs in hybrid_cache.items():
        flat = []
        for eid, sc in pairs:
            s = sc["bm25"] if not np.isnan(sc["bm25"]) else float(sc["dense"])
            flat.append((eid, float(s)))
        out[cid] = flat
    return out


# Build the BM25 + dense pools for the remaining splits (dev/test already done above).
print(f"Retrieving BM25 top-{BM25_CANDIDATE_K} for train...")
t0 = time.time()
train_bm25_candidates = compute_or_load_bm25_candidates(train_claims, "train", BM25_CANDIDATE_K)
print(f"BM25 train retrieval time: {time.time() - t0:.1f}s")

test_dense_candidates = compute_or_load_dense_candidates(test_claims, "test")
train_dense_candidates = compute_or_load_dense_candidates(train_claims, "train")

train_hybrid_full = build_hybrid_candidates(train_bm25_candidates, train_dense_candidates)
dev_hybrid_full   = build_hybrid_candidates(dev_bm25_candidates,   dev_dense_candidates)
test_hybrid_full  = build_hybrid_candidates(test_bm25_candidates,  test_dense_candidates) if test_dense_candidates else None

# Diagnostics: candidate-pool size and retrieval recall ceiling.
dev_sizes = [len(v) for v in dev_hybrid_full.values()]
print(f"Dev hybrid pool size: min={min(dev_sizes)} median={int(np.median(dev_sizes))} max={max(dev_sizes)} mean={np.mean(dev_sizes):.1f}")

dev_hybrid_eids = {cid: [eid for eid, _ in pairs] for cid, pairs in dev_hybrid_full.items()}
print(f"Dev hybrid recall (full pool): {evaluate_retrieval_only(dev_hybrid_eids, dev_claims):.4f}")

# Keep BM25-only candidates for *training negatives* (preserved under *_bm25_only_*).
# Dense retrieval pulls semantically-similar evidence that is not in the gold set ~ false
# negatives. If we sample reranker negatives from the hybrid pool, the model is trained to
# push down correct-looking-but-unlabelled items, which collapsed dev F from 0.22 -> 0.04
# in our v7 ablation. So negatives stay BM25-only; inference uses the full hybrid pool.
train_bm25_only_candidates = dict(train_bm25_candidates)
dev_bm25_only_candidates   = dict(dev_bm25_candidates)
test_bm25_only_candidates  = dict(test_bm25_candidates) if test_bm25_candidates is not None else None

# At inference time we want the reranker to score the larger hybrid pool, so rebind the
# canonical names that downstream code reads.
# DISABLED for v7-simple: train_bm25_candidates = hybrid_to_score_pairs(train_hybrid_full)
# DISABLED for v7-simple: dev_bm25_candidates   = hybrid_to_score_pairs(dev_hybrid_full)
# DISABLED for v7-simple: test_bm25_candidates  = hybrid_to_score_pairs(test_hybrid_full) if test_hybrid_full else None
print("Hybrid candidates are now wired into the v3 reranker pipeline as train/dev/test_bm25_candidates.")
print("BM25-only candidates preserved for reranker training negatives as *_bm25_only_candidates.")

# Free dense model + GPU embeddings before training the reranker.
del dense_model, evidence_emb_gpu
torch.cuda.empty_cache()


## Stage-2 cross-encoder reranker — hard-negative sampling from BM25 only

In [ ]:
from torch import nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

# train_bm25_candidates now points at the hybrid pool wired up above.


# dev candidates already computed above.


def build_reranker_examples(claims_dict, bm25_candidates, negatives_per_positive=NEGATIVES_PER_POSITIVE):
    examples = []
    rng = random.Random(SEED)

    for cid, claim in claims_dict.items():
        gold = [eid for eid in claim["evidences"] if eid in evidence]
        gold_set = set(gold)

        # Positives: each annotated evidence.
        for eid in gold:
            examples.append({"cid": cid, "eid": eid, "label": 1.0})

        # Hard negatives: sample from BM25 top-500 non-gold candidates.
        # Do not always take the very top non-gold candidates: they are often noisy negatives.
        candidate_negs = []
        seen = set()
        for eid, _ in bm25_candidates[cid][:BM25_CANDIDATE_K]:
            if eid in gold_set or eid in seen or eid not in evidence:
                continue
            candidate_negs.append(eid)
            seen.add(eid)

        needed = negatives_per_positive * max(1, len(gold))
        if len(candidate_negs) > needed:
            candidate_negs = rng.sample(candidate_negs, needed)

        for eid in candidate_negs:
            examples.append({"cid": cid, "eid": eid, "label": 0.0})

    rng.shuffle(examples)
    return examples


reranker_train_examples = build_reranker_examples(train_claims, train_bm25_only_candidates)
pos = sum(1 for x in reranker_train_examples if x["label"] == 1.0)
neg = len(reranker_train_examples) - pos
print(
    f"Reranker training examples: {len(reranker_train_examples):,} | positives={pos:,} negatives={neg:,} neg/pos={neg / max(pos, 1):.2f}"
    )


## Reranker dataset and model load

In [ ]:
class RerankerPairDataset(Dataset):
    def __init__(self, examples, claims_dict, evidence_dict, tokenizer, max_len=RERANKER_MAX_LEN):
        self.examples = examples
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        cid = ex["cid"]
        eid = ex["eid"]
        enc = self.tokenizer(
            self.claims[cid]["claim_text"],
            self.evidence[eid],
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(ex["label"], dtype=torch.float32)
        return item


print("Loading reranker:", RERANKER_MODEL_NAME)
reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)
reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME,
    num_labels=1,
    ignore_mismatched_sizes=True,
    # Default attention (eager). SDPA + fp32 should be numerically identical for inference,
    # but on this checkpoint we observed catastrophic fine-tuning collapse (dev F 0.22 -> 0.03)
    # when SDPA + fused AdamW + bf16 were combined, so we revert to the safest setup.
).to(DEVICE)
print(f"Reranker attention impl: {getattr(reranker_model.config, '_attn_implementation', '?')}")

reranker_ds = RerankerPairDataset(reranker_train_examples, train_claims, evidence, reranker_tokenizer)
reranker_loader = DataLoader(reranker_ds, batch_size=RERANKER_BATCH_SIZE, shuffle=True)

# For reranking, controlled negative sampling already defines the training balance.
# Extra pos_weight often over-corrects and hurts top-K ranking.
reranker_loss_fn = nn.BCEWithLogitsLoss()
print("Reranker loss: BCEWithLogitsLoss without pos_weight")


## Reranker scoring + selectors with dense-cosine fusion (α-sweep)

In [ ]:
@torch.no_grad()
def score_candidates_with_reranker(claims_dict, bm25_candidates, split_name="dev", dense_candidates=None):
    """Score the BM25 candidates with the fine-tuned cross-encoder reranker.

    For each candidate we additionally record the dense cosine similarity (BGE) as a separate
    feature. Candidates not present in the dense top-500 get NaN; downstream fusion imputes
    those with the per-claim min dense score before normalising.

    v7-broken showed that scoring the *full* hybrid pool (BM25 ∪ Dense) with a reranker
    trained on BM25-only negatives collapses dev F to 0.04, because the reranker assigns
    inflated scores to dense-only candidates it has never been trained against. v8 sticks
    to scoring the BM25 pool with the reranker and uses dense cosine as an *auxiliary*
    selector feature (fused in the selector), which keeps the reranker in distribution
    while still benefiting from the dense signal.
    """
    reranker_model.eval()
    score_cache = {}
    all_items = list(claims_dict.items())

    dense_lookup = {}
    if dense_candidates is not None:
        for cid, pairs in dense_candidates.items():
            dense_lookup[cid] = {eid: float(score) for eid, score in pairs}

    print(f"Scoring {split_name} candidates with cross-encoder reranker...")
    for cid, claim in tqdm(all_items):
        cand_pairs = bm25_candidates[cid]
        cand_eids = [eid for eid, _ in cand_pairs]
        bm25_scores = np.array([score for _, score in cand_pairs], dtype=np.float32)
        if cid in dense_lookup:
            dl = dense_lookup[cid]
            dense_cos = np.array([dl.get(eid, np.nan) for eid in cand_eids], dtype=np.float32)
        else:
            dense_cos = np.full(len(cand_eids), np.nan, dtype=np.float32)
        logits_all = []

        for start in range(0, len(cand_eids), RERANKER_EVAL_BATCH_SIZE):
            batch_eids = cand_eids[start:start + RERANKER_EVAL_BATCH_SIZE]
            claims_batch = [claim["claim_text"]] * len(batch_eids)
            ev_batch = [evidence[eid] for eid in batch_eids]
            enc = reranker_tokenizer(
                claims_batch,
                ev_batch,
                truncation=True,
                padding=True,
                max_length=RERANKER_MAX_LEN,
                return_tensors="pt",
            ).to(DEVICE)
            with torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_BF16):
                logits = reranker_model(**enc).logits.squeeze(-1)
            logits = logits.float()
            logits_all.extend(logits.detach().cpu().tolist())

        logits_np = np.array(logits_all, dtype=np.float32)
        score_cache[cid] = {
            "eids": cand_eids,
            "bm25": bm25_scores,
            "ce_logit": logits_np,
            "ce_prob": 1.0 / (1.0 + np.exp(-logits_np)),
            "dense_cosine": dense_cos,
        }
    return score_cache


# Fusion: alpha * sigmoid(reranker_logit) + (1 - alpha) * dense_norm. Alpha is dev-tuned.
FUSION_ALPHA_GRID = [1.0, 0.85, 0.7, 0.55, 0.4]


def _fused_score(entry, alpha):
    ce = entry["ce_prob"]
    if alpha >= 1.0 - 1e-9:
        return ce
    dense = entry.get("dense_cosine")
    if dense is None or np.isnan(dense).all():
        return ce
    nan_mask = np.isnan(dense)
    min_d = float(np.nanmin(dense))
    dense_imp = np.where(nan_mask, min_d, dense)
    rng = dense_imp.max() - dense_imp.min()
    dense_norm = (dense_imp - dense_imp.min()) / rng if rng > 1e-9 else dense_imp * 0.0
    return alpha * ce + (1.0 - alpha) * dense_norm


def select_fixed_k_ce(score_cache, k, alpha=1.0):
    out = {}
    for cid, entry in score_cache.items():
        score = _fused_score(entry, alpha)
        order = np.argsort(-score)[:k]
        out[cid] = [entry["eids"][int(i)] for i in order]
    return out


def select_dynamic_threshold_ce(score_cache, threshold, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K, alpha=1.0):
    out = {}
    for cid, entry in score_cache.items():
        score = _fused_score(entry, alpha)
        order = np.argsort(-score)
        selected = [int(i) for i in order[:max_k] if score[int(i)] >= threshold]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def select_relative_logit_ce(score_cache, delta, max_k=MAX_FINAL_K, min_k=MIN_FINAL_K, alpha=1.0):
    """Dynamic K by keeping candidates whose fused score is within delta of the top candidate."""
    out = {}
    for cid, entry in score_cache.items():
        score = _fused_score(entry, alpha)
        order = np.argsort(-score)
        top = float(score[int(order[0])])
        selected = [int(i) for i in order[:max_k] if top - float(score[int(i)]) <= delta]
        if len(selected) < min_k:
            selected = [int(i) for i in order[:min_k]]
        out[cid] = [entry["eids"][i] for i in selected[:max_k]]
    return out


def tune_retrieval_from_ce_scores(dev_score_cache):
    rows = []
    for alpha in FUSION_ALPHA_GRID:
        for k in FIXED_K_GRID:
            retr = select_fixed_k_ce(dev_score_cache, k=k, alpha=alpha)
            rows.append({
                "mode": "fixed_k", "k": k, "threshold": np.nan, "delta": np.nan, "alpha": alpha,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            })
        for threshold in THRESHOLD_GRID:
            retr = select_dynamic_threshold_ce(dev_score_cache, threshold=threshold, max_k=MAX_FINAL_K, alpha=alpha)
            rows.append({
                "mode": "dynamic_threshold", "k": np.nan, "threshold": threshold, "delta": np.nan, "alpha": alpha,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            })
        for delta in RELATIVE_LOGIT_DELTA_GRID:
            retr = select_relative_logit_ce(dev_score_cache, delta=delta, max_k=MAX_FINAL_K, alpha=alpha)
            rows.append({
                "mode": "relative_logit", "k": np.nan, "threshold": np.nan, "delta": delta, "alpha": alpha,
                "retrieval_F": evaluate_retrieval_only(retr, dev_claims),
                "avg_pred_evidence": np.mean([len(v) for v in retr.values()]),
            })
    return pd.DataFrame(rows).sort_values("retrieval_F", ascending=False).reset_index(drop=True)


def choose_final_retrieval_setting(results_df):
    """Choose the final selector from dev results with an explicit, reportable policy."""
    results_df = results_df.sort_values("retrieval_F", ascending=False).reset_index(drop=True)
    best = results_df.iloc[0]

    if FINAL_RETRIEVAL_POLICY == "best_dev":
        chosen = best
    elif FINAL_RETRIEVAL_POLICY in {"fixed_k", "dynamic_threshold", "relative_logit"}:
        family = results_df[results_df["mode"] == FINAL_RETRIEVAL_POLICY]
        if family.empty:
            raise ValueError(f"No retrieval rows for policy {FINAL_RETRIEVAL_POLICY}")
        chosen = family.iloc[0]
    elif FINAL_RETRIEVAL_POLICY == "prefer_dynamic":
        dynamic = results_df[results_df["mode"].isin(["dynamic_threshold", "relative_logit"])]
        eligible = dynamic[dynamic["retrieval_F"] >= float(best["retrieval_F"]) - DYNAMIC_RETRIEVAL_TOLERANCE]
        chosen = eligible.iloc[0] if not eligible.empty else best
    else:
        raise ValueError(f"Unknown FINAL_RETRIEVAL_POLICY: {FINAL_RETRIEVAL_POLICY}")

    chosen = chosen.to_dict()
    print("Final retrieval policy:", FINAL_RETRIEVAL_POLICY)
    print("Best dev row:", best.to_dict())
    print("Chosen row:", chosen)
    return chosen


def apply_retrieval_setting(score_cache, row):
    mode = row["mode"]
    alpha_val = row.get("alpha", 1.0)
    alpha = float(alpha_val) if not (isinstance(alpha_val, float) and np.isnan(alpha_val)) else 1.0
    if mode == "fixed_k":
        return select_fixed_k_ce(score_cache, k=int(row["k"]), alpha=alpha)
    if mode == "dynamic_threshold":
        return select_dynamic_threshold_ce(score_cache, threshold=float(row["threshold"]), max_k=MAX_FINAL_K, alpha=alpha)
    if mode == "relative_logit":
        return select_relative_logit_ce(score_cache, delta=float(row["delta"]), max_k=MAX_FINAL_K, alpha=alpha)
    raise ValueError(f"Unknown retrieval mode: {mode}")


# Train cross-encoder reranker. Dev retrieval F is evaluated after each epoch.


## Reranker training loop

In [ ]:
reranker_optimizer = torch.optim.AdamW(reranker_model.parameters(), lr=RERANKER_LR, weight_decay=WEIGHT_DECAY)
reranker_total_steps = len(reranker_loader) * RERANKER_EPOCHS
reranker_scheduler = get_linear_schedule_with_warmup(
    reranker_optimizer,
    num_warmup_steps=int(0.1 * reranker_total_steps),
    num_training_steps=reranker_total_steps,
)

best_reranker_state = None
best_reranker_row = None
best_reranker_F = -1.0

for epoch in range(1, RERANKER_EPOCHS + 1):
    reranker_model.train()
    losses = []
    t0 = time.time()

    for batch in tqdm(reranker_loader, desc=f"Reranker epoch {epoch}/{RERANKER_EPOCHS}"):
        labels = batch.pop("labels").to(DEVICE)
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        # Empirically, bf16 autocast during reranker fine-tuning of bge-reranker-base degrades
        # dev retrieval F from ~0.22 to ~0.04 on this dataset (likely an interaction between
        # fused AdamW and bf16 gradients on this checkpoint). Keep training in fp32; bf16 is
        # still used for the dense encoder, inference scoring, and the classifier (DeBERTa).
        logits = reranker_model(**batch).logits.squeeze(-1)
        loss = reranker_loss_fn(logits.float(), labels)

        reranker_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(reranker_model.parameters(), 1.0)
        reranker_optimizer.step()
        reranker_scheduler.step()
        losses.append(float(loss.item()))

    # Evaluate reranker retrieval quality on dev by sweeping fixed K and dynamic selectors.
    dev_ce_scores_tmp = score_candidates_with_reranker(dev_claims, dev_bm25_candidates, split_name=f"dev_epoch{epoch}", dense_candidates=dev_dense_candidates)
    retrieval_results_tmp = tune_retrieval_from_ce_scores(dev_ce_scores_tmp)
    row = choose_final_retrieval_setting(retrieval_results_tmp)
    print(
        f"Epoch {epoch}: loss={np.mean(losses):.4f} | "
        f"chosen dev retrieval_F={row['retrieval_F']:.4f} | setting={row} | time={time.time() - t0:.1f}s"
    )

    if row["retrieval_F"] > best_reranker_F:
        best_reranker_F = float(row["retrieval_F"])
        best_reranker_row = row
        best_reranker_state = {k: v.detach().cpu().clone() for k, v in reranker_model.state_dict().items()}

print("Best reranker retrieval setting under final policy:")
print(best_reranker_row)
if best_reranker_state is not None:
    reranker_model.load_state_dict(best_reranker_state)


## Apply best reranker to dev/train/test → final retrieval

In [ ]:
# Final retrieval caches after loading the best reranker state.
dev_ce_scores = score_candidates_with_reranker(dev_claims, dev_bm25_candidates, split_name="dev_final", dense_candidates=dev_dense_candidates)
retrieval_results = tune_retrieval_from_ce_scores(dev_ce_scores)
print(retrieval_results.head(30))

best_row = choose_final_retrieval_setting(retrieval_results)

dev_retrieval = apply_retrieval_setting(dev_ce_scores, best_row)
validate_retrieval_coverage(dev_claims, dev_retrieval, split_name="dev")

# Score train candidates as well; classifier will be trained on retrieved evidence.
train_ce_scores = score_candidates_with_reranker(train_claims, train_bm25_candidates, split_name="train_final", dense_candidates=train_dense_candidates)
train_retrieval = apply_retrieval_setting(train_ce_scores, best_row)
validate_retrieval_coverage(train_claims, train_retrieval, split_name="train")

majority = majority_label(train_claims)
print("Majority label:", majority)
dev_majority_predictions = build_predictions(dev_claims, dev_retrieval, default_label=majority)
print("\nDev score with cross-encoder retrieval + majority label:")
retrieval_majority_metrics = evaluate_submission(dev_majority_predictions, dev_claims)
write_predictions(dev_majority_predictions, OUTPUT_DIR / "dev-cross-encoder-retrieval-majority.json")


## Classifier dataset + DataLoader

In [ ]:
class ClaimEvidenceDataset(Dataset):
    def __init__(self, claims_dict, evidence_dict, retrieval_dict, tokenizer, max_len=MAX_SEQ_LEN):
        self.cids = list(claims_dict.keys())
        self.claims = claims_dict
        self.evidence = evidence_dict
        self.retrieval = retrieval_dict
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.has_label = "claim_label" in next(iter(claims_dict.values()))

    def __len__(self):
        return len(self.cids)

    def __getitem__(self, idx):
        cid = self.cids[idx]
        claim = self.claims[cid]
        ev_ids = self.retrieval[cid]
        # Explicit [SEP] between evidence passages helps preserve evidence boundaries.
        sep = f" {self.tokenizer.sep_token} "
        ev_texts = [self.evidence[eid] for eid in ev_ids if eid in self.evidence]
        if not ev_texts:
            ev_texts = [next(iter(self.evidence.values()))]
        ev_text = sep.join(ev_texts)

        enc = self.tokenizer(
            claim["claim_text"],
            ev_text,
            truncation=True,
            padding="max_length",
            max_length=self.max_len,
            return_tensors="pt",
        )
        item = {k: v.squeeze(0) for k, v in enc.items()}
        if self.has_label:
            item["labels"] = torch.tensor(LABEL2ID[claim["claim_label"]], dtype=torch.long)
        return item


def make_loader(claims_dict, retrieval_dict, tokenizer, batch_size, shuffle=False, drop_last=False):
    validate_retrieval_coverage(claims_dict, retrieval_dict, split_name="loader")
    ds = ClaimEvidenceDataset(claims_dict, evidence, retrieval_dict, tokenizer, max_len=MAX_SEQ_LEN)
    return ds, DataLoader(ds, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)


## Classifier model load + class weight setup

In [ ]:
print("Loading classifier:", CLASSIFIER_MODEL_NAME)
# DeBERTa-v3 uses a sentencepiece tokenizer; the slow tokenizer is the reliable choice here.
classifier_tokenizer = AutoTokenizer.from_pretrained(CLASSIFIER_MODEL_NAME)
classifier_model = AutoModelForSequenceClassification.from_pretrained(
    CLASSIFIER_MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    ignore_mismatched_sizes=True,
).to(DEVICE)

n_params = sum(p.numel() for p in classifier_model.parameters())
print(f"Classifier parameters: {n_params:,}")


def build_classifier_train_retrieval(claims_dict, retrieved_dict, source=CLASSIFIER_TRAIN_EVIDENCE_SOURCE):
    """Build the evidence lists used to train the claim classifier."""
    if source not in {"retrieved", "gold", "gold_plus_retrieved"}:
        raise ValueError(f"Unknown CLASSIFIER_TRAIN_EVIDENCE_SOURCE: {source}")

    out = {}
    for cid, claim in claims_dict.items():
        gold = list(claim.get("evidences", []))
        retrieved = list(retrieved_dict.get(cid, []))

        if source == "retrieved":
            chosen = retrieved
        elif source == "gold":
            chosen = gold
        else:
            chosen = []
            seen = set()
            for eid in gold + retrieved:
                if eid not in seen:
                    chosen.append(eid)
                    seen.add(eid)
                if len(chosen) >= MAX_FINAL_K:
                    break

        if not chosen:
            chosen = retrieved[:1] or [next(iter(evidence.keys()))]
        out[cid] = chosen[:MAX_FINAL_K]
    return out


classifier_train_retrieval = build_classifier_train_retrieval(train_claims, train_retrieval)
validate_retrieval_coverage(train_claims, classifier_train_retrieval, split_name="classifier_train")
print("Classifier training evidence source:", CLASSIFIER_TRAIN_EVIDENCE_SOURCE)
print("Classifier train avg evidence count:", np.mean([len(v) for v in classifier_train_retrieval.values()]))

train_ds, train_loader = make_loader(
    train_claims, classifier_train_retrieval, classifier_tokenizer, TRAIN_BATCH_SIZE, shuffle=True
)
dev_ds, dev_loader = make_loader(dev_claims, dev_retrieval, classifier_tokenizer, EVAL_BATCH_SIZE, shuffle=False)

label_counts = Counter(c["claim_label"] for c in train_claims.values())
raw_class_weights = torch.tensor(
    [len(train_claims) / (len(LABELS) * label_counts[label]) for label in LABELS],
    dtype=torch.float32,
    device=DEVICE,
)
if CLASSIFIER_USE_CLASS_WEIGHTS:
    # Optional softening: 1.0 = raw inverse-frequency, 0.5 = sqrt, 0.0 = uniform.
    softened = raw_class_weights ** CLASSIFIER_CLASS_WEIGHT_POWER
    class_weights = softened * (len(LABELS) / softened.sum())
else:
    class_weights = None
print("Raw class weights:", dict(zip(LABELS, raw_class_weights.detach().cpu().tolist())))
if class_weights is not None:
    print("Effective class weights (power=%.2f):" % CLASSIFIER_CLASS_WEIGHT_POWER,
          dict(zip(LABELS, class_weights.detach().cpu().tolist())))
print("Using class weights:", CLASSIFIER_USE_CLASS_WEIGHTS)


## Classifier loss + dev evaluator

In [ ]:
def predict_labels(model, loader, dataset):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items() if k != "labels"}
            with torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_BF16):
                logits = model(**batch).logits
            preds.extend(logits.argmax(dim=-1).detach().cpu().tolist())
    return {dataset.cids[i]: ID2LABEL[p] for i, p in enumerate(preds)}


def evaluate_classifier_on_dev(model):
    pred_labels = predict_labels(model, dev_loader, dev_ds)
    predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=pred_labels)
    return evaluate_submission(predictions, dev_claims, verbose=False), pred_labels, predictions


## Classifier training loop (select by dev A, tie-break on REFUTES/DISPUTED coverage)

In [ ]:
classifier_optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=CLASSIFIER_LR, weight_decay=WEIGHT_DECAY, fused=FUSED_OPTIM)
classifier_total_steps = len(train_loader) * CLASSIFIER_EPOCHS
classifier_scheduler = get_linear_schedule_with_warmup(
    classifier_optimizer,
    num_warmup_steps=int(0.1 * classifier_total_steps),
    num_training_steps=classifier_total_steps,
)
classifier_loss_fn = nn.CrossEntropyLoss(weight=class_weights)

best_classifier_state = None
best_metrics = {"F": 0.0, "A": -1.0, "H": -1.0, "_nonmaj": -1}

for epoch in range(1, CLASSIFIER_EPOCHS + 1):
    classifier_model.train()
    losses = []
    t0 = time.time()

    for batch in tqdm(train_loader, desc=f"Classifier epoch {epoch}/{CLASSIFIER_EPOCHS}"):
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")

        with torch.autocast(device_type=DEVICE.type, dtype=AUTOCAST_DTYPE, enabled=USE_BF16):
            logits = classifier_model(**batch).logits
            loss = classifier_loss_fn(logits.float(), labels)

        classifier_optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(classifier_model.parameters(), 1.0)
        classifier_optimizer.step()
        classifier_scheduler.step()

        losses.append(float(loss.item()))

    metrics, epoch_pred_labels, _ = evaluate_classifier_on_dev(classifier_model)
    pred_counts = Counter(epoch_pred_labels.values())
    pred_summary = ", ".join(f"{label}:{pred_counts.get(label, 0)}" for label in LABELS)
    print(
        f"Epoch {epoch}: loss={np.mean(losses):.4f} | "
        f"dev F={metrics['F']:.4f} A={metrics['A']:.4f} H={metrics['H']:.4f} | "
        f"preds=({pred_summary}) | "
        f"time={time.time() - t0:.1f}s"
    )

    # Retrieval F is constant during classifier training, so H is monotonic in A. Select by A
    # directly. Tie-break on the number of REFUTES+DISPUTED predictions: in the partial v7
    # run, max-H picked epoch 1 (REFUTES=0, DISPUTED=0) over equally-A epochs 2-3 that did
    # learn the minority classes, costing balance and harming the report story.
    nonmaj = sum(1 for v in epoch_pred_labels.values() if v in {"REFUTES", "DISPUTED"})
    cur = (metrics["A"], nonmaj)
    best = (best_metrics["A"], best_metrics.get("_nonmaj", -1))
    if cur > best:
        best_metrics = dict(metrics)
        best_metrics["_nonmaj"] = nonmaj
        best_classifier_state = {k: v.detach().cpu().clone() for k, v in classifier_model.state_dict().items()}

print("Best dev metrics:", best_metrics)
if best_classifier_state is not None:
    classifier_model.load_state_dict(best_classifier_state)

majority_h = retrieval_majority_metrics["H"]
classifier_h = best_metrics["H"]
use_majority_labels_for_final = (
    FALLBACK_TO_MAJORITY_IF_CLASSIFIER_WORSE
    and classifier_h < majority_h + CLASSIFIER_MIN_H_IMPROVEMENT
)
required_h = majority_h + (CLASSIFIER_MIN_H_IMPROVEMENT if FALLBACK_TO_MAJORITY_IF_CLASSIFIER_WORSE else 0.0)
print(
    "Final label source:",
    "majority" if use_majority_labels_for_final else "classifier",
    f"(classifier H={classifier_h:.4f}, majority H={majority_h:.4f}, required H>{required_h:.4f})",
)


## Final dev evaluation + write `dev-predictions.json`

In [ ]:
# Final dev prediction and official-format output.
# Save both classifier-only and selected-final outputs so the notebook result is unambiguous.
classifier_dev_pred_labels = predict_labels(classifier_model, dev_loader, dev_ds)
classifier_dev_predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=classifier_dev_pred_labels)
print("Classifier-only dev score:")
classifier_dev_metrics = evaluate_submission(classifier_dev_predictions, dev_claims)
write_predictions(classifier_dev_predictions, OUTPUT_DIR / "dev-predictions-classifier.json")

if use_majority_labels_for_final:
    final_dev_pred_labels = {cid: majority for cid in dev_claims.keys()}
else:
    final_dev_pred_labels = classifier_dev_pred_labels

dev_predictions = build_predictions(dev_claims, dev_retrieval, label_predictions=final_dev_pred_labels)
print("Selected final dev score:")
final_dev_metrics = evaluate_submission(dev_predictions, dev_claims)
write_predictions(dev_predictions, OUTPUT_DIR / "dev-predictions.json")
print("Selected final label source:", "majority" if use_majority_labels_for_final else "classifier")


## Sanity check with the official `eval.py`

In [ ]:
# Optional: run the official eval.py if it is available in the current directory.
import subprocess
import sys

eval_py = Path("eval.py")
if eval_py.exists():
    cmd = [
        sys.executable, "eval.py", "--predictions", str(OUTPUT_DIR / "dev-predictions.json"), "--groundtruth",
        str(DATA_DIR / "dev-claims.json")
    ]
    print("Running:", " ".join(cmd))
    completed = subprocess.run(cmd, text=True, capture_output=True)
    print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
else:
    print("eval.py not found; skipped official evaluator subprocess.")


## Confusion matrix + per-class breakdown

In [ ]:
# Simple error analysis for the report.
def confusion_matrix_df(gold_claims, pred_labels):
    mat = pd.DataFrame(0, index=LABELS, columns=LABELS)
    for cid, claim in gold_claims.items():
        mat.loc[claim["claim_label"], pred_labels[cid]] += 1
    return mat


print("Confusion matrix for selected final labels: rows=gold, cols=pred")
cm = confusion_matrix_df(dev_claims, final_dev_pred_labels)
print(cm)

if use_majority_labels_for_final:
    print("Classifier-only confusion matrix, for error analysis only:")
    print(confusion_matrix_df(dev_claims, classifier_dev_pred_labels))

per_class_rows = []
for label in LABELS:
    cids = [cid for cid, c in dev_claims.items() if c["claim_label"] == label]
    acc = np.mean([final_dev_pred_labels[cid] == label for cid in cids]) if cids else 0.0
    retr_f = np.mean(
        [evidence_f1_for_claim(dev_retrieval[cid], dev_claims[cid]["evidences"]) for cid in cids]
    ) if cids else 0.0
    per_class_rows.append({"label": label, "n": len(cids), "class_acc": acc, "retrieval_F": retr_f})

per_class = pd.DataFrame(per_class_rows)
print(per_class)


## Test-set predictions for leaderboard

In [ ]:
# Generate test predictions for leaderboard / final submission.
# The test set is unlabeled. Do not inspect or manually modify predictions.

# test_bm25_candidates is the hybrid pool built earlier.
test_ce_scores = score_candidates_with_reranker(test_claims, test_bm25_candidates, split_name="test_final", dense_candidates=test_dense_candidates)
test_retrieval = apply_retrieval_setting(test_ce_scores, best_row)
validate_retrieval_coverage(test_claims, test_retrieval, split_name="test")

if use_majority_labels_for_final:
    test_pred_labels = {cid: majority for cid in test_claims.keys()}
else:
    test_ds, test_loader = make_loader(
        test_claims,
        test_retrieval,
        classifier_tokenizer,
        EVAL_BATCH_SIZE,
        shuffle=False,
        drop_last=False,
    )
    test_pred_labels = predict_labels(classifier_model, test_loader, test_ds)

test_predictions = build_predictions(test_claims, test_retrieval, label_predictions=test_pred_labels)

# Keep the generic name expected by many leaderboard/upload workflows.
write_predictions(test_predictions, OUTPUT_DIR / "test-output.json")
print("Test output ready:", OUTPUT_DIR / "test-output.json")
print("Final retrieval setting used for test:", best_row)
print("Final label source used for test:", "majority" if use_majority_labels_for_final else "classifier")


## Appendix — captured run log

Output produced by running this pipeline as `v8_mini2v2_script.py` on a single
RTX 4060 (8 GB VRAM). tqdm progress-bar carriage returns were collapsed to keep
the markdown readable; everything else is verbatim.

```text
Device: cuda
perf: bf16=True, attn=sdpa, fused_optim=True
train:    1228
dev:      154
test:     153
evidence: 1208827
Train label distribution:
  SUPPORTS               519 (42.26%)
  REFUTES                199 (16.21%)
  NOT_ENOUGH_INFO        386 (31.43%)
  DISPUTED               124 (10.10%)

Ground-truth evidence count per train claim:
  min= 1 max= 5 mean= 3.357
  exact counts: [(1, 210), (2, 223), (3, 191), (4, 127), (5, 477)]
Tokenizing 1,208,827 evidence passages...

Split strings:   0%|          | 0/1208827 [00:00<?, ?it/s]
Split strings:   0%|          | 4360/1208827 [00:00<00:27, 43547.92it/s]
Split strings:   2%|▏         | 19248/1208827 [00:00<00:11, 105176.77it/s]
Split strings:   3%|▎         | 34611/1208827 [00:00<00:09, 126855.37it/s]
Split strings:   4%|▍         | 50144/1208827 [00:00<00:08, 137558.84it/s]
Split strings:   5%|▌         | 63892/1208827 [00:00<00:11, 103540.33it/s]
Split strings:   7%|▋         | 79563/1208827 [00:00<00:09, 117959.48it/s]
Split strings:   8%|▊         | 95972/1208827 [00:00<00:08, 131054.48it/s]
Split strings:   9%|▉         | 112522/1208827 [00:00<00:07, 140831.42it/s]
Split strings:  11%|█         | 129685/1208827 [00:00<00:07, 149342.32it/s]
Split strings:  12%|█▏        | 146551/1208827 [00:01<00:06, 154708.99it/s]
Split strings:  13%|█▎        | 162409/1208827 [00:01<00:09, 115689.69it/s]
Split strings:  15%|█▍        | 178480/1208827 [00:01<00:08, 126348.04it/s]
Split strings:  16%|█▌        | 195214/1208827 [00:01<00:07, 136597.47it/s]
Split strings:  18%|█▊        | 211908/1208827 [00:01<00:06, 144608.04it/s]
Split strings:  19%|█▉        | 228540/1208827 [00:01<00:06, 150450.51it/s]
Split strings:  20%|██        | 245718/1208827 [00:01<00:06, 155896.83it/s]
Split strings:  22%|██▏       | 261841/1208827 [00:02<00:08, 111299.19it/s]
Split strings:  23%|██▎       | 278305/1208827 [00:02<00:07, 123082.86it/s]
Split strings:  24%|██▍       | 295181/1208827 [00:02<00:06, 133976.39it/s]
Split strings:  26%|██▌       | 312216/1208827 [00:02<00:06, 143010.68it/s]
Split strings:  27%|██▋       | 328932/1208827 [00:02<00:05, 149157.18it/s]
Split strings:  29%|██▊       | 344787/1208827 [00:02<00:05, 150985.12it/s]
Split strings:  30%|██▉       | 361530/1208827 [00:02<00:05, 155008.17it/s]
Split strings:  31%|███▏      | 378314/1208827 [00:02<00:05, 158310.92it/s]
Split strings:  33%|███▎      | 394503/1208827 [00:03<00:07, 105006.18it/s]
Split strings:  34%|███▍      | 410865/1208827 [00:03<00:06, 117361.66it/s]
Split strings:  35%|███▌      | 427376/1208827 [00:03<00:06, 128311.11it/s]
Split strings:  37%|███▋      | 444028/1208827 [00:03<00:05, 137581.17it/s]
Split strings:  38%|███▊      | 460721/1208827 [00:03<00:05, 145170.18it/s]
Split strings:  39%|███▉      | 477386/1208827 [00:03<00:04, 150885.41it/s]
Split strings:  41%|████      | 494127/1208827 [00:03<00:04, 155514.71it/s]
Split strings:  42%|████▏     | 510318/1208827 [00:03<00:04, 154435.67it/s]
Split strings:  44%|████▎     | 526665/1208827 [00:03<00:04, 156619.57it/s]
Split strings:  45%|████▍     | 542649/1208827 [00:04<00:06, 97501.57it/s] 
Split strings:  46%|████▌     | 557871/1208827 [00:04<00:05, 108644.72it/s]
Split strings:  47%|████▋     | 573093/1208827 [00:04<00:05, 118263.96it/s]
Split strings:  49%|████▊     | 587053/1208827 [00:04<00:05, 123332.82it/s]
Split strings:  50%|████▉     | 600996/1208827 [00:04<00:04, 125977.93it/s]
Split strings:  51%|█████     | 614754/1208827 [00:04<00:04, 127709.13it/s]
Split strings:  52%|█████▏    | 630734/1208827 [00:04<00:04, 136483.77it/s]
Split strings:  54%|█████▎    | 647401/1208827 [00:04<00:03, 144535.64it/s]
Split strings:  55%|█████▍    | 663821/1208827 [00:04<00:03, 150141.76it/s]
Split strings:  56%|█████▋    | 680653/1208827 [00:05<00:03, 155087.58it/s]
Split strings:  58%|█████▊    | 697483/1208827 [00:05<00:03, 158287.65it/s]
Split strings:  59%|█████▉    | 714147/1208827 [00:05<00:03, 160506.27it/s]
Split strings:  60%|██████    | 730345/1208827 [00:05<00:05, 90864.70it/s] 
Split strings:  62%|██████▏   | 745458/1208827 [00:05<00:04, 102409.45it/s]
Split strings:  63%|██████▎   | 761864/1208827 [00:05<00:03, 115673.51it/s]
Split strings:  64%|██████▍   | 778259/1208827 [00:05<00:03, 127022.49it/s]
Split strings:  66%|██████▌   | 794561/1208827 [00:06<00:03, 136041.37it/s]
Split strings:  67%|██████▋   | 810348/1208827 [00:06<00:02, 141517.67it/s]
Split strings:  68%|██████▊   | 826141/1208827 [00:06<00:02, 145496.60it/s]
Split strings:  70%|██████▉   | 842382/1208827 [00:06<00:02, 150221.37it/s]
Split strings:  71%|███████   | 858971/1208827 [00:06<00:02, 154555.73it/s]
Split strings:  72%|███████▏  | 875553/1208827 [00:06<00:02, 157141.11it/s]
Split strings:  74%|███████▍  | 892250/1208827 [00:06<00:01, 159824.70it/s]
Split strings:  75%|███████▌  | 909049/1208827 [00:06<00:01, 162022.63it/s]
Split strings:  77%|███████▋  | 925698/1208827 [00:06<00:01, 162655.79it/s]
Split strings:  78%|███████▊  | 942246/1208827 [00:06<00:01, 163040.53it/s]
Split strings:  79%|███████▉  | 958969/1208827 [00:07<00:01, 163667.82it/s]
Split strings:  81%|████████  | 975399/1208827 [00:07<00:02, 82809.68it/s] 
Split strings:  82%|████████▏ | 990876/1208827 [00:07<00:02, 95484.44it/s]
Split strings:  83%|████████▎ | 1007331/1208827 [00:07<00:01, 109356.31it/s]
Split strings:  85%|████████▍ | 1023587/1208827 [00:07<00:01, 120972.84it/s]
Split strings:  86%|████████▌ | 1039948/1208827 [00:07<00:01, 131043.59it/s]
Split strings:  87%|████████▋ | 1056404/1208827 [00:07<00:01, 139516.09it/s]
Split strings:  89%|████████▉ | 1073146/1208827 [00:08<00:00, 146511.68it/s]
Split strings:  90%|█████████ | 1089542/1208827 [00:08<00:00, 151215.54it/s]
Split strings:  91%|█████████▏| 1105903/1208827 [00:08<00:00, 154596.42it/s]
Split strings:  93%|█████████▎| 1122024/1208827 [00:08<00:00, 155571.96it/s]
Split strings:  94%|█████████▍| 1138176/1208827 [00:08<00:00, 157165.22it/s]
Split strings:  96%|█████████▌| 1154444/1208827 [00:08<00:00, 158639.76it/s]
Split strings:  97%|█████████▋| 1171023/1208827 [00:08<00:00, 160089.43it/s]
Split strings:  98%|█████████▊| 1187285/1208827 [00:08<00:00, 160822.03it/s]
Split strings: 100%|█████████▉| 1203645/1208827 [00:08<00:00, 161014.52it/s]
                                                                            
Tokenization time: 8.9s
Building BM25 index (k1=1.5, b=0.5)...

BM25S Count Tokens:   0%|          | 0/1208827 [00:00<?, ?it/s]
BM25S Count Tokens:   6%|▌         | 72154/1208827 [00:00<00:01, 715052.15it/s]
BM25S Count Tokens:  12%|█▏        | 143660/1208827 [00:00<00:01, 684683.85it/s]
BM25S Count Tokens:  18%|█▊        | 212206/1208827 [00:00<00:01, 667338.63it/s]
BM25S Count Tokens:  23%|██▎       | 278993/1208827 [00:00<00:01, 657140.67it/s]
BM25S Count Tokens:  29%|██▊       | 344732/1208827 [00:00<00:01, 650871.45it/s]
BM25S Count Tokens:  34%|███▍      | 409827/1208827 [00:00<00:01, 642127.37it/s]
BM25S Count Tokens:  39%|███▉      | 474045/1208827 [00:00<00:01, 632667.87it/s]
BM25S Count Tokens:  44%|████▍     | 537319/1208827 [00:00<00:01, 624119.00it/s]
BM25S Count Tokens:  50%|████▉     | 599737/1208827 [00:00<00:00, 616806.12it/s]
BM25S Count Tokens:  55%|█████▍    | 661420/1208827 [00:01<00:00, 612701.80it/s]
BM25S Count Tokens:  60%|█████▉    | 722686/1208827 [00:01<00:00, 607370.05it/s]
BM25S Count Tokens:  65%|██████▍   | 783650/1208827 [00:01<00:00, 607973.30it/s]
BM25S Count Tokens:  70%|██████▉   | 845352/1208827 [00:01<00:00, 608524.93it/s]
BM25S Count Tokens:  75%|███████▍  | 906204/1208827 [00:01<00:00, 606818.01it/s]
BM25S Count Tokens:  80%|███████▉  | 966884/1208827 [00:01<00:00, 592341.98it/s]
BM25S Count Tokens:  85%|████████▍ | 1026178/1208827 [00:01<00:00, 588850.75it/s]
BM25S Count Tokens:  90%|████████▉ | 1086788/1208827 [00:01<00:00, 592290.90it/s]
BM25S Count Tokens:  95%|█████████▍| 1147121/1208827 [00:01<00:00, 594835.16it/s]
BM25S Count Tokens: 100%|█████████▉| 1207023/1208827 [00:01<00:00, 595871.21it/s]
                                                                                 

BM25S Compute Scores:   0%|          | 0/1208827 [00:00<?, ?it/s]
BM25S Compute Scores:   1%|          | 12610/1208827 [00:00<00:09, 125306.03it/s]
BM25S Compute Scores:   2%|▏         | 26477/1208827 [00:00<00:08, 133105.47it/s]
BM25S Compute Scores:   3%|▎         | 40451/1208827 [00:00<00:08, 135864.67it/s]
BM25S Compute Scores:   4%|▍         | 54311/1208827 [00:00<00:08, 136692.71it/s]
BM25S Compute Scores:   6%|▌         | 68106/1208827 [00:00<00:08, 137001.19it/s]
BM25S Compute Scores:   7%|▋         | 81807/1208827 [00:00<00:08, 136930.69it/s]
BM25S Compute Scores:   8%|▊         | 95593/1208827 [00:00<00:08, 137131.33it/s]
BM25S Compute Scores:   9%|▉         | 109307/1208827 [00:00<00:08, 136990.59it/s]
BM25S Compute Scores:  10%|█         | 123107/1208827 [00:00<00:07, 137242.37it/s]
BM25S Compute Scores:  11%|█▏        | 136832/1208827 [00:01<00:07, 137108.25it/s]
BM25S Compute Scores:  12%|█▏        | 150571/1208827 [00:01<00:07, 137058.20it/s]
BM25S Compute Scores:  14%|█▎        | 164502/1208827 [00:01<00:07, 137371.60it/s]
BM25S Compute Scores:  15%|█▍        | 178372/1208827 [00:01<00:07, 137575.06it/s]
BM25S Compute Scores:  16%|█▌        | 192206/1208827 [00:01<00:07, 137275.09it/s]
BM25S Compute Scores:  17%|█▋        | 205947/1208827 [00:01<00:07, 137269.95it/s]
BM25S Compute Scores:  18%|█▊        | 219706/1208827 [00:01<00:07, 137183.14it/s]
BM25S Compute Scores:  19%|█▉        | 233425/1208827 [00:01<00:07, 136792.29it/s]
BM25S Compute Scores:  20%|██        | 247105/1208827 [00:01<00:07, 136532.49it/s]
BM25S Compute Scores:  22%|██▏       | 260819/1208827 [00:01<00:06, 136372.85it/s]
BM25S Compute Scores:  23%|██▎       | 274550/1208827 [00:02<00:06, 136438.20it/s]
BM25S Compute Scores:  24%|██▍       | 288495/1208827 [00:02<00:06, 136733.27it/s]
BM25S Compute Scores:  25%|██▌       | 302246/1208827 [00:02<00:06, 136824.18it/s]
BM25S Compute Scores:  26%|██▌       | 315929/1208827 [00:02<00:06, 136567.48it/s]
BM25S Compute Scores:  27%|██▋       | 329586/1208827 [00:02<00:06, 136318.96it/s]
BM25S Compute Scores:  28%|██▊       | 343218/1208827 [00:02<00:06, 135622.14it/s]
BM25S Compute Scores:  30%|██▉       | 356781/1208827 [00:02<00:06, 135077.46it/s]
BM25S Compute Scores:  31%|███       | 370435/1208827 [00:02<00:06, 135195.76it/s]
BM25S Compute Scores:  32%|███▏      | 384287/1208827 [00:02<00:06, 135836.30it/s]
BM25S Compute Scores:  33%|███▎      | 398132/1208827 [00:02<00:05, 136406.95it/s]
BM25S Compute Scores:  34%|███▍      | 411980/1208827 [00:03<00:05, 136732.16it/s]
BM25S Compute Scores:  35%|███▌      | 425662/1208827 [00:03<00:05, 136719.99it/s]
BM25S Compute Scores:  36%|███▋      | 439475/1208827 [00:03<00:05, 136864.72it/s]
BM25S Compute Scores:  38%|███▊      | 453343/1208827 [00:03<00:05, 136977.38it/s]
BM25S Compute Scores:  39%|███▊      | 467041/1208827 [00:03<00:05, 136760.32it/s]
BM25S Compute Scores:  40%|███▉      | 480718/1208827 [00:03<00:05, 135345.49it/s]
BM25S Compute Scores:  41%|████      | 494335/1208827 [00:03<00:05, 135582.78it/s]
BM25S Compute Scores:  42%|████▏     | 508016/1208827 [00:03<00:05, 135921.31it/s]
BM25S Compute Scores:  43%|████▎     | 521759/1208827 [00:03<00:05, 135962.18it/s]
BM25S Compute Scores:  44%|████▍     | 535506/1208827 [00:03<00:04, 136043.06it/s]
BM25S Compute Scores:  45%|████▌     | 549300/1208827 [00:04<00:04, 136256.15it/s]
BM25S Compute Scores:  47%|████▋     | 563090/1208827 [00:04<00:04, 136366.24it/s]
BM25S Compute Scores:  48%|████▊     | 576835/1208827 [00:04<00:04, 136667.06it/s]
BM25S Compute Scores:  49%|████▉     | 590705/1208827 [00:04<00:04, 137136.88it/s]
BM25S Compute Scores:  50%|█████     | 604444/1208827 [00:04<00:04, 136992.25it/s]
BM25S Compute Scores:  51%|█████     | 618205/1208827 [00:04<00:04, 136987.06it/s]
BM25S Compute Scores:  52%|█████▏    | 631904/1208827 [00:04<00:04, 136836.05it/s]
BM25S Compute Scores:  53%|█████▎    | 645599/1208827 [00:04<00:04, 136862.16it/s]
BM25S Compute Scores:  55%|█████▍    | 659396/1208827 [00:04<00:04, 137015.34it/s]
BM25S Compute Scores:  56%|█████▌    | 673131/1208827 [00:04<00:03, 137061.15it/s]
BM25S Compute Scores:  57%|█████▋    | 686864/1208827 [00:05<00:03, 137133.85it/s]
BM25S Compute Scores:  58%|█████▊    | 700578/1208827 [00:05<00:03, 136854.43it/s]
BM25S Compute Scores:  59%|█████▉    | 714314/1208827 [00:05<00:03, 136707.10it/s]
BM25S Compute Scores:  60%|██████    | 728041/1208827 [00:05<00:03, 136848.83it/s]
BM25S Compute Scores:  61%|██████▏   | 741727/1208827 [00:05<00:03, 136711.35it/s]
BM25S Compute Scores:  62%|██████▏   | 755411/1208827 [00:05<00:03, 136672.27it/s]
BM25S Compute Scores:  64%|██████▎   | 769099/1208827 [00:05<00:03, 136547.47it/s]
BM25S Compute Scores:  65%|██████▍   | 782754/1208827 [00:05<00:03, 136234.48it/s]
BM25S Compute Scores:  66%|██████▌   | 796432/1208827 [00:05<00:03, 135908.06it/s]
BM25S Compute Scores:  67%|██████▋   | 810198/1208827 [00:05<00:02, 135973.73it/s]
BM25S Compute Scores:  68%|██████▊   | 823852/1208827 [00:06<00:02, 136029.98it/s]
BM25S Compute Scores:  69%|██████▉   | 837456/1208827 [00:06<00:02, 135834.85it/s]
BM25S Compute Scores:  70%|███████   | 851040/1208827 [00:06<00:02, 135534.96it/s]
BM25S Compute Scores:  72%|███████▏  | 864657/1208827 [00:06<00:02, 135713.25it/s]
BM25S Compute Scores:  73%|███████▎  | 878283/1208827 [00:06<00:02, 135847.24it/s]
BM25S Compute Scores:  74%|███████▍  | 892016/1208827 [00:06<00:02, 135970.48it/s]
BM25S Compute Scores:  75%|███████▍  | 905641/1208827 [00:06<00:02, 135965.80it/s]
BM25S Compute Scores:  76%|███████▌  | 919306/1208827 [00:06<00:02, 135984.01it/s]
BM25S Compute Scores:  77%|███████▋  | 932918/1208827 [00:06<00:02, 135937.96it/s]
BM25S Compute Scores:  78%|███████▊  | 946544/1208827 [00:06<00:01, 136010.79it/s]
BM25S Compute Scores:  79%|███████▉  | 960301/1208827 [00:07<00:01, 136108.37it/s]
BM25S Compute Scores:  81%|████████  | 973941/1208827 [00:07<00:01, 136170.28it/s]
BM25S Compute Scores:  82%|████████▏ | 987559/1208827 [00:07<00:01, 136116.51it/s]
BM25S Compute Scores:  83%|████████▎ | 1001171/1208827 [00:07<00:01, 136104.96it/s]
BM25S Compute Scores:  84%|████████▍ | 1014838/1208827 [00:07<00:01, 136131.24it/s]
BM25S Compute Scores:  85%|████████▌ | 1028565/1208827 [00:07<00:01, 136111.27it/s]
BM25S Compute Scores:  86%|████████▌ | 1042321/1208827 [00:07<00:01, 136325.39it/s]
BM25S Compute Scores:  87%|████████▋ | 1056105/1208827 [00:07<00:01, 136563.65it/s]
BM25S Compute Scores:  89%|████████▊ | 1070047/1208827 [00:07<00:01, 136955.24it/s]
BM25S Compute Scores:  90%|████████▉ | 1083936/1208827 [00:07<00:00, 137514.16it/s]
BM25S Compute Scores:  91%|█████████ | 1097997/1208827 [00:08<00:00, 137922.26it/s]
BM25S Compute Scores:  92%|█████████▏| 1111903/1208827 [00:08<00:00, 137977.35it/s]
BM25S Compute Scores:  93%|█████████▎| 1125887/1208827 [00:08<00:00, 138059.61it/s]
BM25S Compute Scores:  94%|█████████▍| 1139693/1208827 [00:08<00:00, 138050.93it/s]
BM25S Compute Scores:  95%|█████████▌| 1153499/1208827 [00:08<00:00, 137300.93it/s]
BM25S Compute Scores:  97%|█████████▋| 1167230/1208827 [00:08<00:00, 137300.55it/s]
BM25S Compute Scores:  98%|█████████▊| 1180961/1208827 [00:08<00:00, 137156.47it/s]
BM25S Compute Scores:  99%|█████████▉| 1194744/1208827 [00:08<00:00, 137024.61it/s]
BM25S Compute Scores: 100%|█████████▉| 1208486/1208827 [00:08<00:00, 137025.06it/s]
                                                                                   
Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
BM25 index time: 12.1s
Retrieving BM25 top-500 for dev...
Loading cached BM25 candidates: outputs_notebook_v7\cache\dev_bm25_top500.pkl
BM25 dev retrieval time: 0.0s
Retrieving BM25 top-500 for test...
Loading cached BM25 candidates: outputs_notebook_v7\cache\test_bm25_top500.pkl
BM25 test retrieval time: 0.0s
Loading dense encoder: BAAI/bge-base-en-v1.5 attn= sdpa

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9732.59it/s]
Loading cached evidence embeddings: outputs_notebook_v7\cache\evidence_bge_fp16.npy
Evidence embedding tensor: (1208827, 768) float16 (1.86 GB)
Evidence embeddings on cuda:0 torch.Size([1208827, 768]) torch.float16
Loading cached dense candidates: outputs_notebook_v7\cache\dev_dense_top500.pkl
Dense dev retrieval time: 0.0s
Retrieving BM25 top-500 for train...
Loading cached BM25 candidates: outputs_notebook_v7\cache\train_bm25_top500.pkl
BM25 train retrieval time: 0.1s
Loading cached dense candidates: outputs_notebook_v7\cache\test_dense_top500.pkl
Loading cached dense candidates: outputs_notebook_v7\cache\train_dense_top500.pkl
Dev hybrid pool size: min=770 median=925 max=992 mean=918.7
Dev hybrid recall (full pool): 0.0060
Hybrid candidates are now wired into the v3 reranker pipeline as train/dev/test_bm25_candidates.
BM25-only candidates preserved for reranker training negatives as *_bm25_only_candidates.
Reranker training examples: 20,610 | positives=4,122 negatives=16,488 neg/pos=4.00
Loading reranker: cross-encoder/ms-marco-MiniLM-L-6-v2

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9759.38it/s]
Reranker attention impl: sdpa
Reranker loss: BCEWithLogitsLoss without pos_weight

Reranker epoch 1/3:   0%|          | 0/645 [00:00<?, ?it/s]
Reranker epoch 1/3:   0%|          | 1/645 [00:00<04:52,  2.20it/s]
Reranker epoch 1/3:   0%|          | 2/645 [00:00<02:57,  3.63it/s]
Reranker epoch 1/3:   0%|          | 3/645 [00:00<02:20,  4.57it/s]
Reranker epoch 1/3:   1%|          | 4/645 [00:00<02:03,  5.21it/s]
Reranker epoch 1/3:   1%|          | 5/645 [00:01<01:53,  5.62it/s]
Reranker epoch 1/3:   1%|          | 6/645 [00:01<01:47,  5.95it/s]
Reranker epoch 1/3:   1%|          | 7/645 [00:01<01:44,  6.13it/s]
Reranker epoch 1/3:   1%|          | 8/645 [00:01<01:41,  6.30it/s]
Reranker epoch 1/3:   1%|▏         | 9/645 [00:01<01:39,  6.39it/s]
Reranker epoch 1/3:   2%|▏         | 10/645 [00:01<01:37,  6.49it/s]
Reranker epoch 1/3:   2%|▏         | 11/645 [00:01<01:35,  6.63it/s]
Reranker epoch 1/3:   2%|▏         | 12/645 [00:02<01:35,  6.65it/s]
Reranker epoch 1/3:   2%|▏         | 13/645 [00:02<01:35,  6.65it/s]
Reranker epoch 1/3:   2%|▏         | 14/645 [00:02<01:34,  6.66it/s]
Reranker epoch 1/3:   2%|▏         | 15/645 [00:02<01:34,  6.66it/s]
Reranker epoch 1/3:   2%|▏         | 16/645 [00:02<01:34,  6.63it/s]
Reranker epoch 1/3:   3%|▎         | 17/645 [00:02<01:34,  6.61it/s]
Reranker epoch 1/3:   3%|▎         | 18/645 [00:03<01:34,  6.64it/s]
Reranker epoch 1/3:   3%|▎         | 19/645 [00:03<01:33,  6.68it/s]
Reranker epoch 1/3:   3%|▎         | 20/645 [00:03<01:32,  6.72it/s]
Reranker epoch 1/3:   3%|▎         | 21/645 [00:03<01:33,  6.68it/s]
Reranker epoch 1/3:   3%|▎         | 22/645 [00:03<01:33,  6.69it/s]
Reranker epoch 1/3:   4%|▎         | 23/645 [00:03<01:33,  6.67it/s]
Reranker epoch 1/3:   4%|▎         | 24/645 [00:03<01:33,  6.66it/s]
Reranker epoch 1/3:   4%|▍         | 25/645 [00:04<01:32,  6.73it/s]
Reranker epoch 1/3:   4%|▍         | 26/645 [00:04<01:32,  6.68it/s]
Reranker epoch 1/3:   4%|▍         | 27/645 [00:04<01:32,  6.67it/s]
Reranker epoch 1/3:   4%|▍         | 28/645 [00:04<01:32,  6.65it/s]
Reranker epoch 1/3:   4%|▍         | 29/645 [00:04<01:32,  6.64it/s]
Reranker epoch 1/3:   5%|▍         | 30/645 [00:04<01:31,  6.72it/s]
Reranker epoch 1/3:   5%|▍         | 31/645 [00:04<01:31,  6.72it/s]
Reranker epoch 1/3:   5%|▍         | 32/645 [00:05<01:31,  6.70it/s]
Reranker epoch 1/3:   5%|▌         | 33/645 [00:05<01:30,  6.73it/s]
Reranker epoch 1/3:   5%|▌         | 34/645 [00:05<01:31,  6.69it/s]
Reranker epoch 1/3:   5%|▌         | 35/645 [00:05<01:30,  6.73it/s]
Reranker epoch 1/3:   6%|▌         | 36/645 [00:05<01:30,  6.71it/s]
Reranker epoch 1/3:   6%|▌         | 37/645 [00:05<01:30,  6.69it/s]
Reranker epoch 1/3:   6%|▌         | 38/645 [00:05<01:30,  6.68it/s]
Reranker epoch 1/3:   6%|▌         | 39/645 [00:06<01:31,  6.64it/s]
Reranker epoch 1/3:   6%|▌         | 40/645 [00:06<01:31,  6.61it/s]
Reranker epoch 1/3:   6%|▋         | 41/645 [00:06<01:31,  6.61it/s]
Reranker epoch 1/3:   7%|▋         | 42/645 [00:06<01:30,  6.64it/s]
Reranker epoch 1/3:   7%|▋         | 43/645 [00:06<01:30,  6.66it/s]
Reranker epoch 1/3:   7%|▋         | 44/645 [00:06<01:30,  6.65it/s]
Reranker epoch 1/3:   7%|▋         | 45/645 [00:07<01:30,  6.66it/s]
Reranker epoch 1/3:   7%|▋         | 46/645 [00:07<01:29,  6.73it/s]
Reranker epoch 1/3:   7%|▋         | 47/645 [00:07<01:29,  6.67it/s]
Reranker epoch 1/3:   7%|▋         | 48/645 [00:07<01:29,  6.64it/s]
Reranker epoch 1/3:   8%|▊         | 49/645 [00:07<01:29,  6.65it/s]
Reranker epoch 1/3:   8%|▊         | 50/645 [00:07<01:29,  6.65it/s]
Reranker epoch 1/3:   8%|▊         | 51/645 [00:07<01:29,  6.63it/s]
Reranker epoch 1/3:   8%|▊         | 52/645 [00:08<01:30,  6.57it/s]
Reranker epoch 1/3:   8%|▊         | 53/645 [00:08<01:29,  6.58it/s]
Reranker epoch 1/3:   8%|▊         | 54/645 [00:08<01:29,  6.63it/s]
Reranker epoch 1/3:   9%|▊         | 55/645 [00:08<01:29,  6.59it/s]
Reranker epoch 1/3:   9%|▊         | 56/645 [00:08<01:28,  6.67it/s]
Reranker epoch 1/3:   9%|▉         | 57/645 [00:08<01:28,  6.62it/s]
Reranker epoch 1/3:   9%|▉         | 58/645 [00:09<01:28,  6.65it/s]
Reranker epoch 1/3:   9%|▉         | 59/645 [00:09<01:28,  6.66it/s]
Reranker epoch 1/3:   9%|▉         | 60/645 [00:09<01:27,  6.67it/s]
Reranker epoch 1/3:   9%|▉         | 61/645 [00:09<01:27,  6.67it/s]
Reranker epoch 1/3:  10%|▉         | 62/645 [00:09<01:27,  6.67it/s]
Reranker epoch 1/3:  10%|▉         | 63/645 [00:09<01:27,  6.68it/s]
Reranker epoch 1/3:  10%|▉         | 64/645 [00:09<01:26,  6.71it/s]
Reranker epoch 1/3:  10%|█         | 65/645 [00:10<01:27,  6.67it/s]
Reranker epoch 1/3:  10%|█         | 66/645 [00:10<01:26,  6.66it/s]
Reranker epoch 1/3:  10%|█         | 67/645 [00:10<01:25,  6.72it/s]
Reranker epoch 1/3:  11%|█         | 68/645 [00:10<01:25,  6.72it/s]
Reranker epoch 1/3:  11%|█         | 69/645 [00:10<01:25,  6.72it/s]
Reranker epoch 1/3:  11%|█         | 70/645 [00:10<01:25,  6.71it/s]
Reranker epoch 1/3:  11%|█         | 71/645 [00:10<01:25,  6.73it/s]
Reranker epoch 1/3:  11%|█         | 72/645 [00:11<01:25,  6.69it/s]
Reranker epoch 1/3:  11%|█▏        | 73/645 [00:11<01:25,  6.67it/s]
Reranker epoch 1/3:  11%|█▏        | 74/645 [00:11<01:25,  6.67it/s]
Reranker epoch 1/3:  12%|█▏        | 75/645 [00:11<01:25,  6.69it/s]
Reranker epoch 1/3:  12%|█▏        | 76/645 [00:11<01:25,  6.68it/s]
Reranker epoch 1/3:  12%|█▏        | 77/645 [00:11<01:24,  6.70it/s]
Reranker epoch 1/3:  12%|█▏        | 78/645 [00:11<01:24,  6.68it/s]
Reranker epoch 1/3:  12%|█▏        | 79/645 [00:12<01:24,  6.70it/s]
Reranker epoch 1/3:  12%|█▏        | 80/645 [00:12<01:24,  6.67it/s]
Reranker epoch 1/3:  13%|█▎        | 81/645 [00:12<01:24,  6.66it/s]
Reranker epoch 1/3:  13%|█▎        | 82/645 [00:12<01:25,  6.62it/s]
Reranker epoch 1/3:  13%|█▎        | 83/645 [00:12<01:24,  6.68it/s]
Reranker epoch 1/3:  13%|█▎        | 84/645 [00:12<01:23,  6.72it/s]
Reranker epoch 1/3:  13%|█▎        | 85/645 [00:13<01:23,  6.71it/s]
Reranker epoch 1/3:  13%|█▎        | 86/645 [00:13<01:23,  6.70it/s]
Reranker epoch 1/3:  13%|█▎        | 87/645 [00:13<01:23,  6.66it/s]
Reranker epoch 1/3:  14%|█▎        | 88/645 [00:13<01:24,  6.61it/s]
Reranker epoch 1/3:  14%|█▍        | 89/645 [00:13<01:23,  6.67it/s]
Reranker epoch 1/3:  14%|█▍        | 90/645 [00:13<01:23,  6.68it/s]
Reranker epoch 1/3:  14%|█▍        | 91/645 [00:13<01:23,  6.67it/s]
Reranker epoch 1/3:  14%|█▍        | 92/645 [00:14<01:22,  6.70it/s]
Reranker epoch 1/3:  14%|█▍        | 93/645 [00:14<01:22,  6.71it/s]
Reranker epoch 1/3:  15%|█▍        | 94/645 [00:14<01:20,  6.80it/s]
Reranker epoch 1/3:  15%|█▍        | 95/645 [00:14<01:21,  6.75it/s]
Reranker epoch 1/3:  15%|█▍        | 96/645 [00:14<01:22,  6.68it/s]
Reranker epoch 1/3:  15%|█▌        | 97/645 [00:14<01:21,  6.73it/s]
Reranker epoch 1/3:  15%|█▌        | 98/645 [00:14<01:22,  6.65it/s]
Reranker epoch 1/3:  15%|█▌        | 99/645 [00:15<01:21,  6.66it/s]
Reranker epoch 1/3:  16%|█▌        | 100/645 [00:15<01:22,  6.61it/s]
Reranker epoch 1/3:  16%|█▌        | 101/645 [00:15<01:20,  6.72it/s]
Reranker epoch 1/3:  16%|█▌        | 102/645 [00:15<01:21,  6.64it/s]
Reranker epoch 1/3:  16%|█▌        | 103/645 [00:15<01:20,  6.71it/s]
Reranker epoch 1/3:  16%|█▌        | 104/645 [00:15<01:20,  6.72it/s]
Reranker epoch 1/3:  16%|█▋        | 105/645 [00:16<01:20,  6.71it/s]
Reranker epoch 1/3:  16%|█▋        | 106/645 [00:16<01:20,  6.66it/s]
Reranker epoch 1/3:  17%|█▋        | 107/645 [00:16<01:20,  6.67it/s]
Reranker epoch 1/3:  17%|█▋        | 108/645 [00:16<01:20,  6.67it/s]
Reranker epoch 1/3:  17%|█▋        | 109/645 [00:16<01:20,  6.67it/s]
Reranker epoch 1/3:  17%|█▋        | 110/645 [00:16<01:20,  6.62it/s]
Reranker epoch 1/3:  17%|█▋        | 111/645 [00:16<01:20,  6.63it/s]
Reranker epoch 1/3:  17%|█▋        | 112/645 [00:17<01:20,  6.63it/s]
Reranker epoch 1/3:  18%|█▊        | 113/645 [00:17<01:19,  6.70it/s]
Reranker epoch 1/3:  18%|█▊        | 114/645 [00:17<01:19,  6.72it/s]
Reranker epoch 1/3:  18%|█▊        | 115/645 [00:17<01:19,  6.66it/s]
Reranker epoch 1/3:  18%|█▊        | 116/645 [00:17<01:19,  6.64it/s]
Reranker epoch 1/3:  18%|█▊        | 117/645 [00:17<01:19,  6.67it/s]
Reranker epoch 1/3:  18%|█▊        | 118/645 [00:17<01:19,  6.65it/s]
Reranker epoch 1/3:  18%|█▊        | 119/645 [00:18<01:18,  6.72it/s]
Reranker epoch 1/3:  19%|█▊        | 120/645 [00:18<01:18,  6.71it/s]
Reranker epoch 1/3:  19%|█▉        | 121/645 [00:18<01:17,  6.75it/s]
Reranker epoch 1/3:  19%|█▉        | 122/645 [00:18<01:17,  6.75it/s]
Reranker epoch 1/3:  19%|█▉        | 123/645 [00:18<01:17,  6.72it/s]
Reranker epoch 1/3:  19%|█▉        | 124/645 [00:18<01:17,  6.74it/s]
Reranker epoch 1/3:  19%|█▉        | 125/645 [00:19<01:17,  6.69it/s]
Reranker epoch 1/3:  20%|█▉        | 126/645 [00:19<01:16,  6.77it/s]
Reranker epoch 1/3:  20%|█▉        | 127/645 [00:19<01:17,  6.71it/s]
Reranker epoch 1/3:  20%|█▉        | 128/645 [00:19<01:17,  6.67it/s]
Reranker epoch 1/3:  20%|██        | 129/645 [00:19<01:17,  6.68it/s]
Reranker epoch 1/3:  20%|██        | 130/645 [00:19<01:17,  6.67it/s]
Reranker epoch 1/3:  20%|██        | 131/645 [00:19<01:16,  6.68it/s]
Reranker epoch 1/3:  20%|██        | 132/645 [00:20<01:16,  6.68it/s]
Reranker epoch 1/3:  21%|██        | 133/645 [00:20<01:15,  6.77it/s]
Reranker epoch 1/3:  21%|██        | 134/645 [00:20<01:15,  6.81it/s]
Reranker epoch 1/3:  21%|██        | 135/645 [00:20<01:14,  6.84it/s]
Reranker epoch 1/3:  21%|██        | 136/645 [00:20<01:15,  6.77it/s]
Reranker epoch 1/3:  21%|██        | 137/645 [00:20<01:15,  6.77it/s]
Reranker epoch 1/3:  21%|██▏       | 138/645 [00:20<01:14,  6.79it/s]
Reranker epoch 1/3:  22%|██▏       | 139/645 [00:21<01:14,  6.77it/s]
Reranker epoch 1/3:  22%|██▏       | 140/645 [00:21<01:15,  6.70it/s]
Reranker epoch 1/3:  22%|██▏       | 141/645 [00:21<01:15,  6.71it/s]
Reranker epoch 1/3:  22%|██▏       | 142/645 [00:21<01:14,  6.73it/s]
Reranker epoch 1/3:  22%|██▏       | 143/645 [00:21<01:14,  6.71it/s]
Reranker epoch 1/3:  22%|██▏       | 144/645 [00:21<01:14,  6.69it/s]
Reranker epoch 1/3:  22%|██▏       | 145/645 [00:21<01:14,  6.70it/s]
Reranker epoch 1/3:  23%|██▎       | 146/645 [00:22<01:14,  6.67it/s]
Reranker epoch 1/3:  23%|██▎       | 147/645 [00:22<01:14,  6.68it/s]
Reranker epoch 1/3:  23%|██▎       | 148/645 [00:22<01:14,  6.65it/s]
Reranker epoch 1/3:  23%|██▎       | 149/645 [00:22<01:14,  6.68it/s]
Reranker epoch 1/3:  23%|██▎       | 150/645 [00:22<01:13,  6.71it/s]
Reranker epoch 1/3:  23%|██▎       | 151/645 [00:22<01:13,  6.71it/s]
Reranker epoch 1/3:  24%|██▎       | 152/645 [00:23<01:13,  6.69it/s]
Reranker epoch 1/3:  24%|██▎       | 153/645 [00:23<01:12,  6.74it/s]
Reranker epoch 1/3:  24%|██▍       | 154/645 [00:23<01:13,  6.71it/s]
Reranker epoch 1/3:  24%|██▍       | 155/645 [00:23<01:13,  6.68it/s]
Reranker epoch 1/3:  24%|██▍       | 156/645 [00:23<01:13,  6.66it/s]
Reranker epoch 1/3:  24%|██▍       | 157/645 [00:23<01:12,  6.74it/s]
Reranker epoch 1/3:  24%|██▍       | 158/645 [00:23<01:12,  6.69it/s]
Reranker epoch 1/3:  25%|██▍       | 159/645 [00:24<01:12,  6.69it/s]
Reranker epoch 1/3:  25%|██▍       | 160/645 [00:24<01:12,  6.69it/s]
Reranker epoch 1/3:  25%|██▍       | 161/645 [00:24<01:11,  6.73it/s]
Reranker epoch 1/3:  25%|██▌       | 162/645 [00:24<01:12,  6.69it/s]
Reranker epoch 1/3:  25%|██▌       | 163/645 [00:24<01:12,  6.62it/s]
Reranker epoch 1/3:  25%|██▌       | 164/645 [00:24<01:12,  6.60it/s]
Reranker epoch 1/3:  26%|██▌       | 165/645 [00:24<01:12,  6.65it/s]
Reranker epoch 1/3:  26%|██▌       | 166/645 [00:25<01:12,  6.63it/s]
Reranker epoch 1/3:  26%|██▌       | 167/645 [00:25<01:11,  6.69it/s]
Reranker epoch 1/3:  26%|██▌       | 168/645 [00:25<01:11,  6.68it/s]
Reranker epoch 1/3:  26%|██▌       | 169/645 [00:25<01:11,  6.69it/s]
Reranker epoch 1/3:  26%|██▋       | 170/645 [00:25<01:10,  6.71it/s]
Reranker epoch 1/3:  27%|██▋       | 171/645 [00:25<01:10,  6.71it/s]
Reranker epoch 1/3:  27%|██▋       | 172/645 [00:26<01:10,  6.74it/s]
Reranker epoch 1/3:  27%|██▋       | 173/645 [00:26<01:10,  6.70it/s]
Reranker epoch 1/3:  27%|██▋       | 174/645 [00:26<01:10,  6.70it/s]
Reranker epoch 1/3:  27%|██▋       | 175/645 [00:26<01:10,  6.68it/s]
Reranker epoch 1/3:  27%|██▋       | 176/645 [00:26<01:09,  6.71it/s]
Reranker epoch 1/3:  27%|██▋       | 177/645 [00:26<01:09,  6.69it/s]
Reranker epoch 1/3:  28%|██▊       | 178/645 [00:26<01:10,  6.62it/s]
Reranker epoch 1/3:  28%|██▊       | 179/645 [00:27<01:10,  6.63it/s]
Reranker epoch 1/3:  28%|██▊       | 180/645 [00:27<01:10,  6.61it/s]
Reranker epoch 1/3:  28%|██▊       | 181/645 [00:27<01:09,  6.71it/s]
Reranker epoch 1/3:  28%|██▊       | 182/645 [00:27<01:09,  6.69it/s]
Reranker epoch 1/3:  28%|██▊       | 183/645 [00:27<01:09,  6.68it/s]
Reranker epoch 1/3:  29%|██▊       | 184/645 [00:27<01:09,  6.65it/s]
Reranker epoch 1/3:  29%|██▊       | 185/645 [00:27<01:09,  6.64it/s]
Reranker epoch 1/3:  29%|██▉       | 186/645 [00:28<01:08,  6.67it/s]
Reranker epoch 1/3:  29%|██▉       | 187/645 [00:28<01:08,  6.69it/s]
Reranker epoch 1/3:  29%|██▉       | 188/645 [00:28<01:08,  6.68it/s]
Reranker epoch 1/3:  29%|██▉       | 189/645 [00:28<01:08,  6.66it/s]
Reranker epoch 1/3:  29%|██▉       | 190/645 [00:28<01:08,  6.68it/s]
Reranker epoch 1/3:  30%|██▉       | 191/645 [00:28<01:08,  6.62it/s]
Reranker epoch 1/3:  30%|██▉       | 192/645 [00:29<01:08,  6.62it/s]
Reranker epoch 1/3:  30%|██▉       | 193/645 [00:29<01:07,  6.67it/s]
Reranker epoch 1/3:  30%|███       | 194/645 [00:29<01:08,  6.62it/s]
Reranker epoch 1/3:  30%|███       | 195/645 [00:29<01:08,  6.59it/s]
Reranker epoch 1/3:  30%|███       | 196/645 [00:29<01:08,  6.59it/s]
Reranker epoch 1/3:  31%|███       | 197/645 [00:29<01:07,  6.61it/s]
Reranker epoch 1/3:  31%|███       | 198/645 [00:29<01:07,  6.63it/s]
Reranker epoch 1/3:  31%|███       | 199/645 [00:30<01:06,  6.68it/s]
Reranker epoch 1/3:  31%|███       | 200/645 [00:30<01:06,  6.68it/s]
Reranker epoch 1/3:  31%|███       | 201/645 [00:30<01:06,  6.69it/s]
Reranker epoch 1/3:  31%|███▏      | 202/645 [00:30<01:06,  6.67it/s]
Reranker epoch 1/3:  31%|███▏      | 203/645 [00:30<01:05,  6.70it/s]
Reranker epoch 1/3:  32%|███▏      | 204/645 [00:30<01:05,  6.71it/s]
Reranker epoch 1/3:  32%|███▏      | 205/645 [00:30<01:06,  6.66it/s]
Reranker epoch 1/3:  32%|███▏      | 206/645 [00:31<01:06,  6.61it/s]
Reranker epoch 1/3:  32%|███▏      | 207/645 [00:31<01:06,  6.62it/s]
Reranker epoch 1/3:  32%|███▏      | 208/645 [00:31<01:06,  6.62it/s]
Reranker epoch 1/3:  32%|███▏      | 209/645 [00:31<01:05,  6.71it/s]
Reranker epoch 1/3:  33%|███▎      | 210/645 [00:31<01:04,  6.70it/s]
Reranker epoch 1/3:  33%|███▎      | 211/645 [00:31<01:05,  6.65it/s]
Reranker epoch 1/3:  33%|███▎      | 212/645 [00:32<01:05,  6.65it/s]
Reranker epoch 1/3:  33%|███▎      | 213/645 [00:32<01:04,  6.73it/s]
Reranker epoch 1/3:  33%|███▎      | 214/645 [00:32<01:04,  6.69it/s]
Reranker epoch 1/3:  33%|███▎      | 215/645 [00:32<01:04,  6.70it/s]
Reranker epoch 1/3:  33%|███▎      | 216/645 [00:32<01:04,  6.66it/s]
Reranker epoch 1/3:  34%|███▎      | 217/645 [00:32<01:03,  6.70it/s]
Reranker epoch 1/3:  34%|███▍      | 218/645 [00:32<01:03,  6.68it/s]
Reranker epoch 1/3:  34%|███▍      | 219/645 [00:33<01:04,  6.66it/s]
Reranker epoch 1/3:  34%|███▍      | 220/645 [00:33<01:04,  6.63it/s]
Reranker epoch 1/3:  34%|███▍      | 221/645 [00:33<01:03,  6.67it/s]
Reranker epoch 1/3:  34%|███▍      | 222/645 [00:33<01:03,  6.66it/s]
Reranker epoch 1/3:  35%|███▍      | 223/645 [00:33<01:02,  6.71it/s]
Reranker epoch 1/3:  35%|███▍      | 224/645 [00:33<01:03,  6.68it/s]
Reranker epoch 1/3:  35%|███▍      | 225/645 [00:33<01:02,  6.70it/s]
Reranker epoch 1/3:  35%|███▌      | 226/645 [00:34<01:02,  6.69it/s]
Reranker epoch 1/3:  35%|███▌      | 227/645 [00:34<01:02,  6.70it/s]
Reranker epoch 1/3:  35%|███▌      | 228/645 [00:34<01:01,  6.76it/s]
Reranker epoch 1/3:  36%|███▌      | 229/645 [00:34<01:02,  6.71it/s]
Reranker epoch 1/3:  36%|███▌      | 230/645 [00:34<01:01,  6.70it/s]
Reranker epoch 1/3:  36%|███▌      | 231/645 [00:34<01:02,  6.67it/s]
Reranker epoch 1/3:  36%|███▌      | 232/645 [00:35<01:01,  6.73it/s]
Reranker epoch 1/3:  36%|███▌      | 233/645 [00:35<01:01,  6.70it/s]
Reranker epoch 1/3:  36%|███▋      | 234/645 [00:35<01:01,  6.69it/s]
Reranker epoch 1/3:  36%|███▋      | 235/645 [00:35<01:01,  6.65it/s]
Reranker epoch 1/3:  37%|███▋      | 236/645 [00:35<01:01,  6.67it/s]
Reranker epoch 1/3:  37%|███▋      | 237/645 [00:35<01:01,  6.64it/s]
Reranker epoch 1/3:  37%|███▋      | 238/645 [00:35<01:00,  6.69it/s]
Reranker epoch 1/3:  37%|███▋      | 239/645 [00:36<01:00,  6.66it/s]
Reranker epoch 1/3:  37%|███▋      | 240/645 [00:36<01:00,  6.68it/s]
Reranker epoch 1/3:  37%|███▋      | 241/645 [00:36<01:00,  6.67it/s]
Reranker epoch 1/3:  38%|███▊      | 242/645 [00:36<00:59,  6.74it/s]
Reranker epoch 1/3:  38%|███▊      | 243/645 [00:36<00:59,  6.76it/s]
Reranker epoch 1/3:  38%|███▊      | 244/645 [00:36<00:59,  6.71it/s]
Reranker epoch 1/3:  38%|███▊      | 245/645 [00:36<00:59,  6.69it/s]
Reranker epoch 1/3:  38%|███▊      | 246/645 [00:37<00:59,  6.70it/s]
Reranker epoch 1/3:  38%|███▊      | 247/645 [00:37<00:59,  6.68it/s]
Reranker epoch 1/3:  38%|███▊      | 248/645 [00:37<00:59,  6.67it/s]
Reranker epoch 1/3:  39%|███▊      | 249/645 [00:37<00:59,  6.62it/s]
Reranker epoch 1/3:  39%|███▉      | 250/645 [00:37<00:59,  6.67it/s]
Reranker epoch 1/3:  39%|███▉      | 251/645 [00:37<00:59,  6.67it/s]
Reranker epoch 1/3:  39%|███▉      | 252/645 [00:38<00:59,  6.65it/s]
Reranker epoch 1/3:  39%|███▉      | 253/645 [00:38<00:58,  6.65it/s]
Reranker epoch 1/3:  39%|███▉      | 254/645 [00:38<00:58,  6.67it/s]
Reranker epoch 1/3:  40%|███▉      | 255/645 [00:38<00:58,  6.67it/s]
Reranker epoch 1/3:  40%|███▉      | 256/645 [00:38<00:58,  6.67it/s]
Reranker epoch 1/3:  40%|███▉      | 257/645 [00:38<00:57,  6.75it/s]
Reranker epoch 1/3:  40%|████      | 258/645 [00:38<00:57,  6.71it/s]
Reranker epoch 1/3:  40%|████      | 259/645 [00:39<00:57,  6.69it/s]
Reranker epoch 1/3:  40%|████      | 260/645 [00:39<00:56,  6.76it/s]
Reranker epoch 1/3:  40%|████      | 261/645 [00:39<00:57,  6.74it/s]
Reranker epoch 1/3:  41%|████      | 262/645 [00:39<00:57,  6.69it/s]
Reranker epoch 1/3:  41%|████      | 263/645 [00:39<00:57,  6.66it/s]
Reranker epoch 1/3:  41%|████      | 264/645 [00:39<00:57,  6.67it/s]
Reranker epoch 1/3:  41%|████      | 265/645 [00:39<00:57,  6.64it/s]
Reranker epoch 1/3:  41%|████      | 266/645 [00:40<00:57,  6.65it/s]
Reranker epoch 1/3:  41%|████▏     | 267/645 [00:40<00:56,  6.64it/s]
Reranker epoch 1/3:  42%|████▏     | 268/645 [00:40<00:56,  6.66it/s]
Reranker epoch 1/3:  42%|████▏     | 269/645 [00:40<00:56,  6.66it/s]
Reranker epoch 1/3:  42%|████▏     | 270/645 [00:40<00:56,  6.64it/s]
Reranker epoch 1/3:  42%|████▏     | 271/645 [00:40<00:55,  6.71it/s]
Reranker epoch 1/3:  42%|████▏     | 272/645 [00:41<00:55,  6.69it/s]
Reranker epoch 1/3:  42%|████▏     | 273/645 [00:41<00:55,  6.76it/s]
Reranker epoch 1/3:  42%|████▏     | 274/645 [00:41<00:55,  6.73it/s]
Reranker epoch 1/3:  43%|████▎     | 275/645 [00:41<00:55,  6.72it/s]
Reranker epoch 1/3:  43%|████▎     | 276/645 [00:41<00:54,  6.73it/s]
Reranker epoch 1/3:  43%|████▎     | 277/645 [00:41<00:55,  6.67it/s]
Reranker epoch 1/3:  43%|████▎     | 278/645 [00:41<00:55,  6.64it/s]
Reranker epoch 1/3:  43%|████▎     | 279/645 [00:42<00:55,  6.63it/s]
Reranker epoch 1/3:  43%|████▎     | 280/645 [00:42<00:54,  6.68it/s]
Reranker epoch 1/3:  44%|████▎     | 281/645 [00:42<00:54,  6.66it/s]
Reranker epoch 1/3:  44%|████▎     | 282/645 [00:42<00:54,  6.65it/s]
Reranker epoch 1/3:  44%|████▍     | 283/645 [00:42<00:54,  6.65it/s]
Reranker epoch 1/3:  44%|████▍     | 284/645 [00:42<00:53,  6.70it/s]
Reranker epoch 1/3:  44%|████▍     | 285/645 [00:42<00:53,  6.69it/s]
Reranker epoch 1/3:  44%|████▍     | 286/645 [00:43<00:53,  6.70it/s]
Reranker epoch 1/3:  44%|████▍     | 287/645 [00:43<00:53,  6.66it/s]
Reranker epoch 1/3:  45%|████▍     | 288/645 [00:43<00:53,  6.69it/s]
Reranker epoch 1/3:  45%|████▍     | 289/645 [00:43<00:53,  6.65it/s]
Reranker epoch 1/3:  45%|████▍     | 290/645 [00:43<00:53,  6.65it/s]
Reranker epoch 1/3:  45%|████▌     | 291/645 [00:43<00:53,  6.62it/s]
Reranker epoch 1/3:  45%|████▌     | 292/645 [00:44<00:52,  6.71it/s]
Reranker epoch 1/3:  45%|████▌     | 293/645 [00:44<00:52,  6.66it/s]
Reranker epoch 1/3:  46%|████▌     | 294/645 [00:44<00:52,  6.65it/s]
Reranker epoch 1/3:  46%|████▌     | 295/645 [00:44<00:52,  6.67it/s]
Reranker epoch 1/3:  46%|████▌     | 296/645 [00:44<00:52,  6.67it/s]
Reranker epoch 1/3:  46%|████▌     | 297/645 [00:44<00:52,  6.62it/s]
Reranker epoch 1/3:  46%|████▌     | 298/645 [00:44<00:52,  6.60it/s]
Reranker epoch 1/3:  46%|████▋     | 299/645 [00:45<00:52,  6.63it/s]
Reranker epoch 1/3:  47%|████▋     | 300/645 [00:45<00:51,  6.67it/s]
Reranker epoch 1/3:  47%|████▋     | 301/645 [00:45<00:51,  6.66it/s]
Reranker epoch 1/3:  47%|████▋     | 302/645 [00:45<00:51,  6.67it/s]
Reranker epoch 1/3:  47%|████▋     | 303/645 [00:45<00:51,  6.62it/s]
Reranker epoch 1/3:  47%|████▋     | 304/645 [00:45<00:51,  6.64it/s]
Reranker epoch 1/3:  47%|████▋     | 305/645 [00:45<00:50,  6.74it/s]
Reranker epoch 1/3:  47%|████▋     | 306/645 [00:46<00:50,  6.75it/s]
Reranker epoch 1/3:  48%|████▊     | 307/645 [00:46<00:50,  6.73it/s]
Reranker epoch 1/3:  48%|████▊     | 308/645 [00:46<00:49,  6.78it/s]
Reranker epoch 1/3:  48%|████▊     | 309/645 [00:46<00:49,  6.75it/s]
Reranker epoch 1/3:  48%|████▊     | 310/645 [00:46<00:49,  6.70it/s]
Reranker epoch 1/3:  48%|████▊     | 311/645 [00:46<00:49,  6.68it/s]
Reranker epoch 1/3:  48%|████▊     | 312/645 [00:47<00:49,  6.72it/s]
Reranker epoch 1/3:  49%|████▊     | 313/645 [00:47<00:49,  6.69it/s]
Reranker epoch 1/3:  49%|████▊     | 314/645 [00:47<00:49,  6.69it/s]
Reranker epoch 1/3:  49%|████▉     | 315/645 [00:47<00:49,  6.69it/s]
Reranker epoch 1/3:  49%|████▉     | 316/645 [00:47<00:48,  6.72it/s]
Reranker epoch 1/3:  49%|████▉     | 317/645 [00:47<00:48,  6.72it/s]
Reranker epoch 1/3:  49%|████▉     | 318/645 [00:47<00:48,  6.70it/s]
Reranker epoch 1/3:  49%|████▉     | 319/645 [00:48<00:48,  6.67it/s]
Reranker epoch 1/3:  50%|████▉     | 320/645 [00:48<00:48,  6.65it/s]
Reranker epoch 1/3:  50%|████▉     | 321/645 [00:48<00:49,  6.55it/s]
Reranker epoch 1/3:  50%|████▉     | 322/645 [00:48<00:49,  6.58it/s]
Reranker epoch 1/3:  50%|█████     | 323/645 [00:48<00:48,  6.59it/s]
Reranker epoch 1/3:  50%|█████     | 324/645 [00:48<00:48,  6.60it/s]
Reranker epoch 1/3:  50%|█████     | 325/645 [00:48<00:48,  6.62it/s]
Reranker epoch 1/3:  51%|█████     | 326/645 [00:49<00:48,  6.62it/s]
Reranker epoch 1/3:  51%|█████     | 327/645 [00:49<00:47,  6.63it/s]
Reranker epoch 1/3:  51%|█████     | 328/645 [00:49<00:47,  6.68it/s]
Reranker epoch 1/3:  51%|█████     | 329/645 [00:49<00:47,  6.65it/s]
Reranker epoch 1/3:  51%|█████     | 330/645 [00:49<00:47,  6.66it/s]
Reranker epoch 1/3:  51%|█████▏    | 331/645 [00:49<00:47,  6.59it/s]
Reranker epoch 1/3:  51%|█████▏    | 332/645 [00:50<00:47,  6.61it/s]
Reranker epoch 1/3:  52%|█████▏    | 333/645 [00:50<00:46,  6.69it/s]
Reranker epoch 1/3:  52%|█████▏    | 334/645 [00:50<00:46,  6.72it/s]
Reranker epoch 1/3:  52%|█████▏    | 335/645 [00:50<00:46,  6.71it/s]
Reranker epoch 1/3:  52%|█████▏    | 336/645 [00:50<00:45,  6.72it/s]
Reranker epoch 1/3:  52%|█████▏    | 337/645 [00:50<00:46,  6.66it/s]
Reranker epoch 1/3:  52%|█████▏    | 338/645 [00:50<00:46,  6.66it/s]
Reranker epoch 1/3:  53%|█████▎    | 339/645 [00:51<00:46,  6.62it/s]
Reranker epoch 1/3:  53%|█████▎    | 340/645 [00:51<00:45,  6.69it/s]
Reranker epoch 1/3:  53%|█████▎    | 341/645 [00:51<00:45,  6.74it/s]
Reranker epoch 1/3:  53%|█████▎    | 342/645 [00:51<00:45,  6.72it/s]
Reranker epoch 1/3:  53%|█████▎    | 343/645 [00:51<00:45,  6.68it/s]
Reranker epoch 1/3:  53%|█████▎    | 344/645 [00:51<00:44,  6.70it/s]
Reranker epoch 1/3:  53%|█████▎    | 345/645 [00:51<00:44,  6.67it/s]
Reranker epoch 1/3:  54%|█████▎    | 346/645 [00:52<00:44,  6.67it/s]
Reranker epoch 1/3:  54%|█████▍    | 347/645 [00:52<00:44,  6.67it/s]
Reranker epoch 1/3:  54%|█████▍    | 348/645 [00:52<00:44,  6.70it/s]
Reranker epoch 1/3:  54%|█████▍    | 349/645 [00:52<00:44,  6.67it/s]
Reranker epoch 1/3:  54%|█████▍    | 350/645 [00:52<00:44,  6.65it/s]
Reranker epoch 1/3:  54%|█████▍    | 351/645 [00:52<00:44,  6.64it/s]
Reranker epoch 1/3:  55%|█████▍    | 352/645 [00:53<00:43,  6.67it/s]
Reranker epoch 1/3:  55%|█████▍    | 353/645 [00:53<00:44,  6.63it/s]
Reranker epoch 1/3:  55%|█████▍    | 354/645 [00:53<00:43,  6.71it/s]
Reranker epoch 1/3:  55%|█████▌    | 355/645 [00:53<00:43,  6.74it/s]
Reranker epoch 1/3:  55%|█████▌    | 356/645 [00:53<00:42,  6.74it/s]
Reranker epoch 1/3:  55%|█████▌    | 357/645 [00:53<00:42,  6.70it/s]
Reranker epoch 1/3:  56%|█████▌    | 358/645 [00:53<00:42,  6.68it/s]
Reranker epoch 1/3:  56%|█████▌    | 359/645 [00:54<00:42,  6.66it/s]
Reranker epoch 1/3:  56%|█████▌    | 360/645 [00:54<00:42,  6.70it/s]
Reranker epoch 1/3:  56%|█████▌    | 361/645 [00:54<00:42,  6.70it/s]
Reranker epoch 1/3:  56%|█████▌    | 362/645 [00:54<00:42,  6.67it/s]
Reranker epoch 1/3:  56%|█████▋    | 363/645 [00:54<00:41,  6.72it/s]
Reranker epoch 1/3:  56%|█████▋    | 364/645 [00:54<00:41,  6.71it/s]
Reranker epoch 1/3:  57%|█████▋    | 365/645 [00:54<00:42,  6.67it/s]
Reranker epoch 1/3:  57%|█████▋    | 366/645 [00:55<00:41,  6.70it/s]
Reranker epoch 1/3:  57%|█████▋    | 367/645 [00:55<00:41,  6.71it/s]
Reranker epoch 1/3:  57%|█████▋    | 368/645 [00:55<00:41,  6.69it/s]
Reranker epoch 1/3:  57%|█████▋    | 369/645 [00:55<00:41,  6.65it/s]
Reranker epoch 1/3:  57%|█████▋    | 370/645 [00:55<00:41,  6.66it/s]
Reranker epoch 1/3:  58%|█████▊    | 371/645 [00:55<00:41,  6.67it/s]
Reranker epoch 1/3:  58%|█████▊    | 372/645 [00:56<00:40,  6.70it/s]
Reranker epoch 1/3:  58%|█████▊    | 373/645 [00:56<00:40,  6.67it/s]
Reranker epoch 1/3:  58%|█████▊    | 374/645 [00:56<00:40,  6.67it/s]
Reranker epoch 1/3:  58%|█████▊    | 375/645 [00:56<00:40,  6.67it/s]
Reranker epoch 1/3:  58%|█████▊    | 376/645 [00:56<00:40,  6.66it/s]
Reranker epoch 1/3:  58%|█████▊    | 377/645 [00:56<00:40,  6.63it/s]
Reranker epoch 1/3:  59%|█████▊    | 378/645 [00:56<00:40,  6.64it/s]
Reranker epoch 1/3:  59%|█████▉    | 379/645 [00:57<00:39,  6.65it/s]
Reranker epoch 1/3:  59%|█████▉    | 380/645 [00:57<00:39,  6.68it/s]
Reranker epoch 1/3:  59%|█████▉    | 381/645 [00:57<00:39,  6.77it/s]
Reranker epoch 1/3:  59%|█████▉    | 382/645 [00:57<00:39,  6.69it/s]
Reranker epoch 1/3:  59%|█████▉    | 383/645 [00:57<00:39,  6.56it/s]
Reranker epoch 1/3:  60%|█████▉    | 384/645 [00:57<00:40,  6.47it/s]
Reranker epoch 1/3:  60%|█████▉    | 385/645 [00:57<00:40,  6.41it/s]
Reranker epoch 1/3:  60%|█████▉    | 386/645 [00:58<00:40,  6.36it/s]
Reranker epoch 1/3:  60%|██████    | 387/645 [00:58<00:40,  6.41it/s]
Reranker epoch 1/3:  60%|██████    | 388/645 [00:58<00:40,  6.40it/s]
Reranker epoch 1/3:  60%|██████    | 389/645 [00:58<00:39,  6.44it/s]
Reranker epoch 1/3:  60%|██████    | 390/645 [00:58<00:39,  6.42it/s]
Reranker epoch 1/3:  61%|██████    | 391/645 [00:58<00:39,  6.40it/s]
Reranker epoch 1/3:  61%|██████    | 392/645 [00:59<00:39,  6.43it/s]
Reranker epoch 1/3:  61%|██████    | 393/645 [00:59<00:39,  6.46it/s]
Reranker epoch 1/3:  61%|██████    | 394/645 [00:59<00:38,  6.47it/s]
Reranker epoch 1/3:  61%|██████    | 395/645 [00:59<00:38,  6.47it/s]
Reranker epoch 1/3:  61%|██████▏   | 396/645 [00:59<00:38,  6.50it/s]
Reranker epoch 1/3:  62%|██████▏   | 397/645 [00:59<00:37,  6.55it/s]
Reranker epoch 1/3:  62%|██████▏   | 398/645 [00:59<00:37,  6.54it/s]
Reranker epoch 1/3:  62%|██████▏   | 399/645 [01:00<00:37,  6.54it/s]
Reranker epoch 1/3:  62%|██████▏   | 400/645 [01:00<00:37,  6.56it/s]
Reranker epoch 1/3:  62%|██████▏   | 401/645 [01:00<00:37,  6.53it/s]
Reranker epoch 1/3:  62%|██████▏   | 402/645 [01:00<00:36,  6.57it/s]
Reranker epoch 1/3:  62%|██████▏   | 403/645 [01:00<00:36,  6.63it/s]
Reranker epoch 1/3:  63%|██████▎   | 404/645 [01:00<00:36,  6.65it/s]
Reranker epoch 1/3:  63%|██████▎   | 405/645 [01:01<00:36,  6.66it/s]
Reranker epoch 1/3:  63%|██████▎   | 406/645 [01:01<00:35,  6.66it/s]
Reranker epoch 1/3:  63%|██████▎   | 407/645 [01:01<00:35,  6.63it/s]
Reranker epoch 1/3:  63%|██████▎   | 408/645 [01:01<00:35,  6.65it/s]
Reranker epoch 1/3:  63%|██████▎   | 409/645 [01:01<00:35,  6.64it/s]
Reranker epoch 1/3:  64%|██████▎   | 410/645 [01:01<00:35,  6.66it/s]
Reranker epoch 1/3:  64%|██████▎   | 411/645 [01:01<00:34,  6.70it/s]
Reranker epoch 1/3:  64%|██████▍   | 412/645 [01:02<00:34,  6.69it/s]
Reranker epoch 1/3:  64%|██████▍   | 413/645 [01:02<00:34,  6.67it/s]
Reranker epoch 1/3:  64%|██████▍   | 414/645 [01:02<00:34,  6.71it/s]
Reranker epoch 1/3:  64%|██████▍   | 415/645 [01:02<00:34,  6.67it/s]
Reranker epoch 1/3:  64%|██████▍   | 416/645 [01:02<00:34,  6.67it/s]
Reranker epoch 1/3:  65%|██████▍   | 417/645 [01:02<00:34,  6.69it/s]
Reranker epoch 1/3:  65%|██████▍   | 418/645 [01:02<00:33,  6.74it/s]
Reranker epoch 1/3:  65%|██████▍   | 419/645 [01:03<00:33,  6.68it/s]
Reranker epoch 1/3:  65%|██████▌   | 420/645 [01:03<00:33,  6.70it/s]
Reranker epoch 1/3:  65%|██████▌   | 421/645 [01:03<00:33,  6.68it/s]
Reranker epoch 1/3:  65%|██████▌   | 422/645 [01:03<00:33,  6.65it/s]
Reranker epoch 1/3:  66%|██████▌   | 423/645 [01:03<00:33,  6.65it/s]
Reranker epoch 1/3:  66%|██████▌   | 424/645 [01:03<00:33,  6.64it/s]
Reranker epoch 1/3:  66%|██████▌   | 425/645 [01:04<00:33,  6.62it/s]
Reranker epoch 1/3:  66%|██████▌   | 426/645 [01:04<00:33,  6.63it/s]
Reranker epoch 1/3:  66%|██████▌   | 427/645 [01:04<00:32,  6.68it/s]
Reranker epoch 1/3:  66%|██████▋   | 428/645 [01:04<00:32,  6.64it/s]
Reranker epoch 1/3:  67%|██████▋   | 429/645 [01:04<00:32,  6.63it/s]
Reranker epoch 1/3:  67%|██████▋   | 430/645 [01:04<00:32,  6.66it/s]
Reranker epoch 1/3:  67%|██████▋   | 431/645 [01:04<00:32,  6.66it/s]
Reranker epoch 1/3:  67%|██████▋   | 432/645 [01:05<00:32,  6.64it/s]
Reranker epoch 1/3:  67%|██████▋   | 433/645 [01:05<00:32,  6.61it/s]
Reranker epoch 1/3:  67%|██████▋   | 434/645 [01:05<00:31,  6.67it/s]
Reranker epoch 1/3:  67%|██████▋   | 435/645 [01:05<00:31,  6.61it/s]
Reranker epoch 1/3:  68%|██████▊   | 436/645 [01:05<00:31,  6.60it/s]
Reranker epoch 1/3:  68%|██████▊   | 437/645 [01:05<00:31,  6.61it/s]
Reranker epoch 1/3:  68%|██████▊   | 438/645 [01:05<00:30,  6.69it/s]
Reranker epoch 1/3:  68%|██████▊   | 439/645 [01:06<00:30,  6.66it/s]
Reranker epoch 1/3:  68%|██████▊   | 440/645 [01:06<00:30,  6.65it/s]
Reranker epoch 1/3:  68%|██████▊   | 441/645 [01:06<00:30,  6.65it/s]
Reranker epoch 1/3:  69%|██████▊   | 442/645 [01:06<00:30,  6.72it/s]
Reranker epoch 1/3:  69%|██████▊   | 443/645 [01:06<00:30,  6.68it/s]
Reranker epoch 1/3:  69%|██████▉   | 444/645 [01:06<00:30,  6.66it/s]
Reranker epoch 1/3:  69%|██████▉   | 445/645 [01:07<00:30,  6.65it/s]
Reranker epoch 1/3:  69%|██████▉   | 446/645 [01:07<00:29,  6.68it/s]
Reranker epoch 1/3:  69%|██████▉   | 447/645 [01:07<00:29,  6.67it/s]
Reranker epoch 1/3:  69%|██████▉   | 448/645 [01:07<00:29,  6.66it/s]
Reranker epoch 1/3:  70%|██████▉   | 449/645 [01:07<00:29,  6.64it/s]
Reranker epoch 1/3:  70%|██████▉   | 450/645 [01:07<00:28,  6.74it/s]
Reranker epoch 1/3:  70%|██████▉   | 451/645 [01:07<00:28,  6.69it/s]
Reranker epoch 1/3:  70%|███████   | 452/645 [01:08<00:28,  6.73it/s]
Reranker epoch 1/3:  70%|███████   | 453/645 [01:08<00:28,  6.68it/s]
Reranker epoch 1/3:  70%|███████   | 454/645 [01:08<00:28,  6.72it/s]
Reranker epoch 1/3:  71%|███████   | 455/645 [01:08<00:28,  6.67it/s]
Reranker epoch 1/3:  71%|███████   | 456/645 [01:08<00:28,  6.63it/s]
Reranker epoch 1/3:  71%|███████   | 457/645 [01:08<00:28,  6.56it/s]
Reranker epoch 1/3:  71%|███████   | 458/645 [01:08<00:28,  6.65it/s]
Reranker epoch 1/3:  71%|███████   | 459/645 [01:09<00:28,  6.63it/s]
Reranker epoch 1/3:  71%|███████▏  | 460/645 [01:09<00:27,  6.62it/s]
Reranker epoch 1/3:  71%|███████▏  | 461/645 [01:09<00:27,  6.63it/s]
Reranker epoch 1/3:  72%|███████▏  | 462/645 [01:09<00:27,  6.64it/s]
Reranker epoch 1/3:  72%|███████▏  | 463/645 [01:09<00:27,  6.62it/s]
Reranker epoch 1/3:  72%|███████▏  | 464/645 [01:09<00:27,  6.60it/s]
Reranker epoch 1/3:  72%|███████▏  | 465/645 [01:10<00:26,  6.68it/s]
Reranker epoch 1/3:  72%|███████▏  | 466/645 [01:10<00:26,  6.66it/s]
Reranker epoch 1/3:  72%|███████▏  | 467/645 [01:10<00:26,  6.64it/s]
Reranker epoch 1/3:  73%|███████▎  | 468/645 [01:10<00:26,  6.67it/s]
Reranker epoch 1/3:  73%|███████▎  | 469/645 [01:10<00:26,  6.65it/s]
Reranker epoch 1/3:  73%|███████▎  | 470/645 [01:10<00:26,  6.69it/s]
Reranker epoch 1/3:  73%|███████▎  | 471/645 [01:10<00:26,  6.67it/s]
Reranker epoch 1/3:  73%|███████▎  | 472/645 [01:11<00:26,  6.65it/s]
Reranker epoch 1/3:  73%|███████▎  | 473/645 [01:11<00:25,  6.63it/s]
Reranker epoch 1/3:  73%|███████▎  | 474/645 [01:11<00:25,  6.67it/s]
Reranker epoch 1/3:  74%|███████▎  | 475/645 [01:11<00:25,  6.65it/s]
Reranker epoch 1/3:  74%|███████▍  | 476/645 [01:11<00:25,  6.62it/s]
Reranker epoch 1/3:  74%|███████▍  | 477/645 [01:11<00:25,  6.62it/s]
Reranker epoch 1/3:  74%|███████▍  | 478/645 [01:12<00:25,  6.65it/s]
Reranker epoch 1/3:  74%|███████▍  | 479/645 [01:12<00:25,  6.63it/s]
Reranker epoch 1/3:  74%|███████▍  | 480/645 [01:12<00:25,  6.59it/s]
Reranker epoch 1/3:  75%|███████▍  | 481/645 [01:12<00:24,  6.63it/s]
Reranker epoch 1/3:  75%|███████▍  | 482/645 [01:12<00:24,  6.65it/s]
Reranker epoch 1/3:  75%|███████▍  | 483/645 [01:12<00:24,  6.59it/s]
Reranker epoch 1/3:  75%|███████▌  | 484/645 [01:12<00:24,  6.68it/s]
Reranker epoch 1/3:  75%|███████▌  | 485/645 [01:13<00:24,  6.63it/s]
Reranker epoch 1/3:  75%|███████▌  | 486/645 [01:13<00:23,  6.63it/s]
Reranker epoch 1/3:  76%|███████▌  | 487/645 [01:13<00:23,  6.61it/s]
Reranker epoch 1/3:  76%|███████▌  | 488/645 [01:13<00:23,  6.61it/s]
Reranker epoch 1/3:  76%|███████▌  | 489/645 [01:13<00:23,  6.59it/s]
Reranker epoch 1/3:  76%|███████▌  | 490/645 [01:13<00:23,  6.63it/s]
Reranker epoch 1/3:  76%|███████▌  | 491/645 [01:13<00:23,  6.65it/s]
Reranker epoch 1/3:  76%|███████▋  | 492/645 [01:14<00:23,  6.63it/s]
Reranker epoch 1/3:  76%|███████▋  | 493/645 [01:14<00:22,  6.65it/s]
Reranker epoch 1/3:  77%|███████▋  | 494/645 [01:14<00:22,  6.60it/s]
Reranker epoch 1/3:  77%|███████▋  | 495/645 [01:14<00:22,  6.60it/s]
Reranker epoch 1/3:  77%|███████▋  | 496/645 [01:14<00:22,  6.60it/s]
Reranker epoch 1/3:  77%|███████▋  | 497/645 [01:14<00:22,  6.66it/s]
Reranker epoch 1/3:  77%|███████▋  | 498/645 [01:15<00:22,  6.68it/s]
Reranker epoch 1/3:  77%|███████▋  | 499/645 [01:15<00:21,  6.66it/s]
Reranker epoch 1/3:  78%|███████▊  | 500/645 [01:15<00:21,  6.65it/s]
Reranker epoch 1/3:  78%|███████▊  | 501/645 [01:15<00:21,  6.63it/s]
Reranker epoch 1/3:  78%|███████▊  | 502/645 [01:15<00:21,  6.63it/s]
Reranker epoch 1/3:  78%|███████▊  | 503/645 [01:15<00:21,  6.62it/s]
Reranker epoch 1/3:  78%|███████▊  | 504/645 [01:15<00:21,  6.68it/s]
Reranker epoch 1/3:  78%|███████▊  | 505/645 [01:16<00:20,  6.70it/s]
Reranker epoch 1/3:  78%|███████▊  | 506/645 [01:16<00:20,  6.67it/s]
Reranker epoch 1/3:  79%|███████▊  | 507/645 [01:16<00:20,  6.65it/s]
Reranker epoch 1/3:  79%|███████▉  | 508/645 [01:16<00:20,  6.57it/s]
Reranker epoch 1/3:  79%|███████▉  | 509/645 [01:16<00:20,  6.60it/s]
Reranker epoch 1/3:  79%|███████▉  | 510/645 [01:16<00:20,  6.56it/s]
Reranker epoch 1/3:  79%|███████▉  | 511/645 [01:16<00:20,  6.63it/s]
Reranker epoch 1/3:  79%|███████▉  | 512/645 [01:17<00:20,  6.61it/s]
Reranker epoch 1/3:  80%|███████▉  | 513/645 [01:17<00:19,  6.67it/s]
Reranker epoch 1/3:  80%|███████▉  | 514/645 [01:17<00:19,  6.66it/s]
Reranker epoch 1/3:  80%|███████▉  | 515/645 [01:17<00:19,  6.70it/s]
Reranker epoch 1/3:  80%|████████  | 516/645 [01:17<00:19,  6.70it/s]
Reranker epoch 1/3:  80%|████████  | 517/645 [01:17<00:19,  6.71it/s]
Reranker epoch 1/3:  80%|████████  | 518/645 [01:18<00:19,  6.68it/s]
Reranker epoch 1/3:  80%|████████  | 519/645 [01:18<00:18,  6.73it/s]
Reranker epoch 1/3:  81%|████████  | 520/645 [01:18<00:18,  6.71it/s]
Reranker epoch 1/3:  81%|████████  | 521/645 [01:18<00:18,  6.69it/s]
Reranker epoch 1/3:  81%|████████  | 522/645 [01:18<00:18,  6.65it/s]
Reranker epoch 1/3:  81%|████████  | 523/645 [01:18<00:18,  6.65it/s]
Reranker epoch 1/3:  81%|████████  | 524/645 [01:18<00:18,  6.61it/s]
Reranker epoch 1/3:  81%|████████▏ | 525/645 [01:19<00:18,  6.60it/s]
Reranker epoch 1/3:  82%|████████▏ | 526/645 [01:19<00:17,  6.68it/s]
Reranker epoch 1/3:  82%|████████▏ | 527/645 [01:19<00:17,  6.72it/s]
Reranker epoch 1/3:  82%|████████▏ | 528/645 [01:19<00:17,  6.73it/s]
Reranker epoch 1/3:  82%|████████▏ | 529/645 [01:19<00:17,  6.66it/s]
Reranker epoch 1/3:  82%|████████▏ | 530/645 [01:19<00:17,  6.65it/s]
Reranker epoch 1/3:  82%|████████▏ | 531/645 [01:19<00:17,  6.66it/s]
Reranker epoch 1/3:  82%|████████▏ | 532/645 [01:20<00:16,  6.72it/s]
Reranker epoch 1/3:  83%|████████▎ | 533/645 [01:20<00:16,  6.68it/s]
Reranker epoch 1/3:  83%|████████▎ | 534/645 [01:20<00:16,  6.66it/s]
Reranker epoch 1/3:  83%|████████▎ | 535/645 [01:20<00:16,  6.69it/s]
Reranker epoch 1/3:  83%|████████▎ | 536/645 [01:20<00:16,  6.61it/s]
Reranker epoch 1/3:  83%|████████▎ | 537/645 [01:20<00:16,  6.55it/s]
Reranker epoch 1/3:  83%|████████▎ | 538/645 [01:21<00:16,  6.56it/s]
Reranker epoch 1/3:  84%|████████▎ | 539/645 [01:21<00:16,  6.61it/s]
Reranker epoch 1/3:  84%|████████▎ | 540/645 [01:21<00:15,  6.63it/s]
Reranker epoch 1/3:  84%|████████▍ | 541/645 [01:21<00:15,  6.60it/s]
Reranker epoch 1/3:  84%|████████▍ | 542/645 [01:21<00:15,  6.60it/s]
Reranker epoch 1/3:  84%|████████▍ | 543/645 [01:21<00:15,  6.71it/s]
Reranker epoch 1/3:  84%|████████▍ | 544/645 [01:21<00:15,  6.67it/s]
Reranker epoch 1/3:  84%|████████▍ | 545/645 [01:22<00:14,  6.71it/s]
Reranker epoch 1/3:  85%|████████▍ | 546/645 [01:22<00:14,  6.68it/s]
Reranker epoch 1/3:  85%|████████▍ | 547/645 [01:22<00:14,  6.72it/s]
Reranker epoch 1/3:  85%|████████▍ | 548/645 [01:22<00:14,  6.61it/s]
Reranker epoch 1/3:  85%|████████▌ | 549/645 [01:22<00:14,  6.60it/s]
Reranker epoch 1/3:  85%|████████▌ | 550/645 [01:22<00:14,  6.65it/s]
Reranker epoch 1/3:  85%|████████▌ | 551/645 [01:22<00:13,  6.75it/s]
Reranker epoch 1/3:  86%|████████▌ | 552/645 [01:23<00:14,  6.64it/s]
Reranker epoch 1/3:  86%|████████▌ | 553/645 [01:23<00:13,  6.63it/s]
Reranker epoch 1/3:  86%|████████▌ | 554/645 [01:23<00:13,  6.62it/s]
Reranker epoch 1/3:  86%|████████▌ | 555/645 [01:23<00:13,  6.70it/s]
Reranker epoch 1/3:  86%|████████▌ | 556/645 [01:23<00:13,  6.66it/s]
Reranker epoch 1/3:  86%|████████▋ | 557/645 [01:23<00:13,  6.66it/s]
Reranker epoch 1/3:  87%|████████▋ | 558/645 [01:24<00:13,  6.64it/s]
Reranker epoch 1/3:  87%|████████▋ | 559/645 [01:24<00:12,  6.72it/s]
Reranker epoch 1/3:  87%|████████▋ | 560/645 [01:24<00:12,  6.67it/s]
Reranker epoch 1/3:  87%|████████▋ | 561/645 [01:24<00:12,  6.66it/s]
Reranker epoch 1/3:  87%|████████▋ | 562/645 [01:24<00:12,  6.63it/s]
Reranker epoch 1/3:  87%|████████▋ | 563/645 [01:24<00:12,  6.63it/s]
Reranker epoch 1/3:  87%|████████▋ | 564/645 [01:24<00:12,  6.63it/s]
Reranker epoch 1/3:  88%|████████▊ | 565/645 [01:25<00:12,  6.64it/s]
Reranker epoch 1/3:  88%|████████▊ | 566/645 [01:25<00:11,  6.60it/s]
Reranker epoch 1/3:  88%|████████▊ | 567/645 [01:25<00:11,  6.65it/s]
Reranker epoch 1/3:  88%|████████▊ | 568/645 [01:25<00:11,  6.66it/s]
Reranker epoch 1/3:  88%|████████▊ | 569/645 [01:25<00:11,  6.63it/s]
Reranker epoch 1/3:  88%|████████▊ | 570/645 [01:25<00:11,  6.55it/s]
Reranker epoch 1/3:  89%|████████▊ | 571/645 [01:26<00:11,  6.61it/s]
Reranker epoch 1/3:  89%|████████▊ | 572/645 [01:26<00:11,  6.58it/s]
Reranker epoch 1/3:  89%|████████▉ | 573/645 [01:26<00:10,  6.63it/s]
Reranker epoch 1/3:  89%|████████▉ | 574/645 [01:26<00:10,  6.64it/s]
Reranker epoch 1/3:  89%|████████▉ | 575/645 [01:26<00:10,  6.70it/s]
Reranker epoch 1/3:  89%|████████▉ | 576/645 [01:26<00:10,  6.73it/s]
Reranker epoch 1/3:  89%|████████▉ | 577/645 [01:26<00:10,  6.69it/s]
Reranker epoch 1/3:  90%|████████▉ | 578/645 [01:27<00:09,  6.72it/s]
Reranker epoch 1/3:  90%|████████▉ | 579/645 [01:27<00:09,  6.75it/s]
Reranker epoch 1/3:  90%|████████▉ | 580/645 [01:27<00:09,  6.67it/s]
Reranker epoch 1/3:  90%|█████████ | 581/645 [01:27<00:09,  6.61it/s]
Reranker epoch 1/3:  90%|█████████ | 582/645 [01:27<00:09,  6.60it/s]
Reranker epoch 1/3:  90%|█████████ | 583/645 [01:27<00:09,  6.62it/s]
Reranker epoch 1/3:  91%|█████████ | 584/645 [01:27<00:09,  6.53it/s]
Reranker epoch 1/3:  91%|█████████ | 585/645 [01:28<00:09,  6.55it/s]
Reranker epoch 1/3:  91%|█████████ | 586/645 [01:28<00:09,  6.53it/s]
Reranker epoch 1/3:  91%|█████████ | 587/645 [01:28<00:08,  6.49it/s]
Reranker epoch 1/3:  91%|█████████ | 588/645 [01:28<00:08,  6.55it/s]
Reranker epoch 1/3:  91%|█████████▏| 589/645 [01:28<00:08,  6.56it/s]
Reranker epoch 1/3:  91%|█████████▏| 590/645 [01:28<00:08,  6.63it/s]
Reranker epoch 1/3:  92%|█████████▏| 591/645 [01:29<00:08,  6.62it/s]
Reranker epoch 1/3:  92%|█████████▏| 592/645 [01:29<00:08,  6.59it/s]
Reranker epoch 1/3:  92%|█████████▏| 593/645 [01:29<00:07,  6.62it/s]
Reranker epoch 1/3:  92%|█████████▏| 594/645 [01:29<00:07,  6.59it/s]
Reranker epoch 1/3:  92%|█████████▏| 595/645 [01:29<00:07,  6.61it/s]
Reranker epoch 1/3:  92%|█████████▏| 596/645 [01:29<00:07,  6.65it/s]
Reranker epoch 1/3:  93%|█████████▎| 597/645 [01:29<00:07,  6.62it/s]
Reranker epoch 1/3:  93%|█████████▎| 598/645 [01:30<00:07,  6.61it/s]
Reranker epoch 1/3:  93%|█████████▎| 599/645 [01:30<00:06,  6.58it/s]
Reranker epoch 1/3:  93%|█████████▎| 600/645 [01:30<00:06,  6.63it/s]
Reranker epoch 1/3:  93%|█████████▎| 601/645 [01:30<00:06,  6.61it/s]
Reranker epoch 1/3:  93%|█████████▎| 602/645 [01:30<00:06,  6.61it/s]
Reranker epoch 1/3:  93%|█████████▎| 603/645 [01:30<00:06,  6.63it/s]
Reranker epoch 1/3:  94%|█████████▎| 604/645 [01:30<00:06,  6.70it/s]
Reranker epoch 1/3:  94%|█████████▍| 605/645 [01:31<00:06,  6.65it/s]
Reranker epoch 1/3:  94%|█████████▍| 606/645 [01:31<00:05,  6.67it/s]
Reranker epoch 1/3:  94%|█████████▍| 607/645 [01:31<00:05,  6.63it/s]
Reranker epoch 1/3:  94%|█████████▍| 608/645 [01:31<00:05,  6.66it/s]
Reranker epoch 1/3:  94%|█████████▍| 609/645 [01:31<00:05,  6.60it/s]
Reranker epoch 1/3:  95%|█████████▍| 610/645 [01:31<00:05,  6.60it/s]
Reranker epoch 1/3:  95%|█████████▍| 611/645 [01:32<00:05,  6.60it/s]
Reranker epoch 1/3:  95%|█████████▍| 612/645 [01:32<00:04,  6.64it/s]
Reranker epoch 1/3:  95%|█████████▌| 613/645 [01:32<00:04,  6.71it/s]
Reranker epoch 1/3:  95%|█████████▌| 614/645 [01:32<00:04,  6.67it/s]
Reranker epoch 1/3:  95%|█████████▌| 615/645 [01:32<00:04,  6.69it/s]
Reranker epoch 1/3:  96%|█████████▌| 616/645 [01:32<00:04,  6.70it/s]
Reranker epoch 1/3:  96%|█████████▌| 617/645 [01:32<00:04,  6.80it/s]
Reranker epoch 1/3:  96%|█████████▌| 618/645 [01:33<00:04,  6.70it/s]
Reranker epoch 1/3:  96%|█████████▌| 619/645 [01:33<00:03,  6.65it/s]
Reranker epoch 1/3:  96%|█████████▌| 620/645 [01:33<00:03,  6.65it/s]
Reranker epoch 1/3:  96%|█████████▋| 621/645 [01:33<00:03,  6.63it/s]
Reranker epoch 1/3:  96%|█████████▋| 622/645 [01:33<00:03,  6.57it/s]
Reranker epoch 1/3:  97%|█████████▋| 623/645 [01:33<00:03,  6.56it/s]
Reranker epoch 1/3:  97%|█████████▋| 624/645 [01:33<00:03,  6.65it/s]
Reranker epoch 1/3:  97%|█████████▋| 625/645 [01:34<00:03,  6.61it/s]
Reranker epoch 1/3:  97%|█████████▋| 626/645 [01:34<00:02,  6.62it/s]
Reranker epoch 1/3:  97%|█████████▋| 627/645 [01:34<00:02,  6.59it/s]
Reranker epoch 1/3:  97%|█████████▋| 628/645 [01:34<00:02,  6.60it/s]
Reranker epoch 1/3:  98%|█████████▊| 629/645 [01:34<00:02,  6.67it/s]
Reranker epoch 1/3:  98%|█████████▊| 630/645 [01:34<00:02,  6.65it/s]
Reranker epoch 1/3:  98%|█████████▊| 631/645 [01:35<00:02,  6.63it/s]
Reranker epoch 1/3:  98%|█████████▊| 632/645 [01:35<00:01,  6.64it/s]
Reranker epoch 1/3:  98%|█████████▊| 633/645 [01:35<00:01,  6.61it/s]
Reranker epoch 1/3:  98%|█████████▊| 634/645 [01:35<00:01,  6.56it/s]
Reranker epoch 1/3:  98%|█████████▊| 635/645 [01:35<00:01,  6.57it/s]
Reranker epoch 1/3:  99%|█████████▊| 636/645 [01:35<00:01,  6.61it/s]
Reranker epoch 1/3:  99%|█████████▉| 637/645 [01:35<00:01,  6.60it/s]
Reranker epoch 1/3:  99%|█████████▉| 638/645 [01:36<00:01,  6.63it/s]
Reranker epoch 1/3:  99%|█████████▉| 639/645 [01:36<00:00,  6.64it/s]
Reranker epoch 1/3:  99%|█████████▉| 640/645 [01:36<00:00,  6.62it/s]
Reranker epoch 1/3:  99%|█████████▉| 641/645 [01:36<00:00,  6.57it/s]
Reranker epoch 1/3: 100%|█████████▉| 642/645 [01:36<00:00,  6.61it/s]
Reranker epoch 1/3: 100%|█████████▉| 643/645 [01:36<00:00,  6.65it/s]
Reranker epoch 1/3: 100%|█████████▉| 644/645 [01:37<00:00,  6.66it/s]
Reranker epoch 1/3: 100%|██████████| 645/645 [01:37<00:00,  6.65it/s]
Scoring dev_epoch1 candidates with cross-encoder reranker...

  0%|          | 0/154 [00:00<?, ?it/s]
  1%|          | 1/154 [00:00<00:28,  5.31it/s]
  1%|▏         | 2/154 [00:00<00:30,  5.07it/s]
  2%|▏         | 3/154 [00:00<00:24,  6.09it/s]
  3%|▎         | 4/154 [00:00<00:23,  6.35it/s]
  3%|▎         | 5/154 [00:00<00:21,  7.04it/s]
  4%|▍         | 6/154 [00:00<00:19,  7.50it/s]
  5%|▍         | 7/154 [00:01<00:20,  7.30it/s]
  5%|▌         | 8/154 [00:01<00:20,  6.97it/s]
  6%|▌         | 9/154 [00:01<00:24,  5.99it/s]
  6%|▋         | 10/154 [00:01<00:22,  6.47it/s]
  7%|▋         | 11/154 [00:01<00:22,  6.30it/s]
  8%|▊         | 12/154 [00:01<00:21,  6.46it/s]
  8%|▊         | 13/154 [00:01<00:21,  6.66it/s]
  9%|▉         | 14/154 [00:02<00:20,  6.76it/s]
 10%|▉         | 15/154 [00:02<00:21,  6.61it/s]
 10%|█         | 16/154 [00:02<00:20,  6.82it/s]
 11%|█         | 17/154 [00:02<00:19,  6.93it/s]
 12%|█▏        | 18/154 [00:02<00:20,  6.77it/s]
 12%|█▏        | 19/154 [00:02<00:18,  7.19it/s]
 13%|█▎        | 20/154 [00:03<00:20,  6.40it/s]
 14%|█▎        | 21/154 [00:03<00:20,  6.51it/s]
 14%|█▍        | 22/154 [00:03<00:19,  6.73it/s]
 15%|█▍        | 23/154 [00:03<00:18,  7.18it/s]
 16%|█▌        | 24/154 [00:03<00:17,  7.49it/s]
 16%|█▌        | 25/154 [00:03<00:19,  6.56it/s]
 17%|█▋        | 26/154 [00:03<00:19,  6.51it/s]
 18%|█▊        | 27/154 [00:04<00:18,  6.81it/s]
 18%|█▊        | 28/154 [00:04<00:18,  6.90it/s]
 19%|█▉        | 29/154 [00:04<00:18,  6.84it/s]
 19%|█▉        | 30/154 [00:04<00:19,  6.26it/s]
 20%|██        | 31/154 [00:04<00:19,  6.19it/s]
 21%|██        | 32/154 [00:04<00:18,  6.65it/s]
 21%|██▏       | 33/154 [00:04<00:16,  7.14it/s]
 22%|██▏       | 34/154 [00:05<00:16,  7.14it/s]
 23%|██▎       | 35/154 [00:05<00:16,  7.41it/s]
 23%|██▎       | 36/154 [00:05<00:16,  7.19it/s]
 24%|██▍       | 37/154 [00:05<00:17,  6.60it/s]
 25%|██▍       | 38/154 [00:05<00:18,  6.17it/s]
 25%|██▌       | 39/154 [00:05<00:17,  6.46it/s]
 26%|██▌       | 40/154 [00:06<00:17,  6.50it/s]
 27%|██▋       | 41/154 [00:06<00:19,  5.86it/s]
 27%|██▋       | 42/154 [00:06<00:18,  5.98it/s]
 28%|██▊       | 43/154 [00:06<00:17,  6.46it/s]
 29%|██▊       | 44/154 [00:06<00:18,  5.99it/s]
 29%|██▉       | 45/154 [00:06<00:18,  5.85it/s]
 30%|██▉       | 46/154 [00:07<00:17,  6.28it/s]
 31%|███       | 47/154 [00:07<00:16,  6.56it/s]
 31%|███       | 48/154 [00:07<00:17,  6.15it/s]
 32%|███▏      | 49/154 [00:07<00:17,  5.87it/s]
 32%|███▏      | 50/154 [00:07<00:17,  5.88it/s]
 33%|███▎      | 51/154 [00:07<00:16,  6.12it/s]
 34%|███▍      | 52/154 [00:07<00:16,  6.33it/s]
 34%|███▍      | 53/154 [00:08<00:17,  5.89it/s]
 35%|███▌      | 54/154 [00:08<00:16,  6.12it/s]
 36%|███▌      | 55/154 [00:08<00:15,  6.51it/s]
 36%|███▋      | 56/154 [00:08<00:15,  6.45it/s]
 37%|███▋      | 57/154 [00:08<00:15,  6.07it/s]
 38%|███▊      | 58/154 [00:08<00:15,  6.34it/s]
 38%|███▊      | 59/154 [00:09<00:15,  6.04it/s]
 39%|███▉      | 60/154 [00:09<00:17,  5.46it/s]
 40%|███▉      | 61/154 [00:09<00:18,  4.99it/s]
 40%|████      | 62/154 [00:09<00:17,  5.21it/s]
 41%|████      | 63/154 [00:09<00:16,  5.48it/s]
 42%|████▏     | 64/154 [00:10<00:16,  5.59it/s]
 42%|████▏     | 65/154 [00:10<00:15,  5.86it/s]
 43%|████▎     | 66/154 [00:10<00:14,  5.95it/s]
 44%|████▎     | 67/154 [00:10<00:14,  5.81it/s]
 44%|████▍     | 68/154 [00:10<00:13,  6.33it/s]
 45%|████▍     | 69/154 [00:10<00:14,  6.02it/s]
 45%|████▌     | 70/154 [00:11<00:15,  5.54it/s]
 46%|████▌     | 71/154 [00:11<00:13,  5.95it/s]
 47%|████▋     | 72/154 [00:11<00:12,  6.34it/s]
 47%|████▋     | 73/154 [00:11<00:12,  6.48it/s]
 48%|████▊     | 74/154 [00:11<00:11,  7.14it/s]
 49%|████▊     | 75/154 [00:11<00:11,  6.83it/s]
 49%|████▉     | 76/154 [00:11<00:11,  7.04it/s]
 50%|█████     | 77/154 [00:12<00:10,  7.50it/s]
 51%|█████     | 78/154 [00:12<00:11,  6.64it/s]
 51%|█████▏    | 79/154 [00:12<00:11,  6.33it/s]
 52%|█████▏    | 80/154 [00:12<00:12,  5.96it/s]
 53%|█████▎    | 81/154 [00:12<00:11,  6.44it/s]
 53%|█████▎    | 82/154 [00:12<00:10,  6.55it/s]
 54%|█████▍    | 83/154 [00:13<00:11,  6.25it/s]
 55%|█████▍    | 84/154 [00:13<00:11,  6.14it/s]
 55%|█████▌    | 85/154 [00:13<00:11,  6.02it/s]
 56%|█████▌    | 86/154 [00:13<00:10,  6.27it/s]
 56%|█████▋    | 87/154 [00:13<00:11,  5.68it/s]
 57%|█████▋    | 88/154 [00:13<00:11,  5.62it/s]
 58%|█████▊    | 89/154 [00:14<00:10,  6.02it/s]
 58%|█████▊    | 90/154 [00:14<00:10,  6.06it/s]
 59%|█████▉    | 91/154 [00:14<00:09,  6.35it/s]
 60%|█████▉    | 92/154 [00:14<00:08,  7.09it/s]
 60%|██████    | 93/154 [00:14<00:08,  7.05it/s]
 61%|██████    | 94/154 [00:14<00:08,  7.02it/s]
 62%|██████▏   | 95/154 [00:14<00:08,  7.22it/s]
 62%|██████▏   | 96/154 [00:15<00:08,  7.09it/s]
 63%|██████▎   | 97/154 [00:15<00:08,  6.97it/s]
 64%|██████▎   | 98/154 [00:15<00:08,  6.66it/s]
 64%|██████▍   | 99/154 [00:15<00:08,  6.33it/s]
 65%|██████▍   | 100/154 [00:15<00:07,  6.80it/s]
 66%|██████▌   | 101/154 [00:15<00:07,  6.78it/s]
 66%|██████▌   | 102/154 [00:15<00:07,  6.79it/s]
 67%|██████▋   | 103/154 [00:16<00:07,  6.96it/s]
 68%|██████▊   | 104/154 [00:16<00:07,  7.06it/s]
 68%|██████▊   | 105/154 [00:16<00:07,  6.80it/s]
 69%|██████▉   | 106/154 [00:16<00:07,  6.80it/s]
 69%|██████▉   | 107/154 [00:16<00:07,  6.52it/s]
 70%|███████   | 108/154 [00:16<00:06,  6.61it/s]
 71%|███████   | 109/154 [00:17<00:08,  5.33it/s]
 71%|███████▏  | 110/154 [00:17<00:07,  5.79it/s]
 72%|███████▏  | 111/154 [00:17<00:07,  5.81it/s]
 73%|███████▎  | 112/154 [00:17<00:07,  5.82it/s]
 73%|███████▎  | 113/154 [00:17<00:06,  6.31it/s]
 74%|███████▍  | 114/154 [00:17<00:06,  6.57it/s]
 75%|███████▍  | 115/154 [00:18<00:06,  6.44it/s]
 75%|███████▌  | 116/154 [00:18<00:06,  5.88it/s]
 76%|███████▌  | 117/154 [00:18<00:07,  5.14it/s]
 77%|███████▋  | 118/154 [00:18<00:06,  5.55it/s]
 77%|███████▋  | 119/154 [00:18<00:05,  5.93it/s]
 78%|███████▊  | 120/154 [00:18<00:06,  5.51it/s]
 79%|███████▊  | 121/154 [00:19<00:05,  6.12it/s]
 79%|███████▉  | 122/154 [00:19<00:05,  6.22it/s]
 80%|███████▉  | 123/154 [00:19<00:05,  5.75it/s]
 81%|████████  | 124/154 [00:19<00:04,  6.18it/s]
 81%|████████  | 125/154 [00:19<00:04,  6.21it/s]
 82%|████████▏ | 126/154 [00:19<00:04,  6.49it/s]
 82%|████████▏ | 127/154 [00:20<00:04,  5.99it/s]
 83%|████████▎ | 128/154 [00:20<00:04,  5.93it/s]
 84%|████████▍ | 129/154 [00:20<00:03,  6.44it/s]
 84%|████████▍ | 130/154 [00:20<00:03,  6.63it/s]
 85%|████████▌ | 131/154 [00:20<00:03,  6.71it/s]
 86%|████████▌ | 132/154 [00:20<00:03,  6.91it/s]
 86%|████████▋ | 133/154 [00:20<00:03,  6.42it/s]
 87%|████████▋ | 134/154 [00:21<00:03,  6.61it/s]
 88%|████████▊ | 135/154 [00:21<00:02,  6.82it/s]
 88%|████████▊ | 136/154 [00:21<00:02,  6.97it/s]
 89%|████████▉ | 137/154 [00:21<00:02,  7.08it/s]
 90%|████████▉ | 138/154 [00:21<00:02,  6.83it/s]
 90%|█████████ | 139/154 [00:21<00:02,  7.34it/s]
 92%|█████████▏| 141/154 [00:22<00:01,  7.46it/s]
 92%|█████████▏| 142/154 [00:22<00:01,  7.68it/s]
 93%|█████████▎| 143/154 [00:22<00:01,  6.99it/s]
 94%|█████████▎| 144/154 [00:22<00:01,  6.85it/s]
 94%|█████████▍| 145/154 [00:22<00:01,  6.80it/s]
 95%|█████████▍| 146/154 [00:22<00:01,  6.45it/s]
 95%|█████████▌| 147/154 [00:23<00:01,  6.09it/s]
 96%|█████████▌| 148/154 [00:23<00:00,  6.08it/s]
 97%|█████████▋| 149/154 [00:23<00:00,  6.33it/s]
 97%|█████████▋| 150/154 [00:23<00:00,  6.08it/s]
 98%|█████████▊| 151/154 [00:23<00:00,  6.33it/s]
 99%|█████████▊| 152/154 [00:23<00:00,  6.88it/s]
 99%|█████████▉| 153/154 [00:24<00:00,  5.65it/s]
100%|██████████| 154/154 [00:24<00:00,  5.97it/s]
100%|██████████| 154/154 [00:24<00:00,  6.37it/s]
Final retrieval policy: prefer_dynamic
Best dev row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.2481756338899196, 'avg_pred_evidence': 4.246753246753247}
Chosen row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.2481756338899196, 'avg_pred_evidence': 4.246753246753247}
Epoch 1: loss=0.3786 | chosen dev retrieval_F=0.2482 | setting={'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.2481756338899196, 'avg_pred_evidence': 4.246753246753247} | time=121.8s

Reranker epoch 2/3:   0%|          | 0/645 [00:00<?, ?it/s]
Reranker epoch 2/3:   0%|          | 1/645 [00:00<01:39,  6.45it/s]
Reranker epoch 2/3:   0%|          | 2/645 [00:00<01:40,  6.39it/s]
Reranker epoch 2/3:   0%|          | 3/645 [00:00<01:39,  6.42it/s]
Reranker epoch 2/3:   1%|          | 4/645 [00:00<01:40,  6.39it/s]
Reranker epoch 2/3:   1%|          | 5/645 [00:00<01:40,  6.35it/s]
Reranker epoch 2/3:   1%|          | 6/645 [00:00<01:40,  6.33it/s]
Reranker epoch 2/3:   1%|          | 7/645 [00:01<01:39,  6.41it/s]
Reranker epoch 2/3:   1%|          | 8/645 [00:01<01:40,  6.35it/s]
Reranker epoch 2/3:   1%|▏         | 9/645 [00:01<01:39,  6.37it/s]
Reranker epoch 2/3:   2%|▏         | 10/645 [00:01<01:38,  6.44it/s]
Reranker epoch 2/3:   2%|▏         | 11/645 [00:01<01:37,  6.50it/s]
Reranker epoch 2/3:   2%|▏         | 12/645 [00:01<01:37,  6.49it/s]
Reranker epoch 2/3:   2%|▏         | 13/645 [00:02<01:35,  6.58it/s]
Reranker epoch 2/3:   2%|▏         | 14/645 [00:02<01:36,  6.53it/s]
Reranker epoch 2/3:   2%|▏         | 15/645 [00:02<01:36,  6.55it/s]
Reranker epoch 2/3:   2%|▏         | 16/645 [00:02<01:36,  6.53it/s]
Reranker epoch 2/3:   3%|▎         | 17/645 [00:02<01:36,  6.49it/s]
Reranker epoch 2/3:   3%|▎         | 18/645 [00:02<01:36,  6.48it/s]
Reranker epoch 2/3:   3%|▎         | 19/645 [00:02<01:36,  6.49it/s]
Reranker epoch 2/3:   3%|▎         | 20/645 [00:03<01:36,  6.48it/s]
Reranker epoch 2/3:   3%|▎         | 21/645 [00:03<01:36,  6.48it/s]
Reranker epoch 2/3:   3%|▎         | 22/645 [00:03<01:35,  6.50it/s]
Reranker epoch 2/3:   4%|▎         | 23/645 [00:03<01:35,  6.52it/s]
Reranker epoch 2/3:   4%|▎         | 24/645 [00:03<01:36,  6.44it/s]
Reranker epoch 2/3:   4%|▍         | 25/645 [00:03<01:35,  6.47it/s]
Reranker epoch 2/3:   4%|▍         | 26/645 [00:04<01:36,  6.42it/s]
Reranker epoch 2/3:   4%|▍         | 27/645 [00:04<01:34,  6.51it/s]
Reranker epoch 2/3:   4%|▍         | 28/645 [00:04<01:34,  6.55it/s]
Reranker epoch 2/3:   4%|▍         | 29/645 [00:04<01:34,  6.55it/s]
Reranker epoch 2/3:   5%|▍         | 30/645 [00:04<01:34,  6.54it/s]
Reranker epoch 2/3:   5%|▍         | 31/645 [00:04<01:33,  6.55it/s]
Reranker epoch 2/3:   5%|▍         | 32/645 [00:04<01:33,  6.53it/s]
Reranker epoch 2/3:   5%|▌         | 33/645 [00:05<01:32,  6.63it/s]
Reranker epoch 2/3:   5%|▌         | 34/645 [00:05<01:32,  6.58it/s]
Reranker epoch 2/3:   5%|▌         | 35/645 [00:05<01:32,  6.57it/s]
Reranker epoch 2/3:   6%|▌         | 36/645 [00:05<01:32,  6.61it/s]
Reranker epoch 2/3:   6%|▌         | 37/645 [00:05<01:32,  6.55it/s]
Reranker epoch 2/3:   6%|▌         | 38/645 [00:05<01:32,  6.54it/s]
Reranker epoch 2/3:   6%|▌         | 39/645 [00:06<01:32,  6.52it/s]
Reranker epoch 2/3:   6%|▌         | 40/645 [00:06<01:33,  6.48it/s]
Reranker epoch 2/3:   6%|▋         | 41/645 [00:06<01:32,  6.56it/s]
Reranker epoch 2/3:   7%|▋         | 42/645 [00:06<01:31,  6.56it/s]
Reranker epoch 2/3:   7%|▋         | 43/645 [00:06<01:31,  6.56it/s]
Reranker epoch 2/3:   7%|▋         | 44/645 [00:06<01:31,  6.59it/s]
Reranker epoch 2/3:   7%|▋         | 45/645 [00:06<01:29,  6.67it/s]
Reranker epoch 2/3:   7%|▋         | 46/645 [00:07<01:30,  6.64it/s]
Reranker epoch 2/3:   7%|▋         | 47/645 [00:07<01:31,  6.56it/s]
Reranker epoch 2/3:   7%|▋         | 48/645 [00:07<01:30,  6.56it/s]
Reranker epoch 2/3:   8%|▊         | 49/645 [00:07<01:30,  6.60it/s]
Reranker epoch 2/3:   8%|▊         | 50/645 [00:07<01:30,  6.57it/s]
Reranker epoch 2/3:   8%|▊         | 51/645 [00:07<01:30,  6.59it/s]
Reranker epoch 2/3:   8%|▊         | 52/645 [00:07<01:30,  6.56it/s]
Reranker epoch 2/3:   8%|▊         | 53/645 [00:08<01:30,  6.55it/s]
Reranker epoch 2/3:   8%|▊         | 54/645 [00:08<01:30,  6.51it/s]
Reranker epoch 2/3:   9%|▊         | 55/645 [00:08<01:30,  6.50it/s]
Reranker epoch 2/3:   9%|▊         | 56/645 [00:08<01:29,  6.59it/s]
Reranker epoch 2/3:   9%|▉         | 57/645 [00:08<01:29,  6.59it/s]
Reranker epoch 2/3:   9%|▉         | 58/645 [00:08<01:29,  6.57it/s]
Reranker epoch 2/3:   9%|▉         | 59/645 [00:09<01:29,  6.53it/s]
Reranker epoch 2/3:   9%|▉         | 60/645 [00:09<01:30,  6.50it/s]
Reranker epoch 2/3:   9%|▉         | 61/645 [00:09<01:29,  6.52it/s]
Reranker epoch 2/3:  10%|▉         | 62/645 [00:09<01:28,  6.62it/s]
Reranker epoch 2/3:  10%|▉         | 63/645 [00:09<01:28,  6.61it/s]
Reranker epoch 2/3:  10%|▉         | 64/645 [00:09<01:27,  6.62it/s]
Reranker epoch 2/3:  10%|█         | 65/645 [00:09<01:28,  6.57it/s]
Reranker epoch 2/3:  10%|█         | 66/645 [00:10<01:27,  6.61it/s]
Reranker epoch 2/3:  10%|█         | 67/645 [00:10<01:27,  6.58it/s]
Reranker epoch 2/3:  11%|█         | 68/645 [00:10<01:28,  6.53it/s]
Reranker epoch 2/3:  11%|█         | 69/645 [00:10<01:28,  6.51it/s]
Reranker epoch 2/3:  11%|█         | 70/645 [00:10<01:28,  6.50it/s]
Reranker epoch 2/3:  11%|█         | 71/645 [00:10<01:28,  6.50it/s]
Reranker epoch 2/3:  11%|█         | 72/645 [00:11<01:27,  6.54it/s]
Reranker epoch 2/3:  11%|█▏        | 73/645 [00:11<01:28,  6.50it/s]
Reranker epoch 2/3:  11%|█▏        | 74/645 [00:11<01:27,  6.50it/s]
Reranker epoch 2/3:  12%|█▏        | 75/645 [00:11<01:27,  6.51it/s]
Reranker epoch 2/3:  12%|█▏        | 76/645 [00:11<01:26,  6.54it/s]
Reranker epoch 2/3:  12%|█▏        | 77/645 [00:11<01:26,  6.60it/s]
Reranker epoch 2/3:  12%|█▏        | 78/645 [00:11<01:25,  6.59it/s]
Reranker epoch 2/3:  12%|█▏        | 79/645 [00:12<01:25,  6.65it/s]
Reranker epoch 2/3:  12%|█▏        | 80/645 [00:12<01:25,  6.62it/s]
Reranker epoch 2/3:  13%|█▎        | 81/645 [00:12<01:25,  6.59it/s]
Reranker epoch 2/3:  13%|█▎        | 82/645 [00:12<01:25,  6.59it/s]
Reranker epoch 2/3:  13%|█▎        | 83/645 [00:12<01:25,  6.60it/s]
Reranker epoch 2/3:  13%|█▎        | 84/645 [00:12<01:25,  6.59it/s]
Reranker epoch 2/3:  13%|█▎        | 85/645 [00:13<01:24,  6.65it/s]
Reranker epoch 2/3:  13%|█▎        | 86/645 [00:13<01:25,  6.57it/s]
Reranker epoch 2/3:  13%|█▎        | 87/645 [00:13<01:24,  6.60it/s]
Reranker epoch 2/3:  14%|█▎        | 88/645 [00:13<01:24,  6.60it/s]
Reranker epoch 2/3:  14%|█▍        | 89/645 [00:13<01:24,  6.61it/s]
Reranker epoch 2/3:  14%|█▍        | 90/645 [00:13<01:23,  6.65it/s]
Reranker epoch 2/3:  14%|█▍        | 91/645 [00:13<01:23,  6.61it/s]
Reranker epoch 2/3:  14%|█▍        | 92/645 [00:14<01:23,  6.62it/s]
Reranker epoch 2/3:  14%|█▍        | 93/645 [00:14<01:23,  6.59it/s]
Reranker epoch 2/3:  15%|█▍        | 94/645 [00:14<01:23,  6.59it/s]
Reranker epoch 2/3:  15%|█▍        | 95/645 [00:14<01:23,  6.60it/s]
Reranker epoch 2/3:  15%|█▍        | 96/645 [00:14<01:23,  6.59it/s]
Reranker epoch 2/3:  15%|█▌        | 97/645 [00:14<01:23,  6.59it/s]
Reranker epoch 2/3:  15%|█▌        | 98/645 [00:14<01:22,  6.59it/s]
Reranker epoch 2/3:  15%|█▌        | 99/645 [00:15<01:22,  6.65it/s]
Reranker epoch 2/3:  16%|█▌        | 100/645 [00:15<01:22,  6.64it/s]
Reranker epoch 2/3:  16%|█▌        | 101/645 [00:15<01:22,  6.59it/s]
Reranker epoch 2/3:  16%|█▌        | 102/645 [00:15<01:21,  6.68it/s]
Reranker epoch 2/3:  16%|█▌        | 103/645 [00:15<01:20,  6.71it/s]
Reranker epoch 2/3:  16%|█▌        | 104/645 [00:15<01:21,  6.67it/s]
Reranker epoch 2/3:  16%|█▋        | 105/645 [00:16<01:21,  6.64it/s]
Reranker epoch 2/3:  16%|█▋        | 106/645 [00:16<01:21,  6.64it/s]
Reranker epoch 2/3:  17%|█▋        | 107/645 [00:16<01:20,  6.65it/s]
Reranker epoch 2/3:  17%|█▋        | 108/645 [00:16<01:22,  6.51it/s]
Reranker epoch 2/3:  17%|█▋        | 109/645 [00:16<01:22,  6.53it/s]
Reranker epoch 2/3:  17%|█▋        | 110/645 [00:16<01:22,  6.52it/s]
Reranker epoch 2/3:  17%|█▋        | 111/645 [00:16<01:21,  6.54it/s]
Reranker epoch 2/3:  17%|█▋        | 112/645 [00:17<01:21,  6.56it/s]
Reranker epoch 2/3:  18%|█▊        | 113/645 [00:17<01:20,  6.58it/s]
Reranker epoch 2/3:  18%|█▊        | 114/645 [00:17<01:20,  6.63it/s]
Reranker epoch 2/3:  18%|█▊        | 115/645 [00:17<01:20,  6.62it/s]
Reranker epoch 2/3:  18%|█▊        | 116/645 [00:17<01:18,  6.70it/s]
Reranker epoch 2/3:  18%|█▊        | 117/645 [00:17<01:20,  6.59it/s]
Reranker epoch 2/3:  18%|█▊        | 118/645 [00:18<01:19,  6.59it/s]
Reranker epoch 2/3:  18%|█▊        | 119/645 [00:18<01:19,  6.58it/s]
Reranker epoch 2/3:  19%|█▊        | 120/645 [00:18<01:19,  6.59it/s]
Reranker epoch 2/3:  19%|█▉        | 121/645 [00:18<01:19,  6.62it/s]
Reranker epoch 2/3:  19%|█▉        | 122/645 [00:18<01:19,  6.59it/s]
Reranker epoch 2/3:  19%|█▉        | 123/645 [00:18<01:19,  6.60it/s]
Reranker epoch 2/3:  19%|█▉        | 124/645 [00:18<01:18,  6.67it/s]
Reranker epoch 2/3:  19%|█▉        | 125/645 [00:19<01:18,  6.59it/s]
Reranker epoch 2/3:  20%|█▉        | 126/645 [00:19<01:18,  6.58it/s]
Reranker epoch 2/3:  20%|█▉        | 127/645 [00:19<01:18,  6.63it/s]
Reranker epoch 2/3:  20%|█▉        | 128/645 [00:19<01:18,  6.61it/s]
Reranker epoch 2/3:  20%|██        | 129/645 [00:19<01:18,  6.60it/s]
Reranker epoch 2/3:  20%|██        | 130/645 [00:19<01:17,  6.62it/s]
Reranker epoch 2/3:  20%|██        | 131/645 [00:19<01:17,  6.60it/s]
Reranker epoch 2/3:  20%|██        | 132/645 [00:20<01:17,  6.59it/s]
Reranker epoch 2/3:  21%|██        | 133/645 [00:20<01:17,  6.57it/s]
Reranker epoch 2/3:  21%|██        | 134/645 [00:20<01:17,  6.58it/s]
Reranker epoch 2/3:  21%|██        | 135/645 [00:20<01:17,  6.61it/s]
Reranker epoch 2/3:  21%|██        | 136/645 [00:20<01:16,  6.63it/s]
Reranker epoch 2/3:  21%|██        | 137/645 [00:20<01:17,  6.56it/s]
Reranker epoch 2/3:  21%|██▏       | 138/645 [00:21<01:17,  6.58it/s]
Reranker epoch 2/3:  22%|██▏       | 139/645 [00:21<01:16,  6.60it/s]
Reranker epoch 2/3:  22%|██▏       | 140/645 [00:21<01:15,  6.66it/s]
Reranker epoch 2/3:  22%|██▏       | 141/645 [00:21<01:15,  6.64it/s]
Reranker epoch 2/3:  22%|██▏       | 142/645 [00:21<01:15,  6.63it/s]
Reranker epoch 2/3:  22%|██▏       | 143/645 [00:21<01:15,  6.62it/s]
Reranker epoch 2/3:  22%|██▏       | 144/645 [00:21<01:15,  6.62it/s]
Reranker epoch 2/3:  22%|██▏       | 145/645 [00:22<01:16,  6.58it/s]
Reranker epoch 2/3:  23%|██▎       | 146/645 [00:22<01:15,  6.58it/s]
Reranker epoch 2/3:  23%|██▎       | 147/645 [00:22<01:15,  6.58it/s]
Reranker epoch 2/3:  23%|██▎       | 148/645 [00:22<01:15,  6.58it/s]
Reranker epoch 2/3:  23%|██▎       | 149/645 [00:22<01:15,  6.58it/s]
Reranker epoch 2/3:  23%|██▎       | 150/645 [00:22<01:15,  6.57it/s]
Reranker epoch 2/3:  23%|██▎       | 151/645 [00:22<01:14,  6.64it/s]
Reranker epoch 2/3:  24%|██▎       | 152/645 [00:23<01:14,  6.65it/s]
Reranker epoch 2/3:  24%|██▎       | 153/645 [00:23<01:14,  6.63it/s]
Reranker epoch 2/3:  24%|██▍       | 154/645 [00:23<01:14,  6.59it/s]
Reranker epoch 2/3:  24%|██▍       | 155/645 [00:23<01:14,  6.59it/s]
Reranker epoch 2/3:  24%|██▍       | 156/645 [00:23<01:14,  6.57it/s]
Reranker epoch 2/3:  24%|██▍       | 157/645 [00:23<01:14,  6.57it/s]
Reranker epoch 2/3:  24%|██▍       | 158/645 [00:24<01:13,  6.59it/s]
Reranker epoch 2/3:  25%|██▍       | 159/645 [00:24<01:13,  6.58it/s]
Reranker epoch 2/3:  25%|██▍       | 160/645 [00:24<01:13,  6.60it/s]
Reranker epoch 2/3:  25%|██▍       | 161/645 [00:24<01:13,  6.58it/s]
Reranker epoch 2/3:  25%|██▌       | 162/645 [00:24<01:13,  6.57it/s]
Reranker epoch 2/3:  25%|██▌       | 163/645 [00:24<01:12,  6.63it/s]
Reranker epoch 2/3:  25%|██▌       | 164/645 [00:24<01:12,  6.61it/s]
Reranker epoch 2/3:  26%|██▌       | 165/645 [00:25<01:12,  6.63it/s]
Reranker epoch 2/3:  26%|██▌       | 166/645 [00:25<01:12,  6.60it/s]
Reranker epoch 2/3:  26%|██▌       | 167/645 [00:25<01:12,  6.55it/s]
Reranker epoch 2/3:  26%|██▌       | 168/645 [00:25<01:13,  6.53it/s]
Reranker epoch 2/3:  26%|██▌       | 169/645 [00:25<01:12,  6.58it/s]
Reranker epoch 2/3:  26%|██▋       | 170/645 [00:25<01:12,  6.57it/s]
Reranker epoch 2/3:  27%|██▋       | 171/645 [00:26<01:12,  6.57it/s]
Reranker epoch 2/3:  27%|██▋       | 172/645 [00:26<01:12,  6.52it/s]
Reranker epoch 2/3:  27%|██▋       | 173/645 [00:26<01:11,  6.61it/s]
Reranker epoch 2/3:  27%|██▋       | 174/645 [00:26<01:11,  6.58it/s]
Reranker epoch 2/3:  27%|██▋       | 175/645 [00:26<01:10,  6.63it/s]
Reranker epoch 2/3:  27%|██▋       | 176/645 [00:26<01:10,  6.64it/s]
Reranker epoch 2/3:  27%|██▋       | 177/645 [00:26<01:10,  6.62it/s]
Reranker epoch 2/3:  28%|██▊       | 178/645 [00:27<01:10,  6.60it/s]
Reranker epoch 2/3:  28%|██▊       | 179/645 [00:27<01:11,  6.54it/s]
Reranker epoch 2/3:  28%|██▊       | 180/645 [00:27<01:11,  6.55it/s]
Reranker epoch 2/3:  28%|██▊       | 181/645 [00:27<01:10,  6.56it/s]
Reranker epoch 2/3:  28%|██▊       | 182/645 [00:27<01:10,  6.59it/s]
Reranker epoch 2/3:  28%|██▊       | 183/645 [00:27<01:09,  6.60it/s]
Reranker epoch 2/3:  29%|██▊       | 184/645 [00:28<01:10,  6.56it/s]
Reranker epoch 2/3:  29%|██▊       | 185/645 [00:28<01:09,  6.60it/s]
Reranker epoch 2/3:  29%|██▉       | 186/645 [00:28<01:09,  6.62it/s]
Reranker epoch 2/3:  29%|██▉       | 187/645 [00:28<01:08,  6.72it/s]
Reranker epoch 2/3:  29%|██▉       | 188/645 [00:28<01:08,  6.71it/s]
Reranker epoch 2/3:  29%|██▉       | 189/645 [00:28<01:08,  6.69it/s]
Reranker epoch 2/3:  29%|██▉       | 190/645 [00:28<01:08,  6.68it/s]
Reranker epoch 2/3:  30%|██▉       | 191/645 [00:29<01:08,  6.65it/s]
Reranker epoch 2/3:  30%|██▉       | 192/645 [00:29<01:08,  6.60it/s]
Reranker epoch 2/3:  30%|██▉       | 193/645 [00:29<01:08,  6.55it/s]
Reranker epoch 2/3:  30%|███       | 194/645 [00:29<01:08,  6.59it/s]
Reranker epoch 2/3:  30%|███       | 195/645 [00:29<01:08,  6.54it/s]
Reranker epoch 2/3:  30%|███       | 196/645 [00:29<01:08,  6.56it/s]
Reranker epoch 2/3:  31%|███       | 197/645 [00:29<01:08,  6.56it/s]
Reranker epoch 2/3:  31%|███       | 198/645 [00:30<01:07,  6.60it/s]
Reranker epoch 2/3:  31%|███       | 199/645 [00:30<01:07,  6.65it/s]
Reranker epoch 2/3:  31%|███       | 200/645 [00:30<01:06,  6.66it/s]
Reranker epoch 2/3:  31%|███       | 201/645 [00:30<01:06,  6.65it/s]
Reranker epoch 2/3:  31%|███▏      | 202/645 [00:30<01:06,  6.67it/s]
Reranker epoch 2/3:  31%|███▏      | 203/645 [00:30<01:06,  6.64it/s]
Reranker epoch 2/3:  32%|███▏      | 204/645 [00:31<01:06,  6.62it/s]
Reranker epoch 2/3:  32%|███▏      | 205/645 [00:31<01:06,  6.63it/s]
Reranker epoch 2/3:  32%|███▏      | 206/645 [00:31<01:05,  6.66it/s]
Reranker epoch 2/3:  32%|███▏      | 207/645 [00:31<01:05,  6.72it/s]
Reranker epoch 2/3:  32%|███▏      | 208/645 [00:31<01:05,  6.69it/s]
Reranker epoch 2/3:  32%|███▏      | 209/645 [00:31<01:05,  6.64it/s]
Reranker epoch 2/3:  33%|███▎      | 210/645 [00:31<01:05,  6.62it/s]
Reranker epoch 2/3:  33%|███▎      | 211/645 [00:32<01:05,  6.59it/s]
Reranker epoch 2/3:  33%|███▎      | 212/645 [00:32<01:05,  6.59it/s]
Reranker epoch 2/3:  33%|███▎      | 213/645 [00:32<01:04,  6.65it/s]
Reranker epoch 2/3:  33%|███▎      | 214/645 [00:32<01:05,  6.62it/s]
Reranker epoch 2/3:  33%|███▎      | 215/645 [00:32<01:04,  6.63it/s]
Reranker epoch 2/3:  33%|███▎      | 216/645 [00:32<01:04,  6.61it/s]
Reranker epoch 2/3:  34%|███▎      | 217/645 [00:32<01:04,  6.61it/s]
Reranker epoch 2/3:  34%|███▍      | 218/645 [00:33<01:04,  6.61it/s]
Reranker epoch 2/3:  34%|███▍      | 219/645 [00:33<01:04,  6.56it/s]
Reranker epoch 2/3:  34%|███▍      | 220/645 [00:33<01:04,  6.63it/s]
Reranker epoch 2/3:  34%|███▍      | 221/645 [00:33<01:04,  6.59it/s]
Reranker epoch 2/3:  34%|███▍      | 222/645 [00:33<01:04,  6.59it/s]
Reranker epoch 2/3:  35%|███▍      | 223/645 [00:33<01:03,  6.61it/s]
Reranker epoch 2/3:  35%|███▍      | 224/645 [00:34<01:04,  6.57it/s]
Reranker epoch 2/3:  35%|███▍      | 225/645 [00:34<01:03,  6.63it/s]
Reranker epoch 2/3:  35%|███▌      | 226/645 [00:34<01:03,  6.61it/s]
Reranker epoch 2/3:  35%|███▌      | 227/645 [00:34<01:03,  6.62it/s]
Reranker epoch 2/3:  35%|███▌      | 228/645 [00:34<01:03,  6.62it/s]
Reranker epoch 2/3:  36%|███▌      | 229/645 [00:34<01:03,  6.58it/s]
Reranker epoch 2/3:  36%|███▌      | 230/645 [00:34<01:03,  6.58it/s]
Reranker epoch 2/3:  36%|███▌      | 231/645 [00:35<01:02,  6.61it/s]
Reranker epoch 2/3:  36%|███▌      | 232/645 [00:35<01:02,  6.61it/s]
Reranker epoch 2/3:  36%|███▌      | 233/645 [00:35<01:02,  6.59it/s]
Reranker epoch 2/3:  36%|███▋      | 234/645 [00:35<01:02,  6.58it/s]
Reranker epoch 2/3:  36%|███▋      | 235/645 [00:35<01:01,  6.62it/s]
Reranker epoch 2/3:  37%|███▋      | 236/645 [00:35<01:02,  6.58it/s]
Reranker epoch 2/3:  37%|███▋      | 237/645 [00:36<01:01,  6.65it/s]
Reranker epoch 2/3:  37%|███▋      | 238/645 [00:36<01:01,  6.63it/s]
Reranker epoch 2/3:  37%|███▋      | 239/645 [00:36<01:00,  6.66it/s]
Reranker epoch 2/3:  37%|███▋      | 240/645 [00:36<01:00,  6.65it/s]
Reranker epoch 2/3:  37%|███▋      | 241/645 [00:36<01:00,  6.63it/s]
Reranker epoch 2/3:  38%|███▊      | 242/645 [00:36<01:00,  6.61it/s]
Reranker epoch 2/3:  38%|███▊      | 243/645 [00:36<01:00,  6.65it/s]
Reranker epoch 2/3:  38%|███▊      | 244/645 [00:37<01:00,  6.68it/s]
Reranker epoch 2/3:  38%|███▊      | 245/645 [00:37<01:00,  6.65it/s]
Reranker epoch 2/3:  38%|███▊      | 246/645 [00:37<01:00,  6.64it/s]
Reranker epoch 2/3:  38%|███▊      | 247/645 [00:37<00:59,  6.68it/s]
Reranker epoch 2/3:  38%|███▊      | 248/645 [00:37<01:00,  6.59it/s]
Reranker epoch 2/3:  39%|███▊      | 249/645 [00:37<01:00,  6.58it/s]
Reranker epoch 2/3:  39%|███▉      | 250/645 [00:37<00:59,  6.63it/s]
Reranker epoch 2/3:  39%|███▉      | 251/645 [00:38<00:59,  6.63it/s]
Reranker epoch 2/3:  39%|███▉      | 252/645 [00:38<00:59,  6.61it/s]
Reranker epoch 2/3:  39%|███▉      | 253/645 [00:38<00:59,  6.62it/s]
Reranker epoch 2/3:  39%|███▉      | 254/645 [00:38<00:59,  6.61it/s]
Reranker epoch 2/3:  40%|███▉      | 255/645 [00:38<00:59,  6.60it/s]
Reranker epoch 2/3:  40%|███▉      | 256/645 [00:38<00:59,  6.56it/s]
Reranker epoch 2/3:  40%|███▉      | 257/645 [00:39<00:58,  6.61it/s]
Reranker epoch 2/3:  40%|████      | 258/645 [00:39<00:58,  6.59it/s]
Reranker epoch 2/3:  40%|████      | 259/645 [00:39<00:58,  6.65it/s]
Reranker epoch 2/3:  40%|████      | 260/645 [00:39<00:57,  6.65it/s]
Reranker epoch 2/3:  40%|████      | 261/645 [00:39<00:57,  6.62it/s]
Reranker epoch 2/3:  41%|████      | 262/645 [00:39<00:57,  6.61it/s]
Reranker epoch 2/3:  41%|████      | 263/645 [00:39<00:57,  6.59it/s]
Reranker epoch 2/3:  41%|████      | 264/645 [00:40<00:57,  6.61it/s]
Reranker epoch 2/3:  41%|████      | 265/645 [00:40<00:57,  6.63it/s]
Reranker epoch 2/3:  41%|████      | 266/645 [00:40<00:57,  6.61it/s]
Reranker epoch 2/3:  41%|████▏     | 267/645 [00:40<00:57,  6.59it/s]
Reranker epoch 2/3:  42%|████▏     | 268/645 [00:40<00:56,  6.65it/s]
Reranker epoch 2/3:  42%|████▏     | 269/645 [00:40<00:56,  6.62it/s]
Reranker epoch 2/3:  42%|████▏     | 270/645 [00:40<00:56,  6.65it/s]
Reranker epoch 2/3:  42%|████▏     | 271/645 [00:41<00:56,  6.57it/s]
Reranker epoch 2/3:  42%|████▏     | 272/645 [00:41<00:56,  6.60it/s]
Reranker epoch 2/3:  42%|████▏     | 273/645 [00:41<00:56,  6.61it/s]
Reranker epoch 2/3:  42%|████▏     | 274/645 [00:41<00:56,  6.56it/s]
Reranker epoch 2/3:  43%|████▎     | 275/645 [00:41<00:55,  6.66it/s]
Reranker epoch 2/3:  43%|████▎     | 276/645 [00:41<00:55,  6.66it/s]
Reranker epoch 2/3:  43%|████▎     | 277/645 [00:42<00:55,  6.63it/s]
Reranker epoch 2/3:  43%|████▎     | 278/645 [00:42<00:55,  6.62it/s]
Reranker epoch 2/3:  43%|████▎     | 279/645 [00:42<00:55,  6.57it/s]
Reranker epoch 2/3:  43%|████▎     | 280/645 [00:42<00:55,  6.60it/s]
Reranker epoch 2/3:  44%|████▎     | 281/645 [00:42<00:55,  6.56it/s]
Reranker epoch 2/3:  44%|████▎     | 282/645 [00:42<00:54,  6.60it/s]
Reranker epoch 2/3:  44%|████▍     | 283/645 [00:42<00:55,  6.56it/s]
Reranker epoch 2/3:  44%|████▍     | 284/645 [00:43<00:54,  6.61it/s]
Reranker epoch 2/3:  44%|████▍     | 285/645 [00:43<00:54,  6.59it/s]
Reranker epoch 2/3:  44%|████▍     | 286/645 [00:43<00:55,  6.51it/s]
Reranker epoch 2/3:  44%|████▍     | 287/645 [00:43<00:54,  6.60it/s]
Reranker epoch 2/3:  45%|████▍     | 288/645 [00:43<00:54,  6.60it/s]
Reranker epoch 2/3:  45%|████▍     | 289/645 [00:43<00:54,  6.58it/s]
Reranker epoch 2/3:  45%|████▍     | 290/645 [00:44<00:53,  6.58it/s]
Reranker epoch 2/3:  45%|████▌     | 291/645 [00:44<00:53,  6.56it/s]
Reranker epoch 2/3:  45%|████▌     | 292/645 [00:44<00:53,  6.57it/s]
Reranker epoch 2/3:  45%|████▌     | 293/645 [00:44<00:52,  6.65it/s]
Reranker epoch 2/3:  46%|████▌     | 294/645 [00:44<00:53,  6.62it/s]
Reranker epoch 2/3:  46%|████▌     | 295/645 [00:44<00:53,  6.59it/s]
Reranker epoch 2/3:  46%|████▌     | 296/645 [00:44<00:53,  6.58it/s]
Reranker epoch 2/3:  46%|████▌     | 297/645 [00:45<00:52,  6.61it/s]
Reranker epoch 2/3:  46%|████▌     | 298/645 [00:45<00:52,  6.61it/s]
Reranker epoch 2/3:  46%|████▋     | 299/645 [00:45<00:51,  6.68it/s]
Reranker epoch 2/3:  47%|████▋     | 300/645 [00:45<00:51,  6.64it/s]
Reranker epoch 2/3:  47%|████▋     | 301/645 [00:45<00:51,  6.65it/s]
Reranker epoch 2/3:  47%|████▋     | 302/645 [00:45<00:51,  6.62it/s]
Reranker epoch 2/3:  47%|████▋     | 303/645 [00:45<00:52,  6.58it/s]
Reranker epoch 2/3:  47%|████▋     | 304/645 [00:46<00:51,  6.56it/s]
Reranker epoch 2/3:  47%|████▋     | 305/645 [00:46<00:51,  6.62it/s]
Reranker epoch 2/3:  47%|████▋     | 306/645 [00:46<00:51,  6.63it/s]
Reranker epoch 2/3:  48%|████▊     | 307/645 [00:46<00:51,  6.59it/s]
Reranker epoch 2/3:  48%|████▊     | 308/645 [00:46<00:51,  6.58it/s]
Reranker epoch 2/3:  48%|████▊     | 309/645 [00:46<00:51,  6.58it/s]
Reranker epoch 2/3:  48%|████▊     | 310/645 [00:47<00:51,  6.56it/s]
Reranker epoch 2/3:  48%|████▊     | 311/645 [00:47<00:50,  6.62it/s]
Reranker epoch 2/3:  48%|████▊     | 312/645 [00:47<00:50,  6.61it/s]
Reranker epoch 2/3:  49%|████▊     | 313/645 [00:47<00:50,  6.63it/s]
Reranker epoch 2/3:  49%|████▊     | 314/645 [00:47<00:50,  6.59it/s]
Reranker epoch 2/3:  49%|████▉     | 315/645 [00:47<00:50,  6.58it/s]
Reranker epoch 2/3:  49%|████▉     | 316/645 [00:47<00:49,  6.59it/s]
Reranker epoch 2/3:  49%|████▉     | 317/645 [00:48<00:49,  6.62it/s]
Reranker epoch 2/3:  49%|████▉     | 318/645 [00:48<00:48,  6.73it/s]
Reranker epoch 2/3:  49%|████▉     | 319/645 [00:48<00:49,  6.65it/s]
Reranker epoch 2/3:  50%|████▉     | 320/645 [00:48<00:48,  6.65it/s]
Reranker epoch 2/3:  50%|████▉     | 321/645 [00:48<00:48,  6.66it/s]
Reranker epoch 2/3:  50%|████▉     | 322/645 [00:48<00:49,  6.57it/s]
Reranker epoch 2/3:  50%|█████     | 323/645 [00:49<00:49,  6.54it/s]
Reranker epoch 2/3:  50%|█████     | 324/645 [00:49<00:48,  6.61it/s]
Reranker epoch 2/3:  50%|█████     | 325/645 [00:49<00:48,  6.59it/s]
Reranker epoch 2/3:  51%|█████     | 326/645 [00:49<00:48,  6.60it/s]
Reranker epoch 2/3:  51%|█████     | 327/645 [00:49<00:48,  6.59it/s]
Reranker epoch 2/3:  51%|█████     | 328/645 [00:49<00:47,  6.65it/s]
Reranker epoch 2/3:  51%|█████     | 329/645 [00:49<00:47,  6.63it/s]
Reranker epoch 2/3:  51%|█████     | 330/645 [00:50<00:47,  6.58it/s]
Reranker epoch 2/3:  51%|█████▏    | 331/645 [00:50<00:47,  6.64it/s]
Reranker epoch 2/3:  51%|█████▏    | 332/645 [00:50<00:47,  6.59it/s]
Reranker epoch 2/3:  52%|█████▏    | 333/645 [00:50<00:47,  6.60it/s]
Reranker epoch 2/3:  52%|█████▏    | 334/645 [00:50<00:47,  6.61it/s]
Reranker epoch 2/3:  52%|█████▏    | 335/645 [00:50<00:47,  6.58it/s]
Reranker epoch 2/3:  52%|█████▏    | 336/645 [00:50<00:46,  6.63it/s]
Reranker epoch 2/3:  52%|█████▏    | 337/645 [00:51<00:46,  6.58it/s]
Reranker epoch 2/3:  52%|█████▏    | 338/645 [00:51<00:46,  6.64it/s]
Reranker epoch 2/3:  53%|█████▎    | 339/645 [00:51<00:46,  6.59it/s]
Reranker epoch 2/3:  53%|█████▎    | 340/645 [00:51<00:46,  6.53it/s]
Reranker epoch 2/3:  53%|█████▎    | 341/645 [00:51<00:46,  6.53it/s]
Reranker epoch 2/3:  53%|█████▎    | 342/645 [00:51<00:45,  6.60it/s]
Reranker epoch 2/3:  53%|█████▎    | 343/645 [00:52<00:45,  6.59it/s]
Reranker epoch 2/3:  53%|█████▎    | 344/645 [00:52<00:45,  6.60it/s]
Reranker epoch 2/3:  53%|█████▎    | 345/645 [00:52<00:45,  6.57it/s]
Reranker epoch 2/3:  54%|█████▎    | 346/645 [00:52<00:45,  6.61it/s]
Reranker epoch 2/3:  54%|█████▍    | 347/645 [00:52<00:45,  6.58it/s]
Reranker epoch 2/3:  54%|█████▍    | 348/645 [00:52<00:44,  6.63it/s]
Reranker epoch 2/3:  54%|█████▍    | 349/645 [00:52<00:44,  6.60it/s]
Reranker epoch 2/3:  54%|█████▍    | 350/645 [00:53<00:44,  6.61it/s]
Reranker epoch 2/3:  54%|█████▍    | 351/645 [00:53<00:44,  6.60it/s]
Reranker epoch 2/3:  55%|█████▍    | 352/645 [00:53<00:44,  6.58it/s]
Reranker epoch 2/3:  55%|█████▍    | 353/645 [00:53<00:44,  6.55it/s]
Reranker epoch 2/3:  55%|█████▍    | 354/645 [00:53<00:44,  6.55it/s]
Reranker epoch 2/3:  55%|█████▌    | 355/645 [00:53<00:43,  6.64it/s]
Reranker epoch 2/3:  55%|█████▌    | 356/645 [00:54<00:43,  6.62it/s]
Reranker epoch 2/3:  55%|█████▌    | 357/645 [00:54<00:43,  6.60it/s]
Reranker epoch 2/3:  56%|█████▌    | 358/645 [00:54<00:43,  6.61it/s]
Reranker epoch 2/3:  56%|█████▌    | 359/645 [00:54<00:43,  6.60it/s]
Reranker epoch 2/3:  56%|█████▌    | 360/645 [00:54<00:42,  6.68it/s]
Reranker epoch 2/3:  56%|█████▌    | 361/645 [00:54<00:42,  6.69it/s]
Reranker epoch 2/3:  56%|█████▌    | 362/645 [00:54<00:42,  6.71it/s]
Reranker epoch 2/3:  56%|█████▋    | 363/645 [00:55<00:42,  6.62it/s]
Reranker epoch 2/3:  56%|█████▋    | 364/645 [00:55<00:42,  6.57it/s]
Reranker epoch 2/3:  57%|█████▋    | 365/645 [00:55<00:42,  6.58it/s]
Reranker epoch 2/3:  57%|█████▋    | 366/645 [00:55<00:42,  6.58it/s]
Reranker epoch 2/3:  57%|█████▋    | 367/645 [00:55<00:41,  6.66it/s]
Reranker epoch 2/3:  57%|█████▋    | 368/645 [00:55<00:41,  6.62it/s]
Reranker epoch 2/3:  57%|█████▋    | 369/645 [00:55<00:41,  6.65it/s]
Reranker epoch 2/3:  57%|█████▋    | 370/645 [00:56<00:41,  6.62it/s]
Reranker epoch 2/3:  58%|█████▊    | 371/645 [00:56<00:41,  6.56it/s]
Reranker epoch 2/3:  58%|█████▊    | 372/645 [00:56<00:41,  6.54it/s]
Reranker epoch 2/3:  58%|█████▊    | 373/645 [00:56<00:41,  6.59it/s]
Reranker epoch 2/3:  58%|█████▊    | 374/645 [00:56<00:41,  6.56it/s]
Reranker epoch 2/3:  58%|█████▊    | 375/645 [00:56<00:40,  6.62it/s]
Reranker epoch 2/3:  58%|█████▊    | 376/645 [00:57<00:40,  6.59it/s]
Reranker epoch 2/3:  58%|█████▊    | 377/645 [00:57<00:40,  6.56it/s]
Reranker epoch 2/3:  59%|█████▊    | 378/645 [00:57<00:40,  6.55it/s]
Reranker epoch 2/3:  59%|█████▉    | 379/645 [00:57<00:40,  6.65it/s]
Reranker epoch 2/3:  59%|█████▉    | 380/645 [00:57<00:40,  6.62it/s]
Reranker epoch 2/3:  59%|█████▉    | 381/645 [00:57<00:40,  6.58it/s]
Reranker epoch 2/3:  59%|█████▉    | 382/645 [00:57<00:39,  6.58it/s]
Reranker epoch 2/3:  59%|█████▉    | 383/645 [00:58<00:39,  6.63it/s]
Reranker epoch 2/3:  60%|█████▉    | 384/645 [00:58<00:39,  6.57it/s]
Reranker epoch 2/3:  60%|█████▉    | 385/645 [00:58<00:39,  6.58it/s]
Reranker epoch 2/3:  60%|█████▉    | 386/645 [00:58<00:39,  6.57it/s]
Reranker epoch 2/3:  60%|██████    | 387/645 [00:58<00:39,  6.58it/s]
Reranker epoch 2/3:  60%|██████    | 388/645 [00:58<00:39,  6.54it/s]
Reranker epoch 2/3:  60%|██████    | 389/645 [00:59<00:39,  6.56it/s]
Reranker epoch 2/3:  60%|██████    | 390/645 [00:59<00:38,  6.54it/s]
Reranker epoch 2/3:  61%|██████    | 391/645 [00:59<00:38,  6.54it/s]
Reranker epoch 2/3:  61%|██████    | 392/645 [00:59<00:38,  6.54it/s]
Reranker epoch 2/3:  61%|██████    | 393/645 [00:59<00:38,  6.56it/s]
Reranker epoch 2/3:  61%|██████    | 394/645 [00:59<00:38,  6.54it/s]
Reranker epoch 2/3:  61%|██████    | 395/645 [00:59<00:37,  6.59it/s]
Reranker epoch 2/3:  61%|██████▏   | 396/645 [01:00<00:37,  6.56it/s]
Reranker epoch 2/3:  62%|██████▏   | 397/645 [01:00<00:37,  6.57it/s]
Reranker epoch 2/3:  62%|██████▏   | 398/645 [01:00<00:37,  6.57it/s]
Reranker epoch 2/3:  62%|██████▏   | 399/645 [01:00<00:37,  6.56it/s]
Reranker epoch 2/3:  62%|██████▏   | 400/645 [01:00<00:37,  6.57it/s]
Reranker epoch 2/3:  62%|██████▏   | 401/645 [01:00<00:37,  6.58it/s]
Reranker epoch 2/3:  62%|██████▏   | 402/645 [01:01<00:37,  6.56it/s]
Reranker epoch 2/3:  62%|██████▏   | 403/645 [01:01<00:36,  6.58it/s]
Reranker epoch 2/3:  63%|██████▎   | 404/645 [01:01<00:36,  6.65it/s]
Reranker epoch 2/3:  63%|██████▎   | 405/645 [01:01<00:36,  6.55it/s]
Reranker epoch 2/3:  63%|██████▎   | 406/645 [01:01<00:36,  6.54it/s]
Reranker epoch 2/3:  63%|██████▎   | 407/645 [01:01<00:36,  6.56it/s]
Reranker epoch 2/3:  63%|██████▎   | 408/645 [01:01<00:36,  6.57it/s]
Reranker epoch 2/3:  63%|██████▎   | 409/645 [01:02<00:35,  6.57it/s]
Reranker epoch 2/3:  64%|██████▎   | 410/645 [01:02<00:35,  6.56it/s]
Reranker epoch 2/3:  64%|██████▎   | 411/645 [01:02<00:35,  6.57it/s]
Reranker epoch 2/3:  64%|██████▍   | 412/645 [01:02<00:35,  6.53it/s]
Reranker epoch 2/3:  64%|██████▍   | 413/645 [01:02<00:35,  6.54it/s]
Reranker epoch 2/3:  64%|██████▍   | 414/645 [01:02<00:34,  6.61it/s]
Reranker epoch 2/3:  64%|██████▍   | 415/645 [01:02<00:34,  6.58it/s]
Reranker epoch 2/3:  64%|██████▍   | 416/645 [01:03<00:34,  6.60it/s]
Reranker epoch 2/3:  65%|██████▍   | 417/645 [01:03<00:34,  6.58it/s]
Reranker epoch 2/3:  65%|██████▍   | 418/645 [01:03<00:34,  6.59it/s]
Reranker epoch 2/3:  65%|██████▍   | 419/645 [01:03<00:34,  6.61it/s]
Reranker epoch 2/3:  65%|██████▌   | 420/645 [01:03<00:34,  6.61it/s]
Reranker epoch 2/3:  65%|██████▌   | 421/645 [01:03<00:33,  6.61it/s]
Reranker epoch 2/3:  65%|██████▌   | 422/645 [01:04<00:33,  6.57it/s]
Reranker epoch 2/3:  66%|██████▌   | 423/645 [01:04<00:33,  6.58it/s]
Reranker epoch 2/3:  66%|██████▌   | 424/645 [01:04<00:33,  6.55it/s]
Reranker epoch 2/3:  66%|██████▌   | 425/645 [01:04<00:33,  6.58it/s]
Reranker epoch 2/3:  66%|██████▌   | 426/645 [01:04<00:33,  6.56it/s]
Reranker epoch 2/3:  66%|██████▌   | 427/645 [01:04<00:33,  6.60it/s]
Reranker epoch 2/3:  66%|██████▋   | 428/645 [01:04<00:32,  6.63it/s]
Reranker epoch 2/3:  67%|██████▋   | 429/645 [01:05<00:32,  6.60it/s]
Reranker epoch 2/3:  67%|██████▋   | 430/645 [01:05<00:32,  6.58it/s]
Reranker epoch 2/3:  67%|██████▋   | 431/645 [01:05<00:32,  6.65it/s]
Reranker epoch 2/3:  67%|██████▋   | 432/645 [01:05<00:32,  6.63it/s]
Reranker epoch 2/3:  67%|██████▋   | 433/645 [01:05<00:31,  6.65it/s]
Reranker epoch 2/3:  67%|██████▋   | 434/645 [01:05<00:31,  6.63it/s]
Reranker epoch 2/3:  67%|██████▋   | 435/645 [01:06<00:31,  6.61it/s]
Reranker epoch 2/3:  68%|██████▊   | 436/645 [01:06<00:31,  6.56it/s]
Reranker epoch 2/3:  68%|██████▊   | 437/645 [01:06<00:31,  6.62it/s]
Reranker epoch 2/3:  68%|██████▊   | 438/645 [01:06<00:31,  6.60it/s]
Reranker epoch 2/3:  68%|██████▊   | 439/645 [01:06<00:31,  6.58it/s]
Reranker epoch 2/3:  68%|██████▊   | 440/645 [01:06<00:31,  6.58it/s]
Reranker epoch 2/3:  68%|██████▊   | 441/645 [01:06<00:30,  6.59it/s]
Reranker epoch 2/3:  69%|██████▊   | 442/645 [01:07<00:30,  6.56it/s]
Reranker epoch 2/3:  69%|██████▊   | 443/645 [01:07<00:30,  6.66it/s]
Reranker epoch 2/3:  69%|██████▉   | 444/645 [01:07<00:30,  6.61it/s]
Reranker epoch 2/3:  69%|██████▉   | 445/645 [01:07<00:30,  6.62it/s]
Reranker epoch 2/3:  69%|██████▉   | 446/645 [01:07<00:30,  6.62it/s]
Reranker epoch 2/3:  69%|██████▉   | 447/645 [01:07<00:30,  6.57it/s]
Reranker epoch 2/3:  69%|██████▉   | 448/645 [01:07<00:30,  6.56it/s]
Reranker epoch 2/3:  70%|██████▉   | 449/645 [01:08<00:29,  6.56it/s]
Reranker epoch 2/3:  70%|██████▉   | 450/645 [01:08<00:29,  6.65it/s]
Reranker epoch 2/3:  70%|██████▉   | 451/645 [01:08<00:29,  6.58it/s]
Reranker epoch 2/3:  70%|███████   | 452/645 [01:08<00:29,  6.58it/s]
Reranker epoch 2/3:  70%|███████   | 453/645 [01:08<00:29,  6.59it/s]
Reranker epoch 2/3:  70%|███████   | 454/645 [01:08<00:28,  6.60it/s]
Reranker epoch 2/3:  71%|███████   | 455/645 [01:09<00:28,  6.71it/s]
Reranker epoch 2/3:  71%|███████   | 456/645 [01:09<00:28,  6.66it/s]
Reranker epoch 2/3:  71%|███████   | 457/645 [01:09<00:28,  6.62it/s]
Reranker epoch 2/3:  71%|███████   | 458/645 [01:09<00:28,  6.64it/s]
Reranker epoch 2/3:  71%|███████   | 459/645 [01:09<00:28,  6.58it/s]
Reranker epoch 2/3:  71%|███████▏  | 460/645 [01:09<00:28,  6.52it/s]
Reranker epoch 2/3:  71%|███████▏  | 461/645 [01:09<00:28,  6.51it/s]
Reranker epoch 2/3:  72%|███████▏  | 462/645 [01:10<00:28,  6.53it/s]
Reranker epoch 2/3:  72%|███████▏  | 463/645 [01:10<00:28,  6.49it/s]
Reranker epoch 2/3:  72%|███████▏  | 464/645 [01:10<00:27,  6.48it/s]
Reranker epoch 2/3:  72%|███████▏  | 465/645 [01:10<00:27,  6.44it/s]
Reranker epoch 2/3:  72%|███████▏  | 466/645 [01:10<00:27,  6.52it/s]
Reranker epoch 2/3:  72%|███████▏  | 467/645 [01:10<00:27,  6.46it/s]
Reranker epoch 2/3:  73%|███████▎  | 468/645 [01:11<00:27,  6.44it/s]
Reranker epoch 2/3:  73%|███████▎  | 469/645 [01:11<00:27,  6.45it/s]
Reranker epoch 2/3:  73%|███████▎  | 470/645 [01:11<00:27,  6.47it/s]
Reranker epoch 2/3:  73%|███████▎  | 471/645 [01:11<00:26,  6.52it/s]
Reranker epoch 2/3:  73%|███████▎  | 472/645 [01:11<00:26,  6.52it/s]
Reranker epoch 2/3:  73%|███████▎  | 473/645 [01:11<00:26,  6.56it/s]
Reranker epoch 2/3:  73%|███████▎  | 474/645 [01:11<00:25,  6.59it/s]
Reranker epoch 2/3:  74%|███████▎  | 475/645 [01:12<00:25,  6.62it/s]
Reranker epoch 2/3:  74%|███████▍  | 476/645 [01:12<00:25,  6.58it/s]
Reranker epoch 2/3:  74%|███████▍  | 477/645 [01:12<00:25,  6.59it/s]
Reranker epoch 2/3:  74%|███████▍  | 478/645 [01:12<00:25,  6.58it/s]
Reranker epoch 2/3:  74%|███████▍  | 479/645 [01:12<00:24,  6.64it/s]
Reranker epoch 2/3:  74%|███████▍  | 480/645 [01:12<00:25,  6.58it/s]
Reranker epoch 2/3:  75%|███████▍  | 481/645 [01:13<00:25,  6.56it/s]
Reranker epoch 2/3:  75%|███████▍  | 482/645 [01:13<00:24,  6.56it/s]
Reranker epoch 2/3:  75%|███████▍  | 483/645 [01:13<00:24,  6.62it/s]
Reranker epoch 2/3:  75%|███████▌  | 484/645 [01:13<00:24,  6.61it/s]
Reranker epoch 2/3:  75%|███████▌  | 485/645 [01:13<00:24,  6.61it/s]
Reranker epoch 2/3:  75%|███████▌  | 486/645 [01:13<00:24,  6.58it/s]
Reranker epoch 2/3:  76%|███████▌  | 487/645 [01:13<00:23,  6.59it/s]
Reranker epoch 2/3:  76%|███████▌  | 488/645 [01:14<00:23,  6.57it/s]
Reranker epoch 2/3:  76%|███████▌  | 489/645 [01:14<00:23,  6.56it/s]
Reranker epoch 2/3:  76%|███████▌  | 490/645 [01:14<00:23,  6.57it/s]
Reranker epoch 2/3:  76%|███████▌  | 491/645 [01:14<00:23,  6.55it/s]
Reranker epoch 2/3:  76%|███████▋  | 492/645 [01:14<00:23,  6.59it/s]
Reranker epoch 2/3:  76%|███████▋  | 493/645 [01:14<00:23,  6.59it/s]
Reranker epoch 2/3:  77%|███████▋  | 494/645 [01:14<00:22,  6.65it/s]
Reranker epoch 2/3:  77%|███████▋  | 495/645 [01:15<00:22,  6.63it/s]
Reranker epoch 2/3:  77%|███████▋  | 496/645 [01:15<00:22,  6.66it/s]
Reranker epoch 2/3:  77%|███████▋  | 497/645 [01:15<00:22,  6.63it/s]
Reranker epoch 2/3:  77%|███████▋  | 498/645 [01:15<00:22,  6.58it/s]
Reranker epoch 2/3:  77%|███████▋  | 499/645 [01:15<00:21,  6.66it/s]
Reranker epoch 2/3:  78%|███████▊  | 500/645 [01:15<00:21,  6.63it/s]
Reranker epoch 2/3:  78%|███████▊  | 501/645 [01:16<00:21,  6.73it/s]
Reranker epoch 2/3:  78%|███████▊  | 502/645 [01:16<00:21,  6.68it/s]
Reranker epoch 2/3:  78%|███████▊  | 503/645 [01:16<00:21,  6.66it/s]
Reranker epoch 2/3:  78%|███████▊  | 504/645 [01:16<00:21,  6.66it/s]
Reranker epoch 2/3:  78%|███████▊  | 505/645 [01:16<00:21,  6.59it/s]
Reranker epoch 2/3:  78%|███████▊  | 506/645 [01:16<00:21,  6.61it/s]
Reranker epoch 2/3:  79%|███████▊  | 507/645 [01:16<00:20,  6.58it/s]
Reranker epoch 2/3:  79%|███████▉  | 508/645 [01:17<00:20,  6.68it/s]
Reranker epoch 2/3:  79%|███████▉  | 509/645 [01:17<00:20,  6.63it/s]
Reranker epoch 2/3:  79%|███████▉  | 510/645 [01:17<00:20,  6.63it/s]
Reranker epoch 2/3:  79%|███████▉  | 511/645 [01:17<00:20,  6.62it/s]
Reranker epoch 2/3:  79%|███████▉  | 512/645 [01:17<00:20,  6.61it/s]
Reranker epoch 2/3:  80%|███████▉  | 513/645 [01:17<00:20,  6.56it/s]
Reranker epoch 2/3:  80%|███████▉  | 514/645 [01:18<00:19,  6.60it/s]
Reranker epoch 2/3:  80%|███████▉  | 515/645 [01:18<00:19,  6.56it/s]
Reranker epoch 2/3:  80%|████████  | 516/645 [01:18<00:19,  6.61it/s]
Reranker epoch 2/3:  80%|████████  | 517/645 [01:18<00:19,  6.59it/s]
Reranker epoch 2/3:  80%|████████  | 518/645 [01:18<00:19,  6.59it/s]
Reranker epoch 2/3:  80%|████████  | 519/645 [01:18<00:19,  6.60it/s]
Reranker epoch 2/3:  81%|████████  | 520/645 [01:18<00:18,  6.60it/s]
Reranker epoch 2/3:  81%|████████  | 521/645 [01:19<00:18,  6.70it/s]
Reranker epoch 2/3:  81%|████████  | 522/645 [01:19<00:18,  6.67it/s]
Reranker epoch 2/3:  81%|████████  | 523/645 [01:19<00:18,  6.65it/s]
Reranker epoch 2/3:  81%|████████  | 524/645 [01:19<00:18,  6.64it/s]
Reranker epoch 2/3:  81%|████████▏ | 525/645 [01:19<00:18,  6.60it/s]
Reranker epoch 2/3:  82%|████████▏ | 526/645 [01:19<00:18,  6.55it/s]
Reranker epoch 2/3:  82%|████████▏ | 527/645 [01:19<00:18,  6.53it/s]
Reranker epoch 2/3:  82%|████████▏ | 528/645 [01:20<00:17,  6.62it/s]
Reranker epoch 2/3:  82%|████████▏ | 529/645 [01:20<00:17,  6.62it/s]
Reranker epoch 2/3:  82%|████████▏ | 530/645 [01:20<00:17,  6.58it/s]
Reranker epoch 2/3:  82%|████████▏ | 531/645 [01:20<00:17,  6.60it/s]
Reranker epoch 2/3:  82%|████████▏ | 532/645 [01:20<00:17,  6.50it/s]
Reranker epoch 2/3:  83%|████████▎ | 533/645 [01:20<00:16,  6.59it/s]
Reranker epoch 2/3:  83%|████████▎ | 534/645 [01:21<00:17,  6.48it/s]
Reranker epoch 2/3:  83%|████████▎ | 535/645 [01:21<00:17,  6.42it/s]
Reranker epoch 2/3:  83%|████████▎ | 536/645 [01:21<00:16,  6.41it/s]
Reranker epoch 2/3:  83%|████████▎ | 537/645 [01:21<00:16,  6.42it/s]
Reranker epoch 2/3:  83%|████████▎ | 538/645 [01:21<00:16,  6.42it/s]
Reranker epoch 2/3:  84%|████████▎ | 539/645 [01:21<00:16,  6.37it/s]
Reranker epoch 2/3:  84%|████████▎ | 540/645 [01:21<00:16,  6.44it/s]
Reranker epoch 2/3:  84%|████████▍ | 541/645 [01:22<00:16,  6.45it/s]
Reranker epoch 2/3:  84%|████████▍ | 542/645 [01:22<00:15,  6.49it/s]
Reranker epoch 2/3:  84%|████████▍ | 543/645 [01:22<00:15,  6.50it/s]
Reranker epoch 2/3:  84%|████████▍ | 544/645 [01:22<00:15,  6.49it/s]
Reranker epoch 2/3:  84%|████████▍ | 545/645 [01:22<00:15,  6.47it/s]
Reranker epoch 2/3:  85%|████████▍ | 546/645 [01:22<00:15,  6.53it/s]
Reranker epoch 2/3:  85%|████████▍ | 547/645 [01:23<00:14,  6.61it/s]
Reranker epoch 2/3:  85%|████████▍ | 548/645 [01:23<00:14,  6.60it/s]
Reranker epoch 2/3:  85%|████████▌ | 549/645 [01:23<00:14,  6.60it/s]
Reranker epoch 2/3:  85%|████████▌ | 550/645 [01:23<00:14,  6.56it/s]
Reranker epoch 2/3:  85%|████████▌ | 551/645 [01:23<00:14,  6.53it/s]
Reranker epoch 2/3:  86%|████████▌ | 552/645 [01:23<00:14,  6.58it/s]
Reranker epoch 2/3:  86%|████████▌ | 553/645 [01:23<00:14,  6.55it/s]
Reranker epoch 2/3:  86%|████████▌ | 554/645 [01:24<00:13,  6.57it/s]
Reranker epoch 2/3:  86%|████████▌ | 555/645 [01:24<00:13,  6.56it/s]
Reranker epoch 2/3:  86%|████████▌ | 556/645 [01:24<00:13,  6.57it/s]
Reranker epoch 2/3:  86%|████████▋ | 557/645 [01:24<00:13,  6.58it/s]
Reranker epoch 2/3:  87%|████████▋ | 558/645 [01:24<00:13,  6.67it/s]
Reranker epoch 2/3:  87%|████████▋ | 559/645 [01:24<00:12,  6.63it/s]
Reranker epoch 2/3:  87%|████████▋ | 560/645 [01:25<00:12,  6.62it/s]
Reranker epoch 2/3:  87%|████████▋ | 561/645 [01:25<00:12,  6.63it/s]
Reranker epoch 2/3:  87%|████████▋ | 562/645 [01:25<00:12,  6.58it/s]
Reranker epoch 2/3:  87%|████████▋ | 563/645 [01:25<00:12,  6.62it/s]
Reranker epoch 2/3:  87%|████████▋ | 564/645 [01:25<00:12,  6.52it/s]
Reranker epoch 2/3:  88%|████████▊ | 565/645 [01:25<00:12,  6.51it/s]
Reranker epoch 2/3:  88%|████████▊ | 566/645 [01:25<00:12,  6.57it/s]
Reranker epoch 2/3:  88%|████████▊ | 567/645 [01:26<00:11,  6.62it/s]
Reranker epoch 2/3:  88%|████████▊ | 568/645 [01:26<00:11,  6.57it/s]
Reranker epoch 2/3:  88%|████████▊ | 569/645 [01:26<00:11,  6.53it/s]
Reranker epoch 2/3:  88%|████████▊ | 570/645 [01:26<00:11,  6.53it/s]
Reranker epoch 2/3:  89%|████████▊ | 571/645 [01:26<00:11,  6.59it/s]
Reranker epoch 2/3:  89%|████████▊ | 572/645 [01:26<00:11,  6.60it/s]
Reranker epoch 2/3:  89%|████████▉ | 573/645 [01:27<00:10,  6.57it/s]
Reranker epoch 2/3:  89%|████████▉ | 574/645 [01:27<00:10,  6.56it/s]
Reranker epoch 2/3:  89%|████████▉ | 575/645 [01:27<00:10,  6.61it/s]
Reranker epoch 2/3:  89%|████████▉ | 576/645 [01:27<00:10,  6.63it/s]
Reranker epoch 2/3:  89%|████████▉ | 577/645 [01:27<00:10,  6.61it/s]
Reranker epoch 2/3:  90%|████████▉ | 578/645 [01:27<00:10,  6.64it/s]
Reranker epoch 2/3:  90%|████████▉ | 579/645 [01:27<00:09,  6.65it/s]
Reranker epoch 2/3:  90%|████████▉ | 580/645 [01:28<00:09,  6.62it/s]
Reranker epoch 2/3:  90%|█████████ | 581/645 [01:28<00:09,  6.61it/s]
Reranker epoch 2/3:  90%|█████████ | 582/645 [01:28<00:09,  6.67it/s]
Reranker epoch 2/3:  90%|█████████ | 583/645 [01:28<00:09,  6.70it/s]
Reranker epoch 2/3:  91%|█████████ | 584/645 [01:28<00:09,  6.66it/s]
Reranker epoch 2/3:  91%|█████████ | 585/645 [01:28<00:09,  6.64it/s]
Reranker epoch 2/3:  91%|█████████ | 586/645 [01:28<00:08,  6.58it/s]
Reranker epoch 2/3:  91%|█████████ | 587/645 [01:29<00:08,  6.61it/s]
Reranker epoch 2/3:  91%|█████████ | 588/645 [01:29<00:08,  6.59it/s]
Reranker epoch 2/3:  91%|█████████▏| 589/645 [01:29<00:08,  6.59it/s]
Reranker epoch 2/3:  91%|█████████▏| 590/645 [01:29<00:08,  6.59it/s]
Reranker epoch 2/3:  92%|█████████▏| 591/645 [01:29<00:08,  6.58it/s]
Reranker epoch 2/3:  92%|█████████▏| 592/645 [01:29<00:08,  6.58it/s]
Reranker epoch 2/3:  92%|█████████▏| 593/645 [01:30<00:07,  6.55it/s]
Reranker epoch 2/3:  92%|█████████▏| 594/645 [01:30<00:07,  6.61it/s]
Reranker epoch 2/3:  92%|█████████▏| 595/645 [01:30<00:07,  6.62it/s]
Reranker epoch 2/3:  92%|█████████▏| 596/645 [01:30<00:07,  6.62it/s]
Reranker epoch 2/3:  93%|█████████▎| 597/645 [01:30<00:07,  6.61it/s]
Reranker epoch 2/3:  93%|█████████▎| 598/645 [01:30<00:07,  6.61it/s]
Reranker epoch 2/3:  93%|█████████▎| 599/645 [01:30<00:06,  6.57it/s]
Reranker epoch 2/3:  93%|█████████▎| 600/645 [01:31<00:06,  6.59it/s]
Reranker epoch 2/3:  93%|█████████▎| 601/645 [01:31<00:06,  6.59it/s]
Reranker epoch 2/3:  93%|█████████▎| 602/645 [01:31<00:06,  6.59it/s]
Reranker epoch 2/3:  93%|█████████▎| 603/645 [01:31<00:06,  6.59it/s]
Reranker epoch 2/3:  94%|█████████▎| 604/645 [01:31<00:06,  6.60it/s]
Reranker epoch 2/3:  94%|█████████▍| 605/645 [01:31<00:06,  6.59it/s]
Reranker epoch 2/3:  94%|█████████▍| 606/645 [01:31<00:05,  6.64it/s]
Reranker epoch 2/3:  94%|█████████▍| 607/645 [01:32<00:05,  6.62it/s]
Reranker epoch 2/3:  94%|█████████▍| 608/645 [01:32<00:05,  6.64it/s]
Reranker epoch 2/3:  94%|█████████▍| 609/645 [01:32<00:05,  6.65it/s]
Reranker epoch 2/3:  95%|█████████▍| 610/645 [01:32<00:05,  6.70it/s]
Reranker epoch 2/3:  95%|█████████▍| 611/645 [01:32<00:05,  6.67it/s]
Reranker epoch 2/3:  95%|█████████▍| 612/645 [01:32<00:04,  6.68it/s]
Reranker epoch 2/3:  95%|█████████▌| 613/645 [01:33<00:04,  6.60it/s]
Reranker epoch 2/3:  95%|█████████▌| 614/645 [01:33<00:04,  6.59it/s]
Reranker epoch 2/3:  95%|█████████▌| 615/645 [01:33<00:04,  6.59it/s]
Reranker epoch 2/3:  96%|█████████▌| 616/645 [01:33<00:04,  6.59it/s]
Reranker epoch 2/3:  96%|█████████▌| 617/645 [01:33<00:04,  6.56it/s]
Reranker epoch 2/3:  96%|█████████▌| 618/645 [01:33<00:04,  6.54it/s]
Reranker epoch 2/3:  96%|█████████▌| 619/645 [01:33<00:03,  6.61it/s]
Reranker epoch 2/3:  96%|█████████▌| 620/645 [01:34<00:03,  6.63it/s]
Reranker epoch 2/3:  96%|█████████▋| 621/645 [01:34<00:03,  6.57it/s]
Reranker epoch 2/3:  96%|█████████▋| 622/645 [01:34<00:03,  6.56it/s]
Reranker epoch 2/3:  97%|█████████▋| 623/645 [01:34<00:03,  6.56it/s]
Reranker epoch 2/3:  97%|█████████▋| 624/645 [01:34<00:03,  6.57it/s]
Reranker epoch 2/3:  97%|█████████▋| 625/645 [01:34<00:03,  6.66it/s]
Reranker epoch 2/3:  97%|█████████▋| 626/645 [01:35<00:02,  6.61it/s]
Reranker epoch 2/3:  97%|█████████▋| 627/645 [01:35<00:02,  6.60it/s]
Reranker epoch 2/3:  97%|█████████▋| 628/645 [01:35<00:02,  6.57it/s]
Reranker epoch 2/3:  98%|█████████▊| 629/645 [01:35<00:02,  6.59it/s]
Reranker epoch 2/3:  98%|█████████▊| 630/645 [01:35<00:02,  6.55it/s]
Reranker epoch 2/3:  98%|█████████▊| 631/645 [01:35<00:02,  6.58it/s]
Reranker epoch 2/3:  98%|█████████▊| 632/645 [01:35<00:01,  6.57it/s]
Reranker epoch 2/3:  98%|█████████▊| 633/645 [01:36<00:01,  6.63it/s]
Reranker epoch 2/3:  98%|█████████▊| 634/645 [01:36<00:01,  6.60it/s]
Reranker epoch 2/3:  98%|█████████▊| 635/645 [01:36<00:01,  6.61it/s]
Reranker epoch 2/3:  99%|█████████▊| 636/645 [01:36<00:01,  6.57it/s]
Reranker epoch 2/3:  99%|█████████▉| 637/645 [01:36<00:01,  6.63it/s]
Reranker epoch 2/3:  99%|█████████▉| 638/645 [01:36<00:01,  6.60it/s]
Reranker epoch 2/3:  99%|█████████▉| 639/645 [01:36<00:00,  6.56it/s]
Reranker epoch 2/3:  99%|█████████▉| 640/645 [01:37<00:00,  6.58it/s]
Reranker epoch 2/3:  99%|█████████▉| 641/645 [01:37<00:00,  6.63it/s]
Reranker epoch 2/3: 100%|█████████▉| 642/645 [01:37<00:00,  6.57it/s]
Reranker epoch 2/3: 100%|█████████▉| 643/645 [01:37<00:00,  6.59it/s]
Reranker epoch 2/3: 100%|█████████▉| 644/645 [01:37<00:00,  6.57it/s]
Reranker epoch 2/3: 100%|██████████| 645/645 [01:37<00:00,  6.60it/s]
Scoring dev_epoch2 candidates with cross-encoder reranker...

  0%|          | 0/154 [00:00<?, ?it/s]
  1%|          | 1/154 [00:00<00:15,  9.64it/s]
  1%|▏         | 2/154 [00:00<00:24,  6.22it/s]
  2%|▏         | 3/154 [00:00<00:21,  7.04it/s]
  3%|▎         | 4/154 [00:00<00:21,  6.89it/s]
  3%|▎         | 5/154 [00:00<00:19,  7.49it/s]
  4%|▍         | 6/154 [00:00<00:18,  7.82it/s]
  5%|▍         | 7/154 [00:00<00:19,  7.51it/s]
  5%|▌         | 8/154 [00:01<00:20,  7.06it/s]
  6%|▌         | 9/154 [00:01<00:24,  6.03it/s]
  6%|▋         | 10/154 [00:01<00:21,  6.55it/s]
  7%|▋         | 11/154 [00:01<00:22,  6.28it/s]
  8%|▊         | 12/154 [00:01<00:22,  6.38it/s]
  8%|▊         | 13/154 [00:01<00:21,  6.58it/s]
  9%|▉         | 14/154 [00:02<00:21,  6.65it/s]
 10%|▉         | 15/154 [00:02<00:21,  6.56it/s]
 10%|█         | 16/154 [00:02<00:20,  6.77it/s]
 11%|█         | 17/154 [00:02<00:19,  6.85it/s]
 12%|█▏        | 18/154 [00:02<00:20,  6.65it/s]
 12%|█▏        | 19/154 [00:02<00:18,  7.17it/s]
 13%|█▎        | 20/154 [00:02<00:21,  6.36it/s]
 14%|█▎        | 21/154 [00:03<00:20,  6.49it/s]
 14%|█▍        | 22/154 [00:03<00:19,  6.70it/s]
 15%|█▍        | 23/154 [00:03<00:18,  7.15it/s]
 16%|█▌        | 24/154 [00:03<00:17,  7.48it/s]
 16%|█▌        | 25/154 [00:03<00:19,  6.54it/s]
 17%|█▋        | 26/154 [00:03<00:19,  6.46it/s]
 18%|█▊        | 27/154 [00:03<00:18,  6.77it/s]
 18%|█▊        | 28/154 [00:04<00:18,  6.81it/s]
 19%|█▉        | 29/154 [00:04<00:18,  6.77it/s]
 19%|█▉        | 30/154 [00:04<00:20,  6.19it/s]
 20%|██        | 31/154 [00:04<00:19,  6.15it/s]
 21%|██        | 32/154 [00:04<00:18,  6.60it/s]
 21%|██▏       | 33/154 [00:04<00:17,  7.11it/s]
 22%|██▏       | 34/154 [00:05<00:16,  7.14it/s]
 23%|██▎       | 35/154 [00:05<00:15,  7.44it/s]
 23%|██▎       | 36/154 [00:05<00:16,  7.23it/s]
 24%|██▍       | 37/154 [00:05<00:17,  6.60it/s]
 25%|██▍       | 38/154 [00:05<00:18,  6.12it/s]
 25%|██▌       | 39/154 [00:05<00:17,  6.42it/s]
 26%|██▌       | 40/154 [00:05<00:17,  6.52it/s]
 27%|██▋       | 41/154 [00:06<00:19,  5.92it/s]
 27%|██▋       | 42/154 [00:06<00:18,  5.98it/s]
 28%|██▊       | 43/154 [00:06<00:17,  6.49it/s]
 29%|██▊       | 44/154 [00:06<00:18,  6.01it/s]
 29%|██▉       | 45/154 [00:06<00:18,  5.88it/s]
 30%|██▉       | 46/154 [00:06<00:16,  6.37it/s]
 31%|███       | 47/154 [00:07<00:15,  6.73it/s]
 31%|███       | 48/154 [00:07<00:16,  6.32it/s]
 32%|███▏      | 49/154 [00:07<00:17,  6.03it/s]
 32%|███▏      | 50/154 [00:07<00:17,  6.06it/s]
 33%|███▎      | 51/154 [00:07<00:16,  6.36it/s]
 34%|███▍      | 52/154 [00:07<00:15,  6.55it/s]
 34%|███▍      | 53/154 [00:08<00:16,  6.08it/s]
 35%|███▌      | 54/154 [00:08<00:15,  6.33it/s]
 36%|███▌      | 55/154 [00:08<00:14,  6.68it/s]
 36%|███▋      | 56/154 [00:08<00:14,  6.69it/s]
 37%|███▋      | 57/154 [00:08<00:15,  6.31it/s]
 38%|███▊      | 58/154 [00:08<00:14,  6.50it/s]
 38%|███▊      | 59/154 [00:08<00:15,  6.15it/s]
 39%|███▉      | 60/154 [00:09<00:16,  5.53it/s]
 40%|███▉      | 61/154 [00:09<00:18,  5.02it/s]
 40%|████      | 62/154 [00:09<00:17,  5.26it/s]
 41%|████      | 63/154 [00:09<00:16,  5.54it/s]
 42%|████▏     | 64/154 [00:09<00:16,  5.60it/s]
 42%|████▏     | 65/154 [00:10<00:15,  5.91it/s]
 43%|████▎     | 66/154 [00:10<00:14,  5.98it/s]
 44%|████▎     | 67/154 [00:10<00:15,  5.79it/s]
 44%|████▍     | 68/154 [00:10<00:13,  6.30it/s]
 45%|████▍     | 69/154 [00:10<00:14,  6.00it/s]
 45%|████▌     | 70/154 [00:10<00:15,  5.54it/s]
 46%|████▌     | 71/154 [00:11<00:14,  5.92it/s]
 47%|████▋     | 72/154 [00:11<00:12,  6.31it/s]
 47%|████▋     | 73/154 [00:11<00:12,  6.44it/s]
 48%|████▊     | 74/154 [00:11<00:11,  7.12it/s]
 49%|████▊     | 75/154 [00:11<00:11,  6.84it/s]
 49%|████▉     | 76/154 [00:11<00:10,  7.13it/s]
 50%|█████     | 77/154 [00:11<00:10,  7.62it/s]
 51%|█████     | 78/154 [00:12<00:11,  6.72it/s]
 51%|█████▏    | 79/154 [00:12<00:11,  6.35it/s]
 52%|█████▏    | 80/154 [00:12<00:12,  5.96it/s]
 53%|█████▎    | 81/154 [00:12<00:11,  6.42it/s]
 53%|█████▎    | 82/154 [00:12<00:11,  6.48it/s]
 54%|█████▍    | 83/154 [00:12<00:11,  6.22it/s]
 55%|█████▍    | 84/154 [00:13<00:11,  6.09it/s]
 55%|█████▌    | 85/154 [00:13<00:11,  5.97it/s]
 56%|█████▌    | 86/154 [00:13<00:10,  6.20it/s]
 56%|█████▋    | 87/154 [00:13<00:11,  5.64it/s]
 57%|█████▋    | 88/154 [00:13<00:11,  5.60it/s]
 58%|█████▊    | 89/154 [00:13<00:10,  5.99it/s]
 58%|█████▊    | 90/154 [00:14<00:10,  6.08it/s]
 59%|█████▉    | 91/154 [00:14<00:09,  6.38it/s]
 60%|█████▉    | 92/154 [00:14<00:08,  7.07it/s]
 60%|██████    | 93/154 [00:14<00:08,  7.03it/s]
 61%|██████    | 94/154 [00:14<00:08,  7.05it/s]
 62%|██████▏   | 95/154 [00:14<00:08,  7.33it/s]
 62%|██████▏   | 96/154 [00:14<00:08,  7.10it/s]
 63%|██████▎   | 97/154 [00:15<00:08,  6.93it/s]
 64%|██████▎   | 98/154 [00:15<00:08,  6.58it/s]
 64%|██████▍   | 99/154 [00:15<00:08,  6.23it/s]
 65%|██████▍   | 100/154 [00:15<00:08,  6.69it/s]
 66%|██████▌   | 101/154 [00:15<00:08,  6.56it/s]
 66%|██████▌   | 102/154 [00:15<00:07,  6.60it/s]
 67%|██████▋   | 103/154 [00:15<00:07,  6.75it/s]
 68%|██████▊   | 104/154 [00:16<00:07,  6.89it/s]
 68%|██████▊   | 105/154 [00:16<00:07,  6.62it/s]
 69%|██████▉   | 106/154 [00:16<00:07,  6.61it/s]
 69%|██████▉   | 107/154 [00:16<00:07,  6.30it/s]
 70%|███████   | 108/154 [00:16<00:07,  6.40it/s]
 71%|███████   | 109/154 [00:17<00:08,  5.16it/s]
 71%|███████▏  | 110/154 [00:17<00:07,  5.61it/s]
 72%|███████▏  | 111/154 [00:17<00:07,  5.67it/s]
 73%|███████▎  | 112/154 [00:17<00:07,  5.69it/s]
 73%|███████▎  | 113/154 [00:17<00:06,  6.19it/s]
 74%|███████▍  | 114/154 [00:17<00:06,  6.48it/s]
 75%|███████▍  | 115/154 [00:17<00:06,  6.38it/s]
 75%|███████▌  | 116/154 [00:18<00:06,  5.81it/s]
 76%|███████▌  | 117/154 [00:18<00:07,  5.08it/s]
 77%|███████▋  | 118/154 [00:18<00:06,  5.50it/s]
 77%|███████▋  | 119/154 [00:18<00:05,  5.89it/s]
 78%|███████▊  | 120/154 [00:18<00:06,  5.44it/s]
 79%|███████▊  | 121/154 [00:19<00:05,  6.03it/s]
 79%|███████▉  | 122/154 [00:19<00:05,  6.10it/s]
 80%|███████▉  | 123/154 [00:19<00:05,  5.64it/s]
 81%|████████  | 124/154 [00:19<00:04,  6.05it/s]
 81%|████████  | 125/154 [00:19<00:04,  6.15it/s]
 82%|████████▏ | 126/154 [00:19<00:04,  6.41it/s]
 82%|████████▏ | 127/154 [00:20<00:04,  5.95it/s]
 83%|████████▎ | 128/154 [00:20<00:04,  5.91it/s]
 84%|████████▍ | 129/154 [00:20<00:03,  6.39it/s]
 84%|████████▍ | 130/154 [00:20<00:03,  6.60it/s]
 85%|████████▌ | 131/154 [00:20<00:03,  6.68it/s]
 86%|████████▌ | 132/154 [00:20<00:03,  6.91it/s]
 86%|████████▋ | 133/154 [00:20<00:03,  6.44it/s]
 87%|████████▋ | 134/154 [00:21<00:03,  6.63it/s]
 88%|████████▊ | 135/154 [00:21<00:02,  6.87it/s]
 88%|████████▊ | 136/154 [00:21<00:02,  7.07it/s]
 89%|████████▉ | 137/154 [00:21<00:02,  7.17it/s]
 90%|████████▉ | 138/154 [00:21<00:02,  6.98it/s]
 90%|█████████ | 139/154 [00:21<00:02,  7.48it/s]
 92%|█████████▏| 141/154 [00:22<00:01,  7.62it/s]
 92%|█████████▏| 142/154 [00:22<00:01,  7.90it/s]
 93%|█████████▎| 143/154 [00:22<00:01,  7.12it/s]
 94%|█████████▎| 144/154 [00:22<00:01,  6.90it/s]
 94%|█████████▍| 145/154 [00:22<00:01,  6.85it/s]
 95%|█████████▍| 146/154 [00:22<00:01,  6.48it/s]
 95%|█████████▌| 147/154 [00:22<00:01,  6.10it/s]
 96%|█████████▌| 148/154 [00:23<00:00,  6.09it/s]
 97%|█████████▋| 149/154 [00:23<00:00,  6.35it/s]
 97%|█████████▋| 150/154 [00:23<00:00,  6.08it/s]
 98%|█████████▊| 151/154 [00:23<00:00,  6.37it/s]
 99%|█████████▊| 152/154 [00:23<00:00,  6.91it/s]
 99%|█████████▉| 153/154 [00:23<00:00,  5.65it/s]
100%|██████████| 154/154 [00:24<00:00,  5.93it/s]
100%|██████████| 154/154 [00:24<00:00,  6.39it/s]
Final retrieval policy: prefer_dynamic
Best dev row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.55, 'retrieval_F': 0.2506029684601113, 'avg_pred_evidence': 3.9415584415584415}
Chosen row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.55, 'retrieval_F': 0.2506029684601113, 'avg_pred_evidence': 3.9415584415584415}
Epoch 2: loss=0.2574 | chosen dev retrieval_F=0.2506 | setting={'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.55, 'retrieval_F': 0.2506029684601113, 'avg_pred_evidence': 3.9415584415584415} | time=122.5s

Reranker epoch 3/3:   0%|          | 0/645 [00:00<?, ?it/s]
Reranker epoch 3/3:   0%|          | 1/645 [00:00<01:42,  6.29it/s]
Reranker epoch 3/3:   0%|          | 2/645 [00:00<01:41,  6.33it/s]
Reranker epoch 3/3:   0%|          | 3/645 [00:00<01:41,  6.31it/s]
Reranker epoch 3/3:   1%|          | 4/645 [00:00<01:40,  6.37it/s]
Reranker epoch 3/3:   1%|          | 5/645 [00:00<01:39,  6.46it/s]
Reranker epoch 3/3:   1%|          | 6/645 [00:00<01:37,  6.57it/s]
Reranker epoch 3/3:   1%|          | 7/645 [00:01<01:38,  6.49it/s]
Reranker epoch 3/3:   1%|          | 8/645 [00:01<01:39,  6.43it/s]
Reranker epoch 3/3:   1%|▏         | 9/645 [00:01<01:38,  6.43it/s]
Reranker epoch 3/3:   2%|▏         | 10/645 [00:01<01:37,  6.48it/s]
Reranker epoch 3/3:   2%|▏         | 11/645 [00:01<01:37,  6.47it/s]
Reranker epoch 3/3:   2%|▏         | 12/645 [00:01<01:38,  6.44it/s]
Reranker epoch 3/3:   2%|▏         | 13/645 [00:02<01:36,  6.54it/s]
Reranker epoch 3/3:   2%|▏         | 14/645 [00:02<01:36,  6.53it/s]
Reranker epoch 3/3:   2%|▏         | 15/645 [00:02<01:36,  6.50it/s]
Reranker epoch 3/3:   2%|▏         | 16/645 [00:02<01:36,  6.50it/s]
Reranker epoch 3/3:   3%|▎         | 17/645 [00:02<01:35,  6.61it/s]
Reranker epoch 3/3:   3%|▎         | 18/645 [00:02<01:35,  6.58it/s]
Reranker epoch 3/3:   3%|▎         | 19/645 [00:02<01:35,  6.57it/s]
Reranker epoch 3/3:   3%|▎         | 20/645 [00:03<01:35,  6.55it/s]
Reranker epoch 3/3:   3%|▎         | 21/645 [00:03<01:34,  6.59it/s]
Reranker epoch 3/3:   3%|▎         | 22/645 [00:03<01:35,  6.55it/s]
Reranker epoch 3/3:   4%|▎         | 23/645 [00:03<01:35,  6.53it/s]
Reranker epoch 3/3:   4%|▎         | 24/645 [00:03<01:36,  6.47it/s]
Reranker epoch 3/3:   4%|▍         | 25/645 [00:03<01:36,  6.45it/s]
Reranker epoch 3/3:   4%|▍         | 26/645 [00:04<01:36,  6.43it/s]
Reranker epoch 3/3:   4%|▍         | 27/645 [00:04<01:34,  6.56it/s]
Reranker epoch 3/3:   4%|▍         | 28/645 [00:04<01:35,  6.49it/s]
Reranker epoch 3/3:   4%|▍         | 29/645 [00:04<01:33,  6.57it/s]
Reranker epoch 3/3:   5%|▍         | 30/645 [00:04<01:34,  6.54it/s]
Reranker epoch 3/3:   5%|▍         | 31/645 [00:04<01:32,  6.61it/s]
Reranker epoch 3/3:   5%|▍         | 32/645 [00:04<01:32,  6.59it/s]
Reranker epoch 3/3:   5%|▌         | 33/645 [00:05<01:33,  6.56it/s]
Reranker epoch 3/3:   5%|▌         | 34/645 [00:05<01:32,  6.60it/s]
Reranker epoch 3/3:   5%|▌         | 35/645 [00:05<01:31,  6.65it/s]
Reranker epoch 3/3:   6%|▌         | 36/645 [00:05<01:32,  6.62it/s]
Reranker epoch 3/3:   6%|▌         | 37/645 [00:05<01:31,  6.63it/s]
Reranker epoch 3/3:   6%|▌         | 38/645 [00:05<01:31,  6.65it/s]
Reranker epoch 3/3:   6%|▌         | 39/645 [00:05<01:30,  6.68it/s]
Reranker epoch 3/3:   6%|▌         | 40/645 [00:06<01:31,  6.60it/s]
Reranker epoch 3/3:   6%|▋         | 41/645 [00:06<01:31,  6.58it/s]
Reranker epoch 3/3:   7%|▋         | 42/645 [00:06<01:31,  6.57it/s]
Reranker epoch 3/3:   7%|▋         | 43/645 [00:06<01:31,  6.57it/s]
Reranker epoch 3/3:   7%|▋         | 44/645 [00:06<01:30,  6.61it/s]
Reranker epoch 3/3:   7%|▋         | 45/645 [00:06<01:30,  6.59it/s]
Reranker epoch 3/3:   7%|▋         | 46/645 [00:07<01:31,  6.57it/s]
Reranker epoch 3/3:   7%|▋         | 47/645 [00:07<01:30,  6.61it/s]
Reranker epoch 3/3:   7%|▋         | 48/645 [00:07<01:30,  6.57it/s]
Reranker epoch 3/3:   8%|▊         | 49/645 [00:07<01:30,  6.56it/s]
Reranker epoch 3/3:   8%|▊         | 50/645 [00:07<01:30,  6.58it/s]
Reranker epoch 3/3:   8%|▊         | 51/645 [00:07<01:29,  6.62it/s]
Reranker epoch 3/3:   8%|▊         | 52/645 [00:07<01:29,  6.63it/s]
Reranker epoch 3/3:   8%|▊         | 53/645 [00:08<01:31,  6.49it/s]
Reranker epoch 3/3:   8%|▊         | 54/645 [00:08<01:32,  6.42it/s]
Reranker epoch 3/3:   9%|▊         | 55/645 [00:08<01:31,  6.42it/s]
Reranker epoch 3/3:   9%|▊         | 56/645 [00:08<01:32,  6.39it/s]
Reranker epoch 3/3:   9%|▉         | 57/645 [00:08<01:31,  6.39it/s]
Reranker epoch 3/3:   9%|▉         | 58/645 [00:08<01:32,  6.37it/s]
Reranker epoch 3/3:   9%|▉         | 59/645 [00:09<01:31,  6.37it/s]
Reranker epoch 3/3:   9%|▉         | 60/645 [00:09<01:31,  6.38it/s]
Reranker epoch 3/3:   9%|▉         | 61/645 [00:09<01:31,  6.41it/s]
Reranker epoch 3/3:  10%|▉         | 62/645 [00:09<01:31,  6.38it/s]
Reranker epoch 3/3:  10%|▉         | 63/645 [00:09<01:31,  6.38it/s]
Reranker epoch 3/3:  10%|▉         | 64/645 [00:09<01:30,  6.43it/s]
Reranker epoch 3/3:  10%|█         | 65/645 [00:09<01:29,  6.48it/s]
Reranker epoch 3/3:  10%|█         | 66/645 [00:10<01:30,  6.41it/s]
Reranker epoch 3/3:  10%|█         | 67/645 [00:10<01:29,  6.47it/s]
Reranker epoch 3/3:  11%|█         | 68/645 [00:10<01:29,  6.47it/s]
Reranker epoch 3/3:  11%|█         | 69/645 [00:10<01:28,  6.51it/s]
Reranker epoch 3/3:  11%|█         | 70/645 [00:10<01:28,  6.53it/s]
Reranker epoch 3/3:  11%|█         | 71/645 [00:10<01:28,  6.52it/s]
Reranker epoch 3/3:  11%|█         | 72/645 [00:11<01:27,  6.55it/s]
Reranker epoch 3/3:  11%|█▏        | 73/645 [00:11<01:27,  6.54it/s]
Reranker epoch 3/3:  11%|█▏        | 74/645 [00:11<01:26,  6.63it/s]
Reranker epoch 3/3:  12%|█▏        | 75/645 [00:11<01:26,  6.59it/s]
Reranker epoch 3/3:  12%|█▏        | 76/645 [00:11<01:26,  6.57it/s]
Reranker epoch 3/3:  12%|█▏        | 77/645 [00:11<01:25,  6.64it/s]
Reranker epoch 3/3:  12%|█▏        | 78/645 [00:11<01:24,  6.72it/s]
Reranker epoch 3/3:  12%|█▏        | 79/645 [00:12<01:25,  6.65it/s]
Reranker epoch 3/3:  12%|█▏        | 80/645 [00:12<01:25,  6.57it/s]
Reranker epoch 3/3:  13%|█▎        | 81/645 [00:12<01:25,  6.57it/s]
Reranker epoch 3/3:  13%|█▎        | 82/645 [00:12<01:25,  6.61it/s]
Reranker epoch 3/3:  13%|█▎        | 83/645 [00:12<01:25,  6.57it/s]
Reranker epoch 3/3:  13%|█▎        | 84/645 [00:12<01:25,  6.59it/s]
Reranker epoch 3/3:  13%|█▎        | 85/645 [00:13<01:25,  6.56it/s]
Reranker epoch 3/3:  13%|█▎        | 86/645 [00:13<01:24,  6.60it/s]
Reranker epoch 3/3:  13%|█▎        | 87/645 [00:13<01:24,  6.59it/s]
Reranker epoch 3/3:  14%|█▎        | 88/645 [00:13<01:24,  6.60it/s]
Reranker epoch 3/3:  14%|█▍        | 89/645 [00:13<01:23,  6.62it/s]
Reranker epoch 3/3:  14%|█▍        | 90/645 [00:13<01:23,  6.63it/s]
Reranker epoch 3/3:  14%|█▍        | 91/645 [00:13<01:22,  6.69it/s]
Reranker epoch 3/3:  14%|█▍        | 92/645 [00:14<01:23,  6.66it/s]
Reranker epoch 3/3:  14%|█▍        | 93/645 [00:14<01:22,  6.68it/s]
Reranker epoch 3/3:  15%|█▍        | 94/645 [00:14<01:22,  6.70it/s]
Reranker epoch 3/3:  15%|█▍        | 95/645 [00:14<01:22,  6.70it/s]
Reranker epoch 3/3:  15%|█▍        | 96/645 [00:14<01:22,  6.65it/s]
Reranker epoch 3/3:  15%|█▌        | 97/645 [00:14<01:22,  6.64it/s]
Reranker epoch 3/3:  15%|█▌        | 98/645 [00:14<01:22,  6.66it/s]
Reranker epoch 3/3:  15%|█▌        | 99/645 [00:15<01:21,  6.71it/s]
Reranker epoch 3/3:  16%|█▌        | 100/645 [00:15<01:22,  6.64it/s]
Reranker epoch 3/3:  16%|█▌        | 101/645 [00:15<01:22,  6.63it/s]
Reranker epoch 3/3:  16%|█▌        | 102/645 [00:15<01:22,  6.59it/s]
Reranker epoch 3/3:  16%|█▌        | 103/645 [00:15<01:23,  6.53it/s]
Reranker epoch 3/3:  16%|█▌        | 104/645 [00:15<01:22,  6.53it/s]
Reranker epoch 3/3:  16%|█▋        | 105/645 [00:16<01:22,  6.57it/s]
Reranker epoch 3/3:  16%|█▋        | 106/645 [00:16<01:22,  6.55it/s]
Reranker epoch 3/3:  17%|█▋        | 107/645 [00:16<01:21,  6.61it/s]
Reranker epoch 3/3:  17%|█▋        | 108/645 [00:16<01:21,  6.57it/s]
Reranker epoch 3/3:  17%|█▋        | 109/645 [00:16<01:21,  6.61it/s]
Reranker epoch 3/3:  17%|█▋        | 110/645 [00:16<01:21,  6.54it/s]
Reranker epoch 3/3:  17%|█▋        | 111/645 [00:16<01:21,  6.54it/s]
Reranker epoch 3/3:  17%|█▋        | 112/645 [00:17<01:21,  6.55it/s]
Reranker epoch 3/3:  18%|█▊        | 113/645 [00:17<01:20,  6.62it/s]
Reranker epoch 3/3:  18%|█▊        | 114/645 [00:17<01:20,  6.60it/s]
Reranker epoch 3/3:  18%|█▊        | 115/645 [00:17<01:19,  6.63it/s]
Reranker epoch 3/3:  18%|█▊        | 116/645 [00:17<01:20,  6.55it/s]
Reranker epoch 3/3:  18%|█▊        | 117/645 [00:17<01:20,  6.54it/s]
Reranker epoch 3/3:  18%|█▊        | 118/645 [00:18<01:21,  6.50it/s]
Reranker epoch 3/3:  18%|█▊        | 119/645 [00:18<01:19,  6.59it/s]
Reranker epoch 3/3:  19%|█▊        | 120/645 [00:18<01:20,  6.54it/s]
Reranker epoch 3/3:  19%|█▉        | 121/645 [00:18<01:20,  6.50it/s]
Reranker epoch 3/3:  19%|█▉        | 122/645 [00:18<01:19,  6.54it/s]
Reranker epoch 3/3:  19%|█▉        | 123/645 [00:18<01:19,  6.57it/s]
Reranker epoch 3/3:  19%|█▉        | 124/645 [00:18<01:19,  6.56it/s]
Reranker epoch 3/3:  19%|█▉        | 125/645 [00:19<01:18,  6.64it/s]
Reranker epoch 3/3:  20%|█▉        | 126/645 [00:19<01:18,  6.60it/s]
Reranker epoch 3/3:  20%|█▉        | 127/645 [00:19<01:18,  6.60it/s]
Reranker epoch 3/3:  20%|█▉        | 128/645 [00:19<01:18,  6.57it/s]
Reranker epoch 3/3:  20%|██        | 129/645 [00:19<01:19,  6.52it/s]
Reranker epoch 3/3:  20%|██        | 130/645 [00:19<01:18,  6.52it/s]
Reranker epoch 3/3:  20%|██        | 131/645 [00:19<01:18,  6.52it/s]
Reranker epoch 3/3:  20%|██        | 132/645 [00:20<01:18,  6.53it/s]
Reranker epoch 3/3:  21%|██        | 133/645 [00:20<01:18,  6.56it/s]
Reranker epoch 3/3:  21%|██        | 134/645 [00:20<01:18,  6.55it/s]
Reranker epoch 3/3:  21%|██        | 135/645 [00:20<01:18,  6.53it/s]
Reranker epoch 3/3:  21%|██        | 136/645 [00:20<01:17,  6.58it/s]
Reranker epoch 3/3:  21%|██        | 137/645 [00:20<01:17,  6.54it/s]
Reranker epoch 3/3:  21%|██▏       | 138/645 [00:21<01:17,  6.55it/s]
Reranker epoch 3/3:  22%|██▏       | 139/645 [00:21<01:16,  6.60it/s]
Reranker epoch 3/3:  22%|██▏       | 140/645 [00:21<01:17,  6.55it/s]
Reranker epoch 3/3:  22%|██▏       | 141/645 [00:21<01:16,  6.55it/s]
Reranker epoch 3/3:  22%|██▏       | 142/645 [00:21<01:16,  6.61it/s]
Reranker epoch 3/3:  22%|██▏       | 143/645 [00:21<01:15,  6.63it/s]
Reranker epoch 3/3:  22%|██▏       | 144/645 [00:21<01:15,  6.60it/s]
Reranker epoch 3/3:  22%|██▏       | 145/645 [00:22<01:15,  6.63it/s]
Reranker epoch 3/3:  23%|██▎       | 146/645 [00:22<01:15,  6.58it/s]
Reranker epoch 3/3:  23%|██▎       | 147/645 [00:22<01:15,  6.56it/s]
Reranker epoch 3/3:  23%|██▎       | 148/645 [00:22<01:15,  6.62it/s]
Reranker epoch 3/3:  23%|██▎       | 149/645 [00:22<01:15,  6.60it/s]
Reranker epoch 3/3:  23%|██▎       | 150/645 [00:22<01:15,  6.58it/s]
Reranker epoch 3/3:  23%|██▎       | 151/645 [00:23<01:15,  6.56it/s]
Reranker epoch 3/3:  24%|██▎       | 152/645 [00:23<01:15,  6.51it/s]
Reranker epoch 3/3:  24%|██▎       | 153/645 [00:23<01:14,  6.57it/s]
Reranker epoch 3/3:  24%|██▍       | 154/645 [00:23<01:14,  6.58it/s]
Reranker epoch 3/3:  24%|██▍       | 155/645 [00:23<01:14,  6.56it/s]
Reranker epoch 3/3:  24%|██▍       | 156/645 [00:23<01:14,  6.56it/s]
Reranker epoch 3/3:  24%|██▍       | 157/645 [00:23<01:14,  6.57it/s]
Reranker epoch 3/3:  24%|██▍       | 158/645 [00:24<01:14,  6.57it/s]
Reranker epoch 3/3:  25%|██▍       | 159/645 [00:24<01:14,  6.56it/s]
Reranker epoch 3/3:  25%|██▍       | 160/645 [00:24<01:13,  6.61it/s]
Reranker epoch 3/3:  25%|██▍       | 161/645 [00:24<01:12,  6.69it/s]
Reranker epoch 3/3:  25%|██▌       | 162/645 [00:24<01:12,  6.64it/s]
Reranker epoch 3/3:  25%|██▌       | 163/645 [00:24<01:12,  6.63it/s]
Reranker epoch 3/3:  25%|██▌       | 164/645 [00:25<01:12,  6.61it/s]
Reranker epoch 3/3:  26%|██▌       | 165/645 [00:25<01:11,  6.67it/s]
Reranker epoch 3/3:  26%|██▌       | 166/645 [00:25<01:12,  6.63it/s]
Reranker epoch 3/3:  26%|██▌       | 167/645 [00:25<01:12,  6.56it/s]
Reranker epoch 3/3:  26%|██▌       | 168/645 [00:25<01:12,  6.54it/s]
Reranker epoch 3/3:  26%|██▌       | 169/645 [00:25<01:12,  6.57it/s]
Reranker epoch 3/3:  26%|██▋       | 170/645 [00:25<01:12,  6.53it/s]
Reranker epoch 3/3:  27%|██▋       | 171/645 [00:26<01:13,  6.48it/s]
Reranker epoch 3/3:  27%|██▋       | 172/645 [00:26<01:12,  6.51it/s]
Reranker epoch 3/3:  27%|██▋       | 173/645 [00:26<01:13,  6.43it/s]
Reranker epoch 3/3:  27%|██▋       | 174/645 [00:26<01:12,  6.48it/s]
Reranker epoch 3/3:  27%|██▋       | 175/645 [00:26<01:12,  6.46it/s]
Reranker epoch 3/3:  27%|██▋       | 176/645 [00:26<01:12,  6.51it/s]
Reranker epoch 3/3:  27%|██▋       | 177/645 [00:27<01:12,  6.45it/s]
Reranker epoch 3/3:  28%|██▊       | 178/645 [00:27<01:11,  6.51it/s]
Reranker epoch 3/3:  28%|██▊       | 179/645 [00:27<01:11,  6.51it/s]
Reranker epoch 3/3:  28%|██▊       | 180/645 [00:27<01:11,  6.53it/s]
Reranker epoch 3/3:  28%|██▊       | 181/645 [00:27<01:10,  6.58it/s]
Reranker epoch 3/3:  28%|██▊       | 182/645 [00:27<01:10,  6.53it/s]
Reranker epoch 3/3:  28%|██▊       | 183/645 [00:27<01:10,  6.52it/s]
Reranker epoch 3/3:  29%|██▊       | 184/645 [00:28<01:10,  6.55it/s]
Reranker epoch 3/3:  29%|██▊       | 185/645 [00:28<01:10,  6.53it/s]
Reranker epoch 3/3:  29%|██▉       | 186/645 [00:28<01:10,  6.53it/s]
Reranker epoch 3/3:  29%|██▉       | 187/645 [00:28<01:09,  6.61it/s]
Reranker epoch 3/3:  29%|██▉       | 188/645 [00:28<01:09,  6.58it/s]
Reranker epoch 3/3:  29%|██▉       | 189/645 [00:28<01:09,  6.60it/s]
Reranker epoch 3/3:  29%|██▉       | 190/645 [00:28<01:09,  6.56it/s]
Reranker epoch 3/3:  30%|██▉       | 191/645 [00:29<01:08,  6.60it/s]
Reranker epoch 3/3:  30%|██▉       | 192/645 [00:29<01:08,  6.58it/s]
Reranker epoch 3/3:  30%|██▉       | 193/645 [00:29<01:08,  6.61it/s]
Reranker epoch 3/3:  30%|███       | 194/645 [00:29<01:08,  6.58it/s]
Reranker epoch 3/3:  30%|███       | 195/645 [00:29<01:08,  6.60it/s]
Reranker epoch 3/3:  30%|███       | 196/645 [00:29<01:07,  6.61it/s]
Reranker epoch 3/3:  31%|███       | 197/645 [00:30<01:08,  6.57it/s]
Reranker epoch 3/3:  31%|███       | 198/645 [00:30<01:08,  6.55it/s]
Reranker epoch 3/3:  31%|███       | 199/645 [00:30<01:07,  6.59it/s]
Reranker epoch 3/3:  31%|███       | 200/645 [00:30<01:06,  6.68it/s]
Reranker epoch 3/3:  31%|███       | 201/645 [00:30<01:07,  6.62it/s]
Reranker epoch 3/3:  31%|███▏      | 202/645 [00:30<01:07,  6.58it/s]
Reranker epoch 3/3:  31%|███▏      | 203/645 [00:30<01:07,  6.57it/s]
Reranker epoch 3/3:  32%|███▏      | 204/645 [00:31<01:07,  6.50it/s]
Reranker epoch 3/3:  32%|███▏      | 205/645 [00:31<01:07,  6.54it/s]
Reranker epoch 3/3:  32%|███▏      | 206/645 [00:31<01:06,  6.55it/s]
Reranker epoch 3/3:  32%|███▏      | 207/645 [00:31<01:07,  6.53it/s]
Reranker epoch 3/3:  32%|███▏      | 208/645 [00:31<01:06,  6.54it/s]
Reranker epoch 3/3:  32%|███▏      | 209/645 [00:31<01:06,  6.55it/s]
Reranker epoch 3/3:  33%|███▎      | 210/645 [00:32<01:05,  6.64it/s]
Reranker epoch 3/3:  33%|███▎      | 211/645 [00:32<01:05,  6.62it/s]
Reranker epoch 3/3:  33%|███▎      | 212/645 [00:32<01:05,  6.61it/s]
Reranker epoch 3/3:  33%|███▎      | 213/645 [00:32<01:05,  6.61it/s]
Reranker epoch 3/3:  33%|███▎      | 214/645 [00:32<01:05,  6.56it/s]
Reranker epoch 3/3:  33%|███▎      | 215/645 [00:32<01:05,  6.54it/s]
Reranker epoch 3/3:  33%|███▎      | 216/645 [00:32<01:04,  6.63it/s]
Reranker epoch 3/3:  34%|███▎      | 217/645 [00:33<01:04,  6.59it/s]
Reranker epoch 3/3:  34%|███▍      | 218/645 [00:33<01:04,  6.58it/s]
Reranker epoch 3/3:  34%|███▍      | 219/645 [00:33<01:04,  6.58it/s]
Reranker epoch 3/3:  34%|███▍      | 220/645 [00:33<01:04,  6.59it/s]
Reranker epoch 3/3:  34%|███▍      | 221/645 [00:33<01:04,  6.60it/s]
Reranker epoch 3/3:  34%|███▍      | 222/645 [00:33<01:04,  6.60it/s]
Reranker epoch 3/3:  35%|███▍      | 223/645 [00:33<01:03,  6.59it/s]
Reranker epoch 3/3:  35%|███▍      | 224/645 [00:34<01:03,  6.62it/s]
Reranker epoch 3/3:  35%|███▍      | 225/645 [00:34<01:03,  6.63it/s]
Reranker epoch 3/3:  35%|███▌      | 226/645 [00:34<01:03,  6.63it/s]
Reranker epoch 3/3:  35%|███▌      | 227/645 [00:34<01:03,  6.59it/s]
Reranker epoch 3/3:  35%|███▌      | 228/645 [00:34<01:02,  6.63it/s]
Reranker epoch 3/3:  36%|███▌      | 229/645 [00:34<01:02,  6.66it/s]
Reranker epoch 3/3:  36%|███▌      | 230/645 [00:35<01:02,  6.65it/s]
Reranker epoch 3/3:  36%|███▌      | 231/645 [00:35<01:02,  6.62it/s]
Reranker epoch 3/3:  36%|███▌      | 232/645 [00:35<01:02,  6.63it/s]
Reranker epoch 3/3:  36%|███▌      | 233/645 [00:35<01:02,  6.60it/s]
Reranker epoch 3/3:  36%|███▋      | 234/645 [00:35<01:02,  6.59it/s]
Reranker epoch 3/3:  36%|███▋      | 235/645 [00:35<01:01,  6.66it/s]
Reranker epoch 3/3:  37%|███▋      | 236/645 [00:35<01:01,  6.64it/s]
Reranker epoch 3/3:  37%|███▋      | 237/645 [00:36<01:01,  6.63it/s]
Reranker epoch 3/3:  37%|███▋      | 238/645 [00:36<01:01,  6.63it/s]
Reranker epoch 3/3:  37%|███▋      | 239/645 [00:36<01:01,  6.61it/s]
Reranker epoch 3/3:  37%|███▋      | 240/645 [00:36<01:01,  6.61it/s]
Reranker epoch 3/3:  37%|███▋      | 241/645 [00:36<01:01,  6.56it/s]
Reranker epoch 3/3:  38%|███▊      | 242/645 [00:36<01:01,  6.59it/s]
Reranker epoch 3/3:  38%|███▊      | 243/645 [00:37<01:01,  6.57it/s]
Reranker epoch 3/3:  38%|███▊      | 244/645 [00:37<01:01,  6.56it/s]
Reranker epoch 3/3:  38%|███▊      | 245/645 [00:37<01:00,  6.58it/s]
Reranker epoch 3/3:  38%|███▊      | 246/645 [00:37<01:00,  6.57it/s]
Reranker epoch 3/3:  38%|███▊      | 247/645 [00:37<00:59,  6.65it/s]
Reranker epoch 3/3:  38%|███▊      | 248/645 [00:37<01:00,  6.61it/s]
Reranker epoch 3/3:  39%|███▊      | 249/645 [00:37<00:59,  6.63it/s]
Reranker epoch 3/3:  39%|███▉      | 250/645 [00:38<00:59,  6.59it/s]
Reranker epoch 3/3:  39%|███▉      | 251/645 [00:38<01:00,  6.54it/s]
Reranker epoch 3/3:  39%|███▉      | 252/645 [00:38<01:00,  6.51it/s]
Reranker epoch 3/3:  39%|███▉      | 253/645 [00:38<00:59,  6.61it/s]
Reranker epoch 3/3:  39%|███▉      | 254/645 [00:38<00:59,  6.55it/s]
Reranker epoch 3/3:  40%|███▉      | 255/645 [00:38<00:59,  6.58it/s]
Reranker epoch 3/3:  40%|███▉      | 256/645 [00:38<00:59,  6.55it/s]
Reranker epoch 3/3:  40%|███▉      | 257/645 [00:39<00:58,  6.58it/s]
Reranker epoch 3/3:  40%|████      | 258/645 [00:39<00:58,  6.64it/s]
Reranker epoch 3/3:  40%|████      | 259/645 [00:39<00:57,  6.69it/s]
Reranker epoch 3/3:  40%|████      | 260/645 [00:39<00:58,  6.63it/s]
Reranker epoch 3/3:  40%|████      | 261/645 [00:39<00:57,  6.63it/s]
Reranker epoch 3/3:  41%|████      | 262/645 [00:39<00:57,  6.64it/s]
Reranker epoch 3/3:  41%|████      | 263/645 [00:40<00:57,  6.63it/s]
Reranker epoch 3/3:  41%|████      | 264/645 [00:40<00:57,  6.59it/s]
Reranker epoch 3/3:  41%|████      | 265/645 [00:40<00:57,  6.66it/s]
Reranker epoch 3/3:  41%|████      | 266/645 [00:40<00:57,  6.64it/s]
Reranker epoch 3/3:  41%|████▏     | 267/645 [00:40<00:56,  6.68it/s]
Reranker epoch 3/3:  42%|████▏     | 268/645 [00:40<00:56,  6.66it/s]
Reranker epoch 3/3:  42%|████▏     | 269/645 [00:40<00:56,  6.66it/s]
Reranker epoch 3/3:  42%|████▏     | 270/645 [00:41<00:56,  6.61it/s]
Reranker epoch 3/3:  42%|████▏     | 271/645 [00:41<00:56,  6.59it/s]
Reranker epoch 3/3:  42%|████▏     | 272/645 [00:41<00:55,  6.67it/s]
Reranker epoch 3/3:  42%|████▏     | 273/645 [00:41<00:55,  6.69it/s]
Reranker epoch 3/3:  42%|████▏     | 274/645 [00:41<00:55,  6.65it/s]
Reranker epoch 3/3:  43%|████▎     | 275/645 [00:41<00:55,  6.63it/s]
Reranker epoch 3/3:  43%|████▎     | 276/645 [00:42<00:55,  6.61it/s]
Reranker epoch 3/3:  43%|████▎     | 277/645 [00:42<00:55,  6.61it/s]
Reranker epoch 3/3:  43%|████▎     | 278/645 [00:42<00:55,  6.59it/s]
Reranker epoch 3/3:  43%|████▎     | 279/645 [00:42<00:55,  6.58it/s]
Reranker epoch 3/3:  43%|████▎     | 280/645 [00:42<00:55,  6.63it/s]
Reranker epoch 3/3:  44%|████▎     | 281/645 [00:42<00:55,  6.60it/s]
Reranker epoch 3/3:  44%|████▎     | 282/645 [00:42<00:55,  6.57it/s]
Reranker epoch 3/3:  44%|████▍     | 283/645 [00:43<00:55,  6.56it/s]
Reranker epoch 3/3:  44%|████▍     | 284/645 [00:43<00:55,  6.54it/s]
Reranker epoch 3/3:  44%|████▍     | 285/645 [00:43<00:55,  6.47it/s]
Reranker epoch 3/3:  44%|████▍     | 286/645 [00:43<00:55,  6.45it/s]
Reranker epoch 3/3:  44%|████▍     | 287/645 [00:43<00:55,  6.45it/s]
Reranker epoch 3/3:  45%|████▍     | 288/645 [00:43<00:55,  6.44it/s]
Reranker epoch 3/3:  45%|████▍     | 289/645 [00:44<00:54,  6.52it/s]
Reranker epoch 3/3:  45%|████▍     | 290/645 [00:44<00:54,  6.49it/s]
Reranker epoch 3/3:  45%|████▌     | 291/645 [00:44<00:55,  6.43it/s]
Reranker epoch 3/3:  45%|████▌     | 292/645 [00:44<00:54,  6.42it/s]
Reranker epoch 3/3:  45%|████▌     | 293/645 [00:44<00:54,  6.45it/s]
Reranker epoch 3/3:  46%|████▌     | 294/645 [00:44<00:54,  6.45it/s]
Reranker epoch 3/3:  46%|████▌     | 295/645 [00:44<00:53,  6.52it/s]
Reranker epoch 3/3:  46%|████▌     | 296/645 [00:45<00:53,  6.51it/s]
Reranker epoch 3/3:  46%|████▌     | 297/645 [00:45<00:53,  6.53it/s]
Reranker epoch 3/3:  46%|████▌     | 298/645 [00:45<00:53,  6.52it/s]
Reranker epoch 3/3:  46%|████▋     | 299/645 [00:45<00:52,  6.62it/s]
Reranker epoch 3/3:  47%|████▋     | 300/645 [00:45<00:52,  6.62it/s]
Reranker epoch 3/3:  47%|████▋     | 301/645 [00:45<00:52,  6.61it/s]
Reranker epoch 3/3:  47%|████▋     | 302/645 [00:45<00:52,  6.59it/s]
Reranker epoch 3/3:  47%|████▋     | 303/645 [00:46<00:51,  6.61it/s]
Reranker epoch 3/3:  47%|████▋     | 304/645 [00:46<00:51,  6.62it/s]
Reranker epoch 3/3:  47%|████▋     | 305/645 [00:46<00:51,  6.66it/s]
Reranker epoch 3/3:  47%|████▋     | 306/645 [00:46<00:51,  6.64it/s]
Reranker epoch 3/3:  48%|████▊     | 307/645 [00:46<00:50,  6.65it/s]
Reranker epoch 3/3:  48%|████▊     | 308/645 [00:46<00:50,  6.66it/s]
Reranker epoch 3/3:  48%|████▊     | 309/645 [00:47<00:50,  6.62it/s]
Reranker epoch 3/3:  48%|████▊     | 310/645 [00:47<00:50,  6.64it/s]
Reranker epoch 3/3:  48%|████▊     | 311/645 [00:47<00:50,  6.66it/s]
Reranker epoch 3/3:  48%|████▊     | 312/645 [00:47<00:49,  6.72it/s]
Reranker epoch 3/3:  49%|████▊     | 313/645 [00:47<00:49,  6.71it/s]
Reranker epoch 3/3:  49%|████▊     | 314/645 [00:47<00:50,  6.62it/s]
Reranker epoch 3/3:  49%|████▉     | 315/645 [00:47<00:49,  6.65it/s]
Reranker epoch 3/3:  49%|████▉     | 316/645 [00:48<00:50,  6.56it/s]
Reranker epoch 3/3:  49%|████▉     | 317/645 [00:48<00:50,  6.55it/s]
Reranker epoch 3/3:  49%|████▉     | 318/645 [00:48<00:49,  6.61it/s]
Reranker epoch 3/3:  49%|████▉     | 319/645 [00:48<00:49,  6.59it/s]
Reranker epoch 3/3:  50%|████▉     | 320/645 [00:48<00:49,  6.57it/s]
Reranker epoch 3/3:  50%|████▉     | 321/645 [00:48<00:49,  6.56it/s]
Reranker epoch 3/3:  50%|████▉     | 322/645 [00:49<00:49,  6.57it/s]
Reranker epoch 3/3:  50%|█████     | 323/645 [00:49<00:48,  6.59it/s]
Reranker epoch 3/3:  50%|█████     | 324/645 [00:49<00:48,  6.64it/s]
Reranker epoch 3/3:  50%|█████     | 325/645 [00:49<00:47,  6.68it/s]
Reranker epoch 3/3:  51%|█████     | 326/645 [00:49<00:47,  6.65it/s]
Reranker epoch 3/3:  51%|█████     | 327/645 [00:49<00:48,  6.62it/s]
Reranker epoch 3/3:  51%|█████     | 328/645 [00:49<00:47,  6.61it/s]
Reranker epoch 3/3:  51%|█████     | 329/645 [00:50<00:47,  6.61it/s]
Reranker epoch 3/3:  51%|█████     | 330/645 [00:50<00:47,  6.63it/s]
Reranker epoch 3/3:  51%|█████▏    | 331/645 [00:50<00:47,  6.63it/s]
Reranker epoch 3/3:  51%|█████▏    | 332/645 [00:50<00:46,  6.68it/s]
Reranker epoch 3/3:  52%|█████▏    | 333/645 [00:50<00:47,  6.62it/s]
Reranker epoch 3/3:  52%|█████▏    | 334/645 [00:50<00:47,  6.60it/s]
Reranker epoch 3/3:  52%|█████▏    | 335/645 [00:50<00:46,  6.61it/s]
Reranker epoch 3/3:  52%|█████▏    | 336/645 [00:51<00:46,  6.60it/s]
Reranker epoch 3/3:  52%|█████▏    | 337/645 [00:51<00:46,  6.62it/s]
Reranker epoch 3/3:  52%|█████▏    | 338/645 [00:51<00:46,  6.60it/s]
Reranker epoch 3/3:  53%|█████▎    | 339/645 [00:51<00:46,  6.57it/s]
Reranker epoch 3/3:  53%|█████▎    | 340/645 [00:51<00:46,  6.58it/s]
Reranker epoch 3/3:  53%|█████▎    | 341/645 [00:51<00:46,  6.59it/s]
Reranker epoch 3/3:  53%|█████▎    | 342/645 [00:52<00:45,  6.60it/s]
Reranker epoch 3/3:  53%|█████▎    | 343/645 [00:52<00:45,  6.65it/s]
Reranker epoch 3/3:  53%|█████▎    | 344/645 [00:52<00:44,  6.71it/s]
Reranker epoch 3/3:  53%|█████▎    | 345/645 [00:52<00:44,  6.67it/s]
Reranker epoch 3/3:  54%|█████▎    | 346/645 [00:52<00:44,  6.66it/s]
Reranker epoch 3/3:  54%|█████▍    | 347/645 [00:52<00:45,  6.60it/s]
Reranker epoch 3/3:  54%|█████▍    | 348/645 [00:52<00:45,  6.59it/s]
Reranker epoch 3/3:  54%|█████▍    | 349/645 [00:53<00:44,  6.59it/s]
Reranker epoch 3/3:  54%|█████▍    | 350/645 [00:53<00:45,  6.55it/s]
Reranker epoch 3/3:  54%|█████▍    | 351/645 [00:53<00:44,  6.54it/s]
Reranker epoch 3/3:  55%|█████▍    | 352/645 [00:53<00:44,  6.57it/s]
Reranker epoch 3/3:  55%|█████▍    | 353/645 [00:53<00:44,  6.50it/s]
Reranker epoch 3/3:  55%|█████▍    | 354/645 [00:53<00:44,  6.47it/s]
Reranker epoch 3/3:  55%|█████▌    | 355/645 [00:54<00:44,  6.54it/s]
Reranker epoch 3/3:  55%|█████▌    | 356/645 [00:54<00:44,  6.49it/s]
Reranker epoch 3/3:  55%|█████▌    | 357/645 [00:54<00:43,  6.59it/s]
Reranker epoch 3/3:  56%|█████▌    | 358/645 [00:54<00:44,  6.52it/s]
Reranker epoch 3/3:  56%|█████▌    | 359/645 [00:54<00:44,  6.48it/s]
Reranker epoch 3/3:  56%|█████▌    | 360/645 [00:54<00:43,  6.50it/s]
Reranker epoch 3/3:  56%|█████▌    | 361/645 [00:54<00:43,  6.51it/s]
Reranker epoch 3/3:  56%|█████▌    | 362/645 [00:55<00:43,  6.55it/s]
Reranker epoch 3/3:  56%|█████▋    | 363/645 [00:55<00:43,  6.48it/s]
Reranker epoch 3/3:  56%|█████▋    | 364/645 [00:55<00:43,  6.51it/s]
Reranker epoch 3/3:  57%|█████▋    | 365/645 [00:55<00:42,  6.56it/s]
Reranker epoch 3/3:  57%|█████▋    | 366/645 [00:55<00:41,  6.65it/s]
Reranker epoch 3/3:  57%|█████▋    | 367/645 [00:55<00:42,  6.62it/s]
Reranker epoch 3/3:  57%|█████▋    | 368/645 [00:55<00:41,  6.63it/s]
Reranker epoch 3/3:  57%|█████▋    | 369/645 [00:56<00:41,  6.66it/s]
Reranker epoch 3/3:  57%|█████▋    | 370/645 [00:56<00:41,  6.61it/s]
Reranker epoch 3/3:  58%|█████▊    | 371/645 [00:56<00:41,  6.59it/s]
Reranker epoch 3/3:  58%|█████▊    | 372/645 [00:56<00:41,  6.58it/s]
Reranker epoch 3/3:  58%|█████▊    | 373/645 [00:56<00:41,  6.62it/s]
Reranker epoch 3/3:  58%|█████▊    | 374/645 [00:56<00:40,  6.65it/s]
Reranker epoch 3/3:  58%|█████▊    | 375/645 [00:57<00:40,  6.64it/s]
Reranker epoch 3/3:  58%|█████▊    | 376/645 [00:57<00:40,  6.63it/s]
Reranker epoch 3/3:  58%|█████▊    | 377/645 [00:57<00:40,  6.62it/s]
Reranker epoch 3/3:  59%|█████▊    | 378/645 [00:57<00:39,  6.70it/s]
Reranker epoch 3/3:  59%|█████▉    | 379/645 [00:57<00:40,  6.64it/s]
Reranker epoch 3/3:  59%|█████▉    | 380/645 [00:57<00:39,  6.66it/s]
Reranker epoch 3/3:  59%|█████▉    | 381/645 [00:57<00:39,  6.63it/s]
Reranker epoch 3/3:  59%|█████▉    | 382/645 [00:58<00:39,  6.59it/s]
Reranker epoch 3/3:  59%|█████▉    | 383/645 [00:58<00:39,  6.60it/s]
Reranker epoch 3/3:  60%|█████▉    | 384/645 [00:58<00:39,  6.57it/s]
Reranker epoch 3/3:  60%|█████▉    | 385/645 [00:58<00:39,  6.57it/s]
Reranker epoch 3/3:  60%|█████▉    | 386/645 [00:58<00:39,  6.57it/s]
Reranker epoch 3/3:  60%|██████    | 387/645 [00:58<00:39,  6.55it/s]
Reranker epoch 3/3:  60%|██████    | 388/645 [00:59<00:39,  6.56it/s]
Reranker epoch 3/3:  60%|██████    | 389/645 [00:59<00:39,  6.55it/s]
Reranker epoch 3/3:  60%|██████    | 390/645 [00:59<00:38,  6.63it/s]
Reranker epoch 3/3:  61%|██████    | 391/645 [00:59<00:38,  6.61it/s]
Reranker epoch 3/3:  61%|██████    | 392/645 [00:59<00:38,  6.58it/s]
Reranker epoch 3/3:  61%|██████    | 393/645 [00:59<00:38,  6.59it/s]
Reranker epoch 3/3:  61%|██████    | 394/645 [00:59<00:38,  6.60it/s]
Reranker epoch 3/3:  61%|██████    | 395/645 [01:00<00:38,  6.51it/s]
Reranker epoch 3/3:  61%|██████▏   | 396/645 [01:00<00:37,  6.57it/s]
Reranker epoch 3/3:  62%|██████▏   | 397/645 [01:00<00:37,  6.56it/s]
Reranker epoch 3/3:  62%|██████▏   | 398/645 [01:00<00:37,  6.53it/s]
Reranker epoch 3/3:  62%|██████▏   | 399/645 [01:00<00:37,  6.48it/s]
Reranker epoch 3/3:  62%|██████▏   | 400/645 [01:00<00:37,  6.47it/s]
Reranker epoch 3/3:  62%|██████▏   | 401/645 [01:01<00:37,  6.46it/s]
Reranker epoch 3/3:  62%|██████▏   | 402/645 [01:01<00:37,  6.44it/s]
Reranker epoch 3/3:  62%|██████▏   | 403/645 [01:01<00:37,  6.46it/s]
Reranker epoch 3/3:  63%|██████▎   | 404/645 [01:01<00:37,  6.48it/s]
Reranker epoch 3/3:  63%|██████▎   | 405/645 [01:01<00:36,  6.57it/s]
Reranker epoch 3/3:  63%|██████▎   | 406/645 [01:01<00:36,  6.58it/s]
Reranker epoch 3/3:  63%|██████▎   | 407/645 [01:01<00:36,  6.59it/s]
Reranker epoch 3/3:  63%|██████▎   | 408/645 [01:02<00:36,  6.57it/s]
Reranker epoch 3/3:  63%|██████▎   | 409/645 [01:02<00:35,  6.59it/s]
Reranker epoch 3/3:  64%|██████▎   | 410/645 [01:02<00:35,  6.53it/s]
Reranker epoch 3/3:  64%|██████▎   | 411/645 [01:02<00:35,  6.65it/s]
Reranker epoch 3/3:  64%|██████▍   | 412/645 [01:02<00:35,  6.64it/s]
Reranker epoch 3/3:  64%|██████▍   | 413/645 [01:02<00:34,  6.63it/s]
Reranker epoch 3/3:  64%|██████▍   | 414/645 [01:02<00:35,  6.58it/s]
Reranker epoch 3/3:  64%|██████▍   | 415/645 [01:03<00:34,  6.62it/s]
Reranker epoch 3/3:  64%|██████▍   | 416/645 [01:03<00:34,  6.58it/s]
Reranker epoch 3/3:  65%|██████▍   | 417/645 [01:03<00:34,  6.54it/s]
Reranker epoch 3/3:  65%|██████▍   | 418/645 [01:03<00:34,  6.56it/s]
Reranker epoch 3/3:  65%|██████▍   | 419/645 [01:03<00:34,  6.61it/s]
Reranker epoch 3/3:  65%|██████▌   | 420/645 [01:03<00:34,  6.59it/s]
Reranker epoch 3/3:  65%|██████▌   | 421/645 [01:04<00:33,  6.60it/s]
Reranker epoch 3/3:  65%|██████▌   | 422/645 [01:04<00:33,  6.60it/s]
Reranker epoch 3/3:  66%|██████▌   | 423/645 [01:04<00:33,  6.60it/s]
Reranker epoch 3/3:  66%|██████▌   | 424/645 [01:04<00:33,  6.64it/s]
Reranker epoch 3/3:  66%|██████▌   | 425/645 [01:04<00:33,  6.59it/s]
Reranker epoch 3/3:  66%|██████▌   | 426/645 [01:04<00:33,  6.57it/s]
Reranker epoch 3/3:  66%|██████▌   | 427/645 [01:04<00:33,  6.59it/s]
Reranker epoch 3/3:  66%|██████▋   | 428/645 [01:05<00:32,  6.58it/s]
Reranker epoch 3/3:  67%|██████▋   | 429/645 [01:05<00:33,  6.54it/s]
Reranker epoch 3/3:  67%|██████▋   | 430/645 [01:05<00:32,  6.53it/s]
Reranker epoch 3/3:  67%|██████▋   | 431/645 [01:05<00:32,  6.53it/s]
Reranker epoch 3/3:  67%|██████▋   | 432/645 [01:05<00:32,  6.55it/s]
Reranker epoch 3/3:  67%|██████▋   | 433/645 [01:05<00:32,  6.51it/s]
Reranker epoch 3/3:  67%|██████▋   | 434/645 [01:06<00:32,  6.51it/s]
Reranker epoch 3/3:  67%|██████▋   | 435/645 [01:06<00:32,  6.50it/s]
Reranker epoch 3/3:  68%|██████▊   | 436/645 [01:06<00:31,  6.55it/s]
Reranker epoch 3/3:  68%|██████▊   | 437/645 [01:06<00:31,  6.59it/s]
Reranker epoch 3/3:  68%|██████▊   | 438/645 [01:06<00:31,  6.54it/s]
Reranker epoch 3/3:  68%|██████▊   | 439/645 [01:06<00:31,  6.60it/s]
Reranker epoch 3/3:  68%|██████▊   | 440/645 [01:06<00:30,  6.67it/s]
Reranker epoch 3/3:  68%|██████▊   | 441/645 [01:07<00:30,  6.70it/s]
Reranker epoch 3/3:  69%|██████▊   | 442/645 [01:07<00:30,  6.62it/s]
Reranker epoch 3/3:  69%|██████▊   | 443/645 [01:07<00:30,  6.61it/s]
Reranker epoch 3/3:  69%|██████▉   | 444/645 [01:07<00:30,  6.61it/s]
Reranker epoch 3/3:  69%|██████▉   | 445/645 [01:07<00:30,  6.57it/s]
Reranker epoch 3/3:  69%|██████▉   | 446/645 [01:07<00:30,  6.55it/s]
Reranker epoch 3/3:  69%|██████▉   | 447/645 [01:08<00:30,  6.51it/s]
Reranker epoch 3/3:  69%|██████▉   | 448/645 [01:08<00:29,  6.59it/s]
Reranker epoch 3/3:  70%|██████▉   | 449/645 [01:08<00:29,  6.63it/s]
Reranker epoch 3/3:  70%|██████▉   | 450/645 [01:08<00:29,  6.57it/s]
Reranker epoch 3/3:  70%|██████▉   | 451/645 [01:08<00:29,  6.59it/s]
Reranker epoch 3/3:  70%|███████   | 452/645 [01:08<00:29,  6.55it/s]
Reranker epoch 3/3:  70%|███████   | 453/645 [01:08<00:29,  6.54it/s]
Reranker epoch 3/3:  70%|███████   | 454/645 [01:09<00:29,  6.55it/s]
Reranker epoch 3/3:  71%|███████   | 455/645 [01:09<00:29,  6.54it/s]
Reranker epoch 3/3:  71%|███████   | 456/645 [01:09<00:28,  6.55it/s]
Reranker epoch 3/3:  71%|███████   | 457/645 [01:09<00:28,  6.58it/s]
Reranker epoch 3/3:  71%|███████   | 458/645 [01:09<00:27,  6.69it/s]
Reranker epoch 3/3:  71%|███████   | 459/645 [01:09<00:28,  6.63it/s]
Reranker epoch 3/3:  71%|███████▏  | 460/645 [01:09<00:27,  6.65it/s]
Reranker epoch 3/3:  71%|███████▏  | 461/645 [01:10<00:27,  6.64it/s]
Reranker epoch 3/3:  72%|███████▏  | 462/645 [01:10<00:27,  6.60it/s]
Reranker epoch 3/3:  72%|███████▏  | 463/645 [01:10<00:27,  6.57it/s]
Reranker epoch 3/3:  72%|███████▏  | 464/645 [01:10<00:27,  6.54it/s]
Reranker epoch 3/3:  72%|███████▏  | 465/645 [01:10<00:27,  6.63it/s]
Reranker epoch 3/3:  72%|███████▏  | 466/645 [01:10<00:27,  6.59it/s]
Reranker epoch 3/3:  72%|███████▏  | 467/645 [01:11<00:26,  6.61it/s]
Reranker epoch 3/3:  73%|███████▎  | 468/645 [01:11<00:26,  6.60it/s]
Reranker epoch 3/3:  73%|███████▎  | 469/645 [01:11<00:26,  6.58it/s]
Reranker epoch 3/3:  73%|███████▎  | 470/645 [01:11<00:26,  6.66it/s]
Reranker epoch 3/3:  73%|███████▎  | 471/645 [01:11<00:26,  6.64it/s]
Reranker epoch 3/3:  73%|███████▎  | 472/645 [01:11<00:26,  6.65it/s]
Reranker epoch 3/3:  73%|███████▎  | 473/645 [01:11<00:25,  6.70it/s]
Reranker epoch 3/3:  73%|███████▎  | 474/645 [01:12<00:25,  6.62it/s]
Reranker epoch 3/3:  74%|███████▎  | 475/645 [01:12<00:25,  6.64it/s]
Reranker epoch 3/3:  74%|███████▍  | 476/645 [01:12<00:25,  6.60it/s]
Reranker epoch 3/3:  74%|███████▍  | 477/645 [01:12<00:25,  6.64it/s]
Reranker epoch 3/3:  74%|███████▍  | 478/645 [01:12<00:24,  6.71it/s]
Reranker epoch 3/3:  74%|███████▍  | 479/645 [01:12<00:24,  6.66it/s]
Reranker epoch 3/3:  74%|███████▍  | 480/645 [01:12<00:24,  6.61it/s]
Reranker epoch 3/3:  75%|███████▍  | 481/645 [01:13<00:24,  6.58it/s]
Reranker epoch 3/3:  75%|███████▍  | 482/645 [01:13<00:24,  6.59it/s]
Reranker epoch 3/3:  75%|███████▍  | 483/645 [01:13<00:24,  6.63it/s]
Reranker epoch 3/3:  75%|███████▌  | 484/645 [01:13<00:24,  6.66it/s]
Reranker epoch 3/3:  75%|███████▌  | 485/645 [01:13<00:24,  6.64it/s]
Reranker epoch 3/3:  75%|███████▌  | 486/645 [01:13<00:24,  6.61it/s]
Reranker epoch 3/3:  76%|███████▌  | 487/645 [01:14<00:23,  6.62it/s]
Reranker epoch 3/3:  76%|███████▌  | 488/645 [01:14<00:23,  6.60it/s]
Reranker epoch 3/3:  76%|███████▌  | 489/645 [01:14<00:23,  6.61it/s]
Reranker epoch 3/3:  76%|███████▌  | 490/645 [01:14<00:23,  6.66it/s]
Reranker epoch 3/3:  76%|███████▌  | 491/645 [01:14<00:23,  6.63it/s]
Reranker epoch 3/3:  76%|███████▋  | 492/645 [01:14<00:23,  6.60it/s]
Reranker epoch 3/3:  76%|███████▋  | 493/645 [01:14<00:23,  6.57it/s]
Reranker epoch 3/3:  77%|███████▋  | 494/645 [01:15<00:22,  6.58it/s]
Reranker epoch 3/3:  77%|███████▋  | 495/645 [01:15<00:22,  6.56it/s]
Reranker epoch 3/3:  77%|███████▋  | 496/645 [01:15<00:22,  6.64it/s]
Reranker epoch 3/3:  77%|███████▋  | 497/645 [01:15<00:22,  6.62it/s]
Reranker epoch 3/3:  77%|███████▋  | 498/645 [01:15<00:22,  6.61it/s]
Reranker epoch 3/3:  77%|███████▋  | 499/645 [01:15<00:22,  6.61it/s]
Reranker epoch 3/3:  78%|███████▊  | 500/645 [01:16<00:22,  6.57it/s]
Reranker epoch 3/3:  78%|███████▊  | 501/645 [01:16<00:21,  6.55it/s]
Reranker epoch 3/3:  78%|███████▊  | 502/645 [01:16<00:21,  6.57it/s]
Reranker epoch 3/3:  78%|███████▊  | 503/645 [01:16<00:21,  6.61it/s]
Reranker epoch 3/3:  78%|███████▊  | 504/645 [01:16<00:21,  6.60it/s]
Reranker epoch 3/3:  78%|███████▊  | 505/645 [01:16<00:21,  6.57it/s]
Reranker epoch 3/3:  78%|███████▊  | 506/645 [01:16<00:21,  6.60it/s]
Reranker epoch 3/3:  79%|███████▊  | 507/645 [01:17<00:20,  6.57it/s]
Reranker epoch 3/3:  79%|███████▉  | 508/645 [01:17<00:20,  6.63it/s]
Reranker epoch 3/3:  79%|███████▉  | 509/645 [01:17<00:20,  6.63it/s]
Reranker epoch 3/3:  79%|███████▉  | 510/645 [01:17<00:20,  6.61it/s]
Reranker epoch 3/3:  79%|███████▉  | 511/645 [01:17<00:20,  6.63it/s]
Reranker epoch 3/3:  79%|███████▉  | 512/645 [01:17<00:20,  6.60it/s]
Reranker epoch 3/3:  80%|███████▉  | 513/645 [01:17<00:20,  6.54it/s]
Reranker epoch 3/3:  80%|███████▉  | 514/645 [01:18<00:19,  6.61it/s]
Reranker epoch 3/3:  80%|███████▉  | 515/645 [01:18<00:19,  6.56it/s]
Reranker epoch 3/3:  80%|████████  | 516/645 [01:18<00:19,  6.54it/s]
Reranker epoch 3/3:  80%|████████  | 517/645 [01:18<00:19,  6.52it/s]
Reranker epoch 3/3:  80%|████████  | 518/645 [01:18<00:19,  6.51it/s]
Reranker epoch 3/3:  80%|████████  | 519/645 [01:18<00:19,  6.62it/s]
Reranker epoch 3/3:  81%|████████  | 520/645 [01:19<00:19,  6.55it/s]
Reranker epoch 3/3:  81%|████████  | 521/645 [01:19<00:18,  6.58it/s]
Reranker epoch 3/3:  81%|████████  | 522/645 [01:19<00:18,  6.54it/s]
Reranker epoch 3/3:  81%|████████  | 523/645 [01:19<00:18,  6.61it/s]
Reranker epoch 3/3:  81%|████████  | 524/645 [01:19<00:18,  6.55it/s]
Reranker epoch 3/3:  81%|████████▏ | 525/645 [01:19<00:18,  6.59it/s]
Reranker epoch 3/3:  82%|████████▏ | 526/645 [01:19<00:17,  6.64it/s]
Reranker epoch 3/3:  82%|████████▏ | 527/645 [01:20<00:17,  6.61it/s]
Reranker epoch 3/3:  82%|████████▏ | 528/645 [01:20<00:17,  6.61it/s]
Reranker epoch 3/3:  82%|████████▏ | 529/645 [01:20<00:17,  6.60it/s]
Reranker epoch 3/3:  82%|████████▏ | 530/645 [01:20<00:17,  6.63it/s]
Reranker epoch 3/3:  82%|████████▏ | 531/645 [01:20<00:17,  6.65it/s]
Reranker epoch 3/3:  82%|████████▏ | 532/645 [01:20<00:16,  6.74it/s]
Reranker epoch 3/3:  83%|████████▎ | 533/645 [01:21<00:16,  6.65it/s]
Reranker epoch 3/3:  83%|████████▎ | 534/645 [01:21<00:16,  6.59it/s]
Reranker epoch 3/3:  83%|████████▎ | 535/645 [01:21<00:16,  6.55it/s]
Reranker epoch 3/3:  83%|████████▎ | 536/645 [01:21<00:16,  6.43it/s]
Reranker epoch 3/3:  83%|████████▎ | 537/645 [01:21<00:16,  6.40it/s]
Reranker epoch 3/3:  83%|████████▎ | 538/645 [01:21<00:16,  6.39it/s]
Reranker epoch 3/3:  84%|████████▎ | 539/645 [01:21<00:16,  6.40it/s]
Reranker epoch 3/3:  84%|████████▎ | 540/645 [01:22<00:16,  6.42it/s]
Reranker epoch 3/3:  84%|████████▍ | 541/645 [01:22<00:16,  6.40it/s]
Reranker epoch 3/3:  84%|████████▍ | 542/645 [01:22<00:16,  6.40it/s]
Reranker epoch 3/3:  84%|████████▍ | 543/645 [01:22<00:16,  6.32it/s]
Reranker epoch 3/3:  84%|████████▍ | 544/645 [01:22<00:16,  6.25it/s]
Reranker epoch 3/3:  84%|████████▍ | 545/645 [01:22<00:15,  6.26it/s]
Reranker epoch 3/3:  85%|████████▍ | 546/645 [01:23<00:15,  6.24it/s]
Reranker epoch 3/3:  85%|████████▍ | 547/645 [01:23<00:16,  6.08it/s]
Reranker epoch 3/3:  85%|████████▍ | 548/645 [01:23<00:15,  6.13it/s]
Reranker epoch 3/3:  85%|████████▌ | 549/645 [01:23<00:15,  6.12it/s]
Reranker epoch 3/3:  85%|████████▌ | 550/645 [01:23<00:15,  6.23it/s]
Reranker epoch 3/3:  85%|████████▌ | 551/645 [01:23<00:15,  6.14it/s]
Reranker epoch 3/3:  86%|████████▌ | 552/645 [01:24<00:15,  6.17it/s]
Reranker epoch 3/3:  86%|████████▌ | 553/645 [01:24<00:14,  6.20it/s]
Reranker epoch 3/3:  86%|████████▌ | 554/645 [01:24<00:14,  6.24it/s]
Reranker epoch 3/3:  86%|████████▌ | 555/645 [01:24<00:14,  6.28it/s]
Reranker epoch 3/3:  86%|████████▌ | 556/645 [01:24<00:14,  6.31it/s]
Reranker epoch 3/3:  86%|████████▋ | 557/645 [01:24<00:13,  6.33it/s]
Reranker epoch 3/3:  87%|████████▋ | 558/645 [01:25<00:13,  6.30it/s]
Reranker epoch 3/3:  87%|████████▋ | 559/645 [01:25<00:13,  6.34it/s]
Reranker epoch 3/3:  87%|████████▋ | 560/645 [01:25<00:13,  6.35it/s]
Reranker epoch 3/3:  87%|████████▋ | 561/645 [01:25<00:13,  6.31it/s]
Reranker epoch 3/3:  87%|████████▋ | 562/645 [01:25<00:13,  6.33it/s]
Reranker epoch 3/3:  87%|████████▋ | 563/645 [01:25<00:12,  6.31it/s]
Reranker epoch 3/3:  87%|████████▋ | 564/645 [01:25<00:12,  6.42it/s]
Reranker epoch 3/3:  88%|████████▊ | 565/645 [01:26<00:12,  6.41it/s]
Reranker epoch 3/3:  88%|████████▊ | 566/645 [01:26<00:12,  6.48it/s]
Reranker epoch 3/3:  88%|████████▊ | 567/645 [01:26<00:12,  6.44it/s]
Reranker epoch 3/3:  88%|████████▊ | 568/645 [01:26<00:11,  6.44it/s]
Reranker epoch 3/3:  88%|████████▊ | 569/645 [01:26<00:11,  6.47it/s]
Reranker epoch 3/3:  88%|████████▊ | 570/645 [01:26<00:11,  6.45it/s]
Reranker epoch 3/3:  89%|████████▊ | 571/645 [01:27<00:11,  6.45it/s]
Reranker epoch 3/3:  89%|████████▊ | 572/645 [01:27<00:11,  6.52it/s]
Reranker epoch 3/3:  89%|████████▉ | 573/645 [01:27<00:10,  6.60it/s]
Reranker epoch 3/3:  89%|████████▉ | 574/645 [01:27<00:10,  6.57it/s]
Reranker epoch 3/3:  89%|████████▉ | 575/645 [01:27<00:10,  6.57it/s]
Reranker epoch 3/3:  89%|████████▉ | 576/645 [01:27<00:10,  6.61it/s]
Reranker epoch 3/3:  89%|████████▉ | 577/645 [01:27<00:10,  6.56it/s]
Reranker epoch 3/3:  90%|████████▉ | 578/645 [01:28<00:10,  6.66it/s]
Reranker epoch 3/3:  90%|████████▉ | 579/645 [01:28<00:09,  6.60it/s]
Reranker epoch 3/3:  90%|████████▉ | 580/645 [01:28<00:09,  6.56it/s]
Reranker epoch 3/3:  90%|█████████ | 581/645 [01:28<00:09,  6.58it/s]
Reranker epoch 3/3:  90%|█████████ | 582/645 [01:28<00:09,  6.56it/s]
Reranker epoch 3/3:  90%|█████████ | 583/645 [01:28<00:09,  6.62it/s]
Reranker epoch 3/3:  91%|█████████ | 584/645 [01:28<00:09,  6.64it/s]
Reranker epoch 3/3:  91%|█████████ | 585/645 [01:29<00:08,  6.69it/s]
Reranker epoch 3/3:  91%|█████████ | 586/645 [01:29<00:08,  6.69it/s]
Reranker epoch 3/3:  91%|█████████ | 587/645 [01:29<00:08,  6.68it/s]
Reranker epoch 3/3:  91%|█████████ | 588/645 [01:29<00:08,  6.66it/s]
Reranker epoch 3/3:  91%|█████████▏| 589/645 [01:29<00:08,  6.63it/s]
Reranker epoch 3/3:  91%|█████████▏| 590/645 [01:29<00:08,  6.62it/s]
Reranker epoch 3/3:  92%|█████████▏| 591/645 [01:30<00:08,  6.67it/s]
Reranker epoch 3/3:  92%|█████████▏| 592/645 [01:30<00:07,  6.63it/s]
Reranker epoch 3/3:  92%|█████████▏| 593/645 [01:30<00:07,  6.62it/s]
Reranker epoch 3/3:  92%|█████████▏| 594/645 [01:30<00:07,  6.61it/s]
Reranker epoch 3/3:  92%|█████████▏| 595/645 [01:30<00:07,  6.61it/s]
Reranker epoch 3/3:  92%|█████████▏| 596/645 [01:30<00:07,  6.58it/s]
Reranker epoch 3/3:  93%|█████████▎| 597/645 [01:30<00:07,  6.57it/s]
Reranker epoch 3/3:  93%|█████████▎| 598/645 [01:31<00:07,  6.59it/s]
Reranker epoch 3/3:  93%|█████████▎| 599/645 [01:31<00:06,  6.58it/s]
Reranker epoch 3/3:  93%|█████████▎| 600/645 [01:31<00:06,  6.57it/s]
Reranker epoch 3/3:  93%|█████████▎| 601/645 [01:31<00:06,  6.56it/s]
Reranker epoch 3/3:  93%|█████████▎| 602/645 [01:31<00:06,  6.59it/s]
Reranker epoch 3/3:  93%|█████████▎| 603/645 [01:31<00:06,  6.63it/s]
Reranker epoch 3/3:  94%|█████████▎| 604/645 [01:32<00:06,  6.59it/s]
Reranker epoch 3/3:  94%|█████████▍| 605/645 [01:32<00:06,  6.59it/s]
Reranker epoch 3/3:  94%|█████████▍| 606/645 [01:32<00:05,  6.59it/s]
Reranker epoch 3/3:  94%|█████████▍| 607/645 [01:32<00:05,  6.58it/s]
Reranker epoch 3/3:  94%|█████████▍| 608/645 [01:32<00:05,  6.64it/s]
Reranker epoch 3/3:  94%|█████████▍| 609/645 [01:32<00:05,  6.64it/s]
Reranker epoch 3/3:  95%|█████████▍| 610/645 [01:32<00:05,  6.70it/s]
Reranker epoch 3/3:  95%|█████████▍| 611/645 [01:33<00:05,  6.65it/s]
Reranker epoch 3/3:  95%|█████████▍| 612/645 [01:33<00:04,  6.63it/s]
Reranker epoch 3/3:  95%|█████████▌| 613/645 [01:33<00:04,  6.57it/s]
Reranker epoch 3/3:  95%|█████████▌| 614/645 [01:33<00:04,  6.61it/s]
Reranker epoch 3/3:  95%|█████████▌| 615/645 [01:33<00:04,  6.59it/s]
Reranker epoch 3/3:  96%|█████████▌| 616/645 [01:33<00:04,  6.57it/s]
Reranker epoch 3/3:  96%|█████████▌| 617/645 [01:33<00:04,  6.59it/s]
Reranker epoch 3/3:  96%|█████████▌| 618/645 [01:34<00:04,  6.59it/s]
Reranker epoch 3/3:  96%|█████████▌| 619/645 [01:34<00:03,  6.57it/s]
Reranker epoch 3/3:  96%|█████████▌| 620/645 [01:34<00:03,  6.61it/s]
Reranker epoch 3/3:  96%|█████████▋| 621/645 [01:34<00:03,  6.63it/s]
Reranker epoch 3/3:  96%|█████████▋| 622/645 [01:34<00:03,  6.69it/s]
Reranker epoch 3/3:  97%|█████████▋| 623/645 [01:34<00:03,  6.61it/s]
Reranker epoch 3/3:  97%|█████████▋| 624/645 [01:35<00:03,  6.61it/s]
Reranker epoch 3/3:  97%|█████████▋| 625/645 [01:35<00:03,  6.59it/s]
Reranker epoch 3/3:  97%|█████████▋| 626/645 [01:35<00:02,  6.57it/s]
Reranker epoch 3/3:  97%|█████████▋| 627/645 [01:35<00:02,  6.58it/s]
Reranker epoch 3/3:  97%|█████████▋| 628/645 [01:35<00:02,  6.63it/s]
Reranker epoch 3/3:  98%|█████████▊| 629/645 [01:35<00:02,  6.59it/s]
Reranker epoch 3/3:  98%|█████████▊| 630/645 [01:35<00:02,  6.59it/s]
Reranker epoch 3/3:  98%|█████████▊| 631/645 [01:36<00:02,  6.60it/s]
Reranker epoch 3/3:  98%|█████████▊| 632/645 [01:36<00:01,  6.59it/s]
Reranker epoch 3/3:  98%|█████████▊| 633/645 [01:36<00:01,  6.64it/s]
Reranker epoch 3/3:  98%|█████████▊| 634/645 [01:36<00:01,  6.63it/s]
Reranker epoch 3/3:  98%|█████████▊| 635/645 [01:36<00:01,  6.62it/s]
Reranker epoch 3/3:  99%|█████████▊| 636/645 [01:36<00:01,  6.61it/s]
Reranker epoch 3/3:  99%|█████████▉| 637/645 [01:37<00:01,  6.60it/s]
Reranker epoch 3/3:  99%|█████████▉| 638/645 [01:37<00:01,  6.60it/s]
Reranker epoch 3/3:  99%|█████████▉| 639/645 [01:37<00:00,  6.61it/s]
Reranker epoch 3/3:  99%|█████████▉| 640/645 [01:37<00:00,  6.65it/s]
Reranker epoch 3/3:  99%|█████████▉| 641/645 [01:37<00:00,  6.62it/s]
Reranker epoch 3/3: 100%|█████████▉| 642/645 [01:37<00:00,  6.60it/s]
Reranker epoch 3/3: 100%|█████████▉| 643/645 [01:37<00:00,  6.61it/s]
Reranker epoch 3/3: 100%|█████████▉| 644/645 [01:38<00:00,  6.60it/s]
Reranker epoch 3/3: 100%|██████████| 645/645 [01:38<00:00,  6.58it/s]
Scoring dev_epoch3 candidates with cross-encoder reranker...

  0%|          | 0/154 [00:00<?, ?it/s]
  1%|          | 1/154 [00:00<00:15,  9.69it/s]
  1%|▏         | 2/154 [00:00<00:24,  6.17it/s]
  2%|▏         | 3/154 [00:00<00:21,  6.89it/s]
  3%|▎         | 4/154 [00:00<00:22,  6.79it/s]
  3%|▎         | 5/154 [00:00<00:20,  7.42it/s]
  4%|▍         | 6/154 [00:00<00:19,  7.72it/s]
  5%|▍         | 7/154 [00:00<00:19,  7.47it/s]
  5%|▌         | 8/154 [00:01<00:21,  6.95it/s]
  6%|▌         | 9/154 [00:01<00:24,  5.97it/s]
  6%|▋         | 10/154 [00:01<00:22,  6.44it/s]
  7%|▋         | 11/154 [00:01<00:22,  6.24it/s]
  8%|▊         | 12/154 [00:01<00:22,  6.40it/s]
  8%|▊         | 13/154 [00:01<00:21,  6.60it/s]
  9%|▉         | 14/154 [00:02<00:20,  6.70it/s]
 10%|▉         | 15/154 [00:02<00:21,  6.59it/s]
 10%|█         | 16/154 [00:02<00:20,  6.79it/s]
 11%|█         | 17/154 [00:02<00:20,  6.81it/s]
 12%|█▏        | 18/154 [00:02<00:20,  6.61it/s]
 12%|█▏        | 19/154 [00:02<00:19,  7.04it/s]
 13%|█▎        | 20/154 [00:02<00:21,  6.31it/s]
 14%|█▎        | 21/154 [00:03<00:20,  6.44it/s]
 14%|█▍        | 22/154 [00:03<00:19,  6.65it/s]
 15%|█▍        | 23/154 [00:03<00:18,  7.05it/s]
 16%|█▌        | 24/154 [00:03<00:17,  7.31it/s]
 16%|█▌        | 25/154 [00:03<00:20,  6.40it/s]
 17%|█▋        | 26/154 [00:03<00:20,  6.31it/s]
 18%|█▊        | 27/154 [00:04<00:19,  6.57it/s]
 18%|█▊        | 28/154 [00:04<00:18,  6.64it/s]
 19%|█▉        | 29/154 [00:04<00:18,  6.61it/s]
 19%|█▉        | 30/154 [00:04<00:20,  6.01it/s]
 20%|██        | 31/154 [00:04<00:20,  5.97it/s]
 21%|██        | 32/154 [00:04<00:19,  6.38it/s]
 21%|██▏       | 33/154 [00:04<00:17,  6.86it/s]
 22%|██▏       | 34/154 [00:05<00:17,  6.87it/s]
 23%|██▎       | 35/154 [00:05<00:16,  7.28it/s]
 23%|██▎       | 36/154 [00:05<00:16,  7.12it/s]
 24%|██▍       | 37/154 [00:05<00:17,  6.53it/s]
 25%|██▍       | 38/154 [00:05<00:19,  6.08it/s]
 25%|██▌       | 39/154 [00:05<00:18,  6.38it/s]
 26%|██▌       | 40/154 [00:06<00:17,  6.49it/s]
 27%|██▋       | 41/154 [00:06<00:19,  5.88it/s]
 27%|██▋       | 42/154 [00:06<00:18,  5.95it/s]
 28%|██▊       | 43/154 [00:06<00:17,  6.41it/s]
 29%|██▊       | 44/154 [00:06<00:18,  5.96it/s]
 29%|██▉       | 45/154 [00:06<00:18,  5.86it/s]
 30%|██▉       | 46/154 [00:07<00:17,  6.27it/s]
 31%|███       | 47/154 [00:07<00:16,  6.59it/s]
 31%|███       | 48/154 [00:07<00:17,  6.22it/s]
 32%|███▏      | 49/154 [00:07<00:17,  5.94it/s]
 32%|███▏      | 50/154 [00:07<00:17,  5.97it/s]
 33%|███▎      | 51/154 [00:07<00:16,  6.26it/s]
 34%|███▍      | 52/154 [00:07<00:15,  6.47it/s]
 34%|███▍      | 53/154 [00:08<00:16,  6.05it/s]
 35%|███▌      | 54/154 [00:08<00:15,  6.29it/s]
 36%|███▌      | 55/154 [00:08<00:14,  6.68it/s]
 36%|███▋      | 56/154 [00:08<00:14,  6.66it/s]
 37%|███▋      | 57/154 [00:08<00:15,  6.23it/s]
 38%|███▊      | 58/154 [00:08<00:14,  6.43it/s]
 38%|███▊      | 59/154 [00:09<00:15,  6.11it/s]
 39%|███▉      | 60/154 [00:09<00:17,  5.53it/s]
 40%|███▉      | 61/154 [00:09<00:18,  5.02it/s]
 40%|████      | 62/154 [00:09<00:17,  5.25it/s]
 41%|████      | 63/154 [00:09<00:16,  5.52it/s]
 42%|████▏     | 64/154 [00:10<00:16,  5.62it/s]
 42%|████▏     | 65/154 [00:10<00:15,  5.90it/s]
 43%|████▎     | 66/154 [00:10<00:14,  6.00it/s]
 44%|████▎     | 67/154 [00:10<00:14,  5.81it/s]
 44%|████▍     | 68/154 [00:10<00:13,  6.28it/s]
 45%|████▍     | 69/154 [00:10<00:14,  5.92it/s]
 45%|████▌     | 70/154 [00:11<00:15,  5.41it/s]
 46%|████▌     | 71/154 [00:11<00:14,  5.79it/s]
 47%|████▋     | 72/154 [00:11<00:13,  6.16it/s]
 47%|████▋     | 73/154 [00:11<00:12,  6.34it/s]
 48%|████▊     | 74/154 [00:11<00:11,  7.04it/s]
 49%|████▊     | 75/154 [00:11<00:11,  6.77it/s]
 49%|████▉     | 76/154 [00:11<00:10,  7.11it/s]
 50%|█████     | 77/154 [00:12<00:10,  7.51it/s]
 51%|█████     | 78/154 [00:12<00:11,  6.63it/s]
 51%|█████▏    | 79/154 [00:12<00:11,  6.31it/s]
 52%|█████▏    | 80/154 [00:12<00:12,  5.94it/s]
 53%|█████▎    | 81/154 [00:12<00:11,  6.40it/s]
 53%|█████▎    | 82/154 [00:12<00:10,  6.55it/s]
 54%|█████▍    | 83/154 [00:13<00:11,  6.26it/s]
 55%|█████▍    | 84/154 [00:13<00:11,  6.14it/s]
 55%|█████▌    | 85/154 [00:13<00:11,  6.02it/s]
 56%|█████▌    | 86/154 [00:13<00:10,  6.24it/s]
 56%|█████▋    | 87/154 [00:13<00:11,  5.68it/s]
 57%|█████▋    | 88/154 [00:13<00:11,  5.64it/s]
 58%|█████▊    | 89/154 [00:14<00:10,  6.03it/s]
 58%|█████▊    | 90/154 [00:14<00:10,  6.13it/s]
 59%|█████▉    | 91/154 [00:14<00:09,  6.41it/s]
 60%|█████▉    | 92/154 [00:14<00:08,  7.10it/s]
 60%|██████    | 93/154 [00:14<00:08,  7.03it/s]
 61%|██████    | 94/154 [00:14<00:08,  7.05it/s]
 62%|██████▏   | 95/154 [00:14<00:08,  7.27it/s]
 62%|██████▏   | 96/154 [00:15<00:08,  7.03it/s]
 63%|██████▎   | 97/154 [00:15<00:08,  6.90it/s]
 64%|██████▎   | 98/154 [00:15<00:08,  6.61it/s]
 64%|██████▍   | 99/154 [00:15<00:08,  6.29it/s]
 65%|██████▍   | 100/154 [00:15<00:07,  6.76it/s]
 66%|██████▌   | 101/154 [00:15<00:07,  6.65it/s]
 66%|██████▌   | 102/154 [00:15<00:07,  6.68it/s]
 67%|██████▋   | 103/154 [00:16<00:07,  6.85it/s]
 68%|██████▊   | 104/154 [00:16<00:07,  6.99it/s]
 68%|██████▊   | 105/154 [00:16<00:07,  6.73it/s]
 69%|██████▉   | 106/154 [00:16<00:07,  6.73it/s]
 69%|██████▉   | 107/154 [00:16<00:07,  6.48it/s]
 70%|███████   | 108/154 [00:16<00:07,  6.53it/s]
 71%|███████   | 109/154 [00:17<00:08,  5.30it/s]
 71%|███████▏  | 110/154 [00:17<00:07,  5.77it/s]
 72%|███████▏  | 111/154 [00:17<00:07,  5.79it/s]
 73%|███████▎  | 112/154 [00:17<00:07,  5.79it/s]
 73%|███████▎  | 113/154 [00:17<00:06,  6.30it/s]
 74%|███████▍  | 114/154 [00:17<00:06,  6.52it/s]
 75%|███████▍  | 115/154 [00:18<00:06,  6.41it/s]
 75%|███████▌  | 116/154 [00:18<00:06,  5.82it/s]
 76%|███████▌  | 117/154 [00:18<00:07,  5.10it/s]
 77%|███████▋  | 118/154 [00:18<00:06,  5.52it/s]
 77%|███████▋  | 119/154 [00:18<00:05,  5.92it/s]
 78%|███████▊  | 120/154 [00:19<00:06,  5.48it/s]
 79%|███████▊  | 121/154 [00:19<00:05,  6.07it/s]
 79%|███████▉  | 122/154 [00:19<00:05,  6.16it/s]
 80%|███████▉  | 123/154 [00:19<00:05,  5.73it/s]
 81%|████████  | 124/154 [00:19<00:04,  6.12it/s]
 81%|████████  | 125/154 [00:19<00:04,  6.17it/s]
 82%|████████▏ | 126/154 [00:19<00:04,  6.41it/s]
 82%|████████▏ | 127/154 [00:20<00:04,  5.99it/s]
 83%|████████▎ | 128/154 [00:20<00:04,  5.89it/s]
 84%|████████▍ | 129/154 [00:20<00:03,  6.30it/s]
 84%|████████▍ | 130/154 [00:20<00:03,  6.42it/s]
 85%|████████▌ | 131/154 [00:20<00:03,  6.48it/s]
 86%|████████▌ | 132/154 [00:20<00:03,  6.70it/s]
 86%|████████▋ | 133/154 [00:21<00:03,  6.26it/s]
 87%|████████▋ | 134/154 [00:21<00:03,  6.40it/s]
 88%|████████▊ | 135/154 [00:21<00:02,  6.59it/s]
 88%|████████▊ | 136/154 [00:21<00:02,  6.74it/s]
 89%|████████▉ | 137/154 [00:21<00:02,  6.84it/s]
 90%|████████▉ | 138/154 [00:21<00:02,  6.65it/s]
 90%|█████████ | 139/154 [00:21<00:02,  7.08it/s]
 91%|█████████ | 140/154 [00:22<00:01,  7.73it/s]
 92%|█████████▏| 141/154 [00:22<00:01,  7.12it/s]
 92%|█████████▏| 142/154 [00:22<00:01,  7.53it/s]
 93%|█████████▎| 143/154 [00:22<00:01,  6.78it/s]
 94%|█████████▎| 144/154 [00:22<00:01,  6.60it/s]
 94%|█████████▍| 145/154 [00:22<00:01,  6.62it/s]
 95%|█████████▍| 146/154 [00:22<00:01,  6.33it/s]
 95%|█████████▌| 147/154 [00:23<00:01,  6.00it/s]
 96%|█████████▌| 148/154 [00:23<00:01,  6.00it/s]
 97%|█████████▋| 149/154 [00:23<00:00,  6.29it/s]
 97%|█████████▋| 150/154 [00:23<00:00,  6.01it/s]
 98%|█████████▊| 151/154 [00:23<00:00,  6.29it/s]
 99%|█████████▊| 152/154 [00:23<00:00,  6.85it/s]
 99%|█████████▉| 153/154 [00:24<00:00,  5.61it/s]
100%|██████████| 154/154 [00:24<00:00,  5.90it/s]
100%|██████████| 154/154 [00:24<00:00,  6.34it/s]
Final retrieval policy: prefer_dynamic
Best dev row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Chosen row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Epoch 3: loss=0.2289 | chosen dev retrieval_F=0.2557 | setting={'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714} | time=123.0s
Best reranker retrieval setting under final policy:
{'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Scoring dev_final candidates with cross-encoder reranker...

  0%|          | 0/154 [00:00<?, ?it/s]
  1%|          | 1/154 [00:00<00:15,  9.63it/s]
  1%|▏         | 2/154 [00:00<00:25,  6.07it/s]
  2%|▏         | 3/154 [00:00<00:22,  6.78it/s]
  3%|▎         | 4/154 [00:00<00:22,  6.72it/s]
  3%|▎         | 5/154 [00:00<00:20,  7.33it/s]
  4%|▍         | 6/154 [00:00<00:19,  7.64it/s]
  5%|▍         | 7/154 [00:00<00:20,  7.35it/s]
  5%|▌         | 8/154 [00:01<00:21,  6.94it/s]
  6%|▌         | 9/154 [00:01<00:24,  5.95it/s]
  6%|▋         | 10/154 [00:01<00:22,  6.39it/s]
  7%|▋         | 11/154 [00:01<00:23,  6.16it/s]
  8%|▊         | 12/154 [00:01<00:22,  6.32it/s]
  8%|▊         | 13/154 [00:01<00:21,  6.52it/s]
  9%|▉         | 14/154 [00:02<00:21,  6.66it/s]
 10%|▉         | 15/154 [00:02<00:21,  6.52it/s]
 10%|█         | 16/154 [00:02<00:20,  6.74it/s]
 11%|█         | 17/154 [00:02<00:19,  6.86it/s]
 12%|█▏        | 18/154 [00:02<00:20,  6.68it/s]
 12%|█▏        | 19/154 [00:02<00:18,  7.13it/s]
 13%|█▎        | 20/154 [00:03<00:21,  6.34it/s]
 14%|█▎        | 21/154 [00:03<00:20,  6.42it/s]
 14%|█▍        | 22/154 [00:03<00:19,  6.66it/s]
 15%|█▍        | 23/154 [00:03<00:18,  7.14it/s]
 16%|█▌        | 24/154 [00:03<00:17,  7.40it/s]
 16%|█▌        | 25/154 [00:03<00:19,  6.48it/s]
 17%|█▋        | 26/154 [00:03<00:19,  6.41it/s]
 18%|█▊        | 27/154 [00:04<00:18,  6.71it/s]
 18%|█▊        | 28/154 [00:04<00:18,  6.76it/s]
 19%|█▉        | 29/154 [00:04<00:18,  6.71it/s]
 19%|█▉        | 30/154 [00:04<00:20,  6.10it/s]
 20%|██        | 31/154 [00:04<00:20,  6.00it/s]
 21%|██        | 32/154 [00:04<00:19,  6.39it/s]
 21%|██▏       | 33/154 [00:04<00:17,  6.81it/s]
 22%|██▏       | 34/154 [00:05<00:17,  6.82it/s]
 23%|██▎       | 35/154 [00:05<00:16,  7.16it/s]
 23%|██▎       | 36/154 [00:05<00:16,  6.94it/s]
 24%|██▍       | 37/154 [00:05<00:18,  6.40it/s]
 25%|██▍       | 38/154 [00:05<00:19,  5.95it/s]
 25%|██▌       | 39/154 [00:05<00:18,  6.27it/s]
 26%|██▌       | 40/154 [00:06<00:18,  6.27it/s]
 27%|██▋       | 41/154 [00:06<00:19,  5.72it/s]
 27%|██▋       | 42/154 [00:06<00:19,  5.83it/s]
 28%|██▊       | 43/154 [00:06<00:17,  6.33it/s]
 29%|██▊       | 44/154 [00:06<00:18,  5.91it/s]
 29%|██▉       | 45/154 [00:06<00:18,  5.82it/s]
 30%|██▉       | 46/154 [00:07<00:17,  6.25it/s]
 31%|███       | 47/154 [00:07<00:16,  6.58it/s]
 31%|███       | 48/154 [00:07<00:17,  6.18it/s]
 32%|███▏      | 49/154 [00:07<00:17,  5.95it/s]
 32%|███▏      | 50/154 [00:07<00:17,  5.98it/s]
 33%|███▎      | 51/154 [00:07<00:16,  6.25it/s]
 34%|███▍      | 52/154 [00:08<00:15,  6.44it/s]
 34%|███▍      | 53/154 [00:08<00:16,  6.01it/s]
 35%|███▌      | 54/154 [00:08<00:15,  6.25it/s]
 36%|███▌      | 55/154 [00:08<00:14,  6.63it/s]
 36%|███▋      | 56/154 [00:08<00:14,  6.62it/s]
 37%|███▋      | 57/154 [00:08<00:15,  6.24it/s]
 38%|███▊      | 58/154 [00:08<00:14,  6.45it/s]
 38%|███▊      | 59/154 [00:09<00:15,  6.14it/s]
 39%|███▉      | 60/154 [00:09<00:17,  5.52it/s]
 40%|███▉      | 61/154 [00:09<00:18,  5.04it/s]
 40%|████      | 62/154 [00:09<00:17,  5.25it/s]
 41%|████      | 63/154 [00:09<00:16,  5.52it/s]
 42%|████▏     | 64/154 [00:10<00:16,  5.59it/s]
 42%|████▏     | 65/154 [00:10<00:15,  5.91it/s]
 43%|████▎     | 66/154 [00:10<00:14,  5.98it/s]
 44%|████▎     | 67/154 [00:10<00:14,  5.84it/s]
 44%|████▍     | 68/154 [00:10<00:13,  6.30it/s]
 45%|████▍     | 69/154 [00:10<00:14,  5.98it/s]
 45%|████▌     | 70/154 [00:11<00:15,  5.52it/s]
 46%|████▌     | 71/154 [00:11<00:13,  5.93it/s]
 47%|████▋     | 72/154 [00:11<00:13,  6.30it/s]
 47%|████▋     | 73/154 [00:11<00:12,  6.49it/s]
 48%|████▊     | 74/154 [00:11<00:11,  7.11it/s]
 49%|████▊     | 75/154 [00:11<00:11,  6.76it/s]
 49%|████▉     | 76/154 [00:11<00:11,  7.02it/s]
 50%|█████     | 77/154 [00:12<00:10,  7.53it/s]
 51%|█████     | 78/154 [00:12<00:11,  6.68it/s]
 51%|█████▏    | 79/154 [00:12<00:11,  6.35it/s]
 52%|█████▏    | 80/154 [00:12<00:12,  5.98it/s]
 53%|█████▎    | 81/154 [00:12<00:11,  6.40it/s]
 53%|█████▎    | 82/154 [00:12<00:11,  6.49it/s]
 54%|█████▍    | 83/154 [00:13<00:11,  6.21it/s]
 55%|█████▍    | 84/154 [00:13<00:11,  6.11it/s]
 55%|█████▌    | 85/154 [00:13<00:11,  5.94it/s]
 56%|█████▌    | 86/154 [00:13<00:11,  6.07it/s]
 56%|█████▋    | 87/154 [00:13<00:12,  5.49it/s]
 57%|█████▋    | 88/154 [00:13<00:12,  5.43it/s]
 58%|█████▊    | 89/154 [00:14<00:11,  5.80it/s]
 58%|█████▊    | 90/154 [00:14<00:10,  5.87it/s]
 59%|█████▉    | 91/154 [00:14<00:10,  6.16it/s]
 60%|█████▉    | 92/154 [00:14<00:09,  6.80it/s]
 60%|██████    | 93/154 [00:14<00:09,  6.74it/s]
 61%|██████    | 94/154 [00:14<00:08,  6.80it/s]
 62%|██████▏   | 95/154 [00:14<00:08,  7.00it/s]
 62%|██████▏   | 96/154 [00:15<00:08,  6.83it/s]
 63%|██████▎   | 97/154 [00:15<00:08,  6.75it/s]
 64%|██████▎   | 98/154 [00:15<00:08,  6.55it/s]
 64%|██████▍   | 99/154 [00:15<00:08,  6.24it/s]
 65%|██████▍   | 100/154 [00:15<00:07,  6.76it/s]
 66%|██████▌   | 101/154 [00:15<00:07,  6.69it/s]
 66%|██████▌   | 102/154 [00:16<00:07,  6.69it/s]
 67%|██████▋   | 103/154 [00:16<00:07,  6.92it/s]
 68%|██████▊   | 104/154 [00:16<00:07,  6.98it/s]
 68%|██████▊   | 105/154 [00:16<00:07,  6.76it/s]
 69%|██████▉   | 106/154 [00:16<00:07,  6.77it/s]
 69%|██████▉   | 107/154 [00:16<00:07,  6.50it/s]
 70%|███████   | 108/154 [00:16<00:06,  6.61it/s]
 71%|███████   | 109/154 [00:17<00:08,  5.32it/s]
 71%|███████▏  | 110/154 [00:17<00:07,  5.75it/s]
 72%|███████▏  | 111/154 [00:17<00:07,  5.79it/s]
 73%|███████▎  | 112/154 [00:17<00:07,  5.80it/s]
 73%|███████▎  | 113/154 [00:17<00:06,  6.30it/s]
 74%|███████▍  | 114/154 [00:17<00:06,  6.57it/s]
 75%|███████▍  | 115/154 [00:18<00:06,  6.48it/s]
 75%|███████▌  | 116/154 [00:18<00:06,  5.88it/s]
 76%|███████▌  | 117/154 [00:18<00:07,  5.13it/s]
 77%|███████▋  | 118/154 [00:18<00:06,  5.54it/s]
 77%|███████▋  | 119/154 [00:18<00:05,  5.92it/s]
 78%|███████▊  | 120/154 [00:19<00:06,  5.45it/s]
 79%|███████▊  | 121/154 [00:19<00:05,  6.04it/s]
 79%|███████▉  | 122/154 [00:19<00:05,  6.11it/s]
 80%|███████▉  | 123/154 [00:19<00:05,  5.66it/s]
 81%|████████  | 124/154 [00:19<00:04,  6.08it/s]
 81%|████████  | 125/154 [00:19<00:04,  6.12it/s]
 82%|████████▏ | 126/154 [00:20<00:04,  6.35it/s]
 82%|████████▏ | 127/154 [00:20<00:04,  5.94it/s]
 83%|████████▎ | 128/154 [00:20<00:04,  5.89it/s]
 84%|████████▍ | 129/154 [00:20<00:03,  6.38it/s]
 84%|████████▍ | 130/154 [00:20<00:03,  6.50it/s]
 85%|████████▌ | 131/154 [00:20<00:03,  6.52it/s]
 86%|████████▌ | 132/154 [00:20<00:03,  6.72it/s]
 86%|████████▋ | 133/154 [00:21<00:03,  6.25it/s]
 87%|████████▋ | 134/154 [00:21<00:03,  6.39it/s]
 88%|████████▊ | 135/154 [00:21<00:02,  6.56it/s]
 88%|████████▊ | 136/154 [00:21<00:02,  6.73it/s]
 89%|████████▉ | 137/154 [00:21<00:02,  6.80it/s]
 90%|████████▉ | 138/154 [00:21<00:02,  6.65it/s]
 90%|█████████ | 139/154 [00:21<00:02,  7.10it/s]
 91%|█████████ | 140/154 [00:22<00:01,  7.74it/s]
 92%|█████████▏| 141/154 [00:22<00:01,  7.09it/s]
 92%|█████████▏| 142/154 [00:22<00:01,  7.46it/s]
 93%|█████████▎| 143/154 [00:22<00:01,  6.66it/s]
 94%|█████████▎| 144/154 [00:22<00:01,  6.54it/s]
 94%|█████████▍| 145/154 [00:22<00:01,  6.59it/s]
 95%|█████████▍| 146/154 [00:23<00:01,  6.30it/s]
 95%|█████████▌| 147/154 [00:23<00:01,  5.99it/s]
 96%|█████████▌| 148/154 [00:23<00:01,  5.97it/s]
 97%|█████████▋| 149/154 [00:23<00:00,  6.30it/s]
 97%|█████████▋| 150/154 [00:23<00:00,  6.06it/s]
 98%|█████████▊| 151/154 [00:23<00:00,  6.31it/s]
 99%|█████████▊| 152/154 [00:23<00:00,  6.83it/s]
 99%|█████████▉| 153/154 [00:24<00:00,  5.59it/s]
100%|██████████| 154/154 [00:24<00:00,  5.95it/s]
100%|██████████| 154/154 [00:24<00:00,  6.32it/s]
                 mode    k  threshold  ...  alpha  retrieval_F  avg_pred_evidence
0      relative_logit  NaN        NaN  ...   0.70     0.255726           4.214286
1      relative_logit  NaN        NaN  ...   0.55     0.247052           3.889610
2      relative_logit  NaN        NaN  ...   0.85     0.242651           4.435065
3   dynamic_threshold  NaN       0.70  ...   0.70     0.242357           3.454545
4      relative_logit  NaN        NaN  ...   0.70     0.238291           4.889610
5   dynamic_threshold  NaN       0.24  ...   0.70     0.235292           4.980519
6   dynamic_threshold  NaN       0.22  ...   0.70     0.234859           4.993506
7   dynamic_threshold  NaN       0.02  ...   0.70     0.234550           5.000000
8   dynamic_threshold  NaN       0.04  ...   0.70     0.234550           5.000000
9   dynamic_threshold  NaN       0.06  ...   0.70     0.234550           5.000000
10            fixed_k  5.0        NaN  ...   0.70     0.234550           5.000000
11  dynamic_threshold  NaN       0.12  ...   0.70     0.234550           5.000000
12  dynamic_threshold  NaN       0.16  ...   0.70     0.234550           5.000000
13  dynamic_threshold  NaN       0.14  ...   0.70     0.234550           5.000000
14  dynamic_threshold  NaN       0.20  ...   0.70     0.234550           5.000000
15  dynamic_threshold  NaN       0.18  ...   0.70     0.234550           5.000000
16     relative_logit  NaN        NaN  ...   0.70     0.234550           5.000000
17  dynamic_threshold  NaN       0.08  ...   0.70     0.234550           5.000000
18  dynamic_threshold  NaN       0.10  ...   0.70     0.234550           5.000000
19     relative_logit  NaN        NaN  ...   0.70     0.234550           5.000000
20     relative_logit  NaN        NaN  ...   0.70     0.234550           5.000000
21     relative_logit  NaN        NaN  ...   0.70     0.234550           5.000000
22     relative_logit  NaN        NaN  ...   0.70     0.234550           5.000000
23  dynamic_threshold  NaN       0.26  ...   0.70     0.234426           4.974026
24  dynamic_threshold  NaN       0.50  ...   0.70     0.234163           4.532468
25  dynamic_threshold  NaN       0.60  ...   0.70     0.234024           4.136364
26            fixed_k  4.0        NaN  ...   0.70     0.233911           4.000000
27  dynamic_threshold  NaN       0.48  ...   0.70     0.232777           4.597403
28            fixed_k  4.0        NaN  ...   0.85     0.232524           4.000000
29  dynamic_threshold  NaN       0.28  ...   0.70     0.231828           4.954545

[30 rows x 7 columns]
Final retrieval policy: prefer_dynamic
Best dev row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Chosen row: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Scoring train_final candidates with cross-encoder reranker...

  0%|          | 0/1228 [00:00<?, ?it/s]
  0%|          | 1/1228 [00:00<03:49,  5.34it/s]
  0%|          | 2/1228 [00:00<03:26,  5.93it/s]
  0%|          | 3/1228 [00:00<02:52,  7.12it/s]
  0%|          | 4/1228 [00:00<03:00,  6.78it/s]
  0%|          | 5/1228 [00:00<03:02,  6.70it/s]
  0%|          | 6/1228 [00:00<03:15,  6.24it/s]
  1%|          | 7/1228 [00:01<03:03,  6.66it/s]
  1%|          | 8/1228 [00:01<02:51,  7.12it/s]
  1%|          | 9/1228 [00:01<02:59,  6.78it/s]
  1%|          | 10/1228 [00:01<03:15,  6.24it/s]
  1%|          | 11/1228 [00:01<03:27,  5.87it/s]
  1%|          | 12/1228 [00:01<03:35,  5.65it/s]
  1%|          | 13/1228 [00:02<03:29,  5.81it/s]
  1%|          | 14/1228 [00:02<03:13,  6.29it/s]
  1%|          | 15/1228 [00:02<03:29,  5.78it/s]
  1%|▏         | 16/1228 [00:02<03:21,  6.02it/s]
  1%|▏         | 17/1228 [00:02<03:06,  6.50it/s]
  1%|▏         | 18/1228 [00:02<03:09,  6.39it/s]
  2%|▏         | 19/1228 [00:03<03:20,  6.02it/s]
  2%|▏         | 20/1228 [00:03<03:19,  6.04it/s]
  2%|▏         | 21/1228 [00:03<03:22,  5.95it/s]
  2%|▏         | 22/1228 [00:03<03:27,  5.80it/s]
  2%|▏         | 23/1228 [00:03<03:30,  5.72it/s]
  2%|▏         | 24/1228 [00:03<03:21,  5.97it/s]
  2%|▏         | 25/1228 [00:04<03:07,  6.43it/s]
  2%|▏         | 26/1228 [00:04<02:58,  6.73it/s]
  2%|▏         | 27/1228 [00:04<03:09,  6.33it/s]
  2%|▏         | 28/1228 [00:04<03:14,  6.16it/s]
  2%|▏         | 29/1228 [00:04<03:07,  6.41it/s]
  2%|▏         | 30/1228 [00:04<02:49,  7.06it/s]
  3%|▎         | 31/1228 [00:04<03:04,  6.49it/s]
  3%|▎         | 32/1228 [00:05<03:25,  5.81it/s]
  3%|▎         | 33/1228 [00:05<03:27,  5.75it/s]
  3%|▎         | 34/1228 [00:05<03:47,  5.25it/s]
  3%|▎         | 35/1228 [00:05<03:24,  5.83it/s]
  3%|▎         | 36/1228 [00:05<03:24,  5.84it/s]
  3%|▎         | 37/1228 [00:05<03:08,  6.33it/s]
  3%|▎         | 38/1228 [00:06<03:14,  6.13it/s]
  3%|▎         | 39/1228 [00:06<03:12,  6.19it/s]
  3%|▎         | 40/1228 [00:06<03:11,  6.21it/s]
  3%|▎         | 41/1228 [00:06<03:14,  6.12it/s]
  3%|▎         | 42/1228 [00:06<02:58,  6.66it/s]
  4%|▎         | 43/1228 [00:06<02:55,  6.75it/s]
  4%|▎         | 44/1228 [00:07<02:51,  6.91it/s]
  4%|▎         | 45/1228 [00:07<03:10,  6.20it/s]
  4%|▎         | 46/1228 [00:07<03:00,  6.54it/s]
  4%|▍         | 47/1228 [00:07<02:57,  6.65it/s]
  4%|▍         | 48/1228 [00:07<02:51,  6.87it/s]
  4%|▍         | 49/1228 [00:07<02:52,  6.84it/s]
  4%|▍         | 50/1228 [00:07<02:44,  7.15it/s]
  4%|▍         | 51/1228 [00:08<02:34,  7.60it/s]
  4%|▍         | 52/1228 [00:08<02:33,  7.66it/s]
  4%|▍         | 53/1228 [00:08<02:43,  7.18it/s]
  4%|▍         | 54/1228 [00:08<03:08,  6.22it/s]
  4%|▍         | 55/1228 [00:08<02:51,  6.84it/s]
  5%|▍         | 56/1228 [00:08<02:52,  6.78it/s]
  5%|▍         | 57/1228 [00:08<02:59,  6.52it/s]
  5%|▍         | 58/1228 [00:09<02:56,  6.62it/s]
  5%|▍         | 59/1228 [00:09<02:58,  6.56it/s]
  5%|▍         | 60/1228 [00:09<03:01,  6.42it/s]
  5%|▍         | 61/1228 [00:09<02:58,  6.52it/s]
  5%|▌         | 62/1228 [00:09<03:26,  5.64it/s]
  5%|▌         | 63/1228 [00:09<03:01,  6.41it/s]
  5%|▌         | 64/1228 [00:10<03:15,  5.96it/s]
  5%|▌         | 65/1228 [00:10<03:06,  6.24it/s]
  5%|▌         | 66/1228 [00:10<03:01,  6.40it/s]
  5%|▌         | 67/1228 [00:10<03:05,  6.25it/s]
  6%|▌         | 68/1228 [00:10<03:05,  6.25it/s]
  6%|▌         | 69/1228 [00:10<02:55,  6.60it/s]
  6%|▌         | 70/1228 [00:11<02:55,  6.59it/s]
  6%|▌         | 71/1228 [00:11<03:04,  6.26it/s]
  6%|▌         | 72/1228 [00:11<02:56,  6.54it/s]
  6%|▌         | 73/1228 [00:11<02:47,  6.91it/s]
  6%|▌         | 74/1228 [00:11<02:43,  7.07it/s]
  6%|▌         | 75/1228 [00:11<02:49,  6.79it/s]
  6%|▌         | 76/1228 [00:11<02:46,  6.92it/s]
  6%|▋         | 77/1228 [00:12<02:43,  7.04it/s]
  6%|▋         | 78/1228 [00:12<02:40,  7.16it/s]
  6%|▋         | 79/1228 [00:12<02:57,  6.46it/s]
  7%|▋         | 80/1228 [00:12<03:06,  6.17it/s]
  7%|▋         | 81/1228 [00:12<02:49,  6.78it/s]
  7%|▋         | 82/1228 [00:12<02:35,  7.35it/s]
  7%|▋         | 83/1228 [00:12<02:41,  7.07it/s]
  7%|▋         | 84/1228 [00:13<02:34,  7.41it/s]
  7%|▋         | 85/1228 [00:13<02:44,  6.95it/s]
  7%|▋         | 86/1228 [00:13<02:52,  6.63it/s]
  7%|▋         | 87/1228 [00:13<02:47,  6.80it/s]
  7%|▋         | 88/1228 [00:13<03:07,  6.09it/s]
  7%|▋         | 89/1228 [00:13<03:06,  6.10it/s]
  7%|▋         | 90/1228 [00:14<02:53,  6.55it/s]
  7%|▋         | 91/1228 [00:14<02:45,  6.89it/s]
  7%|▋         | 92/1228 [00:14<02:45,  6.88it/s]
  8%|▊         | 93/1228 [00:14<02:44,  6.91it/s]
  8%|▊         | 94/1228 [00:14<02:49,  6.69it/s]
  8%|▊         | 95/1228 [00:14<02:57,  6.40it/s]
  8%|▊         | 96/1228 [00:14<02:45,  6.85it/s]
  8%|▊         | 97/1228 [00:15<02:39,  7.07it/s]
  8%|▊         | 98/1228 [00:15<02:37,  7.19it/s]
  8%|▊         | 99/1228 [00:15<02:54,  6.48it/s]
  8%|▊         | 100/1228 [00:15<03:04,  6.11it/s]
  8%|▊         | 101/1228 [00:15<03:05,  6.06it/s]
  8%|▊         | 102/1228 [00:15<03:06,  6.03it/s]
  8%|▊         | 103/1228 [00:16<03:06,  6.03it/s]
  8%|▊         | 104/1228 [00:16<03:10,  5.91it/s]
  9%|▊         | 105/1228 [00:16<03:03,  6.11it/s]
  9%|▊         | 106/1228 [00:16<03:07,  5.97it/s]
  9%|▊         | 107/1228 [00:16<02:54,  6.44it/s]
  9%|▉         | 108/1228 [00:16<02:59,  6.23it/s]
  9%|▉         | 109/1228 [00:16<03:05,  6.03it/s]
  9%|▉         | 110/1228 [00:17<02:59,  6.23it/s]
  9%|▉         | 111/1228 [00:17<02:53,  6.42it/s]
  9%|▉         | 112/1228 [00:17<02:48,  6.63it/s]
  9%|▉         | 113/1228 [00:17<02:46,  6.70it/s]
  9%|▉         | 114/1228 [00:17<02:43,  6.82it/s]
  9%|▉         | 115/1228 [00:17<02:48,  6.62it/s]
  9%|▉         | 116/1228 [00:18<02:49,  6.55it/s]
 10%|▉         | 118/1228 [00:18<02:50,  6.52it/s]
 10%|▉         | 119/1228 [00:18<02:47,  6.61it/s]
 10%|▉         | 120/1228 [00:18<02:49,  6.53it/s]
 10%|▉         | 121/1228 [00:18<02:43,  6.75it/s]
 10%|▉         | 122/1228 [00:18<02:44,  6.74it/s]
 10%|█         | 123/1228 [00:19<02:49,  6.52it/s]
 10%|█         | 124/1228 [00:19<02:59,  6.16it/s]
 10%|█         | 125/1228 [00:19<03:08,  5.85it/s]
 10%|█         | 126/1228 [00:19<02:56,  6.25it/s]
 10%|█         | 127/1228 [00:19<02:48,  6.53it/s]
 10%|█         | 128/1228 [00:19<02:38,  6.96it/s]
 11%|█         | 129/1228 [00:19<02:29,  7.34it/s]
 11%|█         | 130/1228 [00:20<02:24,  7.59it/s]
 11%|█         | 131/1228 [00:20<02:28,  7.38it/s]
 11%|█         | 132/1228 [00:20<02:51,  6.39it/s]
 11%|█         | 133/1228 [00:20<02:50,  6.42it/s]
 11%|█         | 134/1228 [00:20<02:56,  6.18it/s]
 11%|█         | 135/1228 [00:20<02:47,  6.51it/s]
 11%|█         | 136/1228 [00:21<02:34,  7.05it/s]
 11%|█         | 137/1228 [00:21<02:26,  7.46it/s]
 11%|█         | 138/1228 [00:21<02:27,  7.37it/s]
 11%|█▏        | 139/1228 [00:21<02:28,  7.31it/s]
 11%|█▏        | 140/1228 [00:21<02:22,  7.65it/s]
 11%|█▏        | 141/1228 [00:21<02:38,  6.85it/s]
 12%|█▏        | 142/1228 [00:21<02:46,  6.53it/s]
 12%|█▏        | 143/1228 [00:22<02:40,  6.77it/s]
 12%|█▏        | 144/1228 [00:22<02:33,  7.08it/s]
 12%|█▏        | 145/1228 [00:22<02:51,  6.31it/s]
 12%|█▏        | 146/1228 [00:22<02:49,  6.39it/s]
 12%|█▏        | 147/1228 [00:22<02:50,  6.33it/s]
 12%|█▏        | 148/1228 [00:22<02:36,  6.90it/s]
 12%|█▏        | 149/1228 [00:22<02:41,  6.70it/s]
 12%|█▏        | 150/1228 [00:23<02:47,  6.45it/s]
 12%|█▏        | 151/1228 [00:23<02:54,  6.17it/s]
 12%|█▏        | 152/1228 [00:23<02:55,  6.12it/s]
 12%|█▏        | 153/1228 [00:23<03:02,  5.89it/s]
 13%|█▎        | 154/1228 [00:23<02:48,  6.38it/s]
 13%|█▎        | 155/1228 [00:23<02:45,  6.48it/s]
 13%|█▎        | 156/1228 [00:24<02:54,  6.14it/s]
 13%|█▎        | 157/1228 [00:24<03:04,  5.79it/s]
 13%|█▎        | 158/1228 [00:24<03:02,  5.87it/s]
 13%|█▎        | 159/1228 [00:24<02:59,  5.96it/s]
 13%|█▎        | 160/1228 [00:24<03:02,  5.86it/s]
 13%|█▎        | 161/1228 [00:24<02:53,  6.16it/s]
 13%|█▎        | 162/1228 [00:25<02:41,  6.58it/s]
 13%|█▎        | 163/1228 [00:25<03:02,  5.85it/s]
 13%|█▎        | 164/1228 [00:25<03:06,  5.69it/s]
 13%|█▎        | 165/1228 [00:25<03:02,  5.82it/s]
 14%|█▎        | 166/1228 [00:25<03:01,  5.84it/s]
 14%|█▎        | 167/1228 [00:25<02:47,  6.34it/s]
 14%|█▎        | 168/1228 [00:26<02:41,  6.56it/s]
 14%|█▍        | 169/1228 [00:26<02:40,  6.61it/s]
 14%|█▍        | 170/1228 [00:26<02:43,  6.46it/s]
 14%|█▍        | 171/1228 [00:26<02:45,  6.39it/s]
 14%|█▍        | 172/1228 [00:26<02:35,  6.81it/s]
 14%|█▍        | 173/1228 [00:26<02:39,  6.62it/s]
 14%|█▍        | 174/1228 [00:26<02:28,  7.11it/s]
 14%|█▍        | 175/1228 [00:27<02:37,  6.69it/s]
 14%|█▍        | 176/1228 [00:27<02:41,  6.53it/s]
 14%|█▍        | 177/1228 [00:27<02:47,  6.29it/s]
 14%|█▍        | 178/1228 [00:27<02:45,  6.33it/s]
 15%|█▍        | 179/1228 [00:27<02:54,  6.01it/s]
 15%|█▍        | 180/1228 [00:27<02:58,  5.88it/s]
 15%|█▍        | 181/1228 [00:28<02:53,  6.04it/s]
 15%|█▍        | 182/1228 [00:28<02:47,  6.24it/s]
 15%|█▍        | 183/1228 [00:28<02:48,  6.22it/s]
 15%|█▍        | 184/1228 [00:28<02:34,  6.76it/s]
 15%|█▌        | 185/1228 [00:28<02:42,  6.44it/s]
 15%|█▌        | 186/1228 [00:28<02:33,  6.78it/s]
 15%|█▌        | 187/1228 [00:29<02:55,  5.93it/s]
 15%|█▌        | 188/1228 [00:29<02:47,  6.23it/s]
 15%|█▌        | 189/1228 [00:29<02:50,  6.08it/s]
 15%|█▌        | 190/1228 [00:29<02:51,  6.06it/s]
 16%|█▌        | 191/1228 [00:29<02:44,  6.29it/s]
 16%|█▌        | 192/1228 [00:29<02:57,  5.83it/s]
 16%|█▌        | 193/1228 [00:30<03:00,  5.72it/s]
 16%|█▌        | 194/1228 [00:30<02:58,  5.78it/s]
 16%|█▌        | 195/1228 [00:30<02:46,  6.20it/s]
 16%|█▌        | 196/1228 [00:30<02:42,  6.34it/s]
 16%|█▌        | 197/1228 [00:30<02:50,  6.04it/s]
 16%|█▌        | 198/1228 [00:30<02:48,  6.11it/s]
 16%|█▌        | 199/1228 [00:31<02:38,  6.49it/s]
 16%|█▋        | 200/1228 [00:31<02:32,  6.72it/s]
 16%|█▋        | 201/1228 [00:31<02:49,  6.05it/s]
 16%|█▋        | 202/1228 [00:31<02:50,  6.02it/s]
 17%|█▋        | 203/1228 [00:31<02:42,  6.31it/s]
 17%|█▋        | 204/1228 [00:31<02:36,  6.55it/s]
 17%|█▋        | 205/1228 [00:31<02:44,  6.21it/s]
 17%|█▋        | 206/1228 [00:32<02:42,  6.27it/s]
 17%|█▋        | 207/1228 [00:32<02:49,  6.02it/s]
 17%|█▋        | 208/1228 [00:32<02:34,  6.59it/s]
 17%|█▋        | 209/1228 [00:32<02:32,  6.69it/s]
 17%|█▋        | 210/1228 [00:32<02:32,  6.66it/s]
 17%|█▋        | 211/1228 [00:32<02:34,  6.60it/s]
 17%|█▋        | 212/1228 [00:33<02:31,  6.70it/s]
 17%|█▋        | 213/1228 [00:33<02:29,  6.77it/s]
 17%|█▋        | 214/1228 [00:33<02:31,  6.70it/s]
 18%|█▊        | 215/1228 [00:33<02:46,  6.09it/s]
 18%|█▊        | 216/1228 [00:33<02:41,  6.29it/s]
 18%|█▊        | 217/1228 [00:33<02:42,  6.21it/s]
 18%|█▊        | 218/1228 [00:33<02:42,  6.20it/s]
 18%|█▊        | 219/1228 [00:34<02:37,  6.42it/s]
 18%|█▊        | 220/1228 [00:34<02:47,  6.01it/s]
 18%|█▊        | 221/1228 [00:34<02:42,  6.18it/s]
 18%|█▊        | 222/1228 [00:34<02:45,  6.07it/s]
 18%|█▊        | 223/1228 [00:34<02:40,  6.28it/s]
 18%|█▊        | 224/1228 [00:34<02:45,  6.08it/s]
 18%|█▊        | 225/1228 [00:35<02:43,  6.14it/s]
 18%|█▊        | 226/1228 [00:35<02:33,  6.53it/s]
 18%|█▊        | 227/1228 [00:35<02:46,  6.00it/s]
 19%|█▊        | 228/1228 [00:35<02:31,  6.60it/s]
 19%|█▊        | 229/1228 [00:35<02:36,  6.39it/s]
 19%|█▊        | 230/1228 [00:35<02:36,  6.39it/s]
 19%|█▉        | 231/1228 [00:36<02:41,  6.17it/s]
 19%|█▉        | 232/1228 [00:36<02:53,  5.73it/s]
 19%|█▉        | 233/1228 [00:36<02:42,  6.13it/s]
 19%|█▉        | 234/1228 [00:36<02:51,  5.78it/s]
 19%|█▉        | 235/1228 [00:36<02:39,  6.24it/s]
 19%|█▉        | 236/1228 [00:36<02:30,  6.60it/s]
 19%|█▉        | 237/1228 [00:37<02:35,  6.36it/s]
 19%|█▉        | 238/1228 [00:37<02:35,  6.36it/s]
 19%|█▉        | 239/1228 [00:37<02:27,  6.73it/s]
 20%|█▉        | 240/1228 [00:37<02:28,  6.65it/s]
 20%|█▉        | 241/1228 [00:37<02:26,  6.76it/s]
 20%|█▉        | 242/1228 [00:37<02:25,  6.79it/s]
 20%|█▉        | 243/1228 [00:37<02:16,  7.20it/s]
 20%|█▉        | 244/1228 [00:38<02:45,  5.96it/s]
 20%|█▉        | 245/1228 [00:38<02:45,  5.93it/s]
 20%|██        | 246/1228 [00:38<02:47,  5.85it/s]
 20%|██        | 247/1228 [00:38<02:42,  6.03it/s]
 20%|██        | 248/1228 [00:38<02:47,  5.86it/s]
 20%|██        | 249/1228 [00:38<02:44,  5.96it/s]
 20%|██        | 250/1228 [00:39<02:35,  6.27it/s]
 20%|██        | 251/1228 [00:39<02:34,  6.34it/s]
 21%|██        | 252/1228 [00:39<02:55,  5.57it/s]
 21%|██        | 253/1228 [00:39<02:49,  5.74it/s]
 21%|██        | 254/1228 [00:39<02:31,  6.42it/s]
 21%|██        | 256/1228 [00:40<02:24,  6.72it/s]
 21%|██        | 257/1228 [00:40<02:26,  6.62it/s]
 21%|██        | 258/1228 [00:40<02:19,  6.97it/s]
 21%|██        | 259/1228 [00:40<02:32,  6.36it/s]
 21%|██        | 260/1228 [00:40<02:30,  6.42it/s]
 21%|██▏       | 261/1228 [00:40<02:39,  6.05it/s]
 21%|██▏       | 262/1228 [00:41<02:42,  5.94it/s]
 21%|██▏       | 263/1228 [00:41<02:40,  6.01it/s]
 21%|██▏       | 264/1228 [00:41<02:35,  6.21it/s]
 22%|██▏       | 265/1228 [00:41<02:41,  5.96it/s]
 22%|██▏       | 266/1228 [00:41<02:31,  6.34it/s]
 22%|██▏       | 267/1228 [00:41<02:25,  6.60it/s]
 22%|██▏       | 268/1228 [00:41<02:24,  6.67it/s]
 22%|██▏       | 269/1228 [00:42<02:13,  7.17it/s]
 22%|██▏       | 270/1228 [00:42<02:05,  7.61it/s]
 22%|██▏       | 271/1228 [00:42<02:27,  6.50it/s]
 22%|██▏       | 272/1228 [00:42<02:20,  6.82it/s]
 22%|██▏       | 273/1228 [00:42<02:31,  6.29it/s]
 22%|██▏       | 274/1228 [00:42<02:35,  6.15it/s]
 22%|██▏       | 275/1228 [00:43<02:29,  6.35it/s]
 22%|██▏       | 276/1228 [00:43<02:34,  6.15it/s]
 23%|██▎       | 277/1228 [00:43<02:28,  6.42it/s]
 23%|██▎       | 279/1228 [00:43<02:06,  7.51it/s]
 23%|██▎       | 280/1228 [00:43<02:20,  6.76it/s]
 23%|██▎       | 281/1228 [00:43<02:35,  6.10it/s]
 23%|██▎       | 282/1228 [00:44<02:19,  6.76it/s]
 23%|██▎       | 283/1228 [00:44<02:19,  6.76it/s]
 23%|██▎       | 284/1228 [00:44<02:28,  6.34it/s]
 23%|██▎       | 285/1228 [00:44<02:31,  6.21it/s]
 23%|██▎       | 286/1228 [00:44<02:41,  5.82it/s]
 23%|██▎       | 287/1228 [00:44<02:43,  5.76it/s]
 23%|██▎       | 288/1228 [00:45<02:40,  5.85it/s]
 24%|██▎       | 289/1228 [00:45<02:33,  6.10it/s]
 24%|██▎       | 290/1228 [00:45<02:27,  6.35it/s]
 24%|██▎       | 291/1228 [00:45<02:34,  6.05it/s]
 24%|██▍       | 292/1228 [00:45<02:28,  6.29it/s]
 24%|██▍       | 293/1228 [00:45<02:24,  6.49it/s]
 24%|██▍       | 294/1228 [00:45<02:16,  6.86it/s]
 24%|██▍       | 295/1228 [00:46<02:34,  6.02it/s]
 24%|██▍       | 296/1228 [00:46<02:22,  6.55it/s]
 24%|██▍       | 297/1228 [00:46<02:15,  6.87it/s]
 24%|██▍       | 298/1228 [00:46<02:24,  6.42it/s]
 24%|██▍       | 299/1228 [00:46<02:19,  6.65it/s]
 24%|██▍       | 300/1228 [00:46<02:25,  6.36it/s]
 25%|██▍       | 301/1228 [00:47<02:23,  6.45it/s]
 25%|██▍       | 302/1228 [00:47<02:22,  6.52it/s]
 25%|██▍       | 303/1228 [00:47<02:25,  6.35it/s]
 25%|██▍       | 304/1228 [00:47<02:14,  6.89it/s]
 25%|██▍       | 305/1228 [00:47<02:22,  6.48it/s]
 25%|██▍       | 306/1228 [00:47<02:22,  6.49it/s]
 25%|██▌       | 307/1228 [00:48<02:28,  6.21it/s]
 25%|██▌       | 308/1228 [00:48<02:29,  6.14it/s]
 25%|██▌       | 309/1228 [00:48<02:22,  6.45it/s]
 25%|██▌       | 310/1228 [00:48<02:13,  6.86it/s]
 25%|██▌       | 311/1228 [00:48<02:04,  7.35it/s]
 25%|██▌       | 312/1228 [00:48<02:07,  7.19it/s]
 25%|██▌       | 313/1228 [00:48<02:04,  7.33it/s]
 26%|██▌       | 314/1228 [00:49<02:11,  6.96it/s]
 26%|██▌       | 315/1228 [00:49<02:24,  6.33it/s]
 26%|██▌       | 316/1228 [00:49<02:31,  6.01it/s]
 26%|██▌       | 317/1228 [00:49<02:15,  6.71it/s]
 26%|██▌       | 318/1228 [00:49<02:13,  6.83it/s]
 26%|██▌       | 319/1228 [00:49<02:17,  6.60it/s]
 26%|██▌       | 320/1228 [00:49<02:12,  6.84it/s]
 26%|██▌       | 321/1228 [00:50<02:23,  6.31it/s]
 26%|██▌       | 322/1228 [00:50<02:25,  6.25it/s]
 26%|██▋       | 323/1228 [00:50<02:15,  6.67it/s]
 26%|██▋       | 324/1228 [00:50<02:16,  6.64it/s]
 26%|██▋       | 325/1228 [00:50<02:15,  6.66it/s]
 27%|██▋       | 326/1228 [00:50<02:16,  6.62it/s]
 27%|██▋       | 327/1228 [00:50<02:06,  7.10it/s]
 27%|██▋       | 328/1228 [00:51<02:15,  6.65it/s]
 27%|██▋       | 329/1228 [00:51<02:12,  6.80it/s]
 27%|██▋       | 330/1228 [00:51<02:03,  7.30it/s]
 27%|██▋       | 331/1228 [00:51<02:06,  7.11it/s]
 27%|██▋       | 332/1228 [00:51<02:05,  7.15it/s]
 27%|██▋       | 333/1228 [00:51<02:25,  6.17it/s]
 27%|██▋       | 334/1228 [00:52<02:21,  6.32it/s]
 27%|██▋       | 335/1228 [00:52<02:19,  6.41it/s]
 27%|██▋       | 336/1228 [00:52<02:24,  6.17it/s]
 27%|██▋       | 337/1228 [00:52<02:24,  6.15it/s]
 28%|██▊       | 338/1228 [00:52<02:20,  6.33it/s]
 28%|██▊       | 339/1228 [00:52<02:38,  5.61it/s]
 28%|██▊       | 340/1228 [00:53<02:34,  5.73it/s]
 28%|██▊       | 341/1228 [00:53<02:23,  6.20it/s]
 28%|██▊       | 342/1228 [00:53<02:12,  6.68it/s]
 28%|██▊       | 343/1228 [00:53<02:09,  6.83it/s]
 28%|██▊       | 344/1228 [00:53<02:13,  6.60it/s]
 28%|██▊       | 345/1228 [00:53<02:17,  6.42it/s]
 28%|██▊       | 346/1228 [00:53<02:19,  6.33it/s]
 28%|██▊       | 347/1228 [00:54<02:15,  6.49it/s]
 28%|██▊       | 348/1228 [00:54<02:16,  6.46it/s]
 28%|██▊       | 349/1228 [00:54<02:19,  6.30it/s]
 29%|██▊       | 350/1228 [00:54<02:09,  6.79it/s]
 29%|██▊       | 351/1228 [00:54<02:05,  6.97it/s]
 29%|██▊       | 352/1228 [00:54<02:06,  6.91it/s]
 29%|██▊       | 353/1228 [00:54<01:59,  7.32it/s]
 29%|██▉       | 354/1228 [00:55<02:16,  6.42it/s]
 29%|██▉       | 355/1228 [00:55<02:06,  6.90it/s]
 29%|██▉       | 356/1228 [00:55<02:14,  6.49it/s]
 29%|██▉       | 357/1228 [00:55<02:10,  6.67it/s]
 29%|██▉       | 358/1228 [00:55<02:29,  5.82it/s]
 29%|██▉       | 359/1228 [00:56<02:36,  5.54it/s]
 29%|██▉       | 360/1228 [00:56<02:23,  6.04it/s]
 29%|██▉       | 361/1228 [00:56<02:24,  6.02it/s]
 29%|██▉       | 362/1228 [00:56<02:16,  6.34it/s]
 30%|██▉       | 363/1228 [00:56<02:15,  6.40it/s]
 30%|██▉       | 364/1228 [00:56<02:09,  6.69it/s]
 30%|██▉       | 365/1228 [00:56<02:25,  5.95it/s]
 30%|██▉       | 366/1228 [00:57<02:11,  6.54it/s]
 30%|██▉       | 367/1228 [00:57<02:15,  6.37it/s]
 30%|██▉       | 368/1228 [00:57<02:25,  5.90it/s]
 30%|███       | 369/1228 [00:57<02:19,  6.18it/s]
 30%|███       | 370/1228 [00:57<02:09,  6.60it/s]
 30%|███       | 371/1228 [00:57<02:22,  6.02it/s]
 30%|███       | 372/1228 [00:58<02:10,  6.55it/s]
 30%|███       | 373/1228 [00:58<02:12,  6.44it/s]
 30%|███       | 374/1228 [00:58<02:14,  6.36it/s]
 31%|███       | 375/1228 [00:58<02:17,  6.19it/s]
 31%|███       | 376/1228 [00:58<02:14,  6.35it/s]
 31%|███       | 377/1228 [00:58<02:20,  6.05it/s]
 31%|███       | 378/1228 [00:58<02:13,  6.38it/s]
 31%|███       | 379/1228 [00:59<02:13,  6.34it/s]
 31%|███       | 380/1228 [00:59<02:07,  6.65it/s]
 31%|███       | 381/1228 [00:59<02:27,  5.73it/s]
 31%|███       | 382/1228 [00:59<02:26,  5.78it/s]
 31%|███       | 383/1228 [00:59<02:14,  6.27it/s]
 31%|███▏      | 384/1228 [00:59<02:10,  6.48it/s]
 31%|███▏      | 385/1228 [01:00<01:59,  7.08it/s]
 31%|███▏      | 386/1228 [01:00<01:56,  7.25it/s]
 32%|███▏      | 387/1228 [01:00<01:58,  7.11it/s]
 32%|███▏      | 388/1228 [01:00<02:13,  6.28it/s]
 32%|███▏      | 389/1228 [01:00<02:05,  6.70it/s]
 32%|███▏      | 390/1228 [01:00<01:58,  7.06it/s]
 32%|███▏      | 391/1228 [01:00<02:05,  6.65it/s]
 32%|███▏      | 392/1228 [01:01<02:07,  6.57it/s]
 32%|███▏      | 393/1228 [01:01<02:03,  6.78it/s]
 32%|███▏      | 394/1228 [01:01<02:10,  6.40it/s]
 32%|███▏      | 395/1228 [01:01<02:11,  6.34it/s]
 32%|███▏      | 396/1228 [01:01<02:30,  5.53it/s]
 32%|███▏      | 397/1228 [01:01<02:22,  5.85it/s]
 32%|███▏      | 398/1228 [01:02<02:17,  6.06it/s]
 32%|███▏      | 399/1228 [01:02<02:17,  6.03it/s]
 33%|███▎      | 400/1228 [01:02<02:08,  6.43it/s]
 33%|███▎      | 401/1228 [01:02<02:07,  6.50it/s]
 33%|███▎      | 402/1228 [01:02<02:11,  6.29it/s]
 33%|███▎      | 403/1228 [01:02<02:14,  6.12it/s]
 33%|███▎      | 404/1228 [01:03<02:06,  6.52it/s]
 33%|███▎      | 405/1228 [01:03<02:07,  6.45it/s]
 33%|███▎      | 406/1228 [01:03<02:13,  6.16it/s]
 33%|███▎      | 407/1228 [01:03<02:21,  5.78it/s]
 33%|███▎      | 408/1228 [01:03<02:20,  5.82it/s]
 33%|███▎      | 409/1228 [01:03<02:17,  5.94it/s]
 33%|███▎      | 410/1228 [01:04<02:26,  5.59it/s]
 33%|███▎      | 411/1228 [01:04<02:17,  5.94it/s]
 34%|███▎      | 412/1228 [01:04<02:13,  6.11it/s]
 34%|███▎      | 413/1228 [01:04<02:16,  5.99it/s]
 34%|███▎      | 414/1228 [01:04<02:10,  6.22it/s]
 34%|███▍      | 415/1228 [01:04<02:07,  6.39it/s]
 34%|███▍      | 416/1228 [01:05<02:06,  6.42it/s]
 34%|███▍      | 417/1228 [01:05<02:16,  5.95it/s]
 34%|███▍      | 418/1228 [01:05<02:22,  5.69it/s]
 34%|███▍      | 419/1228 [01:05<02:11,  6.14it/s]
 34%|███▍      | 420/1228 [01:05<02:13,  6.06it/s]
 34%|███▍      | 421/1228 [01:05<02:03,  6.51it/s]
 34%|███▍      | 422/1228 [01:05<01:55,  7.00it/s]
 34%|███▍      | 423/1228 [01:06<01:48,  7.42it/s]
 35%|███▍      | 424/1228 [01:06<01:57,  6.85it/s]
 35%|███▍      | 425/1228 [01:06<02:02,  6.56it/s]
 35%|███▍      | 426/1228 [01:06<01:58,  6.77it/s]
 35%|███▍      | 427/1228 [01:06<01:55,  6.91it/s]
 35%|███▍      | 428/1228 [01:06<01:56,  6.89it/s]
 35%|███▍      | 429/1228 [01:07<02:08,  6.24it/s]
 35%|███▌      | 430/1228 [01:07<02:04,  6.41it/s]
 35%|███▌      | 431/1228 [01:07<01:52,  7.10it/s]
 35%|███▌      | 432/1228 [01:07<01:56,  6.86it/s]
 35%|███▌      | 433/1228 [01:07<01:54,  6.93it/s]
 35%|███▌      | 434/1228 [01:07<01:59,  6.67it/s]
 35%|███▌      | 435/1228 [01:07<01:54,  6.90it/s]
 36%|███▌      | 436/1228 [01:08<02:03,  6.42it/s]
 36%|███▌      | 437/1228 [01:08<01:57,  6.72it/s]
 36%|███▌      | 438/1228 [01:08<01:58,  6.66it/s]
 36%|███▌      | 439/1228 [01:08<01:54,  6.90it/s]
 36%|███▌      | 440/1228 [01:08<01:56,  6.74it/s]
 36%|███▌      | 441/1228 [01:08<02:00,  6.55it/s]
 36%|███▌      | 442/1228 [01:09<02:19,  5.65it/s]
 36%|███▌      | 443/1228 [01:09<02:08,  6.09it/s]
 36%|███▌      | 444/1228 [01:09<01:57,  6.66it/s]
 36%|███▌      | 445/1228 [01:09<01:55,  6.79it/s]
 36%|███▋      | 446/1228 [01:09<01:55,  6.75it/s]
 36%|███▋      | 447/1228 [01:09<01:47,  7.28it/s]
 36%|███▋      | 448/1228 [01:09<01:55,  6.73it/s]
 37%|███▋      | 449/1228 [01:10<02:00,  6.49it/s]
 37%|███▋      | 450/1228 [01:10<01:51,  6.96it/s]
 37%|███▋      | 451/1228 [01:10<01:57,  6.62it/s]
 37%|███▋      | 452/1228 [01:10<01:55,  6.71it/s]
 37%|███▋      | 453/1228 [01:10<01:59,  6.50it/s]
 37%|███▋      | 454/1228 [01:10<02:04,  6.22it/s]
 37%|███▋      | 455/1228 [01:10<01:57,  6.58it/s]
 37%|███▋      | 456/1228 [01:11<01:57,  6.58it/s]
 37%|███▋      | 457/1228 [01:11<01:49,  7.02it/s]
 37%|███▋      | 458/1228 [01:11<01:49,  7.03it/s]
 37%|███▋      | 459/1228 [01:11<01:56,  6.60it/s]
 37%|███▋      | 460/1228 [01:11<01:47,  7.17it/s]
 38%|███▊      | 461/1228 [01:11<01:49,  7.00it/s]
 38%|███▊      | 462/1228 [01:11<01:52,  6.82it/s]
 38%|███▊      | 463/1228 [01:12<01:49,  7.00it/s]
 38%|███▊      | 464/1228 [01:12<01:53,  6.71it/s]
 38%|███▊      | 465/1228 [01:12<01:57,  6.49it/s]
 38%|███▊      | 466/1228 [01:12<01:54,  6.66it/s]
 38%|███▊      | 467/1228 [01:12<01:54,  6.67it/s]
 38%|███▊      | 468/1228 [01:12<02:02,  6.22it/s]
 38%|███▊      | 469/1228 [01:13<02:00,  6.31it/s]
 38%|███▊      | 470/1228 [01:13<02:04,  6.07it/s]
 38%|███▊      | 471/1228 [01:13<01:57,  6.47it/s]
 38%|███▊      | 472/1228 [01:13<02:04,  6.10it/s]
 39%|███▊      | 473/1228 [01:13<02:02,  6.19it/s]
 39%|███▊      | 474/1228 [01:13<02:06,  5.96it/s]
 39%|███▊      | 475/1228 [01:14<02:04,  6.03it/s]
 39%|███▉      | 476/1228 [01:14<02:01,  6.20it/s]
 39%|███▉      | 477/1228 [01:14<01:59,  6.27it/s]
 39%|███▉      | 478/1228 [01:14<02:07,  5.87it/s]
 39%|███▉      | 479/1228 [01:14<02:12,  5.67it/s]
 39%|███▉      | 480/1228 [01:14<02:03,  6.07it/s]
 39%|███▉      | 481/1228 [01:15<02:07,  5.84it/s]
 39%|███▉      | 482/1228 [01:15<02:03,  6.04it/s]
 39%|███▉      | 483/1228 [01:15<02:04,  5.98it/s]
 39%|███▉      | 484/1228 [01:15<02:04,  5.99it/s]
 39%|███▉      | 485/1228 [01:15<02:05,  5.92it/s]
 40%|███▉      | 486/1228 [01:15<02:03,  6.01it/s]
 40%|███▉      | 487/1228 [01:16<02:02,  6.03it/s]
 40%|███▉      | 488/1228 [01:16<02:02,  6.03it/s]
 40%|███▉      | 489/1228 [01:16<01:52,  6.57it/s]
 40%|███▉      | 490/1228 [01:16<01:50,  6.70it/s]
 40%|███▉      | 491/1228 [01:16<01:42,  7.17it/s]
 40%|████      | 492/1228 [01:16<01:48,  6.80it/s]
 40%|████      | 493/1228 [01:16<01:49,  6.70it/s]
 40%|████      | 494/1228 [01:17<01:52,  6.55it/s]
 40%|████      | 495/1228 [01:17<01:46,  6.90it/s]
 40%|████      | 496/1228 [01:17<01:43,  7.05it/s]
 40%|████      | 497/1228 [01:17<01:50,  6.59it/s]
 41%|████      | 498/1228 [01:17<01:52,  6.51it/s]
 41%|████      | 499/1228 [01:17<01:51,  6.55it/s]
 41%|████      | 500/1228 [01:17<01:44,  6.94it/s]
 41%|████      | 501/1228 [01:18<01:48,  6.71it/s]
 41%|████      | 502/1228 [01:18<01:48,  6.72it/s]
 41%|████      | 503/1228 [01:18<01:46,  6.83it/s]
 41%|████      | 504/1228 [01:18<01:47,  6.72it/s]
 41%|████      | 505/1228 [01:18<01:55,  6.23it/s]
 41%|████      | 506/1228 [01:18<01:53,  6.35it/s]
 41%|████▏     | 507/1228 [01:19<01:53,  6.36it/s]
 41%|████▏     | 508/1228 [01:19<01:51,  6.44it/s]
 41%|████▏     | 509/1228 [01:19<01:42,  7.00it/s]
 42%|████▏     | 510/1228 [01:19<01:47,  6.66it/s]
 42%|████▏     | 511/1228 [01:19<01:41,  7.05it/s]
 42%|████▏     | 512/1228 [01:19<01:56,  6.17it/s]
 42%|████▏     | 513/1228 [01:20<02:04,  5.76it/s]
 42%|████▏     | 514/1228 [01:20<01:51,  6.39it/s]
 42%|████▏     | 515/1228 [01:20<01:52,  6.34it/s]
 42%|████▏     | 516/1228 [01:20<01:51,  6.39it/s]
 42%|████▏     | 517/1228 [01:20<01:53,  6.28it/s]
 42%|████▏     | 518/1228 [01:20<01:52,  6.33it/s]
 42%|████▏     | 519/1228 [01:20<01:51,  6.38it/s]
 42%|████▏     | 520/1228 [01:21<01:52,  6.31it/s]
 42%|████▏     | 521/1228 [01:21<01:45,  6.68it/s]
 43%|████▎     | 522/1228 [01:21<01:54,  6.18it/s]
 43%|████▎     | 523/1228 [01:21<01:58,  5.95it/s]
 43%|████▎     | 524/1228 [01:21<02:07,  5.54it/s]
 43%|████▎     | 525/1228 [01:21<01:57,  5.98it/s]
 43%|████▎     | 526/1228 [01:22<01:55,  6.07it/s]
 43%|████▎     | 527/1228 [01:22<01:53,  6.18it/s]
 43%|████▎     | 528/1228 [01:22<01:49,  6.41it/s]
 43%|████▎     | 529/1228 [01:22<01:51,  6.27it/s]
 43%|████▎     | 530/1228 [01:22<01:53,  6.17it/s]
 43%|████▎     | 531/1228 [01:22<01:50,  6.30it/s]
 43%|████▎     | 532/1228 [01:23<01:47,  6.46it/s]
 43%|████▎     | 533/1228 [01:23<01:52,  6.16it/s]
 43%|████▎     | 534/1228 [01:23<01:51,  6.20it/s]
 44%|████▎     | 535/1228 [01:23<01:47,  6.46it/s]
 44%|████▎     | 536/1228 [01:23<01:45,  6.56it/s]
 44%|████▎     | 537/1228 [01:23<01:51,  6.17it/s]
 44%|████▍     | 538/1228 [01:23<01:52,  6.11it/s]
 44%|████▍     | 539/1228 [01:24<01:44,  6.62it/s]
 44%|████▍     | 540/1228 [01:24<01:46,  6.47it/s]
 44%|████▍     | 541/1228 [01:24<01:46,  6.45it/s]
 44%|████▍     | 542/1228 [01:24<01:46,  6.45it/s]
 44%|████▍     | 543/1228 [01:24<01:47,  6.35it/s]
 44%|████▍     | 544/1228 [01:24<01:42,  6.68it/s]
 44%|████▍     | 545/1228 [01:25<01:41,  6.70it/s]
 44%|████▍     | 546/1228 [01:25<01:38,  6.89it/s]
 45%|████▍     | 547/1228 [01:25<01:38,  6.92it/s]
 45%|████▍     | 548/1228 [01:25<01:36,  7.03it/s]
 45%|████▍     | 549/1228 [01:25<01:36,  7.01it/s]
 45%|████▍     | 550/1228 [01:25<01:36,  7.04it/s]
 45%|████▍     | 551/1228 [01:25<01:44,  6.46it/s]
 45%|████▍     | 552/1228 [01:26<01:46,  6.37it/s]
 45%|████▌     | 553/1228 [01:26<01:52,  6.02it/s]
 45%|████▌     | 555/1228 [01:26<01:44,  6.46it/s]
 45%|████▌     | 556/1228 [01:26<01:45,  6.38it/s]
 45%|████▌     | 557/1228 [01:26<01:47,  6.24it/s]
 45%|████▌     | 558/1228 [01:27<01:47,  6.21it/s]
 46%|████▌     | 559/1228 [01:27<01:44,  6.41it/s]
 46%|████▌     | 560/1228 [01:27<01:45,  6.34it/s]
 46%|████▌     | 561/1228 [01:27<01:41,  6.57it/s]
 46%|████▌     | 562/1228 [01:27<01:43,  6.41it/s]
 46%|████▌     | 563/1228 [01:27<01:41,  6.53it/s]
 46%|████▌     | 564/1228 [01:27<01:36,  6.91it/s]
 46%|████▌     | 565/1228 [01:28<01:40,  6.60it/s]
 46%|████▌     | 566/1228 [01:28<01:39,  6.67it/s]
 46%|████▌     | 567/1228 [01:28<01:45,  6.27it/s]
 46%|████▋     | 568/1228 [01:28<01:40,  6.54it/s]
 46%|████▋     | 569/1228 [01:28<01:33,  7.04it/s]
 46%|████▋     | 570/1228 [01:28<01:35,  6.89it/s]
 46%|████▋     | 571/1228 [01:29<01:48,  6.04it/s]
 47%|████▋     | 572/1228 [01:29<01:48,  6.05it/s]
 47%|████▋     | 573/1228 [01:29<01:45,  6.23it/s]
 47%|████▋     | 574/1228 [01:29<01:42,  6.38it/s]
 47%|████▋     | 575/1228 [01:29<01:53,  5.74it/s]
 47%|████▋     | 576/1228 [01:29<01:46,  6.13it/s]
 47%|████▋     | 577/1228 [01:30<01:45,  6.17it/s]
 47%|████▋     | 578/1228 [01:30<01:42,  6.36it/s]
 47%|████▋     | 579/1228 [01:30<01:46,  6.11it/s]
 47%|████▋     | 580/1228 [01:30<01:43,  6.24it/s]
 47%|████▋     | 581/1228 [01:30<01:40,  6.41it/s]
 47%|████▋     | 582/1228 [01:30<01:44,  6.21it/s]
 47%|████▋     | 583/1228 [01:30<01:49,  5.87it/s]
 48%|████▊     | 584/1228 [01:31<01:46,  6.05it/s]
 48%|████▊     | 585/1228 [01:31<01:47,  5.98it/s]
 48%|████▊     | 586/1228 [01:31<01:41,  6.32it/s]
 48%|████▊     | 587/1228 [01:31<01:37,  6.59it/s]
 48%|████▊     | 588/1228 [01:31<01:40,  6.38it/s]
 48%|████▊     | 589/1228 [01:31<01:44,  6.09it/s]
 48%|████▊     | 590/1228 [01:32<01:36,  6.62it/s]
 48%|████▊     | 591/1228 [01:32<01:33,  6.83it/s]
 48%|████▊     | 592/1228 [01:32<01:36,  6.57it/s]
 48%|████▊     | 593/1228 [01:32<01:29,  7.06it/s]
 48%|████▊     | 594/1228 [01:32<01:31,  6.94it/s]
 48%|████▊     | 595/1228 [01:32<01:31,  6.92it/s]
 49%|████▊     | 596/1228 [01:32<01:30,  6.95it/s]
 49%|████▊     | 597/1228 [01:33<01:26,  7.26it/s]
 49%|████▊     | 598/1228 [01:33<01:29,  7.01it/s]
 49%|████▉     | 599/1228 [01:33<01:34,  6.69it/s]
 49%|████▉     | 600/1228 [01:33<01:39,  6.28it/s]
 49%|████▉     | 601/1228 [01:33<01:40,  6.22it/s]
 49%|████▉     | 602/1228 [01:33<01:45,  5.95it/s]
 49%|████▉     | 603/1228 [01:34<01:38,  6.35it/s]
 49%|████▉     | 604/1228 [01:34<01:46,  5.86it/s]
 49%|████▉     | 605/1228 [01:34<01:44,  5.95it/s]
 49%|████▉     | 606/1228 [01:34<01:47,  5.80it/s]
 49%|████▉     | 607/1228 [01:34<01:41,  6.13it/s]
 50%|████▉     | 608/1228 [01:34<01:41,  6.11it/s]
 50%|████▉     | 609/1228 [01:35<01:44,  5.93it/s]
 50%|████▉     | 610/1228 [01:35<01:42,  6.02it/s]
 50%|████▉     | 611/1228 [01:35<01:41,  6.06it/s]
 50%|████▉     | 612/1228 [01:35<01:38,  6.24it/s]
 50%|████▉     | 613/1228 [01:35<01:34,  6.50it/s]
 50%|█████     | 614/1228 [01:35<01:37,  6.31it/s]
 50%|█████     | 615/1228 [01:36<01:53,  5.40it/s]
 50%|█████     | 616/1228 [01:36<01:53,  5.38it/s]
 50%|█████     | 617/1228 [01:36<01:44,  5.84it/s]
 50%|█████     | 618/1228 [01:36<01:41,  5.98it/s]
 50%|█████     | 619/1228 [01:36<01:41,  5.99it/s]
 50%|█████     | 620/1228 [01:36<01:49,  5.54it/s]
 51%|█████     | 621/1228 [01:37<01:42,  5.95it/s]
 51%|█████     | 622/1228 [01:37<01:36,  6.25it/s]
 51%|█████     | 623/1228 [01:37<01:36,  6.24it/s]
 51%|█████     | 624/1228 [01:37<01:40,  6.01it/s]
 51%|█████     | 625/1228 [01:37<01:33,  6.42it/s]
 51%|█████     | 626/1228 [01:37<01:27,  6.88it/s]
 51%|█████     | 627/1228 [01:38<01:34,  6.34it/s]
 51%|█████     | 628/1228 [01:38<01:31,  6.58it/s]
 51%|█████     | 629/1228 [01:38<01:26,  6.96it/s]
 51%|█████▏    | 630/1228 [01:38<01:30,  6.61it/s]
 51%|█████▏    | 631/1228 [01:38<01:37,  6.12it/s]
 51%|█████▏    | 632/1228 [01:38<01:33,  6.35it/s]
 52%|█████▏    | 633/1228 [01:38<01:38,  6.06it/s]
 52%|█████▏    | 634/1228 [01:39<01:28,  6.69it/s]
 52%|█████▏    | 635/1228 [01:39<01:28,  6.67it/s]
 52%|█████▏    | 636/1228 [01:39<01:36,  6.16it/s]
 52%|█████▏    | 637/1228 [01:39<01:31,  6.46it/s]
 52%|█████▏    | 638/1228 [01:39<01:27,  6.77it/s]
 52%|█████▏    | 639/1228 [01:39<01:35,  6.19it/s]
 52%|█████▏    | 640/1228 [01:40<01:35,  6.13it/s]
 52%|█████▏    | 641/1228 [01:40<01:34,  6.24it/s]
 52%|█████▏    | 642/1228 [01:40<01:32,  6.31it/s]
 52%|█████▏    | 643/1228 [01:40<01:34,  6.20it/s]
 52%|█████▏    | 644/1228 [01:40<01:36,  6.04it/s]
 53%|█████▎    | 645/1228 [01:40<01:43,  5.63it/s]
 53%|█████▎    | 646/1228 [01:41<01:36,  6.05it/s]
 53%|█████▎    | 647/1228 [01:41<01:28,  6.56it/s]
 53%|█████▎    | 648/1228 [01:41<01:31,  6.32it/s]
 53%|█████▎    | 649/1228 [01:41<01:31,  6.34it/s]
 53%|█████▎    | 650/1228 [01:41<01:42,  5.64it/s]
 53%|█████▎    | 651/1228 [01:41<01:34,  6.08it/s]
 53%|█████▎    | 652/1228 [01:42<01:38,  5.86it/s]
 53%|█████▎    | 653/1228 [01:42<01:34,  6.11it/s]
 53%|█████▎    | 654/1228 [01:42<01:25,  6.74it/s]
 53%|█████▎    | 655/1228 [01:42<01:41,  5.62it/s]
 53%|█████▎    | 656/1228 [01:42<01:42,  5.61it/s]
 54%|█████▎    | 657/1228 [01:42<01:42,  5.58it/s]
 54%|█████▎    | 658/1228 [01:43<01:42,  5.55it/s]
 54%|█████▎    | 659/1228 [01:43<01:35,  5.99it/s]
 54%|█████▎    | 660/1228 [01:43<01:27,  6.47it/s]
 54%|█████▍    | 661/1228 [01:43<01:21,  6.94it/s]
 54%|█████▍    | 662/1228 [01:43<01:27,  6.44it/s]
 54%|█████▍    | 663/1228 [01:43<01:29,  6.28it/s]
 54%|█████▍    | 664/1228 [01:43<01:26,  6.55it/s]
 54%|█████▍    | 665/1228 [01:44<01:17,  7.26it/s]
 54%|█████▍    | 666/1228 [01:44<01:19,  7.05it/s]
 54%|█████▍    | 667/1228 [01:44<01:19,  7.06it/s]
 54%|█████▍    | 668/1228 [01:44<01:24,  6.63it/s]
 54%|█████▍    | 669/1228 [01:44<01:27,  6.39it/s]
 55%|█████▍    | 670/1228 [01:44<01:30,  6.17it/s]
 55%|█████▍    | 671/1228 [01:45<01:29,  6.25it/s]
 55%|█████▍    | 672/1228 [01:45<01:29,  6.18it/s]
 55%|█████▍    | 673/1228 [01:45<01:22,  6.71it/s]
 55%|█████▍    | 674/1228 [01:45<01:22,  6.72it/s]
 55%|█████▍    | 675/1228 [01:45<01:21,  6.82it/s]
 55%|█████▌    | 676/1228 [01:45<01:18,  7.05it/s]
 55%|█████▌    | 677/1228 [01:45<01:18,  7.02it/s]
 55%|█████▌    | 678/1228 [01:46<01:20,  6.82it/s]
 55%|█████▌    | 679/1228 [01:46<01:22,  6.66it/s]
 55%|█████▌    | 680/1228 [01:46<01:24,  6.49it/s]
 55%|█████▌    | 681/1228 [01:46<01:28,  6.15it/s]
 56%|█████▌    | 682/1228 [01:46<01:26,  6.30it/s]
 56%|█████▌    | 683/1228 [01:46<01:28,  6.19it/s]
 56%|█████▌    | 684/1228 [01:46<01:25,  6.33it/s]
 56%|█████▌    | 685/1228 [01:47<01:28,  6.16it/s]
 56%|█████▌    | 686/1228 [01:47<01:24,  6.45it/s]
 56%|█████▌    | 687/1228 [01:47<01:18,  6.87it/s]
 56%|█████▌    | 688/1228 [01:47<01:17,  6.97it/s]
 56%|█████▌    | 689/1228 [01:47<01:12,  7.47it/s]
 56%|█████▌    | 690/1228 [01:47<01:13,  7.37it/s]
 56%|█████▋    | 691/1228 [01:47<01:19,  6.74it/s]
 56%|█████▋    | 692/1228 [01:48<01:17,  6.93it/s]
 56%|█████▋    | 693/1228 [01:48<01:14,  7.19it/s]
 57%|█████▋    | 694/1228 [01:48<01:21,  6.58it/s]
 57%|█████▋    | 695/1228 [01:48<01:19,  6.72it/s]
 57%|█████▋    | 696/1228 [01:48<01:16,  6.92it/s]
 57%|█████▋    | 697/1228 [01:48<01:19,  6.65it/s]
 57%|█████▋    | 698/1228 [01:49<01:22,  6.40it/s]
 57%|█████▋    | 699/1228 [01:49<01:25,  6.17it/s]
 57%|█████▋    | 700/1228 [01:49<01:27,  6.03it/s]
 57%|█████▋    | 701/1228 [01:49<01:23,  6.30it/s]
 57%|█████▋    | 702/1228 [01:49<01:30,  5.79it/s]
 57%|█████▋    | 703/1228 [01:49<01:27,  6.01it/s]
 57%|█████▋    | 704/1228 [01:50<01:27,  5.98it/s]
 57%|█████▋    | 705/1228 [01:50<01:23,  6.26it/s]
 57%|█████▋    | 706/1228 [01:50<01:17,  6.70it/s]
 58%|█████▊    | 707/1228 [01:50<01:17,  6.69it/s]
 58%|█████▊    | 708/1228 [01:50<01:22,  6.28it/s]
 58%|█████▊    | 709/1228 [01:50<01:18,  6.57it/s]
 58%|█████▊    | 710/1228 [01:50<01:12,  7.11it/s]
 58%|█████▊    | 711/1228 [01:51<01:14,  6.94it/s]
 58%|█████▊    | 712/1228 [01:51<01:16,  6.72it/s]
 58%|█████▊    | 713/1228 [01:51<01:20,  6.41it/s]
 58%|█████▊    | 714/1228 [01:51<01:21,  6.28it/s]
 58%|█████▊    | 715/1228 [01:51<01:24,  6.04it/s]
 58%|█████▊    | 716/1228 [01:51<01:21,  6.28it/s]
 58%|█████▊    | 717/1228 [01:52<01:25,  5.95it/s]
 58%|█████▊    | 718/1228 [01:52<01:26,  5.89it/s]
 59%|█████▊    | 719/1228 [01:52<01:26,  5.90it/s]
 59%|█████▊    | 720/1228 [01:52<01:22,  6.16it/s]
 59%|█████▊    | 721/1228 [01:52<01:18,  6.46it/s]
 59%|█████▉    | 722/1228 [01:52<01:14,  6.77it/s]
 59%|█████▉    | 723/1228 [01:53<01:19,  6.37it/s]
 59%|█████▉    | 724/1228 [01:53<01:17,  6.51it/s]
 59%|█████▉    | 725/1228 [01:53<01:13,  6.89it/s]
 59%|█████▉    | 726/1228 [01:53<01:10,  7.15it/s]
 59%|█████▉    | 727/1228 [01:53<01:16,  6.51it/s]
 59%|█████▉    | 728/1228 [01:53<01:16,  6.55it/s]
 59%|█████▉    | 729/1228 [01:53<01:16,  6.49it/s]
 59%|█████▉    | 730/1228 [01:54<01:20,  6.20it/s]
 60%|█████▉    | 731/1228 [01:54<01:18,  6.29it/s]
 60%|█████▉    | 732/1228 [01:54<01:18,  6.31it/s]
 60%|█████▉    | 733/1228 [01:54<01:17,  6.37it/s]
 60%|█████▉    | 734/1228 [01:54<01:17,  6.34it/s]
 60%|█████▉    | 735/1228 [01:54<01:13,  6.69it/s]
 60%|█████▉    | 736/1228 [01:54<01:12,  6.75it/s]
 60%|██████    | 737/1228 [01:55<01:18,  6.29it/s]
 60%|██████    | 738/1228 [01:55<01:15,  6.52it/s]
 60%|██████    | 739/1228 [01:55<01:13,  6.64it/s]
 60%|██████    | 740/1228 [01:55<01:12,  6.71it/s]
 60%|██████    | 741/1228 [01:55<01:08,  7.15it/s]
 60%|██████    | 742/1228 [01:55<01:09,  7.00it/s]
 61%|██████    | 743/1228 [01:56<01:11,  6.79it/s]
 61%|██████    | 744/1228 [01:56<01:12,  6.66it/s]
 61%|██████    | 745/1228 [01:56<01:06,  7.29it/s]
 61%|██████    | 746/1228 [01:56<01:04,  7.43it/s]
 61%|██████    | 747/1228 [01:56<01:02,  7.71it/s]
 61%|██████    | 748/1228 [01:56<01:02,  7.71it/s]
 61%|██████    | 749/1228 [01:56<01:02,  7.70it/s]
 61%|██████    | 750/1228 [01:56<01:01,  7.77it/s]
 61%|██████    | 751/1228 [01:57<01:09,  6.82it/s]
 61%|██████    | 752/1228 [01:57<01:19,  5.97it/s]
 61%|██████▏   | 753/1228 [01:57<01:22,  5.74it/s]
 61%|██████▏   | 754/1228 [01:57<01:16,  6.20it/s]
 61%|██████▏   | 755/1228 [01:57<01:21,  5.80it/s]
 62%|██████▏   | 756/1228 [01:58<01:21,  5.79it/s]
 62%|██████▏   | 757/1228 [01:58<01:15,  6.23it/s]
 62%|██████▏   | 758/1228 [01:58<01:23,  5.66it/s]
 62%|██████▏   | 759/1228 [01:58<01:15,  6.22it/s]
 62%|██████▏   | 760/1228 [01:58<01:22,  5.65it/s]
 62%|██████▏   | 761/1228 [01:58<01:18,  5.93it/s]
 62%|██████▏   | 762/1228 [01:59<01:18,  5.92it/s]
 62%|██████▏   | 763/1228 [01:59<01:15,  6.15it/s]
 62%|██████▏   | 764/1228 [01:59<01:09,  6.69it/s]
 62%|██████▏   | 765/1228 [01:59<01:21,  5.66it/s]
 62%|██████▏   | 766/1228 [01:59<01:13,  6.27it/s]
 62%|██████▏   | 767/1228 [01:59<01:09,  6.66it/s]
 63%|██████▎   | 768/1228 [01:59<01:04,  7.15it/s]
 63%|██████▎   | 769/1228 [02:00<01:06,  6.87it/s]
 63%|██████▎   | 770/1228 [02:00<01:05,  7.04it/s]
 63%|██████▎   | 771/1228 [02:00<01:09,  6.56it/s]
 63%|██████▎   | 772/1228 [02:00<01:06,  6.83it/s]
 63%|██████▎   | 773/1228 [02:00<01:04,  7.09it/s]
 63%|██████▎   | 774/1228 [02:00<01:03,  7.15it/s]
 63%|██████▎   | 775/1228 [02:00<01:02,  7.29it/s]
 63%|██████▎   | 776/1228 [02:01<01:04,  7.04it/s]
 63%|██████▎   | 777/1228 [02:01<01:05,  6.85it/s]
 63%|██████▎   | 778/1228 [02:01<01:06,  6.79it/s]
 63%|██████▎   | 779/1228 [02:01<01:02,  7.21it/s]
 64%|██████▎   | 780/1228 [02:01<01:01,  7.26it/s]
 64%|██████▎   | 781/1228 [02:01<00:58,  7.61it/s]
 64%|██████▎   | 782/1228 [02:01<01:08,  6.51it/s]
 64%|██████▍   | 783/1228 [02:02<01:09,  6.44it/s]
 64%|██████▍   | 784/1228 [02:02<01:06,  6.71it/s]
 64%|██████▍   | 785/1228 [02:02<01:07,  6.58it/s]
 64%|██████▍   | 786/1228 [02:02<01:17,  5.72it/s]
 64%|██████▍   | 787/1228 [02:02<01:13,  5.98it/s]
 64%|██████▍   | 788/1228 [02:02<01:13,  6.03it/s]
 64%|██████▍   | 789/1228 [02:03<01:06,  6.58it/s]
 64%|██████▍   | 790/1228 [02:03<01:10,  6.23it/s]
 64%|██████▍   | 791/1228 [02:03<01:16,  5.73it/s]
 64%|██████▍   | 792/1228 [02:03<01:14,  5.87it/s]
 65%|██████▍   | 793/1228 [02:03<01:10,  6.13it/s]
 65%|██████▍   | 794/1228 [02:03<01:14,  5.79it/s]
 65%|██████▍   | 795/1228 [02:04<01:11,  6.01it/s]
 65%|██████▍   | 796/1228 [02:04<01:12,  5.95it/s]
 65%|██████▍   | 797/1228 [02:04<01:10,  6.13it/s]
 65%|██████▍   | 798/1228 [02:04<01:08,  6.28it/s]
 65%|██████▌   | 799/1228 [02:04<01:07,  6.31it/s]
 65%|██████▌   | 800/1228 [02:04<01:08,  6.26it/s]
 65%|██████▌   | 801/1228 [02:05<01:05,  6.54it/s]
 65%|██████▌   | 802/1228 [02:05<01:03,  6.71it/s]
 65%|██████▌   | 803/1228 [02:05<01:03,  6.71it/s]
 65%|██████▌   | 804/1228 [02:05<01:08,  6.23it/s]
 66%|██████▌   | 805/1228 [02:05<01:14,  5.69it/s]
 66%|██████▌   | 806/1228 [02:05<01:07,  6.25it/s]
 66%|██████▌   | 807/1228 [02:05<01:07,  6.23it/s]
 66%|██████▌   | 808/1228 [02:06<01:09,  6.02it/s]
 66%|██████▌   | 809/1228 [02:06<01:07,  6.17it/s]
 66%|██████▌   | 810/1228 [02:06<01:08,  6.10it/s]
 66%|██████▌   | 811/1228 [02:06<01:11,  5.83it/s]
 66%|██████▌   | 812/1228 [02:06<01:07,  6.15it/s]
 66%|██████▌   | 813/1228 [02:06<01:06,  6.22it/s]
 66%|██████▋   | 814/1228 [02:07<01:08,  6.04it/s]
 66%|██████▋   | 815/1228 [02:07<01:15,  5.51it/s]
 66%|██████▋   | 816/1228 [02:07<01:12,  5.71it/s]
 67%|██████▋   | 817/1228 [02:07<01:06,  6.22it/s]
 67%|██████▋   | 818/1228 [02:07<01:06,  6.14it/s]
 67%|██████▋   | 819/1228 [02:07<01:01,  6.63it/s]
 67%|██████▋   | 820/1228 [02:08<01:02,  6.57it/s]
 67%|██████▋   | 821/1228 [02:08<01:02,  6.46it/s]
 67%|██████▋   | 822/1228 [02:08<01:00,  6.67it/s]
 67%|██████▋   | 823/1228 [02:08<01:01,  6.62it/s]
 67%|██████▋   | 824/1228 [02:08<00:58,  6.88it/s]
 67%|██████▋   | 825/1228 [02:08<00:57,  7.03it/s]
 67%|██████▋   | 826/1228 [02:09<01:03,  6.32it/s]
 67%|██████▋   | 827/1228 [02:09<00:57,  7.01it/s]
 67%|██████▋   | 828/1228 [02:09<00:58,  6.85it/s]
 68%|██████▊   | 829/1228 [02:09<01:03,  6.25it/s]
 68%|██████▊   | 830/1228 [02:09<01:02,  6.39it/s]
 68%|██████▊   | 831/1228 [02:09<01:03,  6.21it/s]
 68%|██████▊   | 832/1228 [02:09<01:04,  6.11it/s]
 68%|██████▊   | 833/1228 [02:10<01:00,  6.56it/s]
 68%|██████▊   | 834/1228 [02:10<01:00,  6.48it/s]
 68%|██████▊   | 835/1228 [02:10<00:59,  6.64it/s]
 68%|██████▊   | 836/1228 [02:10<00:55,  7.01it/s]
 68%|██████▊   | 837/1228 [02:10<00:57,  6.81it/s]
 68%|██████▊   | 838/1228 [02:10<00:53,  7.29it/s]
 68%|██████▊   | 839/1228 [02:10<00:55,  7.02it/s]
 68%|██████▊   | 840/1228 [02:11<00:56,  6.90it/s]
 68%|██████▊   | 841/1228 [02:11<00:59,  6.48it/s]
 69%|██████▊   | 842/1228 [02:11<01:02,  6.15it/s]
 69%|██████▊   | 843/1228 [02:11<01:08,  5.62it/s]
 69%|██████▊   | 844/1228 [02:11<01:03,  6.01it/s]
 69%|██████▉   | 845/1228 [02:11<00:58,  6.57it/s]
 69%|██████▉   | 846/1228 [02:12<00:52,  7.25it/s]
 69%|██████▉   | 847/1228 [02:12<00:50,  7.50it/s]
 69%|██████▉   | 848/1228 [02:12<00:49,  7.65it/s]
 69%|██████▉   | 849/1228 [02:12<00:48,  7.88it/s]
 69%|██████▉   | 850/1228 [02:12<00:53,  7.11it/s]
 69%|██████▉   | 851/1228 [02:12<00:55,  6.85it/s]
 69%|██████▉   | 852/1228 [02:12<00:57,  6.54it/s]
 69%|██████▉   | 853/1228 [02:13<00:55,  6.78it/s]
 70%|██████▉   | 854/1228 [02:13<00:55,  6.70it/s]
 70%|██████▉   | 855/1228 [02:13<00:56,  6.62it/s]
 70%|██████▉   | 856/1228 [02:13<00:56,  6.56it/s]
 70%|██████▉   | 857/1228 [02:13<00:55,  6.68it/s]
 70%|██████▉   | 858/1228 [02:13<00:58,  6.31it/s]
 70%|██████▉   | 859/1228 [02:14<01:02,  5.92it/s]
 70%|███████   | 860/1228 [02:14<01:01,  6.01it/s]
 70%|███████   | 861/1228 [02:14<01:02,  5.87it/s]
 70%|███████   | 862/1228 [02:14<00:57,  6.33it/s]
 70%|███████   | 863/1228 [02:14<00:58,  6.21it/s]
 70%|███████   | 864/1228 [02:14<01:01,  5.91it/s]
 70%|███████   | 865/1228 [02:14<00:59,  6.15it/s]
 71%|███████   | 866/1228 [02:15<01:04,  5.65it/s]
 71%|███████   | 867/1228 [02:15<01:01,  5.86it/s]
 71%|███████   | 868/1228 [02:15<00:59,  6.01it/s]
 71%|███████   | 869/1228 [02:15<00:57,  6.20it/s]
 71%|███████   | 870/1228 [02:15<00:56,  6.39it/s]
 71%|███████   | 871/1228 [02:15<00:57,  6.19it/s]
 71%|███████   | 872/1228 [02:16<00:56,  6.31it/s]
 71%|███████   | 873/1228 [02:16<00:55,  6.44it/s]
 71%|███████   | 874/1228 [02:16<00:52,  6.78it/s]
 71%|███████▏  | 875/1228 [02:16<00:47,  7.43it/s]
 71%|███████▏  | 876/1228 [02:16<00:48,  7.28it/s]
 71%|███████▏  | 877/1228 [02:16<00:49,  7.10it/s]
 71%|███████▏  | 878/1228 [02:16<00:50,  6.90it/s]
 72%|███████▏  | 879/1228 [02:17<00:51,  6.72it/s]
 72%|███████▏  | 880/1228 [02:17<00:51,  6.70it/s]
 72%|███████▏  | 881/1228 [02:17<00:47,  7.28it/s]
 72%|███████▏  | 882/1228 [02:17<00:46,  7.50it/s]
 72%|███████▏  | 883/1228 [02:17<00:47,  7.28it/s]
 72%|███████▏  | 884/1228 [02:17<00:49,  6.96it/s]
 72%|███████▏  | 885/1228 [02:17<00:51,  6.65it/s]
 72%|███████▏  | 886/1228 [02:18<00:52,  6.49it/s]
 72%|███████▏  | 887/1228 [02:18<00:51,  6.58it/s]
 72%|███████▏  | 888/1228 [02:18<00:46,  7.24it/s]
 72%|███████▏  | 889/1228 [02:18<00:50,  6.66it/s]
 72%|███████▏  | 890/1228 [02:18<00:50,  6.72it/s]
 73%|███████▎  | 891/1228 [02:18<00:55,  6.13it/s]
 73%|███████▎  | 892/1228 [02:19<00:51,  6.54it/s]
 73%|███████▎  | 893/1228 [02:19<00:48,  6.95it/s]
 73%|███████▎  | 894/1228 [02:19<00:48,  6.89it/s]
 73%|███████▎  | 895/1228 [02:19<00:52,  6.34it/s]
 73%|███████▎  | 896/1228 [02:19<00:50,  6.56it/s]
 73%|███████▎  | 897/1228 [02:19<00:48,  6.76it/s]
 73%|███████▎  | 898/1228 [02:19<00:49,  6.71it/s]
 73%|███████▎  | 899/1228 [02:20<00:53,  6.14it/s]
 73%|███████▎  | 900/1228 [02:20<00:51,  6.35it/s]
 73%|███████▎  | 901/1228 [02:20<00:55,  5.89it/s]
 73%|███████▎  | 902/1228 [02:20<00:54,  6.00it/s]
 74%|███████▎  | 903/1228 [02:20<00:51,  6.25it/s]
 74%|███████▎  | 904/1228 [02:20<00:48,  6.66it/s]
 74%|███████▎  | 905/1228 [02:21<00:50,  6.43it/s]
 74%|███████▍  | 906/1228 [02:21<00:49,  6.50it/s]
 74%|███████▍  | 907/1228 [02:21<00:49,  6.50it/s]
 74%|███████▍  | 908/1228 [02:21<00:53,  5.95it/s]
 74%|███████▍  | 909/1228 [02:21<00:52,  6.10it/s]
 74%|███████▍  | 910/1228 [02:21<00:50,  6.33it/s]
 74%|███████▍  | 911/1228 [02:22<00:51,  6.19it/s]
 74%|███████▍  | 912/1228 [02:22<00:47,  6.69it/s]
 74%|███████▍  | 913/1228 [02:22<00:48,  6.46it/s]
 74%|███████▍  | 914/1228 [02:22<00:48,  6.42it/s]
 75%|███████▍  | 915/1228 [02:22<00:47,  6.55it/s]
 75%|███████▍  | 916/1228 [02:22<00:44,  7.00it/s]
 75%|███████▍  | 917/1228 [02:22<00:43,  7.20it/s]
 75%|███████▍  | 918/1228 [02:22<00:42,  7.27it/s]
 75%|███████▍  | 919/1228 [02:23<00:41,  7.46it/s]
 75%|███████▍  | 920/1228 [02:23<00:45,  6.83it/s]
 75%|███████▌  | 921/1228 [02:23<00:47,  6.40it/s]
 75%|███████▌  | 922/1228 [02:23<00:46,  6.54it/s]
 75%|███████▌  | 923/1228 [02:23<00:44,  6.88it/s]
 75%|███████▌  | 924/1228 [02:23<00:44,  6.84it/s]
 75%|███████▌  | 925/1228 [02:24<00:46,  6.51it/s]
 75%|███████▌  | 926/1228 [02:24<00:46,  6.46it/s]
 75%|███████▌  | 927/1228 [02:24<00:46,  6.53it/s]
 76%|███████▌  | 928/1228 [02:24<00:53,  5.61it/s]
 76%|███████▌  | 929/1228 [02:24<00:52,  5.71it/s]
 76%|███████▌  | 930/1228 [02:24<00:49,  6.04it/s]
 76%|███████▌  | 931/1228 [02:25<00:52,  5.65it/s]
 76%|███████▌  | 932/1228 [02:25<00:48,  6.09it/s]
 76%|███████▌  | 933/1228 [02:25<00:46,  6.39it/s]
 76%|███████▌  | 934/1228 [02:25<00:45,  6.51it/s]
 76%|███████▌  | 935/1228 [02:25<00:46,  6.34it/s]
 76%|███████▌  | 936/1228 [02:25<00:44,  6.61it/s]
 76%|███████▋  | 937/1228 [02:26<00:47,  6.12it/s]
 76%|███████▋  | 938/1228 [02:26<00:45,  6.43it/s]
 76%|███████▋  | 939/1228 [02:26<00:46,  6.22it/s]
 77%|███████▋  | 940/1228 [02:26<00:45,  6.36it/s]
 77%|███████▋  | 941/1228 [02:26<00:46,  6.11it/s]
 77%|███████▋  | 942/1228 [02:26<00:47,  6.07it/s]
 77%|███████▋  | 943/1228 [02:27<00:49,  5.71it/s]
 77%|███████▋  | 944/1228 [02:27<00:46,  6.05it/s]
 77%|███████▋  | 945/1228 [02:27<00:44,  6.38it/s]
 77%|███████▋  | 946/1228 [02:27<00:44,  6.30it/s]
 77%|███████▋  | 947/1228 [02:27<00:43,  6.42it/s]
 77%|███████▋  | 948/1228 [02:27<00:40,  6.91it/s]
 77%|███████▋  | 949/1228 [02:27<00:39,  7.10it/s]
 77%|███████▋  | 950/1228 [02:28<00:41,  6.76it/s]
 77%|███████▋  | 951/1228 [02:28<00:40,  6.82it/s]
 78%|███████▊  | 952/1228 [02:28<00:40,  6.83it/s]
 78%|███████▊  | 953/1228 [02:28<00:42,  6.52it/s]
 78%|███████▊  | 954/1228 [02:28<00:38,  7.19it/s]
 78%|███████▊  | 955/1228 [02:28<00:41,  6.58it/s]
 78%|███████▊  | 956/1228 [02:28<00:39,  6.87it/s]
 78%|███████▊  | 957/1228 [02:29<00:39,  6.78it/s]
 78%|███████▊  | 958/1228 [02:29<00:36,  7.33it/s]
 78%|███████▊  | 959/1228 [02:29<00:37,  7.18it/s]
 78%|███████▊  | 960/1228 [02:29<00:36,  7.36it/s]
 78%|███████▊  | 961/1228 [02:29<00:36,  7.32it/s]
 78%|███████▊  | 962/1228 [02:29<00:35,  7.48it/s]
 78%|███████▊  | 963/1228 [02:29<00:38,  6.97it/s]
 79%|███████▊  | 964/1228 [02:30<00:39,  6.61it/s]
 79%|███████▊  | 965/1228 [02:30<00:41,  6.37it/s]
 79%|███████▊  | 966/1228 [02:30<00:41,  6.31it/s]
 79%|███████▊  | 967/1228 [02:30<00:41,  6.31it/s]
 79%|███████▉  | 968/1228 [02:30<00:39,  6.63it/s]
 79%|███████▉  | 969/1228 [02:30<00:40,  6.33it/s]
 79%|███████▉  | 970/1228 [02:31<00:38,  6.68it/s]
 79%|███████▉  | 971/1228 [02:31<00:43,  5.96it/s]
 79%|███████▉  | 972/1228 [02:31<00:43,  5.95it/s]
 79%|███████▉  | 973/1228 [02:31<00:43,  5.90it/s]
 79%|███████▉  | 974/1228 [02:31<00:44,  5.65it/s]
 79%|███████▉  | 975/1228 [02:31<00:41,  6.07it/s]
 79%|███████▉  | 976/1228 [02:32<00:41,  6.02it/s]
 80%|███████▉  | 977/1228 [02:32<00:38,  6.50it/s]
 80%|███████▉  | 978/1228 [02:32<00:37,  6.60it/s]
 80%|███████▉  | 979/1228 [02:32<00:38,  6.51it/s]
 80%|███████▉  | 980/1228 [02:32<00:39,  6.23it/s]
 80%|███████▉  | 981/1228 [02:32<00:36,  6.68it/s]
 80%|███████▉  | 982/1228 [02:32<00:38,  6.41it/s]
 80%|████████  | 983/1228 [02:33<00:38,  6.29it/s]
 80%|████████  | 984/1228 [02:33<00:36,  6.77it/s]
 80%|████████  | 985/1228 [02:33<00:40,  6.03it/s]
 80%|████████  | 986/1228 [02:33<00:39,  6.13it/s]
 80%|████████  | 987/1228 [02:33<00:43,  5.59it/s]
 80%|████████  | 988/1228 [02:34<00:44,  5.45it/s]
 81%|████████  | 989/1228 [02:34<00:42,  5.63it/s]
 81%|████████  | 990/1228 [02:34<00:39,  6.03it/s]
 81%|████████  | 991/1228 [02:34<00:40,  5.90it/s]
 81%|████████  | 992/1228 [02:34<00:39,  5.93it/s]
 81%|████████  | 993/1228 [02:34<00:40,  5.81it/s]
 81%|████████  | 994/1228 [02:34<00:38,  6.13it/s]
 81%|████████  | 995/1228 [02:35<00:35,  6.53it/s]
 81%|████████  | 996/1228 [02:35<00:37,  6.26it/s]
 81%|████████  | 997/1228 [02:35<00:36,  6.34it/s]
 81%|████████▏ | 998/1228 [02:35<00:37,  6.15it/s]
 81%|████████▏ | 999/1228 [02:35<00:37,  6.03it/s]
 81%|████████▏ | 1000/1228 [02:35<00:37,  6.11it/s]
 82%|████████▏ | 1001/1228 [02:36<00:36,  6.16it/s]
 82%|████████▏ | 1002/1228 [02:36<00:34,  6.56it/s]
 82%|████████▏ | 1003/1228 [02:36<00:35,  6.31it/s]
 82%|████████▏ | 1004/1228 [02:36<00:36,  6.20it/s]
 82%|████████▏ | 1005/1228 [02:36<00:38,  5.74it/s]
 82%|████████▏ | 1006/1228 [02:36<00:38,  5.71it/s]
 82%|████████▏ | 1007/1228 [02:37<00:39,  5.53it/s]
 82%|████████▏ | 1008/1228 [02:37<00:39,  5.56it/s]
 82%|████████▏ | 1009/1228 [02:37<00:37,  5.78it/s]
 82%|████████▏ | 1010/1228 [02:37<00:37,  5.86it/s]
 82%|████████▏ | 1011/1228 [02:37<00:36,  5.94it/s]
 82%|████████▏ | 1012/1228 [02:37<00:34,  6.32it/s]
 82%|████████▏ | 1013/1228 [02:38<00:33,  6.50it/s]
 83%|████████▎ | 1014/1228 [02:38<00:35,  6.06it/s]
 83%|████████▎ | 1015/1228 [02:38<00:35,  6.05it/s]
 83%|████████▎ | 1016/1228 [02:38<00:35,  6.01it/s]
 83%|████████▎ | 1017/1228 [02:38<00:37,  5.59it/s]
 83%|████████▎ | 1018/1228 [02:39<00:37,  5.56it/s]
 83%|████████▎ | 1019/1228 [02:39<00:36,  5.72it/s]
 83%|████████▎ | 1020/1228 [02:39<00:36,  5.66it/s]
 83%|████████▎ | 1021/1228 [02:39<00:34,  6.04it/s]
 83%|████████▎ | 1022/1228 [02:39<00:32,  6.26it/s]
 83%|████████▎ | 1023/1228 [02:39<00:34,  5.86it/s]
 83%|████████▎ | 1024/1228 [02:39<00:31,  6.41it/s]
 83%|████████▎ | 1025/1228 [02:40<00:32,  6.21it/s]
 84%|████████▎ | 1026/1228 [02:40<00:32,  6.24it/s]
 84%|████████▎ | 1027/1228 [02:40<00:33,  6.06it/s]
 84%|████████▎ | 1028/1228 [02:40<00:32,  6.20it/s]
 84%|████████▍ | 1029/1228 [02:40<00:32,  6.09it/s]
 84%|████████▍ | 1030/1228 [02:40<00:33,  5.99it/s]
 84%|████████▍ | 1031/1228 [02:41<00:32,  6.07it/s]
 84%|████████▍ | 1032/1228 [02:41<00:30,  6.41it/s]
 84%|████████▍ | 1033/1228 [02:41<00:28,  6.79it/s]
 84%|████████▍ | 1034/1228 [02:41<00:29,  6.50it/s]
 84%|████████▍ | 1035/1228 [02:41<00:28,  6.86it/s]
 84%|████████▍ | 1036/1228 [02:41<00:27,  7.09it/s]
 84%|████████▍ | 1037/1228 [02:41<00:26,  7.24it/s]
 85%|████████▍ | 1038/1228 [02:42<00:25,  7.32it/s]
 85%|████████▍ | 1039/1228 [02:42<00:26,  7.25it/s]
 85%|████████▍ | 1040/1228 [02:42<00:25,  7.44it/s]
 85%|████████▍ | 1041/1228 [02:42<00:26,  7.10it/s]
 85%|████████▍ | 1042/1228 [02:42<00:27,  6.71it/s]
 85%|████████▍ | 1043/1228 [02:42<00:29,  6.34it/s]
 85%|████████▌ | 1044/1228 [02:42<00:26,  6.88it/s]
 85%|████████▌ | 1045/1228 [02:43<00:26,  6.95it/s]
 85%|████████▌ | 1046/1228 [02:43<00:29,  6.23it/s]
 85%|████████▌ | 1047/1228 [02:43<00:28,  6.35it/s]
 85%|████████▌ | 1048/1228 [02:43<00:30,  5.99it/s]
 85%|████████▌ | 1049/1228 [02:43<00:28,  6.26it/s]
 86%|████████▌ | 1050/1228 [02:43<00:28,  6.18it/s]
 86%|████████▌ | 1051/1228 [02:44<00:28,  6.22it/s]
 86%|████████▌ | 1052/1228 [02:44<00:25,  6.78it/s]
 86%|████████▌ | 1053/1228 [02:44<00:27,  6.27it/s]
 86%|████████▌ | 1054/1228 [02:44<00:28,  6.03it/s]
 86%|████████▌ | 1055/1228 [02:44<00:28,  6.15it/s]
 86%|████████▌ | 1056/1228 [02:44<00:28,  6.12it/s]
 86%|████████▌ | 1057/1228 [02:45<01:01,  2.80it/s]
 86%|████████▌ | 1058/1228 [02:45<00:49,  3.41it/s]
 86%|████████▌ | 1059/1228 [02:46<00:42,  3.97it/s]
 86%|████████▋ | 1060/1228 [02:46<00:37,  4.50it/s]
 86%|████████▋ | 1061/1228 [02:46<00:33,  5.01it/s]
 86%|████████▋ | 1062/1228 [02:46<00:30,  5.45it/s]
 87%|████████▋ | 1063/1228 [02:46<00:27,  5.96it/s]
 87%|████████▋ | 1064/1228 [02:46<00:28,  5.70it/s]
 87%|████████▋ | 1065/1228 [02:46<00:28,  5.71it/s]
 87%|████████▋ | 1066/1228 [02:47<00:28,  5.74it/s]
 87%|████████▋ | 1067/1228 [02:47<00:26,  6.03it/s]
 87%|████████▋ | 1068/1228 [02:47<00:25,  6.27it/s]
 87%|████████▋ | 1069/1228 [02:47<00:25,  6.18it/s]
 87%|████████▋ | 1070/1228 [02:47<00:24,  6.38it/s]
 87%|████████▋ | 1071/1228 [02:47<00:26,  6.01it/s]
 87%|████████▋ | 1072/1228 [02:48<00:26,  5.93it/s]
 87%|████████▋ | 1073/1228 [02:48<00:24,  6.23it/s]
 87%|████████▋ | 1074/1228 [02:48<00:23,  6.56it/s]
 88%|████████▊ | 1075/1228 [02:48<00:23,  6.46it/s]
 88%|████████▊ | 1076/1228 [02:48<00:21,  6.93it/s]
 88%|████████▊ | 1077/1228 [02:48<00:20,  7.29it/s]
 88%|████████▊ | 1078/1228 [02:48<00:22,  6.78it/s]
 88%|████████▊ | 1079/1228 [02:49<00:22,  6.57it/s]
 88%|████████▊ | 1080/1228 [02:49<00:24,  6.07it/s]
 88%|████████▊ | 1081/1228 [02:49<00:25,  5.68it/s]
 88%|████████▊ | 1082/1228 [02:49<00:26,  5.57it/s]
 88%|████████▊ | 1083/1228 [02:49<00:24,  5.93it/s]
 88%|████████▊ | 1084/1228 [02:49<00:23,  6.06it/s]
 88%|████████▊ | 1085/1228 [02:50<00:21,  6.54it/s]
 88%|████████▊ | 1086/1228 [02:50<00:20,  6.79it/s]
 89%|████████▊ | 1087/1228 [02:50<00:22,  6.30it/s]
 89%|████████▊ | 1088/1228 [02:50<00:22,  6.28it/s]
 89%|████████▊ | 1089/1228 [02:50<00:22,  6.20it/s]
 89%|████████▉ | 1090/1228 [02:50<00:20,  6.77it/s]
 89%|████████▉ | 1091/1228 [02:51<00:20,  6.80it/s]
 89%|████████▉ | 1092/1228 [02:51<00:21,  6.44it/s]
 89%|████████▉ | 1093/1228 [02:51<00:20,  6.59it/s]
 89%|████████▉ | 1094/1228 [02:51<00:20,  6.69it/s]
 89%|████████▉ | 1095/1228 [02:51<00:19,  6.72it/s]
 89%|████████▉ | 1096/1228 [02:51<00:19,  6.92it/s]
 89%|████████▉ | 1097/1228 [02:51<00:19,  6.69it/s]
 89%|████████▉ | 1098/1228 [02:52<00:17,  7.37it/s]
 89%|████████▉ | 1099/1228 [02:52<00:17,  7.19it/s]
 90%|████████▉ | 1100/1228 [02:52<00:18,  6.96it/s]
 90%|████████▉ | 1101/1228 [02:52<00:17,  7.18it/s]
 90%|████████▉ | 1102/1228 [02:52<00:17,  7.04it/s]
 90%|████████▉ | 1103/1228 [02:52<00:18,  6.92it/s]
 90%|████████▉ | 1104/1228 [02:52<00:19,  6.45it/s]
 90%|████████▉ | 1105/1228 [02:53<00:18,  6.62it/s]
 90%|█████████ | 1106/1228 [02:53<00:18,  6.46it/s]
 90%|█████████ | 1107/1228 [02:53<00:20,  5.91it/s]
 90%|█████████ | 1108/1228 [02:53<00:21,  5.63it/s]
 90%|█████████ | 1109/1228 [02:53<00:19,  6.03it/s]
 90%|█████████ | 1110/1228 [02:53<00:19,  5.95it/s]
 90%|█████████ | 1111/1228 [02:54<00:18,  6.47it/s]
 91%|█████████ | 1112/1228 [02:54<00:17,  6.59it/s]
 91%|█████████ | 1113/1228 [02:54<00:17,  6.54it/s]
 91%|█████████ | 1114/1228 [02:54<00:17,  6.41it/s]
 91%|█████████ | 1115/1228 [02:54<00:16,  6.65it/s]
 91%|█████████ | 1116/1228 [02:54<00:16,  6.94it/s]
 91%|█████████ | 1117/1228 [02:55<00:19,  5.78it/s]
 91%|█████████ | 1118/1228 [02:55<00:18,  6.00it/s]
 91%|█████████ | 1119/1228 [02:55<00:18,  5.86it/s]
 91%|█████████ | 1120/1228 [02:55<00:16,  6.46it/s]
 91%|█████████▏| 1121/1228 [02:55<00:16,  6.64it/s]
 91%|█████████▏| 1122/1228 [02:55<00:15,  6.96it/s]
 91%|█████████▏| 1123/1228 [02:55<00:15,  6.78it/s]
 92%|█████████▏| 1124/1228 [02:56<00:15,  6.56it/s]
 92%|█████████▏| 1125/1228 [02:56<00:15,  6.55it/s]
 92%|█████████▏| 1126/1228 [02:56<00:15,  6.58it/s]
 92%|█████████▏| 1127/1228 [02:56<00:16,  6.20it/s]
 92%|█████████▏| 1128/1228 [02:56<00:16,  6.07it/s]
 92%|█████████▏| 1129/1228 [02:56<00:16,  6.09it/s]
 92%|█████████▏| 1130/1228 [02:57<00:15,  6.36it/s]
 92%|█████████▏| 1131/1228 [02:57<00:14,  6.76it/s]
 92%|█████████▏| 1132/1228 [02:57<00:14,  6.67it/s]
 92%|█████████▏| 1133/1228 [02:57<00:14,  6.70it/s]
 92%|█████████▏| 1134/1228 [02:57<00:14,  6.41it/s]
 92%|█████████▏| 1135/1228 [02:57<00:14,  6.57it/s]
 93%|█████████▎| 1136/1228 [02:57<00:13,  6.60it/s]
 93%|█████████▎| 1137/1228 [02:58<00:14,  6.14it/s]
 93%|█████████▎| 1138/1228 [02:58<00:13,  6.49it/s]
 93%|█████████▎| 1139/1228 [02:58<00:13,  6.40it/s]
 93%|█████████▎| 1140/1228 [02:58<00:13,  6.44it/s]
 93%|█████████▎| 1141/1228 [02:58<00:14,  6.06it/s]
 93%|█████████▎| 1142/1228 [02:58<00:14,  5.80it/s]
 93%|█████████▎| 1143/1228 [02:59<00:13,  6.50it/s]
 93%|█████████▎| 1144/1228 [02:59<00:13,  6.35it/s]
 93%|█████████▎| 1145/1228 [02:59<00:14,  5.80it/s]
 93%|█████████▎| 1146/1228 [02:59<00:14,  5.73it/s]
 93%|█████████▎| 1147/1228 [02:59<00:12,  6.33it/s]
 93%|█████████▎| 1148/1228 [02:59<00:12,  6.25it/s]
 94%|█████████▎| 1149/1228 [03:00<00:12,  6.40it/s]
 94%|█████████▎| 1150/1228 [03:00<00:13,  5.69it/s]
 94%|█████████▎| 1151/1228 [03:00<00:12,  6.18it/s]
 94%|█████████▍| 1152/1228 [03:00<00:11,  6.44it/s]
 94%|█████████▍| 1153/1228 [03:00<00:11,  6.35it/s]
 94%|█████████▍| 1154/1228 [03:00<00:11,  6.42it/s]
 94%|█████████▍| 1155/1228 [03:01<00:12,  6.07it/s]
 94%|█████████▍| 1156/1228 [03:01<00:11,  6.53it/s]
 94%|█████████▍| 1157/1228 [03:01<00:11,  6.45it/s]
 94%|█████████▍| 1158/1228 [03:01<00:10,  6.43it/s]
 94%|█████████▍| 1159/1228 [03:01<00:10,  6.39it/s]
 94%|█████████▍| 1160/1228 [03:01<00:10,  6.29it/s]
 95%|█████████▍| 1161/1228 [03:01<00:09,  6.82it/s]
 95%|█████████▍| 1162/1228 [03:02<00:09,  7.01it/s]
 95%|█████████▍| 1163/1228 [03:02<00:09,  7.04it/s]
 95%|█████████▍| 1164/1228 [03:02<00:09,  6.98it/s]
 95%|█████████▍| 1165/1228 [03:02<00:09,  6.52it/s]
 95%|█████████▍| 1166/1228 [03:02<00:09,  6.42it/s]
 95%|█████████▌| 1167/1228 [03:02<00:09,  6.78it/s]
 95%|█████████▌| 1168/1228 [03:02<00:09,  6.56it/s]
 95%|█████████▌| 1169/1228 [03:03<00:09,  6.14it/s]
 95%|█████████▌| 1170/1228 [03:03<00:08,  6.47it/s]
 95%|█████████▌| 1171/1228 [03:03<00:08,  6.77it/s]
 95%|█████████▌| 1172/1228 [03:03<00:07,  7.05it/s]
 96%|█████████▌| 1173/1228 [03:03<00:08,  6.29it/s]
 96%|█████████▌| 1174/1228 [03:03<00:09,  5.88it/s]
 96%|█████████▌| 1175/1228 [03:04<00:08,  5.96it/s]
 96%|█████████▌| 1176/1228 [03:04<00:08,  6.45it/s]
 96%|█████████▌| 1177/1228 [03:04<00:08,  6.34it/s]
 96%|█████████▌| 1178/1228 [03:04<00:07,  6.55it/s]
 96%|█████████▌| 1179/1228 [03:04<00:07,  6.36it/s]
 96%|█████████▌| 1180/1228 [03:04<00:07,  6.65it/s]
 96%|█████████▌| 1181/1228 [03:05<00:07,  6.53it/s]
 96%|█████████▋| 1182/1228 [03:05<00:07,  6.23it/s]
 96%|█████████▋| 1183/1228 [03:05<00:07,  6.16it/s]
 96%|█████████▋| 1184/1228 [03:05<00:06,  6.53it/s]
 96%|█████████▋| 1185/1228 [03:05<00:06,  6.33it/s]
 97%|█████████▋| 1186/1228 [03:05<00:06,  6.64it/s]
 97%|█████████▋| 1187/1228 [03:05<00:06,  6.38it/s]
 97%|█████████▋| 1188/1228 [03:06<00:05,  6.79it/s]
 97%|█████████▋| 1189/1228 [03:06<00:05,  7.03it/s]
 97%|█████████▋| 1190/1228 [03:06<00:05,  7.02it/s]
 97%|█████████▋| 1191/1228 [03:06<00:05,  7.32it/s]
 97%|█████████▋| 1192/1228 [03:06<00:04,  7.49it/s]
 97%|█████████▋| 1193/1228 [03:06<00:04,  7.38it/s]
 97%|█████████▋| 1194/1228 [03:06<00:05,  6.61it/s]
 97%|█████████▋| 1195/1228 [03:07<00:04,  6.95it/s]
 97%|█████████▋| 1196/1228 [03:07<00:04,  7.19it/s]
 97%|█████████▋| 1197/1228 [03:07<00:04,  6.91it/s]
 98%|█████████▊| 1198/1228 [03:07<00:04,  6.55it/s]
 98%|█████████▊| 1199/1228 [03:07<00:04,  7.16it/s]
 98%|█████████▊| 1200/1228 [03:07<00:03,  7.25it/s]
 98%|█████████▊| 1201/1228 [03:07<00:03,  7.39it/s]
 98%|█████████▊| 1202/1228 [03:08<00:03,  6.78it/s]
 98%|█████████▊| 1203/1228 [03:08<00:03,  6.53it/s]
 98%|█████████▊| 1204/1228 [03:08<00:03,  6.25it/s]
 98%|█████████▊| 1205/1228 [03:08<00:03,  6.25it/s]
 98%|█████████▊| 1206/1228 [03:08<00:03,  6.11it/s]
 98%|█████████▊| 1207/1228 [03:08<00:03,  6.17it/s]
 98%|█████████▊| 1208/1228 [03:09<00:03,  6.23it/s]
 98%|█████████▊| 1209/1228 [03:09<00:03,  5.87it/s]
 99%|█████████▊| 1210/1228 [03:09<00:02,  6.41it/s]
 99%|█████████▊| 1211/1228 [03:09<00:02,  6.04it/s]
 99%|█████████▊| 1212/1228 [03:09<00:02,  6.07it/s]
 99%|█████████▉| 1213/1228 [03:09<00:02,  6.19it/s]
 99%|█████████▉| 1214/1228 [03:10<00:02,  6.41it/s]
 99%|█████████▉| 1215/1228 [03:10<00:02,  6.13it/s]
 99%|█████████▉| 1216/1228 [03:10<00:02,  5.84it/s]
 99%|█████████▉| 1217/1228 [03:10<00:01,  5.68it/s]
 99%|█████████▉| 1218/1228 [03:10<00:01,  5.69it/s]
 99%|█████████▉| 1219/1228 [03:10<00:01,  6.32it/s]
 99%|█████████▉| 1220/1228 [03:11<00:01,  6.35it/s]
 99%|█████████▉| 1221/1228 [03:11<00:01,  6.02it/s]
100%|█████████▉| 1222/1228 [03:11<00:00,  6.49it/s]
100%|█████████▉| 1223/1228 [03:11<00:00,  6.11it/s]
100%|█████████▉| 1224/1228 [03:11<00:00,  5.83it/s]
100%|█████████▉| 1225/1228 [03:11<00:00,  5.30it/s]
100%|█████████▉| 1226/1228 [03:12<00:00,  5.61it/s]
100%|█████████▉| 1227/1228 [03:12<00:00,  5.69it/s]
100%|██████████| 1228/1228 [03:12<00:00,  5.92it/s]
100%|██████████| 1228/1228 [03:12<00:00,  6.38it/s]
[transformers] You passed `num_labels=4` which is incompatible to the `id2label` map of length `1`.
Majority label: SUPPORTS

Dev score with cross-encoder retrieval + majority label:
Evidence Retrieval F-score (F)    = 0.255726
Claim Classification Accuracy (A) = 0.441558
Harmonic Mean of F and A          = 0.323879
Wrote outputs_notebook_v8_mini2v2\dev-cross-encoder-retrieval-majority.json
Loading classifier: cross-encoder/ms-marco-MiniLM-L-6-v2

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]
Loading weights: 100%|██████████| 105/105 [00:00<00:00, 8612.87it/s]
[transformers] [1mBertForSequenceClassification LOAD REPORT[0m from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key               | Status   |                                                                                       
------------------+----------+---------------------------------------------------------------------------------------
classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1, 384]) vs model:torch.Size([4, 384])
classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([1]) vs model:torch.Size([4])          

Notes:
- MISMATCH:	ckpt weights were loaded, but they did not match the original empty weight shapes.
Classifier parameters: 22,714,756
Classifier training evidence source: retrieved
Classifier train avg evidence count: 4.268729641693811
Raw class weights: {'SUPPORTS': 0.5915221571922302, 'REFUTES': 1.5427135229110718, 'NOT_ENOUGH_INFO': 0.7953367829322815, 'DISPUTED': 2.475806474685669}
Using class weights: False

Classifier epoch 1/4:   0%|          | 0/77 [00:00<?, ?it/s]
Classifier epoch 1/4:   1%|▏         | 1/77 [00:00<00:09,  7.77it/s]
Classifier epoch 1/4:   3%|▎         | 2/77 [00:00<00:09,  8.08it/s]
Classifier epoch 1/4:   5%|▌         | 4/77 [00:00<00:06, 11.48it/s]
Classifier epoch 1/4:   9%|▉         | 7/77 [00:00<00:04, 16.25it/s]
Classifier epoch 1/4:  13%|█▎        | 10/77 [00:00<00:03, 19.57it/s]
Classifier epoch 1/4:  17%|█▋        | 13/77 [00:00<00:02, 21.42it/s]
Classifier epoch 1/4:  21%|██        | 16/77 [00:00<00:02, 22.76it/s]
Classifier epoch 1/4:  25%|██▍       | 19/77 [00:00<00:02, 23.89it/s]
Classifier epoch 1/4:  29%|██▊       | 22/77 [00:01<00:02, 24.58it/s]
Classifier epoch 1/4:  32%|███▏      | 25/77 [00:01<00:02, 24.97it/s]
Classifier epoch 1/4:  36%|███▋      | 28/77 [00:01<00:01, 25.22it/s]
Classifier epoch 1/4:  40%|████      | 31/77 [00:01<00:01, 25.67it/s]
Classifier epoch 1/4:  44%|████▍     | 34/77 [00:01<00:01, 25.48it/s]
Classifier epoch 1/4:  48%|████▊     | 37/77 [00:01<00:01, 25.69it/s]
Classifier epoch 1/4:  52%|█████▏    | 40/77 [00:01<00:01, 25.74it/s]
Classifier epoch 1/4:  56%|█████▌    | 43/77 [00:01<00:01, 25.87it/s]
Classifier epoch 1/4:  60%|█████▉    | 46/77 [00:02<00:01, 26.10it/s]
Classifier epoch 1/4:  64%|██████▎   | 49/77 [00:02<00:01, 26.30it/s]
Classifier epoch 1/4:  68%|██████▊   | 52/77 [00:02<00:00, 26.44it/s]
Classifier epoch 1/4:  71%|███████▏  | 55/77 [00:02<00:00, 26.29it/s]
Classifier epoch 1/4:  75%|███████▌  | 58/77 [00:02<00:00, 26.41it/s]
Classifier epoch 1/4:  79%|███████▉  | 61/77 [00:02<00:00, 26.64it/s]
Classifier epoch 1/4:  83%|████████▎ | 64/77 [00:02<00:00, 26.74it/s]
Classifier epoch 1/4:  87%|████████▋ | 67/77 [00:02<00:00, 26.60it/s]
Classifier epoch 1/4:  91%|█████████ | 70/77 [00:02<00:00, 26.63it/s]
Classifier epoch 1/4:  95%|█████████▍| 73/77 [00:03<00:00, 26.69it/s]
Classifier epoch 1/4:  99%|█████████▊| 76/77 [00:03<00:00, 26.60it/s]
Classifier epoch 1/4: 100%|██████████| 77/77 [00:03<00:00, 24.33it/s]
Epoch 1: loss=1.3928 | dev F=0.2557 A=0.2857 H=0.2699 | preds=(SUPPORTS:0, REFUTES:32, NOT_ENOUGH_INFO:122, DISPUTED:0) | time=3.3s

Classifier epoch 2/4:   0%|          | 0/77 [00:00<?, ?it/s]
Classifier epoch 2/4:   4%|▍         | 3/77 [00:00<00:03, 22.50it/s]
Classifier epoch 2/4:   8%|▊         | 6/77 [00:00<00:03, 22.39it/s]
Classifier epoch 2/4:  12%|█▏        | 9/77 [00:00<00:02, 23.59it/s]
Classifier epoch 2/4:  16%|█▌        | 12/77 [00:00<00:02, 24.45it/s]
Classifier epoch 2/4:  19%|█▉        | 15/77 [00:00<00:02, 24.91it/s]
Classifier epoch 2/4:  23%|██▎       | 18/77 [00:00<00:02, 25.04it/s]
Classifier epoch 2/4:  27%|██▋       | 21/77 [00:00<00:02, 25.29it/s]
Classifier epoch 2/4:  31%|███       | 24/77 [00:00<00:02, 25.45it/s]
Classifier epoch 2/4:  35%|███▌      | 27/77 [00:01<00:01, 25.59it/s]
Classifier epoch 2/4:  39%|███▉      | 30/77 [00:01<00:01, 25.68it/s]
Classifier epoch 2/4:  43%|████▎     | 33/77 [00:01<00:01, 25.73it/s]
Classifier epoch 2/4:  47%|████▋     | 36/77 [00:01<00:01, 25.69it/s]
Classifier epoch 2/4:  51%|█████     | 39/77 [00:01<00:01, 25.94it/s]
Classifier epoch 2/4:  55%|█████▍    | 42/77 [00:01<00:01, 26.03it/s]
Classifier epoch 2/4:  58%|█████▊    | 45/77 [00:01<00:01, 25.97it/s]
Classifier epoch 2/4:  62%|██████▏   | 48/77 [00:01<00:01, 25.88it/s]
Classifier epoch 2/4:  66%|██████▌   | 51/77 [00:02<00:01, 25.88it/s]
Classifier epoch 2/4:  70%|███████   | 54/77 [00:02<00:00, 25.86it/s]
Classifier epoch 2/4:  74%|███████▍  | 57/77 [00:02<00:00, 25.79it/s]
Classifier epoch 2/4:  78%|███████▊  | 60/77 [00:02<00:00, 25.50it/s]
Classifier epoch 2/4:  82%|████████▏ | 63/77 [00:02<00:00, 25.58it/s]
Classifier epoch 2/4:  86%|████████▌ | 66/77 [00:02<00:00, 25.59it/s]
Classifier epoch 2/4:  90%|████████▉ | 69/77 [00:02<00:00, 25.49it/s]
Classifier epoch 2/4:  94%|█████████▎| 72/77 [00:02<00:00, 25.40it/s]
Classifier epoch 2/4:  97%|█████████▋| 75/77 [00:02<00:00, 25.18it/s]
Classifier epoch 2/4: 100%|██████████| 77/77 [00:03<00:00, 25.40it/s]
Epoch 2: loss=1.3163 | dev F=0.2557 A=0.4416 H=0.3239 | preds=(SUPPORTS:138, REFUTES:8, NOT_ENOUGH_INFO:8, DISPUTED:0) | time=3.2s

Classifier epoch 3/4:   0%|          | 0/77 [00:00<?, ?it/s]
Classifier epoch 3/4:   4%|▍         | 3/77 [00:00<00:03, 24.48it/s]
Classifier epoch 3/4:   8%|▊         | 6/77 [00:00<00:02, 24.42it/s]
Classifier epoch 3/4:  12%|█▏        | 9/77 [00:00<00:02, 24.89it/s]
Classifier epoch 3/4:  16%|█▌        | 12/77 [00:00<00:02, 25.04it/s]
Classifier epoch 3/4:  19%|█▉        | 15/77 [00:00<00:02, 24.90it/s]
Classifier epoch 3/4:  23%|██▎       | 18/77 [00:00<00:02, 25.07it/s]
Classifier epoch 3/4:  27%|██▋       | 21/77 [00:00<00:02, 25.18it/s]
Classifier epoch 3/4:  31%|███       | 24/77 [00:00<00:02, 25.29it/s]
Classifier epoch 3/4:  35%|███▌      | 27/77 [00:01<00:01, 25.34it/s]
Classifier epoch 3/4:  39%|███▉      | 30/77 [00:01<00:01, 25.53it/s]
Classifier epoch 3/4:  43%|████▎     | 33/77 [00:01<00:01, 25.52it/s]
Classifier epoch 3/4:  47%|████▋     | 36/77 [00:01<00:01, 25.80it/s]
Classifier epoch 3/4:  51%|█████     | 39/77 [00:01<00:01, 25.82it/s]
Classifier epoch 3/4:  55%|█████▍    | 42/77 [00:01<00:01, 25.82it/s]
Classifier epoch 3/4:  58%|█████▊    | 45/77 [00:01<00:01, 25.57it/s]
Classifier epoch 3/4:  62%|██████▏   | 48/77 [00:01<00:01, 25.61it/s]
Classifier epoch 3/4:  66%|██████▌   | 51/77 [00:02<00:01, 25.49it/s]
Classifier epoch 3/4:  70%|███████   | 54/77 [00:02<00:00, 25.26it/s]
Classifier epoch 3/4:  74%|███████▍  | 57/77 [00:02<00:00, 25.30it/s]
Classifier epoch 3/4:  78%|███████▊  | 60/77 [00:02<00:00, 25.44it/s]
Classifier epoch 3/4:  82%|████████▏ | 63/77 [00:02<00:00, 25.61it/s]
Classifier epoch 3/4:  86%|████████▌ | 66/77 [00:02<00:00, 25.76it/s]
Classifier epoch 3/4:  90%|████████▉ | 69/77 [00:02<00:00, 25.92it/s]
Classifier epoch 3/4:  94%|█████████▎| 72/77 [00:02<00:00, 25.92it/s]
Classifier epoch 3/4:  97%|█████████▋| 75/77 [00:02<00:00, 25.92it/s]
Classifier epoch 3/4: 100%|██████████| 77/77 [00:03<00:00, 25.53it/s]
Epoch 3: loss=1.2554 | dev F=0.2557 A=0.4286 H=0.3203 | preds=(SUPPORTS:149, REFUTES:5, NOT_ENOUGH_INFO:0, DISPUTED:0) | time=3.2s

Classifier epoch 4/4:   0%|          | 0/77 [00:00<?, ?it/s]
Classifier epoch 4/4:   4%|▍         | 3/77 [00:00<00:06, 12.21it/s]
Classifier epoch 4/4:   6%|▋         | 5/77 [00:00<00:07,  9.76it/s]
Classifier epoch 4/4:   9%|▉         | 7/77 [00:00<00:07,  9.49it/s]
Classifier epoch 4/4:  10%|█         | 8/77 [00:01<00:11,  6.01it/s]
Classifier epoch 4/4:  13%|█▎        | 10/77 [00:01<00:08,  7.90it/s]
Classifier epoch 4/4:  17%|█▋        | 13/77 [00:01<00:05, 11.01it/s]
Classifier epoch 4/4:  21%|██        | 16/77 [00:01<00:04, 13.47it/s]
Classifier epoch 4/4:  25%|██▍       | 19/77 [00:01<00:03, 15.65it/s]
Classifier epoch 4/4:  29%|██▊       | 22/77 [00:01<00:03, 17.64it/s]
Classifier epoch 4/4:  32%|███▏      | 25/77 [00:01<00:02, 18.58it/s]
Classifier epoch 4/4:  36%|███▋      | 28/77 [00:02<00:02, 20.00it/s]
Classifier epoch 4/4:  40%|████      | 31/77 [00:02<00:02, 21.15it/s]
Classifier epoch 4/4:  44%|████▍     | 34/77 [00:02<00:01, 22.12it/s]
Classifier epoch 4/4:  48%|████▊     | 37/77 [00:02<00:01, 22.73it/s]
Classifier epoch 4/4:  52%|█████▏    | 40/77 [00:02<00:01, 23.24it/s]
Classifier epoch 4/4:  56%|█████▌    | 43/77 [00:02<00:01, 23.88it/s]
Classifier epoch 4/4:  60%|█████▉    | 46/77 [00:02<00:01, 24.31it/s]
Classifier epoch 4/4:  64%|██████▎   | 49/77 [00:02<00:01, 24.69it/s]
Classifier epoch 4/4:  68%|██████▊   | 52/77 [00:03<00:00, 25.03it/s]
Classifier epoch 4/4:  71%|███████▏  | 55/77 [00:03<00:00, 25.03it/s]
Classifier epoch 4/4:  75%|███████▌  | 58/77 [00:03<00:00, 25.10it/s]
Classifier epoch 4/4:  79%|███████▉  | 61/77 [00:03<00:00, 25.54it/s]
Classifier epoch 4/4:  83%|████████▎ | 64/77 [00:03<00:00, 25.77it/s]
Classifier epoch 4/4:  87%|████████▋ | 67/77 [00:03<00:00, 26.03it/s]
Classifier epoch 4/4:  91%|█████████ | 70/77 [00:03<00:00, 26.18it/s]
Classifier epoch 4/4:  95%|█████████▍| 73/77 [00:03<00:00, 26.33it/s]
Classifier epoch 4/4:  99%|█████████▊| 76/77 [00:03<00:00, 26.44it/s]
Classifier epoch 4/4: 100%|██████████| 77/77 [00:03<00:00, 19.50it/s]
Epoch 4: loss=1.2488 | dev F=0.2557 A=0.4286 H=0.3203 | preds=(SUPPORTS:149, REFUTES:5, NOT_ENOUGH_INFO:0, DISPUTED:0) | time=4.1s
Best dev metrics: {'F': 0.25572562358276646, 'A': 0.44155844155844154, 'H': 0.3238789281464502, '_nonmaj': 8}
Final label source: majority (classifier H=0.3239, majority H=0.3239, required H>0.3289)
Classifier-only dev score:
Evidence Retrieval F-score (F)    = 0.255726
Claim Classification Accuracy (A) = 0.441558
Harmonic Mean of F and A          = 0.323879
Wrote outputs_notebook_v8_mini2v2\dev-predictions-classifier.json
Selected final dev score:
Evidence Retrieval F-score (F)    = 0.255726
Claim Classification Accuracy (A) = 0.441558
Harmonic Mean of F and A          = 0.323879
Wrote outputs_notebook_v8_mini2v2\dev-predictions.json
Selected final label source: majority
Running: D:\_Search\_Study\COMP90042-NLP\A3_Group\COMP90042_2026\.venv\Scripts\python.exe eval.py --predictions outputs_notebook_v8_mini2v2\dev-predictions.json --groundtruth data\dev-claims.json
Evidence Retrieval F-score (F)    = 0.25572562358276646
Claim Classification Accuracy (A) = 0.44155844155844154
Harmonic Mean of F and A          = 0.3238789281464502

Confusion matrix for selected final labels: rows=gold, cols=pred
                 SUPPORTS  REFUTES  NOT_ENOUGH_INFO  DISPUTED
SUPPORTS               68        0                0         0
REFUTES                27        0                0         0
NOT_ENOUGH_INFO        41        0                0         0
DISPUTED               18        0                0         0
Classifier-only confusion matrix, for error analysis only:
                 SUPPORTS  REFUTES  NOT_ENOUGH_INFO  DISPUTED
SUPPORTS               63        3                2         0
REFUTES                26        0                1         0
NOT_ENOUGH_INFO        36        0                5         0
DISPUTED               13        5                0         0
             label   n  class_acc  retrieval_F
0         SUPPORTS  68        1.0     0.322444
1          REFUTES  27        0.0     0.108907
2  NOT_ENOUGH_INFO  41        0.0     0.211653
3         DISPUTED  18        0.0     0.324295
Scoring test_final candidates with cross-encoder reranker...

  0%|          | 0/153 [00:00<?, ?it/s]
  1%|          | 1/153 [00:00<00:23,  6.55it/s]
  1%|▏         | 2/153 [00:00<00:22,  6.68it/s]
  2%|▏         | 3/153 [00:00<00:20,  7.35it/s]
  3%|▎         | 4/153 [00:00<00:22,  6.66it/s]
  3%|▎         | 5/153 [00:00<00:20,  7.33it/s]
  4%|▍         | 6/153 [00:00<00:21,  6.78it/s]
  5%|▍         | 7/153 [00:01<00:20,  6.98it/s]
  5%|▌         | 8/153 [00:01<00:20,  6.94it/s]
  6%|▌         | 9/153 [00:01<00:22,  6.27it/s]
  7%|▋         | 10/153 [00:01<00:24,  5.85it/s]
  7%|▋         | 11/153 [00:01<00:23,  5.96it/s]
  8%|▊         | 12/153 [00:01<00:24,  5.71it/s]
  8%|▊         | 13/153 [00:02<00:24,  5.71it/s]
  9%|▉         | 14/153 [00:02<00:24,  5.68it/s]
 10%|▉         | 15/153 [00:02<00:22,  6.01it/s]
 10%|█         | 16/153 [00:02<00:22,  6.16it/s]
 11%|█         | 17/153 [00:02<00:24,  5.62it/s]
 12%|█▏        | 18/153 [00:02<00:22,  5.95it/s]
 12%|█▏        | 19/153 [00:03<00:23,  5.75it/s]
 13%|█▎        | 20/153 [00:03<00:21,  6.11it/s]
 14%|█▎        | 21/153 [00:03<00:20,  6.31it/s]
 14%|█▍        | 22/153 [00:03<00:19,  6.81it/s]
 15%|█▌        | 23/153 [00:03<00:20,  6.24it/s]
 16%|█▌        | 24/153 [00:03<00:21,  5.95it/s]
 16%|█▋        | 25/153 [00:04<00:21,  5.91it/s]
 17%|█▋        | 26/153 [00:04<00:21,  6.02it/s]
 18%|█▊        | 27/153 [00:04<00:22,  5.69it/s]
 18%|█▊        | 28/153 [00:04<00:20,  6.11it/s]
 19%|█▉        | 29/153 [00:04<00:19,  6.26it/s]
 20%|█▉        | 30/153 [00:04<00:20,  6.10it/s]
 20%|██        | 31/153 [00:04<00:18,  6.54it/s]
 21%|██        | 32/153 [00:05<00:19,  6.24it/s]
 22%|██▏       | 33/153 [00:05<00:21,  5.54it/s]
 22%|██▏       | 34/153 [00:05<00:21,  5.64it/s]
 23%|██▎       | 35/153 [00:05<00:19,  6.14it/s]
 24%|██▎       | 36/153 [00:05<00:17,  6.66it/s]
 24%|██▍       | 37/153 [00:05<00:16,  6.84it/s]
 25%|██▍       | 38/153 [00:06<00:17,  6.73it/s]
 25%|██▌       | 39/153 [00:06<00:18,  6.02it/s]
 26%|██▌       | 40/153 [00:06<00:17,  6.47it/s]
 27%|██▋       | 41/153 [00:06<00:17,  6.41it/s]
 27%|██▋       | 42/153 [00:06<00:18,  5.87it/s]
 28%|██▊       | 43/153 [00:06<00:17,  6.28it/s]
 29%|██▉       | 44/153 [00:07<00:17,  6.37it/s]
 29%|██▉       | 45/153 [00:07<00:16,  6.74it/s]
 30%|███       | 46/153 [00:07<00:16,  6.45it/s]
 31%|███       | 47/153 [00:07<00:15,  6.73it/s]
 31%|███▏      | 48/153 [00:07<00:16,  6.42it/s]
 32%|███▏      | 49/153 [00:07<00:14,  7.09it/s]
 33%|███▎      | 50/153 [00:07<00:14,  6.87it/s]
 33%|███▎      | 51/153 [00:08<00:15,  6.80it/s]
 34%|███▍      | 52/153 [00:08<00:15,  6.71it/s]
 35%|███▍      | 53/153 [00:08<00:14,  6.73it/s]
 35%|███▌      | 54/153 [00:08<00:16,  6.09it/s]
 36%|███▌      | 55/153 [00:08<00:15,  6.23it/s]
 37%|███▋      | 56/153 [00:08<00:14,  6.70it/s]
 37%|███▋      | 57/153 [00:09<00:14,  6.82it/s]
 38%|███▊      | 58/153 [00:09<00:13,  7.25it/s]
 39%|███▊      | 59/153 [00:09<00:12,  7.26it/s]
 39%|███▉      | 60/153 [00:09<00:12,  7.44it/s]
 40%|███▉      | 61/153 [00:09<00:12,  7.16it/s]
 41%|████      | 62/153 [00:09<00:12,  7.46it/s]
 41%|████      | 63/153 [00:09<00:12,  7.33it/s]
 42%|████▏     | 64/153 [00:09<00:12,  6.91it/s]
 42%|████▏     | 65/153 [00:10<00:12,  7.09it/s]
 43%|████▎     | 66/153 [00:10<00:13,  6.49it/s]
 44%|████▍     | 67/153 [00:10<00:13,  6.48it/s]
 44%|████▍     | 68/153 [00:10<00:14,  6.01it/s]
 45%|████▌     | 69/153 [00:11<00:20,  4.12it/s]
 46%|████▌     | 70/153 [00:11<00:20,  3.97it/s]
 46%|████▋     | 71/153 [00:11<00:26,  3.13it/s]
 47%|████▋     | 72/153 [00:12<00:23,  3.51it/s]
 48%|████▊     | 73/153 [00:12<00:20,  3.88it/s]
 48%|████▊     | 74/153 [00:12<00:18,  4.34it/s]
 49%|████▉     | 75/153 [00:12<00:16,  4.74it/s]
 50%|████▉     | 76/153 [00:12<00:14,  5.19it/s]
 50%|█████     | 77/153 [00:12<00:13,  5.74it/s]
 51%|█████     | 78/153 [00:12<00:12,  6.24it/s]
 52%|█████▏    | 79/153 [00:13<00:11,  6.50it/s]
 52%|█████▏    | 80/153 [00:13<00:11,  6.40it/s]
 53%|█████▎    | 81/153 [00:13<00:11,  6.34it/s]
 54%|█████▎    | 82/153 [00:13<00:10,  6.66it/s]
 54%|█████▍    | 83/153 [00:13<00:11,  6.31it/s]
 55%|█████▍    | 84/153 [00:13<00:11,  6.26it/s]
 56%|█████▌    | 85/153 [00:14<00:10,  6.48it/s]
 56%|█████▌    | 86/153 [00:14<00:10,  6.35it/s]
 57%|█████▋    | 87/153 [00:14<00:10,  6.44it/s]
 58%|█████▊    | 88/153 [00:14<00:10,  6.00it/s]
 58%|█████▊    | 89/153 [00:14<00:10,  6.35it/s]
 59%|█████▉    | 90/153 [00:14<00:09,  6.79it/s]
 59%|█████▉    | 91/153 [00:14<00:09,  6.43it/s]
 60%|██████    | 92/153 [00:15<00:09,  6.69it/s]
 61%|██████    | 93/153 [00:15<00:09,  6.54it/s]
 61%|██████▏   | 94/153 [00:15<00:09,  6.41it/s]
 62%|██████▏   | 95/153 [00:15<00:09,  6.15it/s]
 63%|██████▎   | 96/153 [00:15<00:09,  6.20it/s]
 63%|██████▎   | 97/153 [00:15<00:08,  6.27it/s]
 64%|██████▍   | 98/153 [00:16<00:08,  6.20it/s]
 65%|██████▍   | 99/153 [00:16<00:07,  6.90it/s]
 65%|██████▌   | 100/153 [00:16<00:09,  5.54it/s]
 66%|██████▌   | 101/153 [00:16<00:09,  5.60it/s]
 67%|██████▋   | 102/153 [00:16<00:08,  6.13it/s]
 67%|██████▋   | 103/153 [00:16<00:07,  6.56it/s]
 68%|██████▊   | 104/153 [00:17<00:08,  5.95it/s]
 69%|██████▊   | 105/153 [00:17<00:07,  6.39it/s]
 69%|██████▉   | 106/153 [00:17<00:06,  6.79it/s]
 70%|██████▉   | 107/153 [00:17<00:07,  6.54it/s]
 71%|███████   | 108/153 [00:17<00:07,  5.98it/s]
 71%|███████   | 109/153 [00:17<00:07,  6.20it/s]
 72%|███████▏  | 110/153 [00:18<00:06,  6.19it/s]
 73%|███████▎  | 111/153 [00:18<00:06,  6.56it/s]
 73%|███████▎  | 112/153 [00:18<00:06,  6.66it/s]
 74%|███████▍  | 113/153 [00:18<00:06,  6.65it/s]
 75%|███████▍  | 114/153 [00:18<00:05,  6.51it/s]
 75%|███████▌  | 115/153 [00:18<00:05,  6.43it/s]
 76%|███████▌  | 116/153 [00:18<00:05,  6.91it/s]
 76%|███████▋  | 117/153 [00:19<00:05,  6.44it/s]
 77%|███████▋  | 118/153 [00:19<00:05,  6.23it/s]
 78%|███████▊  | 119/153 [00:19<00:05,  6.38it/s]
 78%|███████▊  | 120/153 [00:19<00:05,  6.31it/s]
 79%|███████▉  | 121/153 [00:19<00:05,  6.30it/s]
 80%|███████▉  | 122/153 [00:19<00:04,  6.66it/s]
 80%|████████  | 123/153 [00:19<00:04,  7.10it/s]
 81%|████████  | 124/153 [00:20<00:04,  7.01it/s]
 82%|████████▏ | 125/153 [00:20<00:04,  6.17it/s]
 82%|████████▏ | 126/153 [00:20<00:04,  6.57it/s]
 83%|████████▎ | 127/153 [00:20<00:03,  6.70it/s]
 84%|████████▎ | 128/153 [00:20<00:03,  7.20it/s]
 84%|████████▍ | 129/153 [00:20<00:03,  6.98it/s]
 85%|████████▍ | 130/153 [00:21<00:03,  7.02it/s]
 86%|████████▌ | 131/153 [00:21<00:03,  6.80it/s]
 86%|████████▋ | 132/153 [00:21<00:03,  6.31it/s]
 87%|████████▋ | 133/153 [00:21<00:03,  6.14it/s]
 88%|████████▊ | 134/153 [00:21<00:03,  5.88it/s]
 88%|████████▊ | 135/153 [00:21<00:02,  6.30it/s]
 89%|████████▉ | 136/153 [00:21<00:02,  6.41it/s]
 90%|████████▉ | 137/153 [00:22<00:02,  6.20it/s]
 90%|█████████ | 138/153 [00:22<00:02,  5.87it/s]
 91%|█████████ | 139/153 [00:22<00:02,  5.89it/s]
 92%|█████████▏| 140/153 [00:22<00:02,  5.76it/s]
 92%|█████████▏| 141/153 [00:22<00:01,  6.08it/s]
 93%|█████████▎| 142/153 [00:22<00:01,  6.34it/s]
 93%|█████████▎| 143/153 [00:23<00:01,  6.61it/s]
 94%|█████████▍| 144/153 [00:23<00:01,  6.74it/s]
 95%|█████████▍| 145/153 [00:23<00:01,  6.85it/s]
 95%|█████████▌| 146/153 [00:23<00:01,  6.33it/s]
 96%|█████████▌| 147/153 [00:23<00:00,  6.09it/s]
 97%|█████████▋| 148/153 [00:23<00:00,  6.17it/s]
 97%|█████████▋| 149/153 [00:24<00:00,  6.55it/s]
 98%|█████████▊| 150/153 [00:24<00:00,  6.61it/s]
 99%|█████████▊| 151/153 [00:24<00:00,  5.41it/s]
 99%|█████████▉| 152/153 [00:24<00:00,  6.06it/s]
100%|██████████| 153/153 [00:24<00:00,  6.28it/s]
100%|██████████| 153/153 [00:24<00:00,  6.18it/s]
Wrote outputs_notebook_v8_mini2v2\test-output.json
Test output ready: outputs_notebook_v8_mini2v2\test-output.json
Final retrieval setting used for test: {'mode': 'relative_logit', 'k': nan, 'threshold': nan, 'delta': 0.25, 'alpha': 0.7, 'retrieval_F': 0.25572562358276646, 'avg_pred_evidence': 4.214285714285714}
Final label source used for test: majority

```
